In [ ]:
# ================================================================
# T2.1 — CELL 1
# PROTOCOL / INPUT DISCOVERY AND PRE-FLIGHT
# ================================================================

from pathlib import Path
import json
import hashlib
import pandas as pd
import numpy as np
import torch
import torchvision

print("=" * 70)
print("T2.1 — SECOND-ARCHITECTURE REPLICATION")
print("CELL 1 — PROTOCOL / INPUT PRE-FLIGHT")
print("=" * 70)

# ---------------------------------------------------------------
# Frozen T2.1 protocol
# ---------------------------------------------------------------

T2_PROTOCOL = {
    "architecture": "ConvNeXt-Tiny",
    "weights_enum": "ConvNeXt_Tiny_Weights.IMAGENET1K_V1",
    "rho_levels": [0.0, 0.5, 1.0],
    "replicates_per_rho": 3,
    "total_models": 9,

    "reuse_primary_replicates": [1, 2, 3],
    "reuse_rhos": [0.0, 1.0],

    "new_rho": 0.5,
    "new_rho_replicates": [1, 2, 3],

    "optimizer": "AdamW",
    "learning_rate": 1e-4,
    "weight_decay": 1e-4,
    "batch_size": 64,
    "epochs": 15,
    "scheduler": "CosineAnnealingLR",
    "scheduler_T_max": 15,
    "loss": "CrossEntropyLoss",

    "augmentation": [
        "RandomHorizontalFlip(p=0.5)",
        "RandomRotation(15deg, nearest, fill=0)",
    ],

    "normalization_mean": [
        0.485,
        0.456,
        0.406,
    ],

    "normalization_std": [
        0.229,
        0.224,
        0.225,
    ],

    "primary_checkpoint": "final_epoch15",
    "sensitivity_checkpoint": "best_eval_ba",

    "endpoint": "delta_z_only",
    "xai": False,

    "tier": "Tier-2 exploratory",

    "architecture_test": "Freedman-Lane permutation",
    "architecture_model": (
        "delta_z = alpha + gamma*A + beta*rho "
        "+ delta*(rho*A) + epsilon"
    ),
    "interaction_parameter": (
        "delta = slope_ConvNeXt - slope_ResNet"
    ),
    "reduced_model": (
        "delta_z = alpha + gamma*A + beta*rho + epsilon"
    ),
    "permutation_unit": "model-level observations",
    "permutation_scope": "all 39 observations",
    "permutation_count": 100_000,
    "permutation_seed": 20260919,
    "architecture_alpha": 0.05,
    "architecture_test_sidedness": "two-sided",

    "bootstrap": (
        "model-level resampling within "
        "architecture x rho cells"
    ),

    "n_resnet": 30,
    "n_convnext": 9,
    "n_total_architecture_comparison": 39,

    "non_significant_interpretation":
        "no detected slope difference, not equivalence",
}

# ---------------------------------------------------------------
# Basic environment discovery
# ---------------------------------------------------------------

print()
print("Environment")
print("-" * 70)

print(
    "Python torch version      :",
    torch.__version__,
)

print(
    "Torchvision version       :",
    torchvision.__version__,
)

print(
    "CUDA available            :",
    torch.cuda.is_available(),
)

if torch.cuda.is_available():

    print(
        "CUDA device               :",
        torch.cuda.get_device_name(0),
    )

# ---------------------------------------------------------------
# Verify exact ConvNeXt constructor
# ---------------------------------------------------------------

from torchvision.models import (
    convnext_tiny,
    ConvNeXt_Tiny_Weights,
)

weights = (
    ConvNeXt_Tiny_Weights.IMAGENET1K_V1
)

model = convnext_tiny(
    weights=weights
)

print()
print("ConvNeXt-Tiny verification")
print("-" * 70)

print(
    "Weights enum               :",
    weights,
)

print(
    "Weights enum name         :",
    weights.name,
)

print(
    "Classifier                 :",
    model.classifier,
)

print(
    "Classifier[2]              :",
    model.classifier[2],
)

assert (
    weights.name
    ==
    "IMAGENET1K_V1"
)

assert isinstance(
    model.classifier[2],
    torch.nn.Linear,
)

assert (
    model.classifier[2].in_features
    ==
    768
)

# ---------------------------------------------------------------
# Verify deterministic ConvNeXt construction requirements
# ---------------------------------------------------------------

# Locate AdaptiveAvgPool2d modules.

adaptive_pool_locations = []

for name, module in model.named_modules():

    if isinstance(
        module,
        torch.nn.AdaptiveAvgPool2d,
    ):

        adaptive_pool_locations.append(
            (
                name,
                repr(module),
            )
        )

print()
print(
    "AdaptiveAvgPool2d locations:"
)

for item in adaptive_pool_locations:

    print(
        " ",
        item,
    )

assert len(
    adaptive_pool_locations
) >= 1

print(
    "Deterministic mean-pool substitution"
    " will be required in training harness."
)

# ---------------------------------------------------------------
# Verify T2.1 model count
# ---------------------------------------------------------------

expected_models = [
    f"{rho:.1f}_{rep}"
    for rho in [0.0, 0.5, 1.0]
    for rep in [1, 2, 3]
]

assert len(expected_models) == 9

print()
print(
    "Expected T2.1 models:",
    expected_models,
)

# ---------------------------------------------------------------
# Locate likely primary project roots
# ---------------------------------------------------------------

candidate_roots = [
    Path(
        "/kaggle/input/datasets/"
        "tejasseshadriiiii/"
        "isic-final-project"
    ),

    Path(
        "/kaggle/input/notebooks/"
        "tejasseshadriiiii/"
        "isic-kaggle-analysis"
    ),

    Path(
        "/kaggle/input/notebooks/"
        "tejasseshadriiiii/"
        "ml-open-lab-30m-trained"
    ),
]

print()
print("Candidate project roots")
print("-" * 70)

for root in candidate_roots:

    print(
        f"{root} :",
        "FOUND" if root.exists()
        else "missing",
    )

# ---------------------------------------------------------------
# Search for the frozen assignment infrastructure
# ---------------------------------------------------------------

search_names = [
    "step1F",
    "step2P_A1_1",
    "rho_assignment",
    "rho-assignment-v1",
    "rho_0.0_rep_1.csv",
]

found = []

for root in candidate_roots:

    if not root.exists():
        continue

    for token in search_names:

        matches = list(
            root.rglob(
                f"*{token}*"
            )
        )

        for p in matches:

            if p.is_file():

                found.append(
                    p
                )

# Deduplicate

found = sorted(
    set(found),
    key=lambda p: str(p),
)

print()
print(
    "Assignment-related files found:",
    len(found),
)

for p in found[:100]:

    print(
        " ",
        p,
    )

# ---------------------------------------------------------------
# Environment lock candidate
# ---------------------------------------------------------------

env_candidates = []

for root in candidate_roots:

    if not root.exists():
        continue

    matches = list(
        root.rglob(
            "*actual_training_environment.json"
        )
    )

    env_candidates.extend(
        matches
    )

env_candidates = sorted(
    set(env_candidates),
    key=lambda p: str(p),
)

print()
print(
    "Existing environment locks:",
    len(env_candidates),
)

for p in env_candidates:

    print(
        " ",
        p,
    )

# ---------------------------------------------------------------
# Print protocol lock summary
# ---------------------------------------------------------------

print()
print("=" * 70)
print("T2.1 PROTOCOL PRE-FLIGHT COMPLETE")
print("=" * 70)

print(
    "Architecture              : ConvNeXt-Tiny"
)

print(
    "Weights                   : "
    "ConvNeXt_Tiny_Weights.IMAGENET1K_V1"
)

print(
    "Models                    : 9"
)

print(
    "rho                       : {0.0, 0.5, 1.0}"
)

print(
    "Replicates                : 3/rho"
)

print(
    "Reuse primary reps        : 1,2,3 at rho=0,1"
)

print(
    "New assignments           : rho=0.5 reps 1,2,3"
)

print(
    "Primary endpoint          : delta_z"
)

print(
    "XAI                       : OUT"
)

print(
    "Primary checkpoint        : final epoch 15"
)

print(
    "Sensitivity checkpoint    : best_eval_ba"
)

print(
    "Architecture test        : Freedman-Lane"
)

print(
    "Permutations              : 100000"
)

print(
    "Permutation seed          : 20260919"
)

print(
    "Two-sided alpha           : 0.05"
)

print(
    "T2.1 status               : PRE-FLIGHT"
)

print("=" * 70)

# Keep protocol available for the next cells.
T2_PROTOCOL

In [ ]:
# ================================================================
# T2.1 — CELL 2
# FROZEN ENVIRONMENT + ASSIGNMENT INFRASTRUCTURE AUDIT
# ================================================================

import os
import json
import hashlib
from pathlib import Path

print("=" * 70)
print("T2.1 — CELL 2")
print("FROZEN ENVIRONMENT + ASSIGNMENT INFRASTRUCTURE AUDIT")
print("=" * 70)

# ------------------------------------------------
# 1. CURRENT LIVE ENVIRONMENT
# ------------------------------------------------

import torch
import torchvision

LIVE_ENV = {
    "python": __import__("sys").version,
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "cuda_available": torch.cuda.is_available(),
    "cuda_version": torch.version.cuda,
    "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
}

print("\nLIVE ENVIRONMENT")
print("-" * 70)
for k, v in LIVE_ENV.items():
    print(f"{k:20s}: {v}")


# ------------------------------------------------
# 2. READ THE FROZEN TRAINING ENVIRONMENT LOCK
# ------------------------------------------------

env_candidates = [
    Path("/kaggle/input/datasets/tejasseshadriiiii/isic-final-project/step1S_environment_lock/actual_training_environment.json"),
    Path("/kaggle/input/notebooks/tejasseshadriiiii/isic-kaggle-analysis/isic_final_project/step1S_environment_lock/actual_training_environment.json"),
    Path("/kaggle/input/notebooks/tejasseshadriiiii/ml-open-lab-30m-trained/isic_final_project/step1S_environment_lock/actual_training_environment.json"),
]

env_path = next((p for p in env_candidates if p.exists()), None)

if env_path is None:
    raise FileNotFoundError(
        "Frozen actual_training_environment.json was not found in any expected root."
    )

print("\nFROZEN ENVIRONMENT LOCK")
print("-" * 70)
print("Path:", env_path)

with open(env_path, "r") as f:
    FROZEN_ENV = json.load(f)

print(json.dumps(FROZEN_ENV, indent=2, sort_keys=True))


# ------------------------------------------------
# 3. EXTRACT IMPORTANT FROZEN VERSION FIELDS
# ------------------------------------------------

def recursive_find(obj, keys):
    found = {}
    if isinstance(obj, dict):
        for k, v in obj.items():
            if k in keys:
                found[k] = v
            found.update(recursive_find(v, keys))
    elif isinstance(obj, list):
        for item in obj:
            found.update(recursive_find(item, keys))
    return found

version_fields = recursive_find(
    FROZEN_ENV,
    {
        "torch_version",
        "torchvision_version",
        "cuda_version",
        "python_version",
        "device",
        "gpu",
        "weights_enum",
    }
)

print("\nEXTRACTED FROZEN FIELDS")
print("-" * 70)
for k, v in version_fields.items():
    print(f"{k:25s}: {v}")


# ------------------------------------------------
# 4. EXPLICIT VERSION COMPARISON
# ------------------------------------------------

print("\nVERSION COMPARISON")
print("-" * 70)

frozen_torch = version_fields.get("torch_version")
frozen_tv = version_fields.get("torchvision_version")

torch_match = (
    frozen_torch is not None
    and torch.__version__ == frozen_torch
)

tv_match = (
    frozen_tv is not None
    and torchvision.__version__ == frozen_tv
)

print("Torch match       :", torch_match)
print("Torchvision match :", tv_match)

if not torch_match or not tv_match:
    print("\n!!! ENVIRONMENT MISMATCH DETECTED !!!")
    print("Training will NOT be started from this cell.")
else:
    print("\nEnvironment versions match the frozen lock.")


# ------------------------------------------------
# 5. LOCATE EXACT ASSIGNMENT INFRASTRUCTURE
# ------------------------------------------------

print("\nASSIGNMENT INFRASTRUCTURE")
print("-" * 70)

project_roots = [
    Path("/kaggle/input/datasets/tejasseshadriiiii/isic-final-project"),
    Path("/kaggle/input/notebooks/tejasseshadriiiii/isic-kaggle-analysis"),
    Path("/kaggle/input/notebooks/tejasseshadriiiii/ml-open-lab-30m-trained"),
]

assignment_keywords = [
    "assignment",
    "rho",
    "A1.1",
    "solver",
    "lesion",
    "step1F",
    "step2P",
]

assignment_files = []

for root in project_roots:
    if not root.exists():
        continue

    for p in root.rglob("*"):
        if not p.is_file():
            continue

        name_lower = p.name.lower()
        path_lower = str(p).lower()

        if any(
            keyword.lower() in name_lower or keyword.lower() in path_lower
            for keyword in assignment_keywords
        ):
            assignment_files.append(p)

# Deduplicate
assignment_files = sorted(set(assignment_files), key=str)

for p in assignment_files:
    print(p)

print("\nAssignment-related file count:", len(assignment_files))


# ------------------------------------------------
# 6. IDENTIFY LIKELY SOLVER / PROTOCOL FILES
# ------------------------------------------------

solver_candidates = [
    p for p in assignment_files
    if (
        p.suffix.lower() in {".py", ".json", ".md", ".csv"}
        and any(
            x in p.name.lower()
            for x in [
                "solver",
                "assignment",
                "protocol",
                "a1",
                "step1f",
                "step2p",
            ]
        )
    )
]

print("\nLIKELY SOLVER / PROTOCOL CANDIDATES")
print("-" * 70)

for p in solver_candidates:
    print(p)


# ------------------------------------------------
# 7. HASH EXISTING PYTHON ASSIGNMENT IMPLEMENTATIONS
# ------------------------------------------------

print("\nPYTHON ASSIGNMENT FILE HASHES")
print("-" * 70)

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

py_assignment_files = [
    p for p in assignment_files
    if p.suffix.lower() == ".py"
]

if not py_assignment_files:
    print("No Python assignment implementation found in scanned roots.")

for p in py_assignment_files:
    print(f"\n{p}")
    print("SHA256:", sha256_file(p))
    print("SIZE  :", p.stat().st_size)


# ------------------------------------------------
# 8. INSPECT EXISTING LOCKED ASSIGNMENT PROTOCOL
# ------------------------------------------------

protocol_candidates = [
    p for p in assignment_files
    if p.name == "step1F_locked_assignment_protocol.json"
]

if protocol_candidates:
    protocol_path = protocol_candidates[0]

    print("\nLOCKED ASSIGNMENT PROTOCOL")
    print("-" * 70)
    print("Path:", protocol_path)

    with open(protocol_path, "r") as f:
        ASSIGNMENT_PROTOCOL = json.load(f)

    print(json.dumps(ASSIGNMENT_PROTOCOL, indent=2, sort_keys=True))

else:
    print("\nNo step1F_locked_assignment_protocol.json found.")


# ------------------------------------------------
# 9. CHECK EXISTING PRIMARY ASSIGNMENT MANIFESTS
# ------------------------------------------------

print("\nPRIMARY ASSIGNMENT FILES")
print("-" * 70)

for rho in ["0.0", "1.0"]:
    for rep in [1, 2, 3]:

        matches = []

        for root in project_roots:
            candidate = (
                root
                / "step1F_lesion_level_rho_assignments"
                / f"rho_{rho}_rep_{rep}.csv"
            )

            # Some roots have the nested isic_final_project layout
            if candidate.exists():
                matches.append(candidate)

            nested = (
                root
                / "isic_final_project"
                / "step1F_lesion_level_rho_assignments"
                / f"rho_{rho}_rep_{rep}.csv"
            )

            if nested.exists():
                matches.append(nested)

        print(f"rho={rho}, rep={rep}:")
        for m in sorted(set(matches), key=str):
            print("  ", m)


# ------------------------------------------------
# 10. FINAL CELL-2 STATUS
# ------------------------------------------------

print("\n" + "=" * 70)
print("CELL 2 COMPLETE")
print("=" * 70)

print("Current Torch       :", torch.__version__)
print("Current Torchvision :", torchvision.__version__)
print("Frozen Torch        :", frozen_torch)
print("Frozen Torchvision  :", frozen_tv)

print("\nEnvironment match:", torch_match and tv_match)

if torch_match and tv_match:
    print("STATUS: READY FOR ASSIGNMENT INFRASTRUCTURE PRE-LOCK")
else:
    print("STATUS: STOP — ENVIRONMENT MISMATCH MUST BE RESOLVED BEFORE TRAINING")

print("=" * 70)

In [ ]:
# ================================================================
# T2.1 — CELL 3
# EXACT A1.1 SOLVER + ASSIGNMENT LOCK INSPECTION
# ================================================================

import json
import hashlib
from pathlib import Path
import pandas as pd

print("=" * 70)
print("T2.1 — CELL 3")
print("EXACT A1.1 SOLVER + ASSIGNMENT LOCK INSPECTION")
print("=" * 70)

BASE = Path(
    "/kaggle/input/datasets/tejasseshadriiiii/isic-final-project"
)

ASSIGN_DIR = BASE / "step1F_lesion_level_rho_assignments"
A11_DIR = BASE / "step2P_A1_1_solver_addendum_lock"

# ------------------------------------------------
# Helper
# ------------------------------------------------

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def print_json(path, title):
    print("\n" + title)
    print("-" * 70)
    print("PATH:", path)
    with open(path, "r") as f:
        obj = json.load(f)
    print(json.dumps(obj, indent=2, sort_keys=True))
    return obj


# ------------------------------------------------
# 1. EXISTING LOCKED ASSIGNMENT PROTOCOL
# ------------------------------------------------

assignment_protocol_path = (
    ASSIGN_DIR / "step1F_locked_assignment_protocol.json"
)

ASSIGNMENT_PROTOCOL = print_json(
    assignment_protocol_path,
    "STEP1F LOCKED ASSIGNMENT PROTOCOL"
)


# ------------------------------------------------
# 2. FINAL A1.1 SOLVER ADDENDUM
# ------------------------------------------------

a11_lock_path = (
    A11_DIR / "FINAL_LOCKED_A1_1_SOLVER_ADDENDUM.json"
)

A11_LOCK = print_json(
    a11_lock_path,
    "FINAL LOCKED A1.1 SOLVER ADDENDUM"
)


# ------------------------------------------------
# 3. A1.1 HASH LOCK
# ------------------------------------------------

a11_hash_path = (
    A11_DIR / "A1_1_FINAL_HASH.json"
)

A11_HASH = print_json(
    a11_hash_path,
    "A1.1 FINAL HASH LOCK"
)


# ------------------------------------------------
# 4. INSPECT EXISTING PRIMARY ASSIGNMENT SCHEMA
# ------------------------------------------------

print("\nEXISTING PRIMARY ASSIGNMENT SCHEMA")
print("-" * 70)

sample_path = ASSIGN_DIR / "rho_0.0_rep_1.csv"

if not sample_path.exists():
    raise FileNotFoundError(sample_path)

sample_df = pd.read_csv(sample_path)

print("File:", sample_path)
print("Rows:", len(sample_df))
print("Columns:")
for c in sample_df.columns:
    print("  ", c)

print("\nDtypes:")
print(sample_df.dtypes)

print("\nFirst 5 rows:")
print(sample_df.head().to_string(index=False))


# ------------------------------------------------
# 5. CHECK PRIMARY REPLICATE FILES + HASHES
# ------------------------------------------------

print("\nPRIMARY ASSIGNMENT HASHES")
print("-" * 70)

primary_manifest = []

for rho in ["0.0", "1.0"]:
    for rep in [1, 2, 3]:

        path = ASSIGN_DIR / f"rho_{rho}_rep_{rep}.csv"

        if not path.exists():
            raise FileNotFoundError(path)

        df = pd.read_csv(path)

        record = {
            "rho": float(rho),
            "replicate": rep,
            "path": str(path),
            "rows": len(df),
            "sha256": sha256_file(path),
            "columns": list(df.columns),
        }

        primary_manifest.append(record)

        print(
            f"rho={rho} rep={rep} | "
            f"rows={len(df)} | "
            f"sha256={record['sha256']}"
        )


# ------------------------------------------------
# 6. LOOK FOR SOLVER SOURCE REFERENCED BY A1.1
# ------------------------------------------------

print("\nREFERENCED SOLVER / IMPLEMENTATION PATHS")
print("-" * 70)

text_blobs = []

for key, obj in [
    ("ASSIGNMENT_PROTOCOL", ASSIGNMENT_PROTOCOL),
    ("A11_LOCK", A11_LOCK),
    ("A11_HASH", A11_HASH),
]:
    text_blobs.append((key, json.dumps(obj, indent=2, sort_keys=True)))

for key, txt in text_blobs:

    print(f"\n[{key}]")

    interesting_lines = []

    for line in txt.splitlines():

        lower = line.lower()

        if any(
            token in lower
            for token in [
                ".py",
                "solver",
                "implementation",
                "source",
                "hash",
                "seed",
                "sha256",
                "fraction",
                "exact",
                "objective",
                "subtype",
                "class",
            ]
        ):
            interesting_lines.append(line)

    for line in interesting_lines:
        print(line)


# ------------------------------------------------
# 7. SEARCH PROJECT FOR PYTHON FILES RELATED TO
#    THE ORIGINAL ASSIGNMENT SOLVER
# ------------------------------------------------

print("\nPROJECT PYTHON FILES WITH ASSIGNMENT/SOLVER TERMS")
print("-" * 70)

candidate_py = []

for p in BASE.rglob("*.py"):

    lower = str(p).lower()

    if any(
        token in lower
        for token in [
            "assign",
            "rho",
            "solver",
            "step1f",
            "a1_1",
        ]
    ):
        candidate_py.append(p)

candidate_py = sorted(set(candidate_py), key=str)

if candidate_py:
    for p in candidate_py:
        print(p)
        print("  SHA256:", sha256_file(p))
        print("  SIZE  :", p.stat().st_size)
else:
    print("No matching Python implementation files found.")


# ------------------------------------------------
# 8. VERIFY THAT PRIMARY ASSIGNMENTS ARE DISTINCT
# ------------------------------------------------

print("\nPRIMARY ASSIGNMENT DIFFERENCE CHECK")
print("-" * 70)

def assignment_signature(df):
    # Use the actual column order and row order exactly as stored.
    payload = df.to_csv(index=False, lineterminator="\n").encode("utf-8")
    return hashlib.sha256(payload).hexdigest()

loaded = {}

for rho in ["0.0", "1.0"]:
    for rep in [1, 2, 3]:

        path = ASSIGN_DIR / f"rho_{rho}_rep_{rep}.csv"
        df = pd.read_csv(path)

        loaded[(rho, rep)] = df

        print(
            f"rho={rho} rep={rep}: "
            f"signature={assignment_signature(df)}"
        )

for rho in ["0.0", "1.0"]:

    a = loaded[(rho, 1)]
    b = loaded[(rho, 2)]
    c = loaded[(rho, 3)]

    same_12 = a.equals(b)
    same_13 = a.equals(c)
    same_23 = b.equals(c)

    print(
        f"\nrho={rho}: "
        f"rep1==rep2={same_12}, "
        f"rep1==rep3={same_13}, "
        f"rep2==rep3={same_23}"
    )


# ------------------------------------------------
# 9. CELL-3 STATUS
# ------------------------------------------------

print("\n" + "=" * 70)
print("CELL 3 COMPLETE")
print("=" * 70)

print("A1.1 lock found              :", a11_lock_path.exists())
print("A1.1 final hash found        :", a11_hash_path.exists())
print("Primary rho 0/1 files found  :", len(primary_manifest), "/ 6")
print("Assignment schema recovered  :", True)
print("Solver-related Python files  :", len(candidate_py))

print("\nNEXT ACTION:")
print("Use the EXACT frozen A1.1 methodology to construct")
print("rho=0.5 replicates 1, 2, 3.")
print("Do NOT implement an approximate replacement solver.")

print("=" * 70)

In [ ]:
# ================================================================
# T2.1 — CELL 4
# A1.1 SOLVER IMPLEMENTATION FORENSIC AUDIT
# ================================================================

from pathlib import Path
import hashlib
import ast
import re
import json

print("=" * 70)
print("T2.1 — CELL 4")
print("A1.1 SOLVER IMPLEMENTATION FORENSIC AUDIT")
print("=" * 70)

BASE = Path(
    "/kaggle/input/datasets/tejasseshadriiiii/isic-final-project"
)

SOLVER_PATH = (
    BASE
    / "step2Q_solver_implementation_v2"
    / "deterministic_evaluation_solver.py"
)

METADATA_PATH = (
    BASE
    / "step2Q_solver_implementation_v2"
    / "solver_implementation_metadata.json"
)

VALIDATION_PATH = (
    BASE
    / "step2Q_solver_implementation_v2"
    / "step2Q_solver_validation_results_v2.json"
)

EDGE_PATH = (
    BASE
    / "step2Q_solver_implementation_v2"
    / "step2Q_solver_edge_case_results.json"
)

EXPECTED_SOLVER_SHA256 = (
    "37451d50afa7dcd8052a00d53cccb853d3ce658adbdd65bc1ce66c14b67fc5d3"
)

# ------------------------------------------------
# 1. EXISTENCE + HASH
# ------------------------------------------------

print("\nSOLVER FILE")
print("-" * 70)

if not SOLVER_PATH.exists():
    raise FileNotFoundError(SOLVER_PATH)

h = hashlib.sha256()

with open(SOLVER_PATH, "rb") as f:
    for chunk in iter(lambda: f.read(1024 * 1024), b""):
        h.update(chunk)

solver_sha256 = h.hexdigest()

print("Path   :", SOLVER_PATH)
print("Size   :", SOLVER_PATH.stat().st_size)
print("SHA256 :", solver_sha256)
print("Expected:", EXPECTED_SOLVER_SHA256)
print("Hash match:", solver_sha256 == EXPECTED_SOLVER_SHA256)

if solver_sha256 != EXPECTED_SOLVER_SHA256:
    raise RuntimeError(
        "Solver hash mismatch. Do not use this implementation."
    )


# ------------------------------------------------
# 2. READ SOURCE
# ------------------------------------------------

source = SOLVER_PATH.read_text(encoding="utf-8")

print("\nSOURCE SUMMARY")
print("-" * 70)
print("Characters:", len(source))
print("Lines     :", len(source.splitlines()))


# ------------------------------------------------
# 3. AST-LEVEL FUNCTION / CLASS INVENTORY
# ------------------------------------------------

tree = ast.parse(source)

functions = []
classes = []

for node in ast.walk(tree):

    if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
        functions.append(node.name)

    elif isinstance(node, ast.ClassDef):
        classes.append(node.name)

print("\nFUNCTIONS")
print("-" * 70)

for name in sorted(set(functions)):
    print(name)

print("\nCLASSES")
print("-" * 70)

for name in sorted(set(classes)):
    print(name)


# ------------------------------------------------
# 4. CRITICAL IMPLEMENTATION FEATURES
# ------------------------------------------------

print("\nCRITICAL IMPLEMENTATION FEATURE SCAN")
print("-" * 70)

checks = {
    "Fraction arithmetic": [
        "from fractions import Fraction",
        "Fraction("
    ],
    "SHA256": [
        "sha256",
        "hashlib.sha256"
    ],
    "PCG64 / NumPy RNG": [
        "PCG64",
        "default_rng",
        "Generator"
    ],
    "built-in hash prohibition": [
        "hash("
    ],
    "exact DP / dynamic programming": [
        "dp",
        "reachable",
        "suffix",
        "state"
    ],
    "subtype logic": [
        "subtype"
    ],
    "class logic": [
        "malignant",
        "class_name",
        "class"
    ],
    "hair count": [
        "hair_count",
        "hair_present"
    ],
    "lexicographic logic": [
        "lexic",
        "lexicographic"
    ],
    "randomness": [
        "random",
        "rng",
        "seed"
    ]
}

for label, patterns in checks.items():

    hits = []

    for pattern in patterns:

        if pattern.lower() in source.lower():
            hits.append(pattern)

    print(
        f"{label:30s}: "
        f"{'FOUND ' + str(hits) if hits else 'NOT FOUND'}"
    )


# ------------------------------------------------
# 5. PRINT IMPORTS
# ------------------------------------------------

print("\nIMPORTS")
print("-" * 70)

for node in tree.body:

    if isinstance(node, ast.Import):
        for alias in node.names:
            print("import", alias.name)

    elif isinstance(node, ast.ImportFrom):
        names = ", ".join(
            alias.name for alias in node.names
        )
        print(f"from {node.module} import {names}")


# ------------------------------------------------
# 6. PRINT TOP-LEVEL CONSTANTS
# ------------------------------------------------

print("\nTOP-LEVEL CONSTANTS")
print("-" * 70)

for node in tree.body:

    if isinstance(node, ast.Assign):

        for target in node.targets:

            if isinstance(target, ast.Name):

                try:
                    value = ast.literal_eval(node.value)

                    if isinstance(
                        value,
                        (str, int, float, bool, tuple, list, dict)
                    ):
                        print(
                            f"{target.id} = "
                            f"{repr(value)[:500]}"
                        )

                except Exception:
                    pass


# ------------------------------------------------
# 7. READ SOLVER METADATA
# ------------------------------------------------

if METADATA_PATH.exists():

    print("\nSOLVER IMPLEMENTATION METADATA")
    print("-" * 70)

    metadata = json.loads(
        METADATA_PATH.read_text(encoding="utf-8")
    )

    print(
        json.dumps(
            metadata,
            indent=2,
            sort_keys=True
        )
    )

else:
    print("\nNo solver metadata file found.")


# ------------------------------------------------
# 8. READ VALIDATION RESULTS
# ------------------------------------------------

if VALIDATION_PATH.exists():

    print("\nSOLVER VALIDATION RESULTS")
    print("-" * 70)

    validation = json.loads(
        VALIDATION_PATH.read_text(encoding="utf-8")
    )

    print(
        json.dumps(
            validation,
            indent=2,
            sort_keys=True
        )
    )

else:
    print("\nNo solver validation results found.")


# ------------------------------------------------
# 9. READ EDGE-CASE RESULTS
# ------------------------------------------------

if EDGE_PATH.exists():

    print("\nSOLVER EDGE-CASE RESULTS")
    print("-" * 70)

    edge = json.loads(
        EDGE_PATH.read_text(encoding="utf-8")
    )

    print(
        json.dumps(
            edge,
            indent=2,
            sort_keys=True
        )
    )

else:
    print("\nNo edge-case results found.")


# ------------------------------------------------
# 10. SEARCH SOURCE FOR FORBIDDEN / RISKY PATTERNS
# ------------------------------------------------

print("\nFORBIDDEN / RISKY PATTERN AUDIT")
print("-" * 70)

for pattern in [
    r"\bhash\s*\(",
    r"random\.random",
    r"np\.random\.random",
    r"np\.random\.rand",
    r"\.shuffle\s*\(",
]:

    matches = re.findall(
        pattern,
        source,
        flags=re.IGNORECASE
    )

    print(
        f"{pattern:30s}: {len(matches)} occurrence(s)"
    )


# ------------------------------------------------
# 11. CELL STATUS
# ------------------------------------------------

print("\n" + "=" * 70)
print("CELL 4 COMPLETE")
print("=" * 70)

print("Solver exists       :", SOLVER_PATH.exists())
print("Solver hash matches :", solver_sha256 == EXPECTED_SOLVER_SHA256)
print("Validation artifact :", VALIDATION_PATH.exists())
print("Edge-case artifact  :", EDGE_PATH.exists())
print("Metadata artifact   :", METADATA_PATH.exists())

print("\nIMPORTANT:")
print("We will NOT generate rho=0.5 until this implementation")
print("is confirmed to satisfy the locked A1.1 specification.")

print("=" * 70)

In [ ]:
# ================================================================
# T2.1 — CELL 5
# SOLVER API + RHO=0.5 POPULATION PRE-GENERATION AUDIT
# ================================================================

from pathlib import Path
import sys
import json
import inspect
import pandas as pd

print("=" * 70)
print("T2.1 — CELL 5")
print("SOLVER API + RHO=0.5 POPULATION PRE-GENERATION AUDIT")
print("=" * 70)

BASE = Path(
    "/kaggle/input/datasets/tejasseshadriiiii/isic-final-project"
)

SOLVER_DIR = BASE / "step2Q_solver_implementation_v2"
SOLVER_PATH = SOLVER_DIR / "deterministic_evaluation_solver.py"

POP_PATH = (
    BASE
    / "step0C_lesion_disjoint_population"
    / "locked_development_population.csv"
)

POP_SUMMARY_PATH = (
    BASE
    / "step0C_lesion_disjoint_population"
    / "step0C_population_summary.json"
)

RHO_TARGET_PATH = (
    BASE
    / "step1D_rho_feasibility_corrected"
    / "step1D_corrected_rho_targets.csv"
)

SUBTYPE_TARGET_PATH = (
    BASE
    / "step1D_rho_feasibility"
    / "step1D_subtype_target_counts.csv"
)

FEASIBILITY_PATH = (
    BASE
    / "step1E_lesion_assignment_feasibility"
    / "step1E_binary_feasibility.csv"
)

# ------------------------------------------------
# 1. IMPORT FROZEN SOLVER
# ------------------------------------------------

print("\nIMPORTING FROZEN SOLVER")
print("-" * 70)

if not SOLVER_PATH.exists():
    raise FileNotFoundError(SOLVER_PATH)

sys.path.insert(0, str(SOLVER_DIR))

import deterministic_evaluation_solver as solver

print("Imported:", solver.__file__)


# ------------------------------------------------
# 2. SOLVER FUNCTION SIGNATURE
# ------------------------------------------------

print("\nSOLVER FUNCTIONS")
print("-" * 70)

for name in [
    "solve_evaluation_assignment",
    "canonical_group_selection",
    "subtype_dp",
    "reachable_sums",
    "eval_hash_key",
]:

    if hasattr(solver, name):

        fn = getattr(solver, name)

        print(f"\n{name}")
        print("Signature:")
        print(inspect.signature(fn))

        print("\nDocstring:")
        print(inspect.getdoc(fn) or "<none>")


# ------------------------------------------------
# 3. PRINT COMPLETE SOLVER FUNCTION SOURCE
# ------------------------------------------------

print("\nsolve_evaluation_assignment SOURCE")
print("-" * 70)

print(
    inspect.getsource(
        solver.solve_evaluation_assignment
    )
)


# ------------------------------------------------
# 4. LOAD LOCKED DEVELOPMENT POPULATION
# ------------------------------------------------

print("\nLOCKED DEVELOPMENT POPULATION")
print("-" * 70)

if not POP_PATH.exists():
    raise FileNotFoundError(POP_PATH)

population = pd.read_csv(POP_PATH)

print("Path :", POP_PATH)
print("Rows :", len(population))
print("Cols :", list(population.columns))

print("\nDtypes:")
print(population.dtypes)

print("\nFirst rows:")
print(population.head().to_string(index=False))


# ------------------------------------------------
# 5. POPULATION SUMMARY
# ------------------------------------------------

if POP_SUMMARY_PATH.exists():

    print("\nPOPULATION SUMMARY")
    print("-" * 70)

    with open(POP_SUMMARY_PATH) as f:
        pop_summary = json.load(f)

    print(
        json.dumps(
            pop_summary,
            indent=2,
            sort_keys=True
        )
    )


# ------------------------------------------------
# 6. CHECK LESION-LEVEL STRUCTURE
# ------------------------------------------------

print("\nLESION-LEVEL STRUCTURE")
print("-" * 70)

required_candidates = [
    "lesion_id",
    "image",
    "malignant",
    "class_name",
]

for col in required_candidates:
    print(
        f"{col:15s}:",
        col in population.columns
    )

if "lesion_id" in population.columns:

    lesion_sizes = (
        population
        .groupby("lesion_id")
        .size()
    )

    print("\nUnique lesions:", len(lesion_sizes))
    print("Images:", len(population))
    print(
        "Images/lesion min:",
        lesion_sizes.min()
    )
    print(
        "Images/lesion max:",
        lesion_sizes.max()
    )


# ------------------------------------------------
# 7. CLASS COUNTS
# ------------------------------------------------

print("\nCLASS COUNTS")
print("-" * 70)

if "malignant" in population.columns:
    print(
        population["malignant"]
        .value_counts()
        .sort_index()
        .to_string()
    )


# ------------------------------------------------
# 8. LOAD RHO TARGETS
# ------------------------------------------------

print("\nRHO TARGET ARTIFACT")
print("-" * 70)

if not RHO_TARGET_PATH.exists():
    raise FileNotFoundError(RHO_TARGET_PATH)

rho_targets = pd.read_csv(RHO_TARGET_PATH)

print("Path:", RHO_TARGET_PATH)
print("Rows:", len(rho_targets))
print("Columns:", list(rho_targets.columns))

print(
    rho_targets.to_string(index=False)
)


# ------------------------------------------------
# 9. IDENTIFY RHO=0.5 TARGET INFORMATION
# ------------------------------------------------

print("\nRHO=0.5 TARGET ROWS")
print("-" * 70)

rho_column_candidates = [
    c for c in rho_targets.columns
    if c.lower() in {"rho", "target_rho"}
]

if rho_column_candidates:

    rho_col = rho_column_candidates[0]

    rho05 = rho_targets[
        rho_targets[rho_col].astype(float) == 0.5
    ]

    print(
        rho05.to_string(index=False)
    )

else:
    print(
        "Could not automatically identify rho column."
    )


# ------------------------------------------------
# 10. SUBTYPE TARGET ARTIFACT
# ------------------------------------------------

print("\nSUBTYPE TARGET ARTIFACT")
print("-" * 70)

if SUBTYPE_TARGET_PATH.exists():

    subtype_targets = pd.read_csv(
        SUBTYPE_TARGET_PATH
    )

    print(
        "Path:",
        SUBTYPE_TARGET_PATH
    )

    print(
        "Rows:",
        len(subtype_targets)
    )

    print(
        "Columns:",
        list(subtype_targets.columns)
    )

    print(
        subtype_targets.head(20)
        .to_string(index=False)
    )

else:
    print(
        "Subtype target artifact not found."
    )


# ------------------------------------------------
# 11. FEASIBILITY ARTIFACT
# ------------------------------------------------

print("\nASSIGNMENT FEASIBILITY ARTIFACT")
print("-" * 70)

if FEASIBILITY_PATH.exists():

    feasibility = pd.read_csv(
        FEASIBILITY_PATH
    )

    print(
        "Path:",
        FEASIBILITY_PATH
    )

    print(
        "Rows:",
        len(feasibility)
    )

    print(
        "Columns:",
        list(feasibility.columns)
    )

    print(
        feasibility.head(20)
        .to_string(index=False)
    )

else:
    print(
        "Binary feasibility artifact not found."
    )


# ------------------------------------------------
# 12. CELL STATUS
# ------------------------------------------------

print("\n" + "=" * 70)
print("CELL 5 COMPLETE")
print("=" * 70)

print("Frozen solver imported       : True")
print("Population loaded            :", len(population))
print("Rho target artifact loaded   :", RHO_TARGET_PATH.exists())
print("Subtype target artifact      :", SUBTYPE_TARGET_PATH.exists())
print("Feasibility artifact         :", FEASIBILITY_PATH.exists())

print("\nNO rho=0.5 assignment has been generated yet.")

print("=" * 70)

In [ ]:
# ================================================================
# T2.1 — CELL 6
# GENERATE EXACT RHO=0.5 ASSIGNMENTS
# ================================================================

from pathlib import Path
import sys
import json
import hashlib
import pandas as pd
from fractions import Fraction

print("=" * 70)
print("T2.1 — CELL 6")
print("GENERATE EXACT RHO=0.5 ASSIGNMENTS")
print("=" * 70)

# ------------------------------------------------
# PATHS
# ------------------------------------------------

BASE = Path(
    "/kaggle/input/datasets/tejasseshadriiiii/isic-final-project"
)

POP_PATH = (
    BASE
    / "step0C_lesion_disjoint_population"
    / "locked_development_population.csv"
)

SOLVER_DIR = (
    BASE
    / "step2Q_solver_implementation_v2"
)

SOLVER_PATH = (
    SOLVER_DIR
    / "deterministic_evaluation_solver.py"
)

OUT_DIR = Path(
    "/kaggle/working/T2_1_ConvNeXt_Architecture_Replication"
)

ASSIGN_DIR = OUT_DIR / "rho_0.5_assignments"

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

ASSIGN_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ------------------------------------------------
# LOCKED PARAMETERS
# ------------------------------------------------

MASTER_SEED = 20260919
RHO = 0.5
REPLICATES = [1, 2, 3]

# Locked seed formula from Step 1F:
# SHA256("rho-assignment-v1"|master_seed|rho|replicate)
#
# First 64 bits interpreted as unsigned integer.

def assignment_seed(master_seed, rho, replicate):

    payload = (
        f"rho-assignment-v1|"
        f"{master_seed}|"
        f"{rho}|"
        f"{replicate}"
    )

    digest = hashlib.sha256(
        payload.encode("utf-8")
    ).digest()

    return int.from_bytes(
        digest[:8],
        byteorder="big",
        signed=False
    )


# ------------------------------------------------
# LOAD SOLVER
# ------------------------------------------------

sys.path.insert(
    0,
    str(SOLVER_DIR)
)

import deterministic_evaluation_solver as solver

print("\nSOLVER")
print("-" * 70)
print("Path :", SOLVER_PATH)
print(
    "SHA256 :",
    hashlib.sha256(
        SOLVER_PATH.read_bytes()
    ).hexdigest()
)

# ------------------------------------------------
# LOAD POPULATION
# ------------------------------------------------

population = pd.read_csv(
    POP_PATH
)

print("\nPOPULATION")
print("-" * 70)
print("Images :", len(population))
print(
    "Unique lesions :",
    population["lesion_id"].nunique()
)

# ------------------------------------------------
# IMPORTANT:
# ASSIGNMENT IS TRAIN-ONLY
# ------------------------------------------------

train = population[
    population["split"] == "train"
].copy()

print(
    "Train images :",
    len(train)
)

print(
    "Train lesions :",
    train["lesion_id"].nunique()
)

if len(train) != 18514:
    raise AssertionError(
        f"Unexpected train image count: {len(train)}"
    )

# ------------------------------------------------
# BUILD LESION-LEVEL INPUT
# ------------------------------------------------

lesions_df = (
    train
    .groupby(
        [
            "lesion_id",
            "malignant",
            "class_name"
        ],
        as_index=False
    )
    .agg(
        n_images=("image", "size")
    )
)

lesions_df = lesions_df.rename(
    columns={
        "class_name": "subtype"
    }
)

lesions_df = lesions_df[
    [
        "lesion_id",
        "malignant",
        "subtype",
        "n_images"
    ]
].copy()

print("\nLESION INPUT")
print("-" * 70)
print(
    "Rows:",
    len(lesions_df)
)

print(
    "Columns:",
    list(lesions_df.columns)
)

# ------------------------------------------------
# EXACT TRAIN POPULATION COUNTS
# ------------------------------------------------

N_M = int(
    lesions_df.loc[
        lesions_df["malignant"] == 1,
        "n_images"
    ].sum()
)

N_B = int(
    lesions_df.loc[
        lesions_df["malignant"] == 0,
        "n_images"
    ].sum()
)

N = N_M + N_B

print("\nCLASS IMAGE COUNTS")
print("-" * 70)
print("N_M :", N_M)
print("N_B :", N_B)
print("N   :", N)

# ------------------------------------------------
# LOCKED TOTAL HAIR COUNT
# ------------------------------------------------

TARGET_TOTAL_HAIR = N // 2

print(
    "\nTarget total hair images:",
    TARGET_TOTAL_HAIR
)

# ------------------------------------------------
# EXPECTED RHO=.5 TARGET PROBABILITIES
# ------------------------------------------------

# From the locked rho construction:
#
# malignant:
#   0.5 + 0.5*rho
#
# benign:
#   (0.5*N - N_M*rho) / N_B
#
# We calculate the exact Fraction values
# rather than floating-point approximations.

rho_fraction = Fraction(1, 2)

p_M = (
    Fraction(1, 2)
    + Fraction(1, 2) * rho_fraction
)

p_B = (
    Fraction(N, 2)
    - Fraction(N_M) * rho_fraction
) / Fraction(N_B)

print("\nRHO=.5 TARGETS")
print("-" * 70)

print(
    "P(H|M) exact:",
    p_M,
    "=",
    float(p_M)
)

print(
    "P(H|B) exact:",
    p_B,
    "=",
    float(p_B)
)

theoretical_M = p_M * N_M
theoretical_B = p_B * N_B

print(
    "Theoretical malignant hair:",
    theoretical_M,
    "=",
    float(theoretical_M)
)

print(
    "Theoretical benign hair:",
    theoretical_B,
    "=",
    float(theoretical_B)
)

print(
    "Theoretical total:",
    theoretical_M + theoretical_B
)

# ------------------------------------------------
# IMPORTANT CONSISTENCY CHECK
# ------------------------------------------------

if theoretical_M + theoretical_B != TARGET_TOTAL_HAIR:
    raise AssertionError(
        "rho=.5 theoretical total does not equal "
        "locked total hair count."
    )

# ------------------------------------------------
# GENERATION
# ------------------------------------------------

print("\n" + "=" * 70)
print("GENERATING RHO=0.5 ASSIGNMENTS")
print("=" * 70)

generation_records = []

conditions = [
    "matched",
    "neutral",
    "reversed",
]

for condition in conditions:

    print(
        f"\nCONDITION: {condition}"
    )
    print("-" * 70)

    for replicate in REPLICATES:

        seed = assignment_seed(
            MASTER_SEED,
            RHO,
            replicate
        )

        print(
            f"\nReplicate {replicate}"
        )

        print(
            "Assignment seed:",
            seed
        )

        result = (
            solver.solve_evaluation_assignment(
                lesions=lesions_df.to_dict(
                    orient="records"
                ),
                condition=condition,
                seed=seed,
                target_hair_count=TARGET_TOTAL_HAIR
            )
        )

        selected = set(
            result["selected_lesions"]
        )

        print(
            "Selected lesions:",
            len(selected)
        )

        print(
            "H_M:",
            result["H_M"]
        )

        print(
            "H_B:",
            result["H_B"]
        )

        print(
            "Total:",
            result["realized"]["total_hair"]
        )

        print(
            "P(H|M):",
            float(
                result["realized"]["P_H_given_M"]
            )
        )

        print(
            "P(H|B):",
            float(
                result["realized"]["P_H_given_B"]
            )
        )

        print(
            "J1:",
            result["J1"]
        )

        print(
            "J2:",
            result["J2"]
        )

        # ------------------------------------------------
        # CONVERT LESION ASSIGNMENT TO IMAGE ASSIGNMENT
        # ------------------------------------------------

        assignment = train[
            [
                "image",
                "lesion_id",
                "class_name",
                "malignant",
                "split"
            ]
        ].copy()

        assignment["rho"] = RHO
        assignment["replicate"] = replicate
        assignment["assignment_seed"] = seed

        assignment["hair_present"] = (
            assignment["lesion_id"]
            .isin(selected)
            .astype("int64")
        )

        assignment["severity"] = "medium"
        assignment["artifact"] = (
            assignment["hair_present"]
            .map({
                0: "clean",
                1: "hair"
            })
        )

        # ------------------------------------------------
        # EXACT SCHEMA CHECK
        # ------------------------------------------------

        expected_columns = [
            "image",
            "lesion_id",
            "class_name",
            "malignant",
            "split",
            "rho",
            "replicate",
            "assignment_seed",
            "hair_present",
            "severity",
            "artifact",
        ]

        assignment = assignment[
            expected_columns
        ]

        # ------------------------------------------------
        # INVARIANTS
        # ------------------------------------------------

        realized_total = int(
            assignment["hair_present"].sum()
        )

        realized_M = int(
            assignment.loc[
                assignment["malignant"] == 1,
                "hair_present"
            ].sum()
        )

        realized_B = int(
            assignment.loc[
                assignment["malignant"] == 0,
                "hair_present"
            ].sum()
        )

        if realized_total != TARGET_TOTAL_HAIR:
            raise AssertionError(
                f"Total hair mismatch: "
                f"{realized_total}"
            )

        if realized_M != result["H_M"]:
            raise AssertionError(
                "Malignant hair mismatch."
            )

        if realized_B != result["H_B"]:
            raise AssertionError(
                "Benign hair mismatch."
            )

        if assignment["lesion_id"].nunique() != len(
            assignment
        ):
            # This would be wrong because lesions can
            # contain multiple images, so do not use
            # this as an actual invariant.
            pass

        # ------------------------------------------------
        # WRITE
        # ------------------------------------------------

        filename = (
            f"rho_0.5_{condition}"
            f"_rep_{replicate}.csv"
        )

        output_path = (
            ASSIGN_DIR / filename
        )

        assignment.to_csv(
            output_path,
            index=False
        )

        sha = hashlib.sha256(
            output_path.read_bytes()
        ).hexdigest()

        generation_records.append(
            {
                "rho": RHO,
                "condition": condition,
                "replicate": replicate,
                "assignment_seed": seed,
                "H_M": realized_M,
                "H_B": realized_B,
                "total_hair": realized_total,
                "J1": str(result["J1"]),
                "J2": str(result["J2"]),
                "rows": len(assignment),
                "sha256": sha,
                "path": str(output_path),
            }
        )

        print(
            "Saved:",
            output_path
        )

        print(
            "SHA256:",
            sha
        )

# ------------------------------------------------
# SAVE GENERATION MANIFEST
# ------------------------------------------------

manifest = pd.DataFrame(
    generation_records
)

manifest_path = (
    OUT_DIR
    / "T2_1_RHO_0_5_ASSIGNMENT_GENERATION_MANIFEST.csv"
)

manifest.to_csv(
    manifest_path,
    index=False
)

# ------------------------------------------------
# FINAL SUMMARY
# ------------------------------------------------

print("\n" + "=" * 70)
print("CELL 6 COMPLETE")
print("=" * 70)

print(
    "Assignments generated:",
    len(generation_records)
)

print(
    "Expected:",
    9
)

print(
    "\nManifest:",
    manifest_path
)

print(
    "\nNOTE:"
)

print(
    "These are the three rho=.5 assignment replicates "
    "for each of matched/neutral/reversed."
)

print(
    "No training has started."
)

print("=" * 70)

In [ ]:
# ================================================================
# T2.1 — CELL 6A
# FORENSIC AUDIT OF EXISTING LOCKED rho=0 / rho=1 ASSIGNMENTS
# ================================================================

from pathlib import Path
import pandas as pd
import hashlib
import json

print("=" * 70)
print("T2.1 — CELL 6A")
print("EXISTING rho=0 / rho=1 ASSIGNMENT FORENSIC AUDIT")
print("=" * 70)

BASE = Path(
    "/kaggle/input/datasets/tejasseshadriiiii/isic-final-project"
)

# ------------------------------------------------
# SEARCH ALL EXISTING ASSIGNMENT CSVs
# ------------------------------------------------

print("\nSEARCHING EXISTING ASSIGNMENTS")
print("-" * 70)

assignment_files = sorted(
    BASE.rglob("rho_*.csv")
)

for p in assignment_files:
    print(p)

# ------------------------------------------------
# TARGET PRIMARY FILES
# ------------------------------------------------

targets = {}

for rho in ["0.0", "1.0"]:
    for rep in [1, 2, 3]:

        matches = [
            p for p in assignment_files
            if p.name == f"rho_{rho}_rep_{rep}.csv"
        ]

        if len(matches) == 0:
            print(
                f"NOT FOUND: rho_{rho}_rep_{rep}.csv"
            )
        elif len(matches) > 1:
            print(
                f"MULTIPLE FOUND: rho_{rho}_rep_{rep}.csv"
            )
            for p in matches:
                print("   ", p)
        else:
            targets[
                f"rho_{rho}_rep_{rep}"
            ] = matches[0]

# ------------------------------------------------
# LOAD AND AUDIT
# ------------------------------------------------

print("\n" + "=" * 70)
print("ASSIGNMENT STATISTICS")
print("=" * 70)

records = []

for key, path in targets.items():

    df = pd.read_csv(path)

    rho = float(
        df["rho"].iloc[0]
    )

    rep = int(
        df["replicate"].iloc[0]
    )

    H_total = int(
        df["hair_present"].sum()
    )

    H_M = int(
        df.loc[
            df["malignant"] == 1,
            "hair_present"
        ].sum()
    )

    H_B = int(
        df.loc[
            df["malignant"] == 0,
            "hair_present"
        ].sum()
    )

    N = len(df)

    N_M = int(
        (df["malignant"] == 1).sum()
    )

    N_B = int(
        (df["malignant"] == 0).sum()
    )

    p_M = H_M / N_M
    p_B = H_B / N_B
    p_total = H_total / N

    lesion_hair = (
        df.groupby("lesion_id")["hair_present"]
        .nunique()
    )

    lesion_consistent = bool(
        (lesion_hair == 1).all()
    )

    sha = hashlib.sha256(
        path.read_bytes()
    ).hexdigest()

    records.append(
        {
            "key": key,
            "path": str(path),
            "rho": rho,
            "replicate": rep,
            "N": N,
            "N_M": N_M,
            "N_B": N_B,
            "H_M": H_M,
            "H_B": H_B,
            "H_total": H_total,
            "p_H_given_M": p_M,
            "p_H_given_B": p_B,
            "p_H": p_total,
            "lesion_assignment_consistent": lesion_consistent,
            "sha256": sha,
        }
    )

stats = pd.DataFrame(records)

print(
    stats[
        [
            "key",
            "rho",
            "replicate",
            "N",
            "N_M",
            "N_B",
            "H_M",
            "H_B",
            "H_total",
            "p_H_given_M",
            "p_H_given_B",
            "p_H",
            "lesion_assignment_consistent",
        ]
    ].to_string(index=False)
)

# ------------------------------------------------
# COMPARE AGAINST LOCKED STEP-1 TARGET TABLE
# ------------------------------------------------

print("\n" + "=" * 70)
print("COMPARE AGAINST LOCKED TARGETS")
print("=" * 70)

target_path = (
    BASE
    / "step1D_rho_feasibility_corrected"
    / "step1D_corrected_rho_targets.csv"
)

target_df = pd.read_csv(
    target_path
)

print(
    target_df[
        [
            "rho",
            "target_p_h_given_m",
            "target_p_h_given_b",
            "target_malignant_hair_images",
            "target_benign_hair_images",
            "target_total_hair_images",
            "actual_p_h_given_m",
            "actual_p_h_given_b",
            "actual_p_h",
            "empirical_phi",
        ]
    ].to_string(index=False)
)

# ------------------------------------------------
# DETERMINE WHICH TARGETS ACTUAL ASSIGNMENTS MATCH
# ------------------------------------------------

print("\n" + "=" * 70)
print("TARGET RECONCILIATION")
print("=" * 70)

for _, row in stats.iterrows():

    rho = row["rho"]

    match = target_df[
        target_df["rho"] == rho
    ]

    if len(match) != 1:
        print(
            f"{row['key']}: NO UNIQUE TARGET ROW"
        )
        continue

    t = match.iloc[0]

    print(f"\n{row['key']}")

    print(
        "Observed H_M:",
        int(row["H_M"]),
        "| target:",
        int(t["target_malignant_hair_images"])
    )

    print(
        "Observed H_B:",
        int(row["H_B"]),
        "| target:",
        int(t["target_benign_hair_images"])
    )

    print(
        "Observed H:",
        int(row["H_total"]),
        "| target:",
        int(t["target_total_hair_images"])
    )

    print(
        "Observed p_M:",
        row["p_H_given_M"],
        "| target:",
        t["target_p_h_given_m"]
    )

    print(
        "Observed p_B:",
        row["p_H_given_B"],
        "| target:",
        t["target_p_h_given_b"]
    )

# ------------------------------------------------
# CHECK ASSIGNMENT SEEDS
# ------------------------------------------------

print("\n" + "=" * 70)
print("ASSIGNMENT SEEDS")
print("=" * 70)

for key, path in targets.items():

    df = pd.read_csv(path)

    seed_values = (
        df["assignment_seed"]
        .drop_duplicates()
        .tolist()
    )

    print(
        key,
        "=>",
        seed_values
    )

# ------------------------------------------------
# FINAL STATUS
# ------------------------------------------------

print("\n" + "=" * 70)
print("CELL 6A COMPLETE")
print("=" * 70)

print(
    "Existing primary assignments audited:",
    len(stats)
)

print(
    "NO new rho=.5 assignment generated."
)

print(
    "NO existing files modified."
)

print("=" * 70)

In [ ]:
# ================================================================
# T2.1 — CELL 6B
# FORENSIC AUDIT OF LOCKED STEP-1F ASSIGNMENT GENERATOR
# ================================================================

from pathlib import Path
import hashlib
import re

print("=" * 70)
print("T2.1 — CELL 6B")
print("LOCKED STEP-1F ASSIGNMENT GENERATOR FORENSIC AUDIT")
print("=" * 70)

BASE = Path(
    "/kaggle/input/datasets/tejasseshadriiiii/isic-final-project"
)

STEP1F_DIR = (
    BASE
    / "step1F_lesion_level_rho_assignments"
)

print("\nSTEP-1F DIRECTORY")
print("-" * 70)
print(STEP1F_DIR)

if not STEP1F_DIR.exists():
    raise FileNotFoundError(
        f"Missing: {STEP1F_DIR}"
    )

# ------------------------------------------------
# LIST ALL FILES
# ------------------------------------------------

files = sorted(
    p for p in STEP1F_DIR.rglob("*")
    if p.is_file()
)

print("\nFILES")
print("-" * 70)

for p in files:
    print(
        p.relative_to(STEP1F_DIR)
    )

# ------------------------------------------------
# IDENTIFY PYTHON / CODE / METADATA ARTIFACTS
# ------------------------------------------------

code_files = [
    p for p in files
    if p.suffix.lower() in {
        ".py",
        ".txt",
        ".md",
        ".json",
        ".yaml",
        ".yml"
    }
]

print("\nCODE / METADATA CANDIDATES")
print("-" * 70)

for p in code_files:
    print(
        p.relative_to(STEP1F_DIR),
        "|",
        p.stat().st_size,
        "bytes"
    )

# ------------------------------------------------
# SEARCH FOR GENERATOR LOGIC
# ------------------------------------------------

patterns = [
    "rho",
    "assignment_seed",
    "rho-assignment-v1",
    "matched",
    "neutral",
    "reversed",
    "target_hair",
    "target_malignant",
    "target_benign",
    "solve_evaluation_assignment",
    "target_probability",
    "round",
    "floor",
    "ceil",
]

print("\n" + "=" * 70)
print("CONTENT SEARCH")
print("=" * 70)

hits = {}

for path in code_files:

    try:
        text = path.read_text(
            encoding="utf-8",
            errors="replace"
        )
    except Exception:
        continue

    path_hits = []

    for pattern in patterns:

        if pattern.lower() in text.lower():
            path_hits.append(pattern)

    if path_hits:
        hits[path] = path_hits

for path, path_hits in hits.items():

    print(
        "\nFILE:",
        path.relative_to(STEP1F_DIR)
    )

    print(
        "MATCHES:",
        ", ".join(path_hits)
    )

# ------------------------------------------------
# PRINT PYTHON SOURCE FILES
# ------------------------------------------------

print("\n" + "=" * 70)
print("PYTHON SOURCE")
print("=" * 70)

py_files = [
    p for p in files
    if p.suffix.lower() == ".py"
]

if not py_files:

    print(
        "NO .py FILES FOUND INSIDE STEP-1F DIRECTORY."
    )

else:

    for path in py_files:

        print("\n")
        print("#" * 70)
        print(
            "# FILE:",
            path.relative_to(STEP1F_DIR)
        )
        print("#" * 70)

        print(
            path.read_text(
                encoding="utf-8",
                errors="replace"
            )
        )

# ------------------------------------------------
# CHECK PARENT / NEIGHBORING STEP FILES
# ------------------------------------------------

print("\n" + "=" * 70)
print("NEIGHBORING ASSIGNMENT IMPLEMENTATION FILES")
print("=" * 70)

parent = STEP1F_DIR.parent

candidate_files = []

for p in parent.rglob("*"):

    if not p.is_file():
        continue

    name = p.name.lower()

    if any(
        token in name
        for token in [
            "assignment",
            "rho",
            "solver",
            "a1",
            "lesion"
        ]
    ):

        candidate_files.append(p)

for p in sorted(candidate_files):

    print(
        p.relative_to(BASE),
        "|",
        p.stat().st_size,
        "bytes"
    )

# ------------------------------------------------
# HASH EXISTING GENERATOR CODE
# ------------------------------------------------

print("\n" + "=" * 70)
print("SOURCE HASHES")
print("=" * 70)

for path in py_files:

    sha = hashlib.sha256(
        path.read_bytes()
    ).hexdigest()

    print(
        path.relative_to(BASE)
    )

    print(
        "SHA256:",
        sha
    )

# ------------------------------------------------
# FINAL STATUS
# ------------------------------------------------

print("\n" + "=" * 70)
print("CELL 6B COMPLETE")
print("=" * 70)

print(
    "Existing assignments inspected : True"
)

print(
    "Generator source searched      : True"
)

print(
    "New rho=.5 assignments         : NONE"
)

print(
    "Existing assignments modified  : NONE"
)

print(
    "\nNEXT STEP:"
)

print(
    "Reuse the actual locked Step-1F "
    "rho-generation implementation."
)

print("=" * 70)

In [ ]:
# ================================================================
# T2.1 — CELL 6C
# LOCATE ORIGINAL STEP-1F GENERATION IMPLEMENTATION
# ================================================================

from pathlib import Path
import re
import hashlib

print("=" * 70)
print("T2.1 — CELL 6C")
print("LOCATE ORIGINAL STEP-1F GENERATION IMPLEMENTATION")
print("=" * 70)

SEARCH_ROOTS = [
    Path("/kaggle/input/datasets/tejasseshadriiiii/isic-final-project"),
    Path("/kaggle/input/notebooks/tejasseshadriiiii/isic-kaggle-analysis"),
    Path("/kaggle/input/notebooks/tejasseshadriiiii/ml-open-lab-30m-trained"),
]

PATTERNS = [
    "rho-assignment-v1",
    "solve_evaluation_assignment",
    "target_malignant_hair_images",
    "target_benign_hair_images",
    "empirical_phi",
    "rho_0.0_rep_1",
    "rho_values",
    "assignment_seed",
    "lesion-level rho",
    "lesion_level_rho",
]

EXTENSIONS = {
    ".py",
    ".ipynb",
    ".json",
    ".md",
    ".txt",
    ".yaml",
    ".yml",
}

# ------------------------------------------------
# SEARCH
# ------------------------------------------------

matches = []

print("\nSEARCH ROOTS")
print("-" * 70)

for root in SEARCH_ROOTS:
    print(root)

print("\nSEARCHING...")
print("-" * 70)

for root in SEARCH_ROOTS:

    if not root.exists():
        print("MISSING ROOT:", root)
        continue

    for path in root.rglob("*"):

        if not path.is_file():
            continue

        if path.suffix.lower() not in EXTENSIONS:
            continue

        # Avoid enormous model/data binaries disguised as text.
        if path.stat().st_size > 50_000_000:
            continue

        try:
            raw = path.read_bytes()

            text = raw.decode(
                "utf-8",
                errors="ignore"
            )

        except Exception:
            continue

        found = []

        lower_text = text.lower()

        for pattern in PATTERNS:

            if pattern.lower() in lower_text:
                found.append(pattern)

        if found:

            sha = hashlib.sha256(
                raw
            ).hexdigest()

            matches.append(
                {
                    "path": path,
                    "size": path.stat().st_size,
                    "sha256": sha,
                    "matches": found,
                }
            )

# ------------------------------------------------
# RESULTS
# ------------------------------------------------

print("\n" + "=" * 70)
print("MATCHING ARTIFACTS")
print("=" * 70)

if not matches:

    print(
        "NO MATCHING SOURCE ARTIFACTS FOUND."
    )

else:

    for item in matches:

        print("\nFILE:")
        print(item["path"])

        print(
            "SIZE:",
            item["size"]
        )

        print(
            "SHA256:",
            item["sha256"]
        )

        print(
            "MATCHES:",
            ", ".join(item["matches"])
        )

# ------------------------------------------------
# HIGH-VALUE CANDIDATES
# ------------------------------------------------

print("\n" + "=" * 70)
print("HIGH-VALUE GENERATOR CANDIDATES")
print("=" * 70)

generator_keywords = [
    "step1f",
    "rho",
    "assignment",
    "lesion",
    "solver",
]

candidates = []

for item in matches:

    name = item["path"].name.lower()

    score = sum(
        keyword in name
        for keyword in generator_keywords
    )

    if score > 0:
        candidates.append(
            (score, item)
        )

for score, item in sorted(
    candidates,
    key=lambda x: (-x[0], str(x[1]["path"]))
):

    print(
        f"\nSCORE {score}:",
        item["path"]
    )

    print(
        "MATCHES:",
        ", ".join(item["matches"])
    )

# ------------------------------------------------
# FINAL STATUS
# ------------------------------------------------

print("\n" + "=" * 70)
print("CELL 6C COMPLETE")
print("=" * 70)

print(
    "Matching artifacts:",
    len(matches)
)

print(
    "No rho=.5 assignment generated."
)

print(
    "No existing assignment modified."
)

print(
    "\nIf a notebook/script containing the original "
    "rho-generation logic is found, we inspect it next."
)

print("=" * 70)

In [ ]:
# ================================================================
# T2.1 — CELL 6D
# EXTRACT ORIGINAL STEP-1F GENERATION LOGIC FROM LOCKED NOTEBOOK
# ================================================================

from pathlib import Path
import json
import re

print("=" * 70)
print("T2.1 — CELL 6D")
print("ORIGINAL STEP-1F GENERATION LOGIC EXTRACTION")
print("=" * 70)

NOTEBOOK = Path(
    "/kaggle/input/notebooks/tejasseshadriiiii/"
    "isic-kaggle-analysis/__notebook__.ipynb"
)

print("\nNOTEBOOK")
print("-" * 70)
print(NOTEBOOK)
print("Exists:", NOTEBOOK.exists())

if not NOTEBOOK.exists():
    raise FileNotFoundError(NOTEBOOK)

with open(
    NOTEBOOK,
    "r",
    encoding="utf-8"
) as f:
    nb = json.load(f)

cells = nb.get("cells", [])

print(
    "Total notebook cells:",
    len(cells)
)

# ------------------------------------------------
# SEARCH TERMS
# ------------------------------------------------

patterns = [
    "rho-assignment-v1",
    "rho_0.0_rep_1",
    "rho_1.0_rep_1",
    "lesion_level_rho",
    "solve_evaluation_assignment",
    "target_malignant_hair",
    "target_benign_hair",
    "target_hair_count",
    "empirical_phi",
    "target_probability",
    "assignment_seed",
    "hair_present",
    "rho_values",
    "matched",
    "neutral",
    "reversed",
]

# ------------------------------------------------
# SEARCH CODE CELLS
# ------------------------------------------------

hits = []

for idx, cell in enumerate(cells):

    source = cell.get("source", [])

    if isinstance(source, list):
        source = "".join(source)

    if not source.strip():
        continue

    lower = source.lower()

    matched = [
        p for p in patterns
        if p.lower() in lower
    ]

    if matched:

        hits.append(
            {
                "index": idx,
                "type": cell.get("cell_type"),
                "matches": matched,
                "source": source,
            }
        )

print("\n" + "=" * 70)
print("MATCHING NOTEBOOK CELLS")
print("=" * 70)

print(
    "Matching cells:",
    len(hits)
)

for item in hits:

    print(
        f"\nCELL {item['index']} "
        f"[{item['type']}]"
    )

    print(
        "MATCHES:",
        ", ".join(item["matches"])
    )

# ------------------------------------------------
# PRINT RELEVANT CODE CELLS
# ------------------------------------------------

print("\n" + "=" * 70)
print("RELEVANT SOURCE CELLS")
print("=" * 70)

for item in hits:

    if item["type"] != "code":
        continue

    print("\n")
    print("#" * 70)
    print(
        f"# NOTEBOOK CELL {item['index']}"
    )
    print(
        "# MATCHES:",
        ", ".join(item["matches"])
    )
    print("#" * 70)

    print(
        item["source"]
    )

# ------------------------------------------------
# SEARCH OUTPUT CELLS TOO
# ------------------------------------------------

print("\n" + "=" * 70)
print("RELEVANT OUTPUT CELLS")
print("=" * 70)

output_hits = []

for idx, cell in enumerate(cells):

    if cell.get("cell_type") != "code":
        continue

    outputs = cell.get("outputs", [])

    for output in outputs:

        text_parts = []

        if "text" in output:

            value = output["text"]

            if isinstance(value, list):
                value = "".join(value)

            text_parts.append(value)

        data = output.get("data", {})

        if "text/plain" in data:

            value = data["text/plain"]

            if isinstance(value, list):
                value = "".join(value)

            text_parts.append(value)

        combined = "\n".join(text_parts)

        if not combined:
            continue

        lower = combined.lower()

        matched = [
            p for p in patterns
            if p.lower() in lower
        ]

        if matched:

            output_hits.append(
                {
                    "cell": idx,
                    "matches": matched,
                    "text": combined,
                }
            )

for item in output_hits:

    print("\n")
    print(
        f"OUTPUT FROM CELL {item['cell']}"
    )

    print(
        "MATCHES:",
        ", ".join(item["matches"])
    )

    print(
        item["text"]
    )

# ------------------------------------------------
# SPECIAL SEARCH:
# FIND CELLS THAT ACTUALLY WRITE rho_*.csv
# ------------------------------------------------

print("\n" + "=" * 70)
print("CSV-GENERATION CELLS")
print("=" * 70)

generation_patterns = [
    r"to_csv\s*\(",
    r"rho_\{",
    r"rho_\{rho",
    r"rho_.*_rep",
    r"assignment.*csv",
    r"to_csv",
]

generation_cells = []

for idx, cell in enumerate(cells):

    if cell.get("cell_type") != "code":
        continue

    source = cell.get("source", [])

    if isinstance(source, list):
        source = "".join(source)

    if not source.strip():
        continue

    matched = [
        p for p in generation_patterns
        if re.search(
            p,
            source,
            flags=re.IGNORECASE
        )
    ]

    if matched:

        generation_cells.append(
            (
                idx,
                matched,
                source
            )
        )

for idx, matched, source in generation_cells:

    print("\n")
    print("#" * 70)
    print(
        f"# GENERATION CANDIDATE CELL {idx}"
    )
    print(
        "# MATCHES:",
        ", ".join(matched)
    )
    print("#" * 70)

    print(source)

# ------------------------------------------------
# FINAL STATUS
# ------------------------------------------------

print("\n" + "=" * 70)
print("CELL 6D COMPLETE")
print("=" * 70)

print(
    "Notebook loaded:",
    True
)

print(
    "Matching cells:",
    len(hits)
)

print(
    "Generation candidates:",
    len(generation_cells)
)

print(
    "\nNO rho=.5 assignment generated."
)

print(
    "NO existing assignment modified."
)

print("=" * 70)

In [ ]:
# ================================================================
# T2.1 — CELL 6E
# SOLVER API + LOCKED EVALUATION POPULATION FORENSIC INSPECTION
# ================================================================

from pathlib import Path
import importlib.util
import inspect
import json
import hashlib
import pandas as pd
import numpy as np

print("=" * 70)
print("T2.1 — CELL 6E")
print("A1.1 SOLVER API + POPULATION INSPECTION")
print("=" * 70)

PROJECT = Path(
    "/kaggle/input/datasets/tejasseshadriiiii/isic-final-project"
)

SOLVER_PATH = (
    PROJECT
    / "step2Q_solver_implementation_v2"
    / "deterministic_evaluation_solver.py"
)

POP_PATH = (
    PROJECT
    / "step2Q_real_evaluation_population"
    / "LOCKED_EVALUATION_LESION_TABLE.csv"
)

A11_PATH = (
    PROJECT
    / "step2P_A1_1_solver_addendum_lock"
    / "FINAL_LOCKED_A1_1_SOLVER_ADDENDUM.json"
)

STEP1F_PROTOCOL = (
    PROJECT
    / "step1F_lesion_level_rho_assignments"
    / "step1F_locked_assignment_protocol.json"
)

print("\nPATHS")
print("-" * 70)

for p in [
    SOLVER_PATH,
    POP_PATH,
    A11_PATH,
    STEP1F_PROTOCOL,
]:
    print(p)
    print("exists:", p.exists())

assert SOLVER_PATH.exists()
assert POP_PATH.exists()
assert A11_PATH.exists()
assert STEP1F_PROTOCOL.exists()

# ------------------------------------------------
# SOLVER HASH
# ------------------------------------------------

def sha256_file(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1 << 20),
            b""
        ):
            h.update(chunk)

    return h.hexdigest()

solver_hash = sha256_file(SOLVER_PATH)

EXPECTED_SOLVER_HASH = (
    "37451d50afa7dcd8052a00d53cccb853d3ce658adbdd65bc1ce66c14b67fc5d3"
)

print("\nSOLVER HASH")
print("-" * 70)
print("Actual  :", solver_hash)
print("Expected:", EXPECTED_SOLVER_HASH)
print("Match   :", solver_hash == EXPECTED_SOLVER_HASH)

assert solver_hash == EXPECTED_SOLVER_HASH

# ------------------------------------------------
# IMPORT SOLVER WITHOUT EXECUTING ANY GENERATION
# ------------------------------------------------

print("\nIMPORTING SOLVER")
print("-" * 70)

spec = importlib.util.spec_from_file_location(
    "deterministic_evaluation_solver",
    SOLVER_PATH
)

solver = importlib.util.module_from_spec(spec)
spec.loader.exec_module(solver)

print(
    "Imported:",
    True
)

# ------------------------------------------------
# PUBLIC CALLABLES
# ------------------------------------------------

print("\nCALLABLES")
print("-" * 70)

for name in [
    "eval_hash_key",
    "reachable_sums",
    "subtype_dp",
    "canonical_group_selection",
    "solve_evaluation_assignment",
]:

    obj = getattr(
        solver,
        name,
        None
    )

    print(
        f"\n{name}"
    )

    print(
        "exists:",
        obj is not None
    )

    if obj is not None:
        print(
            "signature:",
            inspect.signature(obj)
        )

        print(
            "doc:",
            inspect.getdoc(obj)
        )

# ------------------------------------------------
# MODULE NAMESPACE
# ------------------------------------------------

print("\nMODULE GLOBALS")
print("-" * 70)

for name in sorted(solver.__dict__):

    if name.startswith("_"):
        continue

    obj = solver.__dict__[name]

    if callable(obj):
        continue

    if isinstance(
        obj,
        (
            str,
            int,
            float,
            bool,
            tuple,
            list,
            dict,
        ),
    ):
        print(
            name,
            "=",
            repr(obj)
        )

# ------------------------------------------------
# LOAD A1.1 LOCK
# ------------------------------------------------

with open(
    A11_PATH,
    "r",
    encoding="utf-8"
) as f:
    a11 = json.load(f)

print("\nA1.1 LOCK SUMMARY")
print("-" * 70)

print(
    json.dumps(
        a11,
        indent=2
    )
)

# ------------------------------------------------
# LOAD STEP-1F PROTOCOL
# ------------------------------------------------

with open(
    STEP1F_PROTOCOL,
    "r",
    encoding="utf-8"
) as f:
    step1f = json.load(f)

print("\nSTEP-1F PROTOCOL")
print("-" * 70)

print(
    json.dumps(
        step1f,
        indent=2
    )
)

# ------------------------------------------------
# LOAD LOCKED EVALUATION LESION POPULATION
# ------------------------------------------------

print("\nLOCKED EVALUATION LESION TABLE")
print("-" * 70)

pop = pd.read_csv(
    POP_PATH
)

print(
    "Shape:",
    pop.shape
)

print(
    "\nColumns:"
)

for c in pop.columns:
    print(
        f"  {c}: {pop[c].dtype}"
    )

print(
    "\nFirst rows:"
)

print(
    pop.head().to_string(index=False)
)

# ------------------------------------------------
# BASIC POPULATION STRUCTURE
# ------------------------------------------------

print("\nPOPULATION COUNTS")
print("-" * 70)

for col in [
    "malignant",
    "class_name",
    "split",
]:

    if col in pop.columns:

        print(
            f"\n{col}:"
        )

        print(
            pop[col]
            .value_counts(dropna=False)
            .to_string()
        )

# ------------------------------------------------
# LESION UNIQUENESS
# ------------------------------------------------

if "lesion_id" in pop.columns:

    print(
        "\nLESION STRUCTURE"
    )
    print("-" * 70)

    print(
        "Rows:",
        len(pop)
    )

    print(
        "Unique lesions:",
        pop["lesion_id"].nunique()
    )

    image_per_lesion = (
        pop.groupby("lesion_id")
        .size()
    )

    print(
        "Images per lesion:"
    )

    print(
        image_per_lesion.describe()
        .to_string()
    )

# ------------------------------------------------
# CLASS-SPECIFIC IMAGE COUNTS
# ------------------------------------------------

print(
    "\nCLASS-SPECIFIC IMAGE COUNTS"
)
print("-" * 70)

if "malignant" in pop.columns:

    print(
        pop.groupby("malignant")
        .size()
        .rename("N_images")
        .to_string()
    )

# ------------------------------------------------
# EXACT TARGET CALCULATION INFORMATION
# ------------------------------------------------

print(
    "\nTARGET-RELATED COLUMNS"
)
print("-" * 70)

target_keywords = [
    "rho",
    "weight",
    "target",
    "hair",
    "prob",
    "class",
    "subtype",
]

for c in pop.columns:

    if any(
        k in c.lower()
        for k in target_keywords
    ):
        print(
            c
        )

# ------------------------------------------------
# CHECK WHETHER THIS POPULATION IS THE SOLVER INPUT
# ------------------------------------------------

print(
    "\nSOLVER INPUT COMPATIBILITY"
)
print("-" * 70)

required_candidates = [
    "lesion_id",
    "malignant",
]

for c in required_candidates:

    print(
        c,
        "present:",
        c in pop.columns
    )

print(
    "\nNo assignment is generated in this cell."
)

print("\n" + "=" * 70)
print("CELL 6E COMPLETE")
print("=" * 70)

print(
    "Solver imported:",
    True
)

print(
    "Solver hash verified:",
    True
)

print(
    "Population loaded:",
    True
)

print(
    "rho=.5 generated:",
    False
)

print(
    "Existing assignments modified:",
    False
)

print("=" * 70)

In [ ]:
# ================================================================
# T2.1 — CELL 6F
# FORENSICALLY LOCATE ORIGINAL STEP-1F ASSIGNMENT GENERATOR
# ================================================================

from pathlib import Path
import ast
import hashlib
import re
import json

print("=" * 70)
print("T2.1 — CELL 6F")
print("ORIGINAL STEP-1F ASSIGNMENT GENERATOR SEARCH")
print("=" * 70)

INPUT = Path("/kaggle/input")

PROJECT = (
    INPUT
    / "datasets"
    / "tejasseshadriiiii"
    / "isic-final-project"
)

STEP1F = (
    PROJECT
    / "step1F_lesion_level_rho_assignments"
)

print("\nSTEP-1F DIRECTORY")
print("-" * 70)

print(
    "exists:",
    STEP1F.exists()
)

assert STEP1F.exists()

print(
    "path:",
    STEP1F
)

# ------------------------------------------------
# LIST STEP-1F CONTENTS
# ------------------------------------------------

print("\nSTEP-1F CONTENTS")
print("-" * 70)

for p in sorted(STEP1F.iterdir()):

    if p.is_file():

        print(
            f"{p.name:70s}"
            f"{p.stat().st_size:12,d} bytes"
        )

# ------------------------------------------------
# SEARCH ALL PROJECT FILES FOR UNIQUE GENERATION
# SIGNATURES
# ------------------------------------------------

print("\nSEARCHING PROJECT FOR GENERATION LOGIC")
print("-" * 70)

patterns = [
    "rho-assignment-v1",
    "assignment_seed",
    "lesion_level_rho",
    "hair_present",
    "matched",
    "neutral",
    "reversed",
    "subtype",
    "PCG64",
    "SHA256",
]

candidate_files = []

for p in PROJECT.rglob("*"):

    if not p.is_file():
        continue

    # Only inspect likely source/text files.
    if p.suffix.lower() not in {
        ".py",
        ".ipynb",
        ".md",
        ".json",
        ".txt",
        ".yaml",
        ".yml",
    }:
        continue

    try:
        size = p.stat().st_size

        # Avoid giant binary-ish files.
        if size > 20_000_000:
            continue

        text = p.read_text(
            encoding="utf-8",
            errors="ignore"
        )

    except Exception:
        continue

    hits = [
        pat
        for pat in patterns
        if pat in text
    ]

    if hits:

        candidate_files.append(
            (
                p,
                hits
            )
        )

print(
    "Candidate files:",
    len(candidate_files)
)

for p, hits in candidate_files:

    print(
        "\nFILE:",
        p
    )

    print(
        "MATCHES:",
        ", ".join(hits)
    )

# ------------------------------------------------
# PRIORITIZE PYTHON / NOTEBOOK GENERATORS
# ------------------------------------------------

print("\nPOTENTIAL GENERATOR SOURCES")
print("-" * 70)

generator_candidates = []

generation_words = [
    "generate",
    "assignment",
    "rho",
    "lesion",
    "hair_present",
]

for p, hits in candidate_files:

    if p.suffix.lower() not in {
        ".py",
        ".ipynb",
    }:
        continue

    score = 0

    lower_name = p.name.lower()

    if "assign" in lower_name:
        score += 5

    if "rho" in lower_name:
        score += 3

    if "train" in lower_name:
        score += 1

    if "solver" in lower_name:
        score += 1

    for h in hits:
        if h in generation_words:
            score += 1

    generator_candidates.append(
        (
            score,
            p,
            hits
        )
    )

generator_candidates.sort(
    key=lambda x: (-x[0], str(x[1]))
)

for score, p, hits in generator_candidates:

    print(
        f"\nSCORE={score}"
    )

    print(
        "PATH:",
        p
    )

    print(
        "MATCHES:",
        ", ".join(hits)
    )

# ------------------------------------------------
# SEARCH NOTEBOOK CODE CELLS SPECIFICALLY
# ------------------------------------------------

print("\nNOTEBOOK GENERATION-CELL FORENSICS")
print("-" * 70)

notebook_hits = []

for p, hits in candidate_files:

    if p.suffix.lower() != ".ipynb":
        continue

    try:
        nb = json.loads(
            p.read_text(
                encoding="utf-8",
                errors="ignore"
            )
        )
    except Exception:
        continue

    cells = nb.get(
        "cells",
        []
    )

    for i, cell in enumerate(cells):

        if cell.get("cell_type") != "code":
            continue

        source = "".join(
            cell.get(
                "source",
                []
            )
        )

        # Strong generation indicators.
        strong = [
            "assignment_seed",
            "rho-assignment-v1",
            "hair_present",
            "lesion_id",
            "PCG64",
        ]

        matched = [
            s
            for s in strong
            if s in source
        ]

        # Also identify CSV-writing cells.
        csv_write = (
            "to_csv(" in source
            and (
                "rho_" in source
                or "assignment" in source
            )
        )

        if matched or csv_write:

            notebook_hits.append(
                (
                    p,
                    i,
                    matched,
                    csv_write,
                    source
                )
            )

for p, i, matched, csv_write, source in notebook_hits:

    print(
        "\n" + "-" * 70
    )

    print(
        "NOTEBOOK:",
        p
    )

    print(
        "CELL:",
        i
    )

    print(
        "STRONG MATCHES:",
        matched
    )

    print(
        "CSV WRITER:",
        csv_write
    )

    # Print only the relevant source, not the whole notebook.
    lines = source.splitlines()

    for j, line in enumerate(lines):

        if (
            any(s in line for s in matched)
            or "to_csv(" in line
            or "rho_" in line
            or "assignment" in line.lower()
        ):

            lo = max(
                0,
                j - 3
            )

            hi = min(
                len(lines),
                j + 4
            )

            print(
                f"\n--- source lines {lo+1}:{hi} ---"
            )

            print(
                "\n".join(
                    lines[lo:hi]
                )
            )

# ------------------------------------------------
# SEARCH NOTEBOOKS OUTSIDE PROJECT TOO
# ------------------------------------------------

print("\nSEARCHING /KAGGLE/INPUT NOTEBOOK SOURCES")
print("-" * 70)

external_nb_hits = []

for p in INPUT.rglob("*.ipynb"):

    # Don't duplicate project notebook already inspected.
    if str(p).startswith(
        str(PROJECT)
    ):
        continue

    try:

        if p.stat().st_size > 30_000_000:
            continue

        nb = json.loads(
            p.read_text(
                encoding="utf-8",
                errors="ignore"
            )
        )

    except Exception:
        continue

    for i, cell in enumerate(
        nb.get("cells", [])
    ):

        if cell.get("cell_type") != "code":
            continue

        source = "".join(
            cell.get(
                "source",
                []
            )
        )

        score = sum(
            token in source
            for token in [
                "rho-assignment-v1",
                "assignment_seed",
                "hair_present",
                "lesion_id",
                "PCG64",
                "rho_0.0",
                "rho_1.0",
            ]
        )

        if score >= 2:

            external_nb_hits.append(
                (
                    score,
                    p,
                    i,
                    source
                )
            )

external_nb_hits.sort(
    key=lambda x: (
        -x[0],
        str(x[1]),
        x[2]
    )
)

print(
    "External notebook candidate cells:",
    len(external_nb_hits)
)

for score, p, i, source in external_nb_hits[:30]:

    print(
        "\n" + "-" * 70
    )

    print(
        "SCORE:",
        score
    )

    print(
        "NOTEBOOK:",
        p
    )

    print(
        "CELL:",
        i
    )

    lines = source.splitlines()

    print(
        "\n".join(
            lines[:80]
        )
    )

# ------------------------------------------------
# FINAL STATUS
# ------------------------------------------------

print("\n" + "=" * 70)
print("CELL 6F COMPLETE")
print("=" * 70)

print(
    "Step-1F files inspected:",
    True
)

print(
    "Project source searched:",
    True
)

print(
    "Notebook source searched:",
    True
)

print(
    "rho=.5 generated:",
    False
)

print(
    "Existing assignments modified:",
    False
)

print("=" * 70)

In [ ]:
# ================================================================
# T2.1 — CELL 6G
# STEP-1J GENERATOR RECOVERY AUDIT
# Determine whether the recovered generator is:
#   (A) the Step-1F assignment generator,
#   (B) only the hair-artifact generator, or
#   (C) insufficient to reproduce Step-1F assignments.
# ================================================================

from pathlib import Path
import hashlib
import json
import inspect
import importlib.util
import re

print("=" * 70)
print("T2.1 — CELL 6G")
print("STEP-1J GENERATOR RECOVERY AUDIT")
print("=" * 70)

PROJECT = Path(
    "/kaggle/input/datasets/tejasseshadriiiii/isic-final-project"
)

GEN_DIR = PROJECT / "step1J_generator_recovery_lock"

LOCK_PATH = GEN_DIR / "generator_recovery_lock.json"
GEN_PATH = GEN_DIR / "recovered_original_hair_generator.py"

print("\nPATHS")
print("-" * 70)

for p in [GEN_DIR, LOCK_PATH, GEN_PATH]:

    print(
        str(p),
        "\nexists:",
        p.exists()
    )

assert LOCK_PATH.exists()
assert GEN_PATH.exists()

# ------------------------------------------------
# HASH FILES
# ------------------------------------------------

def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(chunk)

    return h.hexdigest()


print("\nFILE HASHES")
print("-" * 70)

lock_hash = sha256_file(LOCK_PATH)
gen_hash = sha256_file(GEN_PATH)

print(
    "generator_recovery_lock.json:",
    lock_hash
)

print(
    "recovered_original_hair_generator.py:",
    gen_hash
)

# ------------------------------------------------
# LOAD RECOVERY LOCK
# ------------------------------------------------

print("\nRECOVERY LOCK")
print("-" * 70)

lock = json.loads(
    LOCK_PATH.read_text(
        encoding="utf-8"
    )
)

print(
    json.dumps(
        lock,
        indent=2,
        sort_keys=True
    )
)

# ------------------------------------------------
# LOAD GENERATOR SOURCE
# ------------------------------------------------

source = GEN_PATH.read_text(
    encoding="utf-8"
)

print("\nGENERATOR SOURCE SUMMARY")
print("-" * 70)

print(
    "Source bytes:",
    len(source.encode("utf-8"))
)

print(
    "Source lines:",
    len(source.splitlines())
)

# ------------------------------------------------
# PRINT FUNCTION / CLASS DEFINITIONS
# ------------------------------------------------

print("\nFUNCTIONS / CLASSES")
print("-" * 70)

tree = ast.parse(source)

for node in tree.body:

    if isinstance(
        node,
        (
            ast.FunctionDef,
            ast.AsyncFunctionDef,
            ast.ClassDef,
        )
    ):

        print(
            type(node).__name__,
            node.name,
            f"(line {node.lineno})"
        )

# ------------------------------------------------
# SEARCH FOR STEP-1F ASSIGNMENT CONCEPTS
# ------------------------------------------------

print("\nSTEP-1F CONCEPT SEARCH")
print("-" * 70)

tokens = [
    "rho-assignment-v1",
    "assignment_seed",
    "rho",
    "replicate",
    "hair_present",
    "lesion_id",
    "malignant",
    "subtype",
    "n_images",
    "PCG64",
    "Generator",
    "default_rng",
    "SHA256",
    "sha256",
    "to_csv",
    "assignment",
    "condition",
    "matched",
    "neutral",
    "reversed",
]

lines = source.splitlines()

for token in tokens:

    matches = [
        i + 1
        for i, line in enumerate(lines)
        if token.lower() in line.lower()
    ]

    print(
        f"{token:25s}:",
        matches[:30],
        "..." if len(matches) > 30 else ""
    )

# ------------------------------------------------
# PRINT RELEVANT SOURCE WINDOWS
# ------------------------------------------------

print("\nRELEVANT SOURCE WINDOWS")
print("-" * 70)

interesting = []

for i, line in enumerate(lines):

    low = line.lower()

    if any(
        token.lower() in low
        for token in [
            "rho-assignment-v1",
            "assignment_seed",
            "lesion_id",
            "hair_present",
            "pcg64",
            "default_rng",
            "rho",
            "subtype",
            "to_csv",
            "assignment",
        ]
    ):

        interesting.append(i)

# Merge nearby line numbers.
windows = []

for idx in interesting:

    start = max(
        0,
        idx - 3
    )

    end = min(
        len(lines),
        idx + 4
    )

    if not windows or start > windows[-1][1]:

        windows.append(
            [start, end]
        )

    else:

        windows[-1][1] = max(
            windows[-1][1],
            end
        )

for start, end in windows:

    print(
        "\n--- lines "
        f"{start+1}:{end} ---"
    )

    print(
        "\n".join(
            f"{i+1:5d}: {lines[i]}"
            for i in range(start, end)
        )
    )

# ------------------------------------------------
# IMPORT MODULE SAFELY
# ------------------------------------------------

print("\nMODULE IMPORT")
print("-" * 70)

spec = importlib.util.spec_from_file_location(
    "recovered_original_hair_generator",
    GEN_PATH
)

mod = importlib.util.module_from_spec(
    spec
)

spec.loader.exec_module(mod)

print(
    "Imported:",
    True
)

# ------------------------------------------------
# PUBLIC CALLABLES
# ------------------------------------------------

print("\nPUBLIC CALLABLES")
print("-" * 70)

for name in sorted(
    dir(mod)
):

    if name.startswith("_"):
        continue

    obj = getattr(
        mod,
        name
    )

    if callable(obj):

        try:
            sig = inspect.signature(obj)
        except Exception:
            sig = "<signature unavailable>"

        print(
            f"{name}: {sig}"
        )

# ------------------------------------------------
# MODULE CONSTANTS RELEVANT TO ASSIGNMENTS
# ------------------------------------------------

print("\nRELEVANT MODULE CONSTANTS")
print("-" * 70)

for name in sorted(
    dir(mod)
):

    if name.startswith("_"):
        continue

    if any(
        token in name.lower()
        for token in [
            "seed",
            "rho",
            "hair",
            "artifact",
            "severity",
            "lesion",
            "assignment",
        ]
    ):

        try:
            value = getattr(
                mod,
                name
            )

            if callable(value):
                continue

            print(
                name,
                "=",
                repr(value)
            )

        except Exception:
            pass

# ------------------------------------------------
# COMPARE WITH EXISTING STEP-1F CSV
# ------------------------------------------------

print("\nEXISTING STEP-1F MANIFEST COMPARISON")
print("-" * 70)

STEP1F = PROJECT / "step1F_lesion_level_rho_assignments"

sample_manifest = (
    STEP1F / "rho_0.0_rep_1.csv"
)

assert sample_manifest.exists()

import pandas as pd

df = pd.read_csv(
    sample_manifest
)

print(
    "Manifest:",
    sample_manifest.name
)

print(
    "Shape:",
    df.shape
)

print(
    "Columns:"
)

for c in df.columns:
    print(
        " ",
        c
    )

print(
    "\nUnique rho:",
    sorted(
        df["rho"].unique()
    )
)

print(
    "Unique replicate:",
    sorted(
        df["replicate"].unique()
    )
)

print(
    "Unique assignment seeds:",
    df["assignment_seed"].nunique()
)

print(
    "Unique lesion IDs:",
    df["lesion_id"].nunique()
)

print(
    "Unique images:",
    df["image"].nunique()
)

print(
    "Hair-present counts:"
)

print(
    df["hair_present"].value_counts(
        dropna=False
    ).sort_index()
)

# ------------------------------------------------
# TEST WHETHER GENERATOR API EVEN ACCEPTS
# STEP-1F-LIKE INPUTS
# ------------------------------------------------

print("\nAPI COMPATIBILITY CHECK")
print("-" * 70)

public_callables = []

for name in dir(mod):

    if name.startswith("_"):
        continue

    obj = getattr(
        mod,
        name
    )

    if callable(obj):

        public_callables.append(
            name
        )

for name in public_callables:

    obj = getattr(
        mod,
        name
    )

    try:
        sig = inspect.signature(obj)
    except Exception:
        continue

    params = list(
        sig.parameters.values()
    )

    names = [
        p.name
        for p in params
    ]

    step1f_tokens = [
        "rho",
        "replicate",
        "assignment",
        "lesion",
        "seed",
        "hair",
    ]

    overlap = [
        n
        for n in names
        if any(
            token in n.lower()
            for token in step1f_tokens
        )
    ]

    if overlap:

        print(
            f"{name}{sig}"
        )

        print(
            "  relevant parameters:",
            overlap
        )

# ------------------------------------------------
# FINAL INTERPRETATION
# ------------------------------------------------

print("\n" + "=" * 70)
print("CELL 6G COMPLETE")
print("=" * 70)

print(
    "Generator lock loaded:",
    True
)

print(
    "Recovered generator imported:",
    True
)

print(
    "Existing Step-1F assignment modified:",
    False
)

print(
    "rho=.5 generated:",
    False
)

print("=" * 70)

In [ ]:
# ================================================================
# T2.1 — CELL 6H
# STEP-1F / STEP-1G / STEP-1P ASSIGNMENT FORENSIC AUDIT
# ================================================================

from pathlib import Path
import json
import hashlib
import pandas as pd

print("=" * 70)
print("T2.1 — CELL 6H")
print("STEP-1F ASSIGNMENT FORENSIC AUDIT")
print("=" * 70)

PROJECT = Path(
    "/kaggle/input/datasets/tejasseshadriiiii/isic-final-project"
)

# ------------------------------------------------
# TARGET ARTIFACTS
# ------------------------------------------------

ARTIFACTS = {
    "STEP1F_PROTOCOL":
        PROJECT /
        "step1F_lesion_level_rho_assignments" /
        "step1F_locked_assignment_protocol.json",

    "STEP1F_AUDIT":
        PROJECT /
        "step1F_lesion_level_rho_assignments" /
        "step1F_assignment_audit.csv",

    "STEP1G_AUDIT":
        PROJECT /
        "step1G_cross_environment_audit" /
        "step1G_cross_environment_audit_summary.json",

    "STEP1P_SEED_MANIFEST":
        PROJECT /
        "step1P_seed_manifest" /
        "seed_manifest.json",

    "STEP1I_PROTOCOL":
        PROJECT /
        "step1I_protocol_lock" /
        "LOCKED_EXPERIMENT_PROTOCOL.json",

    "STEP1A_GENERATOR_SPEC":
        PROJECT /
        "step1A_artifact_spec" /
        "step1A_artifact_generator_spec.json",

    "STEP1D_FEASIBILITY":
        PROJECT /
        "step1D_rho_feasibility" /
        "step1D_feasibility_summary.json",

    "STEP1E_FEASIBILITY":
        PROJECT /
        "step1E_lesion_assignment_feasibility" /
        "step1E_feasibility_summary.json",
}

print("\nARTIFACT EXISTENCE")
print("-" * 70)

for name, path in ARTIFACTS.items():

    print(
        f"{name:25s}",
        path.exists(),
        str(path)
    )

# ------------------------------------------------
# HELPER
# ------------------------------------------------

def load_json(path):

    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:

        return json.load(f)

# ------------------------------------------------
# PROTOCOL
# ------------------------------------------------

print("\nSTEP-1F LOCKED PROTOCOL")
print("-" * 70)

protocol = load_json(
    ARTIFACTS["STEP1F_PROTOCOL"]
)

print(
    json.dumps(
        protocol,
        indent=2,
        sort_keys=True
    )
)

# ------------------------------------------------
# SEED MANIFEST
# ------------------------------------------------

print("\nSTEP-1P SEED MANIFEST")
print("-" * 70)

seed_manifest = load_json(
    ARTIFACTS["STEP1P_SEED_MANIFEST"]
)

print(
    json.dumps(
        seed_manifest,
        indent=2,
        sort_keys=True
    )
)

# ------------------------------------------------
# STEP-1G
# ------------------------------------------------

print("\nSTEP-1G CROSS-ENVIRONMENT AUDIT")
print("-" * 70)

step1g = load_json(
    ARTIFACTS["STEP1G_AUDIT"]
)

print(
    json.dumps(
        step1g,
        indent=2,
        sort_keys=True
    )
)

# ------------------------------------------------
# STEP-1I
# ------------------------------------------------

print("\nSTEP-1I LOCKED EXPERIMENT PROTOCOL")
print("-" * 70)

step1i = load_json(
    ARTIFACTS["STEP1I_PROTOCOL"]
)

# Print only assignment-related sections/keys
# while preserving the actual locked terminology.

def recursively_find(
    obj,
    keywords,
    path=""
):

    results = []

    if isinstance(obj, dict):

        for key, value in obj.items():

            current = (
                f"{path}.{key}"
                if path
                else str(key)
            )

            if any(
                k.lower() in str(key).lower()
                for k in keywords
            ):

                results.append(
                    (
                        current,
                        value
                    )
                )

            results.extend(
                recursively_find(
                    value,
                    keywords,
                    current
                )
            )

    elif isinstance(obj, list):

        for i, value in enumerate(obj):

            results.extend(
                recursively_find(
                    value,
                    keywords,
                    f"{path}[{i}]"
                )
            )

    return results


matches = recursively_find(
    step1i,
    [
        "rho",
        "assignment",
        "seed",
        "lesion",
        "hair",
        "subtype",
        "replicate",
    ]
)

for path, value in matches:

    print(
        "\nPATH:",
        path
    )

    print(
        json.dumps(
            value,
            indent=2,
            sort_keys=True
        )
        if isinstance(
            value,
            (dict, list)
        )
        else repr(value)
    )

# ------------------------------------------------
# STEP-1D / STEP-1E
# ------------------------------------------------

print("\nSTEP-1D / STEP-1E FEASIBILITY")
print("-" * 70)

for name in [
    "STEP1D_FEASIBILITY",
    "STEP1E_FEASIBILITY",
]:

    data = load_json(
        ARTIFACTS[name]
    )

    print(
        "\n==========",
        name,
        "=========="
    )

    print(
        json.dumps(
            data,
            indent=2,
            sort_keys=True
        )
    )

# ------------------------------------------------
# STEP-1F AUDIT CSV
# ------------------------------------------------

print("\nSTEP-1F ASSIGNMENT AUDIT CSV")
print("-" * 70)

audit = pd.read_csv(
    ARTIFACTS["STEP1F_AUDIT"]
)

print(
    "Shape:",
    audit.shape
)

print(
    "Columns:"
)

for c in audit.columns:
    print(
        " ",
        c
    )

print(
    "\nFull audit table:"
)

print(
    audit.to_string(
        index=False
    )
)

# ------------------------------------------------
# EXISTING MANIFEST SEED RECONSTRUCTION
# ------------------------------------------------

print("\nEXISTING ASSIGNMENT SEED RECONSTRUCTION")
print("-" * 70)

STEP1F_DIR = (
    PROJECT /
    "step1F_lesion_level_rho_assignments"
)

for rho, rep in [
    (0.0, 1),
    (0.0, 2),
    (0.0, 3),
    (0.5, 1),
    (1.0, 1),
    (1.0, 2),
    (1.0, 3),
]:

    path = (
        STEP1F_DIR /
        f"rho_{rho:.1f}_rep_{rep}.csv"
    )

    if not path.exists():

        print(
            f"rho={rho:.1f}, rep={rep}: NOT PRESENT"
        )

        continue

    df = pd.read_csv(
        path,
        usecols=[
            "assignment_seed",
            "rho",
            "replicate",
            "hair_present",
            "lesion_id",
        ]
    )

    print(
        f"\nrho={rho:.1f}, rep={rep}"
    )

    print(
        "rows:",
        len(df)
    )

    print(
        "unique assignment_seed:",
        df["assignment_seed"].unique()
    )

    print(
        "manifest rho:",
        df["rho"].unique()
    )

    print(
        "manifest replicate:",
        df["replicate"].unique()
    )

    print(
        "hair_present:",
        df["hair_present"].value_counts(
            sort=False
        ).to_dict()
    )

# ------------------------------------------------
# SEARCH ALL JSON ARTIFACTS FOR EXACT SEED FORMULA
# ------------------------------------------------

print("\nEXACT SEED FORMULA SEARCH")
print("-" * 70)

seed_hits = []

for p in PROJECT.rglob("*.json"):

    try:

        text = p.read_text(
            encoding="utf-8",
            errors="ignore"
        )

    except Exception:
        continue

    if (
        "rho-assignment-v1" in text
        or "assignment_seed" in text
        or "PCG64" in text
    ):

        seed_hits.append(
            p
        )

for p in seed_hits:

    print(
        p
    )

# ------------------------------------------------
# FINAL STATUS
# ------------------------------------------------

print("\n" + "=" * 70)
print("CELL 6H COMPLETE")
print("=" * 70)

print(
    "No recovered hair generator imported:",
    True
)

print(
    "Existing Step-1F assignments modified:",
    False
)

print(
    "rho=.5 generated:",
    False
)

print("=" * 70)

In [ ]:
# ================================================================
# T2.1 — CELL 6I
# EXISTING MANIFEST FORENSIC RECONSTRUCTION
#
# Goal:
#   Determine exactly how existing Step-1F hair assignments
#   were selected from the locked training population.
#
# NO NEW ASSIGNMENTS ARE GENERATED.
# NO EXISTING FILES ARE MODIFIED.
# ================================================================

from pathlib import Path
import hashlib
import json
import math
import numpy as np
import pandas as pd

print("=" * 70)
print("T2.1 — CELL 6I")
print("STEP-1F EXISTING MANIFEST FORENSIC RECONSTRUCTION")
print("=" * 70)

PROJECT = Path(
    "/kaggle/input/datasets/tejasseshadriiiii/isic-final-project"
)

STEP1F = (
    PROJECT /
    "step1F_lesion_level_rho_assignments"
)

# ------------------------------------------------
# LOAD EXISTING MANIFESTS
# ------------------------------------------------

MANIFESTS = {}

for rho in [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]:

    for rep in range(1, 6):

        path = (
            STEP1F /
            f"rho_{rho:.1f}_rep_{rep}.csv"
        )

        assert path.exists(), path

        df = pd.read_csv(path)

        MANIFESTS[
            (rho, rep)
        ] = df

print(
    "\nLoaded manifests:",
    len(MANIFESTS)
)

# ------------------------------------------------
# VERIFY COMMON POPULATION
# ------------------------------------------------

reference = MANIFESTS[(0.0, 1)]

print("\nREFERENCE MANIFEST")
print("-" * 70)

print(
    "Rows:",
    len(reference)
)

print(
    "Columns:",
    list(reference.columns)
)

print(
    "Unique images:",
    reference["image"].nunique()
)

print(
    "Unique lesions:",
    reference["lesion_id"].nunique()
)

print(
    "Hair images:",
    int(reference["hair_present"].sum())
)

# ------------------------------------------------
# CHECK POPULATION IDENTITY ACROSS ALL 30
# ------------------------------------------------

print("\nPOPULATION IDENTITY AUDIT")
print("-" * 70)

reference_images = set(
    reference["image"]
)

reference_lesions = set(
    reference["lesion_id"]
)

for key, df in MANIFESTS.items():

    images = set(
        df["image"]
    )

    lesions = set(
        df["lesion_id"]
    )

    print(
        key,
        "images_identical=",
        images == reference_images,
        "lesions_identical=",
        lesions == reference_lesions
    )

# ------------------------------------------------
# BUILD IMAGE -> LESION MAPPING
# ------------------------------------------------

image_to_lesion = (
    reference[
        [
            "image",
            "lesion_id",
            "malignant",
            "class_name",
        ]
    ]
    .drop_duplicates()
    .copy()
)

print("\nIMAGE / LESION STRUCTURE")
print("-" * 70)

print(
    image_to_lesion
    .groupby("lesion_id")
    .size()
    .describe()
)

# ------------------------------------------------
# VERIFY HAIR IS LESION-INVARIANT
# ------------------------------------------------

print("\nLESION-LEVEL HAIR CONSISTENCY")
print("-" * 70)

for key, df in MANIFESTS.items():

    per_lesion = (
        df
        .groupby("lesion_id")["hair_present"]
        .nunique()
    )

    inconsistent = (
        per_lesion > 1
    ).sum()

    print(
        key,
        "inconsistent lesions:",
        int(inconsistent)
    )

# ------------------------------------------------
# CLASS COUNTS BY RHO
# ------------------------------------------------

print("\nCLASS TARGETS")
print("-" * 70)

for rho in [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]:

    df = MANIFESTS[(rho, 1)]

    stats = (
        df
        .groupby("malignant")["hair_present"]
        .agg(
            ["sum", "count"]
        )
    )

    print(
        "\nrho =",
        rho
    )

    print(
        stats
    )

    for malignant in [0, 1]:

        row = stats.loc[malignant]

        p = (
            row["sum"] /
            row["count"]
        )

        print(
            "  malignant=",
            malignant,
            "p_hair=",
            f"{p:.12f}"
        )

# ------------------------------------------------
# RECONSTRUCT THE THEORETICAL CLASS TARGET
# FROM THE OBSERVED RHO RELATION
# ------------------------------------------------

print("\nTHEORETICAL TARGET RECONSTRUCTION")
print("-" * 70)

N_M = (
    reference
    .loc[
        reference["malignant"] == 1
    ]
    .shape[0]
)

N_B = (
    reference
    .loc[
        reference["malignant"] == 0
    ]
    .shape[0]
)

N = N_M + N_B
H = N // 2

print(
    "N_M =",
    N_M
)

print(
    "N_B =",
    N_B
)

print(
    "N =",
    N
)

print(
    "H =",
    H
)

for rho in [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]:

    df = MANIFESTS[(rho, 1)]

    h_m = int(
        df.loc[
            df["malignant"] == 1,
            "hair_present"
        ].sum()
    )

    h_b = int(
        df.loc[
            df["malignant"] == 0,
            "hair_present"
        ].sum()
    )

    p_m = h_m / N_M
    p_b = h_b / N_B

    empirical_phi = (
        df["malignant"]
        .corr(
            df["hair_present"]
        )
    )

    print(
        f"\nrho={rho:.1f}"
    )

    print(
        "  H_M =",
        h_m,
        "H_B =",
        h_b,
        "H_total =",
        h_m + h_b
    )

    print(
        "  p_M =",
        f"{p_m:.12f}"
    )

    print(
        "  p_B =",
        f"{p_b:.12f}"
    )

    print(
        "  phi =",
        f"{empirical_phi:.12f}"
    )

# ------------------------------------------------
# REPLICATE DIFFERENCE AUDIT
# ------------------------------------------------

print("\nREPLICATE DIFFERENCE AUDIT")
print("-" * 70)

for rho in [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]:

    base = MANIFESTS[(rho, 1)]

    base_hair = (
        base
        .set_index("image")["hair_present"]
    )

    print(
        f"\nrho={rho:.1f}"
    )

    for rep in range(2, 6):

        other = MANIFESTS[
            (rho, rep)
        ]

        other_hair = (
            other
            .set_index("image")["hair_present"]
        )

        identical = (
            base_hair.equals(
                other_hair
            )
        )

        changed = int(
            (
                base_hair !=
                other_hair
            ).sum()
        )

        print(
            f"  rep1 vs rep{rep}: "
            f"identical={identical}, "
            f"changed_images={changed}"
        )

# ------------------------------------------------
# HASH-ORDER TEST
#
# Test whether the selected lesions in a class
# appear to be prefixes of a deterministic hash
# ordering based on plausible seed formulas.
# ------------------------------------------------

print("\nDETERMINISTIC HASH-ORDER FORENSICS")
print("-" * 70)

MASTER_SEED = 20260919

def seed_from_protocol(
    rho,
    replicate
):

    payload = (
        f"rho-assignment-v1|"
        f"{MASTER_SEED}|"
        f"{rho:.1f}|"
        f"{replicate}"
    )

    digest = hashlib.sha256(
        payload.encode()
    ).digest()

    return int.from_bytes(
        digest[:8],
        "big",
        signed=False
    )

def hash_key(
    seed,
    lesion_id,
    variant
):

    if variant == 0:

        payload = (
            f"{seed}|{lesion_id}"
        )

    elif variant == 1:

        payload = (
            f"rho-assignment-v1|"
            f"{seed}|{lesion_id}"
        )

    elif variant == 2:

        payload = (
            f"{MASTER_SEED}|"
            f"{seed}|"
            f"{lesion_id}"
        )

    elif variant == 3:

        payload = (
            f"{MASTER_SEED}|"
            f"rho-assignment-v1|"
            f"{lesion_id}|"
            f"{seed}"
        )

    else:

        raise ValueError

    return int.from_bytes(
        hashlib.sha256(
            payload.encode()
        ).digest()[:8],
        "big",
        signed=False
    )

# For each manifest, calculate whether hair-selected lesions
# resemble a deterministic prefix under several candidate hash
# constructions.
#
# This is ONLY a forensic score.
# It does NOT create or alter assignments.

for rho in [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]:

    for rep in [1, 2, 3]:

        df = MANIFESTS[
            (rho, rep)
        ]

        lesion_table = (
            df[
                [
                    "lesion_id",
                    "malignant",
                    "hair_present",
                ]
            ]
            .drop_duplicates()
        )

        seed = seed_from_protocol(
            rho,
            rep
        )

        print(
            f"\nrho={rho:.1f}, rep={rep}, "
            f"seed={seed}"
        )

        for variant in range(4):

            class_scores = []

            for malignant in [0, 1]:

                sub = lesion_table[
                    lesion_table["malignant"]
                    == malignant
                ].copy()

                sub["key"] = [
                    hash_key(
                        seed,
                        lesion_id,
                        variant
                    )
                    for lesion_id
                    in sub["lesion_id"]
                ]

                sub = sub.sort_values(
                    [
                        "key",
                        "lesion_id",
                    ]
                )

                n_hair = int(
                    sub["hair_present"].sum()
                )

                predicted = set(
                    sub.head(n_hair)[
                        "lesion_id"
                    ]
                )

                observed = set(
                    sub.loc[
                        sub["hair_present"] == 1,
                        "lesion_id"
                    ]
                )

                overlap = (
                    len(
                        predicted &
                        observed
                    )
                    /
                    max(
                        1,
                        len(observed)
                    )
                )

                class_scores.append(
                    overlap
                )

            print(
                f"  variant={variant}: "
                f"benign_overlap={class_scores[0]:.6f}, "
                f"malignant_overlap={class_scores[1]:.6f}"
            )

# ------------------------------------------------
# CHECK WHETHER SEED COLUMN EXACTLY MATCHES FORMULA
# ------------------------------------------------

print("\nSEED FORMULA BIT-EXACT AUDIT")
print("-" * 70)

all_seed_pass = True

for rho in [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]:

    for rep in range(1, 6):

        df = MANIFESTS[
            (rho, rep)
        ]

        expected = seed_from_protocol(
            rho,
            rep
        )

        actual = (
            int(
                df[
                    "assignment_seed"
                ].iloc[0]
            )
        )

        passed = (
            actual ==
            expected
        )

        all_seed_pass &= passed

        print(
            f"rho={rho:.1f}, rep={rep}: "
            f"actual={actual}, "
            f"expected={expected}, "
            f"PASS={passed}"
        )

# ------------------------------------------------
# FINAL STATUS
# ------------------------------------------------

print("\n" + "=" * 70)
print("CELL 6I COMPLETE")
print("=" * 70)

print(
    "Seed formula matches all 30 manifests:",
    all_seed_pass
)

print(
    "Existing assignments modified:",
    False
)

print(
    "rho=.5 generated:",
    False
)

print("=" * 70)

In [ ]:
# ================================================================
# T2.1 — CELL 6J
# EXACT TRAINING SCRIPT ASSIGNMENT-PATH FORENSICS
# ================================================================

from pathlib import Path
import hashlib
import re

print("=" * 70)
print("T2.1 — CELL 6J")
print("TRAIN_ONE_MODEL.PY ASSIGNMENT-PATH FORENSICS")
print("=" * 70)

PROJECT = Path(
    "/kaggle/input/datasets/tejasseshadriiiii/isic-final-project"
)

TRAIN = (
    PROJECT
    / "step2X_training_code_freeze"
    / "train_one_model.py"
)

assert TRAIN.exists(), (
    f"Missing frozen training script:\n{TRAIN}"
)

print("\nTRAINING SCRIPT")
print("-" * 70)

print(
    "Path:",
    TRAIN
)

print(
    "Exists:",
    TRAIN.exists()
)

# ------------------------------------------------
# HASH
# ------------------------------------------------

h = hashlib.sha256()

with open(TRAIN, "rb") as f:

    for chunk in iter(
        lambda: f.read(1 << 22),
        b""
    ):

        h.update(chunk)

train_sha = h.hexdigest()

print(
    "SHA256:",
    train_sha
)

# ------------------------------------------------
# READ SOURCE
# ------------------------------------------------

source = TRAIN.read_text(
    encoding="utf-8"
)

lines = source.splitlines()

print(
    "Lines:",
    len(lines)
)

# ------------------------------------------------
# SEARCH ASSIGNMENT-RELATED TERMS
# ------------------------------------------------

print("\nASSIGNMENT REFERENCES")
print("-" * 70)

terms = [
    "assignment",
    "assignment_seed",
    "hair_present",
    "rho_",
    "rho",
    "replicate",
    "lesion_id",
    "step1F",
    "csv",
    "read_csv",
    "hair",
]

for term in terms:

    matches = [
        i + 1
        for i, line in enumerate(lines)
        if term.lower() in line.lower()
    ]

    print(
        f"{term:20s}:",
        matches[:100],
        "..." if len(matches) > 100 else ""
    )

# ------------------------------------------------
# PRINT ALL RELEVANT SOURCE WINDOWS
# ------------------------------------------------

print("\nRELEVANT SOURCE")
print("-" * 70)

interesting = []

for i, line in enumerate(lines):

    low = line.lower()

    if any(
        term.lower() in low
        for term in [
            "assignment",
            "assignment_seed",
            "hair_present",
            "step1f",
            "rho_",
            "read_csv",
            "replicate",
        ]
    ):

        interesting.append(i)

windows = []

for i in interesting:

    start = max(
        0,
        i - 5
    )

    end = min(
        len(lines),
        i + 6
    )

    if (
        not windows
        or start > windows[-1][1]
    ):

        windows.append(
            [start, end]
        )

    else:

        windows[-1][1] = max(
            windows[-1][1],
            end
        )

for start, end in windows:

    print(
        "\n--- lines "
        f"{start+1}:{end} ---"
    )

    print(
        "\n".join(
            f"{i+1:5d}: {lines[i]}"
            for i in range(start, end)
        )
    )

# ------------------------------------------------
# SEARCH FOR PROJECT FILE PATHS
# ------------------------------------------------

print("\nPROJECT-PATH REFERENCES")
print("-" * 70)

path_lines = []

for i, line in enumerate(lines):

    low = line.lower()

    if any(
        token in low
        for token in [
            "project /",
            "project/",
            "step1f",
            "assign",
            "csv",
            "cache",
        ]
    ):

        path_lines.append(i)

for i in path_lines:

    start = max(
        0,
        i - 2
    )

    end = min(
        len(lines),
        i + 3
    )

    print(
        f"\n--- around line {i+1} ---"
    )

    print(
        "\n".join(
            f"{j+1:5d}: {lines[j]}"
            for j in range(start, end)
        )
    )

# ------------------------------------------------
# CHECK WHETHER TRAINING SCRIPT ITSELF GENERATES
# ASSIGNMENTS OR ONLY LOADS THEM
# ------------------------------------------------

print("\nGENERATION VS LOADING INDICATORS")
print("-" * 70)

generation_terms = [
    "np.random",
    "numpy.random",
    "default_rng",
    "PCG64",
    "sha256",
    "hashlib",
    "assignment_seed",
    "hair_present =",
    "to_csv",
    "save_csv",
]

loading_terms = [
    "pd.read_csv",
    "read_csv(",
    "assignment_path",
    "assignment_file",
    "assignment_csv",
]

for term in generation_terms:

    count = len(
        re.findall(
            re.escape(term),
            source,
            flags=re.IGNORECASE
        )
    )

    print(
        f"GEN {term:25s}: {count}"
    )

for term in loading_terms:

    count = len(
        re.findall(
            re.escape(term),
            source,
            flags=re.IGNORECASE
        )
    )

    print(
        f"LOAD {term:24s}: {count}"
    )

# ------------------------------------------------
# CLI / ARGUMENTS
# ------------------------------------------------

print("\nCLI ARGUMENTS")
print("-" * 70)

for i, line in enumerate(lines):

    low = line.lower()

    if (
        "argparse" in low
        or "add_argument" in low
        or "args.rho" in low
        or "args.rep" in low
        or "args.replicate" in low
    ):

        start = max(
            0,
            i - 4
        )

        end = min(
            len(lines),
            i + 6
        )

        print(
            f"\n--- around line {i+1} ---"
        )

        print(
            "\n".join(
                f"{j+1:5d}: {lines[j]}"
                for j in range(start, end)
            )
        )

# ------------------------------------------------
# FINAL
# ------------------------------------------------

print("\n" + "=" * 70)
print("CELL 6J COMPLETE")
print("=" * 70)

print(
    "Training script inspected:",
    True
)

print(
    "Existing assignments modified:",
    False
)

print(
    "rho=.5 generated:",
    False
)

print("=" * 70)

In [ ]:
# ================================================================
# T2.1 — CELL 6K
# COMPLETE STEP1F ARTIFACT FORENSICS
# ================================================================

from pathlib import Path
import hashlib
import json
import os

PROJECT = Path(
    "/kaggle/input/datasets/tejasseshadriiiii/isic-final-project"
)

STEP1F = PROJECT / "step1F_lesion_level_rho_assignments"

print("=" * 70)
print("T2.1 — CELL 6K")
print("COMPLETE STEP1F ARTIFACT FORENSICS")
print("=" * 70)

assert STEP1F.exists(), f"Missing: {STEP1F}"

print("\nSTEP1F PATH")
print("-" * 70)
print(STEP1F)

# ------------------------------------------------
# FULL RECURSIVE INVENTORY
# ------------------------------------------------

files = sorted(
    [p for p in STEP1F.rglob("*") if p.is_file()]
)

print("\nTOTAL FILES:", len(files))

for p in files:

    size = p.stat().st_size

    h = hashlib.sha256()

    with open(p, "rb") as f:
        for chunk in iter(
            lambda: f.read(1 << 20),
            b""
        ):
            h.update(chunk)

    print(
        f"{p.relative_to(STEP1F)}"
        f" | {size:,} bytes"
        f" | SHA256 {h.hexdigest()}"
    )

# ------------------------------------------------
# DIRECTORY TREE
# ------------------------------------------------

print("\nDIRECTORY TREE")
print("-" * 70)

for root, dirs, fs in os.walk(STEP1F):

    rootp = Path(root)

    depth = len(rootp.relative_to(STEP1F).parts)

    indent = "    " * depth

    print(
        indent + rootp.name + "/"
    )

    for f in sorted(fs):

        print(
            indent + "    " + f
        )

# ------------------------------------------------
# JSON CONTENT
# ------------------------------------------------

print("\nJSON ARTIFACTS")
print("-" * 70)

for p in files:

    if p.suffix.lower() != ".json":
        continue

    print(
        f"\n===== {p.relative_to(STEP1F)} ====="
    )

    try:

        obj = json.loads(
            p.read_text(
                encoding="utf-8"
            )
        )

        print(
            json.dumps(
                obj,
                indent=2,
                sort_keys=True
            )[:30000]
        )

    except Exception as e:

        print(
            "JSON READ ERROR:",
            repr(e)
        )

# ------------------------------------------------
# PYTHON ARTIFACTS
# ------------------------------------------------

print("\nPYTHON ARTIFACTS")
print("-" * 70)

py_files = [
    p for p in files
    if p.suffix.lower() in {
        ".py",
        ".pyw"
    }
]

if not py_files:

    print(
        "NO .py FILES INSIDE STEP1F"
    )

else:

    for p in py_files:

        print(
            f"\n===== {p.relative_to(STEP1F)} ====="
        )

        src = p.read_text(
            encoding="utf-8",
            errors="replace"
        )

        print(
            src[:50000]
        )

# ------------------------------------------------
# NOTEBOOKS / TEXT / MARKDOWN
# ------------------------------------------------

print("\nNON-DATA TEXT ARTIFACTS")
print("-" * 70)

text_exts = {
    ".ipynb",
    ".md",
    ".txt",
    ".yaml",
    ".yml",
    ".toml",
    ".ini",
    ".cfg",
}

for p in files:

    if p.suffix.lower() not in text_exts:
        continue

    print(
        f"\n===== {p.relative_to(STEP1F)} ====="
    )

    try:

        txt = p.read_text(
            encoding="utf-8",
            errors="replace"
        )

        print(
            txt[:30000]
        )

    except Exception as e:

        print(
            "READ ERROR:",
            repr(e)
        )

# ------------------------------------------------
# SEARCH ALL ARTIFACT NAMES FOR GENERATOR CLUES
# ------------------------------------------------

print("\nGENERATOR-RELATED FILENAMES")
print("-" * 70)

keywords = [
    "generat",
    "solver",
    "assign",
    "seed",
    "lock",
    "protocol",
    "audit",
    "manifest",
    "dp",
]

for p in files:

    name = str(
        p.relative_to(STEP1F)
    ).lower()

    hits = [
        k for k in keywords
        if k in name
    ]

    if hits:

        print(
            p.relative_to(STEP1F),
            "=>",
            hits
        )

# ------------------------------------------------
# CSV INVENTORY
# ------------------------------------------------

print("\nCSV INVENTORY")
print("-" * 70)

for p in files:

    if p.suffix.lower() != ".csv":
        continue

    print(
        p.name,
        "|",
        f"{p.stat().st_size:,}",
        "bytes"
    )

print("\n" + "=" * 70)
print("CELL 6K COMPLETE")
print("=" * 70)

print(
    "STEP1F ARTIFACTS:",
    len(files)
)

print(
    "PYTHON FILES:",
    len(py_files)
)

print("=" * 70)

In [ ]:
# ================================================================
# T2.1 — CELL 6L
# RECOVER EXACT SOLVER IMPLEMENTATION
# ================================================================

from pathlib import Path
import hashlib
import inspect
import ast
import re

PROJECT = Path(
    "/kaggle/input/datasets/tejasseshadriiiii/isic-final-project"
)

SOLVER_DIR = (
    PROJECT
    / "step2Q_solver_implementation_v2"
)

print("=" * 70)
print("T2.1 — CELL 6L")
print("EXACT SOLVER IMPLEMENTATION RECOVERY")
print("=" * 70)

assert SOLVER_DIR.exists(), (
    f"Missing solver directory:\n{SOLVER_DIR}"
)

files = sorted(
    p for p in SOLVER_DIR.rglob("*")
    if p.is_file()
)

print("\nSOLVER DIRECTORY")
print("-" * 70)

for p in files:

    h = hashlib.sha256()

    with open(p, "rb") as f:
        for chunk in iter(
            lambda: f.read(1 << 20),
            b""
        ):
            h.update(chunk)

    print(
        p.relative_to(SOLVER_DIR),
        "|",
        f"{p.stat().st_size:,}",
        "bytes | SHA256",
        h.hexdigest()
    )

# ------------------------------------------------
# TARGET SOLVER
# ------------------------------------------------

solver_files = [
    p for p in files
    if p.name == "deterministic_evaluation_solver.py"
]

assert len(solver_files) == 1, (
    f"Expected exactly one deterministic solver, found {solver_files}"
)

SOLVER = solver_files[0]

EXPECTED_SHA = (
    "37451d50afa7dcd8052a00d53cccb853d3ce658adbdd65bc1ce66c14b67fc5d3"
)

h = hashlib.sha256()

with open(SOLVER, "rb") as f:
    for chunk in iter(
        lambda: f.read(1 << 20),
        b""
    ):
        h.update(chunk)

actual_sha = h.hexdigest()

print("\nSOLVER HASH")
print("-" * 70)
print("Expected:", EXPECTED_SHA)
print("Actual:  ", actual_sha)
print("MATCH:   ", actual_sha == EXPECTED_SHA)

assert actual_sha == EXPECTED_SHA, (
    "FROZEN SOLVER HASH MISMATCH"
)

# ------------------------------------------------
# SOURCE
# ------------------------------------------------

source = SOLVER.read_text(
    encoding="utf-8"
)

lines = source.splitlines()

print("\nSOURCE")
print("-" * 70)
print("Lines:", len(lines))

# ------------------------------------------------
# AST FUNCTION INVENTORY
# ------------------------------------------------

print("\nFUNCTION INVENTORY")
print("-" * 70)

tree = ast.parse(source)

for node in tree.body:

    if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):

        args = []

        for a in node.args.args:
            args.append(a.arg)

        print(
            f"{node.name}({', '.join(args)})"
        )

# ------------------------------------------------
# EXACT TARGET FUNCTION
# ------------------------------------------------

print("\nTARGET FUNCTION SEARCH")
print("-" * 70)

for i, line in enumerate(lines):

    if (
        "solve_evaluation_assignment" in line
        or "def solve" in line
        or "def " in line
    ):

        print(
            f"{i+1:5d}: {line}"
        )

# ------------------------------------------------
# IMPORTANT TERMS
# ------------------------------------------------

terms = [
    "target_hair_count",
    "condition",
    "subtype",
    "lesion_id",
    "n_images",
    "malignant",
    "Fraction",
    "reachable",
    "suffix",
    "lexicographic",
    "hash",
    "SHA256",
    "rho",
    "replicate",
    "seed",
    "random",
    "PCG64",
]

print("\nTERM LOCATIONS")
print("-" * 70)

for term in terms:

    matches = [
        i + 1
        for i, line in enumerate(lines)
        if term.lower() in line.lower()
    ]

    print(
        f"{term:20s}: {matches[:80]}"
    )

# ------------------------------------------------
# PRINT COMPLETE SOURCE
# ------------------------------------------------

print("\nCOMPLETE SOLVER SOURCE")
print("-" * 70)

print(source)

print("=" * 70)
print("CELL 6L COMPLETE")
print("=" * 70)

In [ ]:
# ================================================================
# T2.1 — CELL 6M
# EXISTING STEP1F ASSIGNMENT REPRODUCTION TEST
# ================================================================

from pathlib import Path
import sys
import pandas as pd
import numpy as np
import hashlib
import importlib.util
import json

PROJECT = Path(
    "/kaggle/input/datasets/tejasseshadriiiii/isic-final-project"
)

SOLVER = (
    PROJECT
    / "step2Q_solver_implementation_v2"
    / "deterministic_evaluation_solver.py"
)

ASSIGN_DIR = (
    PROJECT
    / "step1F_lesion_level_rho_assignments"
)

print("=" * 70)
print("T2.1 — CELL 6M")
print("EXISTING STEP1F ASSIGNMENT REPRODUCTION TEST")
print("=" * 70)

# ------------------------------------------------
# LOAD FROZEN SOLVER
# ------------------------------------------------

spec = importlib.util.spec_from_file_location(
    "frozen_solver",
    SOLVER
)

mod = importlib.util.module_from_spec(spec)

spec.loader.exec_module(mod)

assert hasattr(
    mod,
    "solve_evaluation_assignment"
)

solver = mod.solve_evaluation_assignment

print("\nSOLVER LOADED")
print("-" * 70)

# ------------------------------------------------
# LOAD ONE EXISTING ASSIGNMENT
# ------------------------------------------------

rho = 0.0
rep = 1

assignment_path = (
    ASSIGN_DIR
    / f"rho_{rho:.1f}_rep_{rep}.csv"
)

a = pd.read_csv(
    assignment_path
)

print(
    "Assignment:",
    assignment_path.name
)

print(
    "Rows:",
    len(a)
)

print(
    "Columns:",
    list(a.columns)
)

# ------------------------------------------------
# BUILD LESION POPULATION
# ------------------------------------------------
#
# IMPORTANT:
# The solver expects lesion-level n_images.
# Therefore derive it from the frozen assignment
# itself, without using hair_present in the solver.
# ------------------------------------------------

lesions_df = (
    a.groupby(
        ["lesion_id", "malignant"],
        sort=True,
        as_index=False
    )
    .agg(
        n_images=("image", "size")
    )
)

# subtype is required by the frozen solver.
# Use the assignment's subtype if present.
if "subtype" in a.columns:

    subtype_map = (
        a[
            [
                "lesion_id",
                "subtype"
            ]
        ]
        .drop_duplicates(
            "lesion_id"
        )
    )

    lesions_df = lesions_df.merge(
        subtype_map,
        on="lesion_id",
        how="left",
        validate="one_to_one"
    )

else:

    raise RuntimeError(
        "Frozen assignment CSV has no subtype column. "
        "Cannot construct the solver population without "
        "recovering subtype from the locked population."
    )

# ------------------------------------------------
# VALIDATE LESION POPULATION
# ------------------------------------------------

assert lesions_df["lesion_id"].is_unique

assert (
    lesions_df["n_images"] > 0
).all()

assert set(
    lesions_df["malignant"].unique()
).issubset({0, 1})

assert (
    lesions_df["subtype"].notna()
).all()

print("\nLESION POPULATION")
print("-" * 70)

print(
    "Unique lesions:",
    len(lesions_df)
)

print(
    "Images:",
    lesions_df["n_images"].sum()
)

print(
    "Malignant images:",
    lesions_df.loc[
        lesions_df["malignant"] == 1,
        "n_images"
    ].sum()
)

print(
    "Benign images:",
    lesions_df.loc[
        lesions_df["malignant"] == 0,
        "n_images"
    ].sum()
)

# ------------------------------------------------
# CONVERT TO SOLVER INPUT
# ------------------------------------------------

lesions = (
    lesions_df[
        [
            "lesion_id",
            "malignant",
            "subtype",
            "n_images"
        ]
    ]
    .to_dict("records")
)

# ------------------------------------------------
# SEEDS TO TEST
# ------------------------------------------------

# Existing Step1F protocol says:
# SHA256(rho-assignment-v1|master_seed|rho|replicate)
#
# We deliberately test multiple candidate seed
# serializations, because the exact locked textual
# serialization is not yet established.
#
# The solver itself must NOT be modified.

MASTER = 20260919

seed_candidates = {}

formats = {

    "pipe_float1":
        f"rho-assignment-v1|{MASTER}|{rho:.1f}|{rep}",

    "pipe_float":
        f"rho-assignment-v1|{MASTER}|{rho}|{rep}",

    "pipe_int_rho":
        f"rho-assignment-v1|{MASTER}|{int(rho)}|{rep}",

}

for name, payload in formats.items():

    seed_candidates[name] = int.from_bytes(
        hashlib.sha256(
            payload.encode("utf-8")
        ).digest()[:8],
        byteorder="big"
    )

print("\nCANDIDATE ASSIGNMENT SEEDS")
print("-" * 70)

for k, v in seed_candidates.items():

    print(
        k,
        ":",
        v
    )

# ------------------------------------------------
# TARGET
# ------------------------------------------------

target_hair = int(
    a["hair_present"].sum()
)

print(
    "\nTARGET HAIR IMAGES:",
    target_hair
)

# ------------------------------------------------
# TEST CONDITIONS
# ------------------------------------------------

conditions = [
    "neutral",
    "matched",
    "reversed",
]

# ------------------------------------------------
# REPRODUCTION TEST
# ------------------------------------------------

results = []

for seed_name, seed in seed_candidates.items():

    for condition in conditions:

        print(
            "\n"
            + "-" * 70
        )

        print(
            "TEST:",
            seed_name,
            "| condition =",
            condition
        )

        try:

            result = solver(
                lesions=lesions,
                condition=condition,
                seed=seed,
                target_hair_count=target_hair
            )

            selected = set(
                result["selected_lesions"]
            )

            # Reconstruct image-level assignment
            reproduced = (
                a["lesion_id"]
                .isin(selected)
                .astype(np.int8)
                .to_numpy()
            )

            observed = (
                a["hair_present"]
                .astype(np.int8)
                .to_numpy()
            )

            exact = np.array_equal(
                reproduced,
                observed
            )

            changed = int(
                np.sum(
                    reproduced != observed
                )
            )

            results.append({
                "seed_name": seed_name,
                "seed": seed,
                "condition": condition,
                "exact_match": exact,
                "changed_images": changed,
                "J1": str(result["J1"]),
                "J2": str(result["J2"]),
                "H_M": int(result["H_M"]),
                "H_B": int(result["H_B"]),
            })

            print(
                "H_M:",
                result["H_M"]
            )

            print(
                "H_B:",
                result["H_B"]
            )

            print(
                "J1:",
                result["J1"]
            )

            print(
                "J2:",
                result["J2"]
            )

            print(
                "Changed images:",
                changed
            )

            print(
                "EXACT MATCH:",
                exact
            )

        except Exception as e:

            print(
                "FAILED:",
                repr(e)
            )

            results.append({
                "seed_name": seed_name,
                "seed": seed,
                "condition": condition,
                "exact_match": False,
                "changed_images": None,
                "J1": None,
                "J2": None,
                "H_M": None,
                "H_B": None,
                "error": repr(e),
            })

# ------------------------------------------------
# SUMMARY
# ------------------------------------------------

summary = pd.DataFrame(
    results
)

print("\n" + "=" * 70)
print("REPRODUCTION SUMMARY")
print("=" * 70)

print(
    summary.to_string(
        index=False
    )
)

exact_hits = summary[
    summary["exact_match"] == True
]

print("\nEXACT REPRODUCTION HITS:", len(exact_hits))

if len(exact_hits):

    print(
        exact_hits.to_string(
            index=False
        )
    )

else:

    print(
        "NO EXACT REPRODUCTION."
    )

print("=" * 70)
print("CELL 6M COMPLETE")
print("=" * 70)

In [ ]:
# ================================================================
# T2.1 — CELL 6N
# EXACT ORIGINAL STEP1F SEED REPRODUCTION AUDIT
# ================================================================

from pathlib import Path
import pandas as pd
import numpy as np
import hashlib

PROJECT = Path(
    "/kaggle/input/datasets/tejasseshadriiiii/isic-final-project"
)

ASSIGN_DIR = (
    PROJECT
    / "step1F_lesion_level_rho_assignments"
)

MASTER_SEED = 20260919

RHO_VALUES = [
    0.0,
    0.2,
    0.4,
    0.6,
    0.8,
    1.0
]

N_REPLICATES = 5


# ------------------------------------------------
# THIS IS THE RECOVERED ORIGINAL FUNCTION
# FROM THE ORIGINAL STEP1F GENERATOR.
# ------------------------------------------------

def original_assignment_seed(
    master_seed,
    rho,
    replicate
):

    key = (
        f"rho-assignment-v1|"
        f"{master_seed}|"
        f"{rho:.1f}|"
        f"replicate-{replicate}"
    ).encode("utf-8")

    return int.from_bytes(
        hashlib.sha256(
            key
        ).digest()[:8],
        byteorder="big",
        signed=False
    )


print("=" * 70)
print("T2.1 — CELL 6N")
print("EXACT ORIGINAL STEP1F SEED REPRODUCTION AUDIT")
print("=" * 70)


results = []


for rho in RHO_VALUES:

    for replicate in range(
        1,
        N_REPLICATES + 1
    ):

        path = (
            ASSIGN_DIR
            / f"rho_{rho:.1f}_rep_{replicate}.csv"
        )

        assert path.exists(), path

        df = pd.read_csv(
            path,
            usecols=[
                "rho",
                "replicate",
                "assignment_seed"
            ]
        )

        assert len(df) == 18514

        # Existing stored seed must be unique within
        # the entire manifest.
        stored = (
            df["assignment_seed"]
            .astype("uint64")
            .unique()
        )

        assert len(stored) == 1

        stored_seed = int(
            stored[0]
        )

        expected_seed = (
            original_assignment_seed(
                MASTER_SEED,
                rho,
                replicate
            )
        )

        exact = (
            stored_seed
            == expected_seed
        )

        results.append({
            "rho": rho,
            "replicate": replicate,
            "stored_seed": stored_seed,
            "expected_seed": expected_seed,
            "exact": exact
        })


audit = pd.DataFrame(
    results
)

print(
    audit.to_string(
        index=False
    )
)

print("\n" + "-" * 70)

print(
    "Total manifests:",
    len(audit)
)

print(
    "Exact seed matches:",
    int(audit["exact"].sum())
)

print(
    "Seed mismatches:",
    int((~audit["exact"]).sum())
)

assert audit["exact"].all()

print(
    "\nPASS — ALL 30 LOCKED STEP1F ASSIGNMENT SEEDS "
    "ARE REPRODUCED EXACTLY."
)

print("=" * 70)
print("CELL 6N COMPLETE")
print("=" * 70)

In [ ]:
# ================================================================
# T2.1 — CELL 6O
# EXACT REPRODUCTION OF ONE LOCKED STEP1F ASSIGNMENT
#
# Target:
#   rho = 0.2
#   replicate = 1
#
# The recovered original Step1F generator is reconstructed here
# from the original generator source.
#
# SUCCESS CRITERION:
#   Candidate CSV == locked CSV byte-for-byte
#
# FAILURE:
#   STOP. Do NOT generate rho=0.5.
# ================================================================

from pathlib import Path
from collections import defaultdict
import pandas as pd
import numpy as np
import hashlib
import shutil


print("=" * 70)
print("T2.1 — CELL 6O")
print("EXACT STEP1F ASSIGNMENT REPRODUCTION")
print("=" * 70)


# ================================================================
# PATHS
# ================================================================

PROJECT_ROOT = Path(
    "/kaggle/input/datasets/tejasseshadriiiii/isic-final-project"
)

LOCKED_MANIFEST = (
    PROJECT_ROOT
    / "step0I_locked_clean_population"
    / "LOCKED_CLEAN_development_population.csv"
)

TARGET_PATH = (
    PROJECT_ROOT
    / "step1D_rho_feasibility_corrected"
    / "step1D_corrected_rho_targets.csv"
)

ASSIGN_DIR = (
    PROJECT_ROOT
    / "step1F_lesion_level_rho_assignments"
)

REPRO_DIR = (
    Path("/kaggle/working")
    / "T2_1_STEP1F_REPRODUCTION_AUDIT"
)

REPRO_DIR.mkdir(
    parents=True,
    exist_ok=True
)


TARGET_RHO = 0.2
TARGET_REPLICATE = 1

MASTER_SEED = 20260919
SEVERITY = "medium"


# ================================================================
# VERIFY REQUIRED LOCKED ARTIFACTS
# ================================================================

assert LOCKED_MANIFEST.exists(), LOCKED_MANIFEST
assert TARGET_PATH.exists(), TARGET_PATH

LOCKED_ASSIGNMENT = (
    ASSIGN_DIR
    / f"rho_{TARGET_RHO:.1f}_rep_{TARGET_REPLICATE}.csv"
)

assert LOCKED_ASSIGNMENT.exists(), LOCKED_ASSIGNMENT

print("\nLocked population:")
print(LOCKED_MANIFEST)

print("\nTarget table:")
print(TARGET_PATH)

print("\nLocked assignment:")
print(LOCKED_ASSIGNMENT)


# ================================================================
# LOAD EXACT LOCKED INPUTS
# ================================================================

manifest = pd.read_csv(
    LOCKED_MANIFEST
)

targets = pd.read_csv(
    TARGET_PATH
)

train = (
    manifest[
        manifest["split"] == "train"
    ]
    .copy()
    .reset_index(drop=True)
)

assert len(train) == 18514
assert train["lesion_id"].notna().all()


# ================================================================
# ORIGINAL STEP1F SEED FUNCTION
# ================================================================

def assignment_seed(
    master_seed,
    rho,
    replicate
):

    key = (
        f"rho-assignment-v1|"
        f"{master_seed}|"
        f"{rho:.1f}|"
        f"replicate-{replicate}"
    ).encode("utf-8")

    return int.from_bytes(
        hashlib.sha256(key).digest()[:8],
        byteorder="big",
        signed=False
    )


# ================================================================
# ORIGINAL STEP1F exact_random_subset()
#
# Recovered verbatim in substance from the original generator.
# ================================================================

def exact_random_subset(
    lesion_table,
    target_images,
    rng
):

    lesion_table = lesion_table.copy()

    # ------------------------------------------------------------
    # Group lesions by number of images.
    # ------------------------------------------------------------

    groups = {}

    for n_images, group in lesion_table.groupby(
        "n_images"
    ):

        ids = (
            group["lesion_id"]
            .astype(str)
            .tolist()
        )

        rng.shuffle(ids)

        groups[int(n_images)] = ids

    sizes = sorted(
        groups.keys()
    )

    # ------------------------------------------------------------
    # Bounded subset-sum over lesion size counts.
    # ------------------------------------------------------------

    predecessor = {
        0: None
    }

    for size in sizes:

        n_available = len(
            groups[size]
        )

        old_states = list(
            predecessor.keys()
        )

        new_states = {}

        for old_sum in old_states:

            max_k = min(
                n_available,
                (target_images - old_sum)
                // size
            )

            for k in range(
                max_k + 1
            ):

                new_sum = (
                    old_sum
                    + k * size
                )

                if new_sum in predecessor:
                    continue

                if new_sum in new_states:
                    continue

                new_states[new_sum] = (
                    old_sum,
                    size,
                    k
                )

        predecessor.update(
            new_states
        )

        if target_images in predecessor:
            break

    assert target_images in predecessor, (
        f"Exact target {target_images} "
        f"is not achievable."
    )

    # ------------------------------------------------------------
    # Reconstruct selected lesion counts.
    # ------------------------------------------------------------

    selected_counts = defaultdict(int)

    current = target_images

    while current != 0:

        prev_sum, size, k = (
            predecessor[current]
        )

        selected_counts[size] += k

        current = prev_sum

    # ------------------------------------------------------------
    # Choose actual lesions.
    # ------------------------------------------------------------

    selected = []

    for size, k in selected_counts.items():

        assert k <= len(
            groups[size]
        )

        selected.extend(
            groups[size][:k]
        )

    assert (
        len(selected) > 0
        or target_images == 0
    )

    return set(selected)


# ================================================================
# PREPARE EXACT LESION TABLE
# ================================================================

lesions = (
    train.groupby(
        [
            "malignant",
            "class_name",
            "lesion_id"
        ],
        as_index=False
    )
    .agg(
        n_images=("image", "size")
    )
)

assert (
    lesions
    .groupby("lesion_id")["class_name"]
    .nunique()
    .max()
    == 1
)

print("\nTraining images:", len(train))
print("Training lesions:", lesions["lesion_id"].nunique())


# ================================================================
# SELECT EXACT TARGET ROW
# ================================================================

target_row = targets[
    np.isclose(
        targets["rho"],
        TARGET_RHO
    )
].iloc[0]

binary_targets = {
    1: int(
        target_row[
            "target_malignant_hair_images"
        ]
    ),
    0: int(
        target_row[
            "target_benign_hair_images"
        ]
    )
}

print("\nBinary targets:")
print(
    binary_targets
)


# ================================================================
# ORIGINAL SUBTYPE TARGET CONSTRUCTION
# ================================================================

subtype_targets = {}

for malignant in [0, 1]:

    class_target = (
        binary_targets[malignant]
    )

    class_lesions = lesions[
        lesions["malignant"] == malignant
    ]

    raw = {}

    for class_name, group in (
        class_lesions
        .groupby("class_name")
    ):

        n_images = int(
            group["n_images"].sum()
        )

        if malignant == 1:

            p_target = float(
                target_row[
                    "target_p_h_given_m"
                ]
            )

        else:

            p_target = float(
                target_row[
                    "target_p_h_given_b"
                ]
            )

        raw[class_name] = (
            n_images * p_target
        )

    rounded = {
        k: int(round(v))
        for k, v in raw.items()
    }

    difference = (
        class_target
        - sum(rounded.values())
    )

    # ------------------------------------------------------------
    # Original fractional-remainder adjustment.
    # ------------------------------------------------------------

    if difference != 0:

        fractional = {
            k: raw[k] - np.floor(raw[k])
            for k in raw
        }

        if difference > 0:

            order = sorted(
                fractional,
                key=fractional.get,
                reverse=True
            )

            for k in order[:difference]:
                rounded[k] += 1

        else:

            order = sorted(
                fractional,
                key=fractional.get
            )

            for k in order[:abs(difference)]:
                rounded[k] -= 1

    assert (
        sum(rounded.values())
        == class_target
    )

    subtype_targets[malignant] = (
        rounded
    )


print("\nSubtype targets:")
for malignant in [0, 1]:
    print(
        f"malignant={malignant}:",
        subtype_targets[malignant]
    )


# ================================================================
# GENERATE EXACT ORIGINAL ASSIGNMENT
# ================================================================

seed = assignment_seed(
    MASTER_SEED,
    TARGET_RHO,
    TARGET_REPLICATE
)

print("\nAssignment seed:")
print(seed)

rng = np.random.Generator(
    np.random.PCG64(seed)
)

selected_hair_lesions = set()


# ------------------------------------------------
# Original malignant × subtype loop
# ------------------------------------------------

for malignant in [0, 1]:

    for class_name, target_count in (
        subtype_targets[malignant].items()
    ):

        stratum = lesions[
            (lesions["malignant"] == malignant)
            &
            (lesions["class_name"] == class_name)
        ].copy()

        selected = exact_random_subset(
            stratum,
            target_count,
            rng
        )

        selected_hair_lesions.update(
            selected
        )

        achieved = int(
            stratum[
                stratum["lesion_id"]
                .astype(str)
                .isin(selected)
            ]["n_images"].sum()
        )

        assert achieved == target_count


# ================================================================
# CONVERT LESION ASSIGNMENT TO IMAGE ASSIGNMENT
# ================================================================

candidate = train.copy()

candidate["rho"] = TARGET_RHO
candidate["replicate"] = TARGET_REPLICATE
candidate["assignment_seed"] = seed

candidate["hair_present"] = (
    candidate["lesion_id"]
    .astype(str)
    .isin(selected_hair_lesions)
    .astype(int)
)

candidate["severity"] = SEVERITY

candidate["artifact"] = np.where(
    candidate["hair_present"] == 1,
    "synthetic_hair",
    "none"
)


# ================================================================
# ORIGINAL GENERATOR AUDITS
# ================================================================

malignant_hair = int(
    candidate.loc[
        candidate["malignant"] == 1,
        "hair_present"
    ].sum()
)

benign_hair = int(
    candidate.loc[
        candidate["malignant"] == 0,
        "hair_present"
    ].sum()
)

total_hair = int(
    candidate["hair_present"].sum()
)

assert (
    malignant_hair
    == binary_targets[1]
)

assert (
    benign_hair
    == binary_targets[0]
)

assert total_hair == 9257

lesion_assignment_counts = (
    candidate
    .groupby("lesion_id")["hair_present"]
    .nunique()
)

assert (
    lesion_assignment_counts.max()
    == 1
)

print("\nGenerated counts:")
print(
    "  malignant hair:",
    malignant_hair
)

print(
    "  benign hair:",
    benign_hair
)

print(
    "  total hair:",
    total_hair
)

print(
    "  selected lesions:",
    len(selected_hair_lesions)
)


# ================================================================
# WRITE ONLY TO WORKING DIRECTORY
# ================================================================

candidate_path = (
    REPRO_DIR
    / "rho_0.2_rep_1_REPRODUCED.csv"
)

candidate.to_csv(
    candidate_path,
    index=False
)

print("\nCandidate written to:")
print(candidate_path)


# ================================================================
# BYTE-FOR-BYTE COMPARISON
# ================================================================

locked_bytes = (
    LOCKED_ASSIGNMENT
    .read_bytes()
)

candidate_bytes = (
    candidate_path
    .read_bytes()
)

byte_exact = (
    locked_bytes
    == candidate_bytes
)

print("\n" + "-" * 70)

print(
    "Locked file bytes:",
    len(locked_bytes)
)

print(
    "Candidate file bytes:",
    len(candidate_bytes)
)

print(
    "BYTE-FOR-BYTE EXACT:",
    byte_exact
)


# ================================================================
# SEMANTIC COMPARISON — ALWAYS RUN EVEN IF BYTE TEST PASSES
# ================================================================

locked_df = pd.read_csv(
    LOCKED_ASSIGNMENT
)

candidate_df = pd.read_csv(
    candidate_path
)

assert list(
    locked_df.columns
) == list(
    candidate_df.columns
)

assert locked_df.shape == candidate_df.shape

semantic_exact = (
    locked_df.equals(candidate_df)
)

print(
    "DATAFRAME EXACT:",
    semantic_exact
)


# ================================================================
# COLUMN-BY-COLUMN AUDIT
# ================================================================

column_results = []

for column in locked_df.columns:

    exact = (
        locked_df[column]
        .equals(
            candidate_df[column]
        )
    )

    column_results.append({
        "column": column,
        "exact": exact
    })

column_audit = pd.DataFrame(
    column_results
)

print("\nColumn audit:")
print(
    column_audit.to_string(
        index=False
    )
)


# ================================================================
# HASHES
# ================================================================

locked_sha256 = hashlib.sha256(
    locked_bytes
).hexdigest()

candidate_sha256 = hashlib.sha256(
    candidate_bytes
).hexdigest()

print("\nLocked SHA256:")
print(locked_sha256)

print("\nCandidate SHA256:")
print(candidate_sha256)


# ================================================================
# FINAL GATE
# ================================================================

assert byte_exact, (
    "STOP: recovered generator did NOT reproduce "
    "the locked CSV byte-for-byte."
)

assert semantic_exact, (
    "STOP: recovered generator did NOT reproduce "
    "the locked assignment exactly."
)

assert column_audit["exact"].all()

print("\n" + "=" * 70)
print("PASS — EXACT ORIGINAL STEP1F REPRODUCTION")
print("=" * 70)
print(
    "rho=0.2, replicate=1 reproduced "
    "byte-for-byte."
)
print(
    "The recovered generator is now validated "
    "against a real locked assignment."
)
print(
    "NO rho=0.5 assignment has been generated."
)
print("=" * 70)
print("CELL 6O COMPLETE")
print("=" * 70)

In [ ]:
# ================================================================
# T2.1 — CELL 6P
# MULTI-CONDITION EXACT STEP1F REPRODUCTION AUDIT
#
# Reproduce several already-locked assignments using the recovered
# original generator and require byte-for-byte equality.
#
# TESTS:
#   rho=0.0, replicate=1
#   rho=0.8, replicate=1
#   rho=1.0, replicate=3
#
# IMPORTANT:
#   NO rho=0.5 assignment is generated here.
# ================================================================

print("=" * 70)
print("T2.1 — CELL 6P")
print("MULTI-CONDITION EXACT STEP1F REPRODUCTION AUDIT")
print("=" * 70)


# ================================================================
# REPRODUCTION FUNCTION
# ================================================================

def reproduce_locked_assignment(
    rho,
    replicate
):

    # ------------------------------------------------------------
    # Load exact target row
    # ------------------------------------------------------------

    target_row = targets[
        np.isclose(
            targets["rho"],
            rho
        )
    ].iloc[0]

    binary_targets = {
        1: int(
            target_row[
                "target_malignant_hair_images"
            ]
        ),
        0: int(
            target_row[
                "target_benign_hair_images"
            ]
        )
    }

    # ------------------------------------------------------------
    # Reconstruct subtype targets exactly as original generator
    # ------------------------------------------------------------

    subtype_targets = {}

    for malignant in [0, 1]:

        class_target = (
            binary_targets[malignant]
        )

        class_lesions = lesions[
            lesions["malignant"] == malignant
        ]

        raw = {}

        for class_name, group in (
            class_lesions
            .groupby("class_name")
        ):

            n_images = int(
                group["n_images"].sum()
            )

            if malignant == 1:

                p_target = float(
                    target_row[
                        "target_p_h_given_m"
                    ]
                )

            else:

                p_target = float(
                    target_row[
                        "target_p_h_given_b"
                    ]
                )

            raw[class_name] = (
                n_images * p_target
            )

        rounded = {
            k: int(round(v))
            for k, v in raw.items()
        }

        difference = (
            class_target
            - sum(rounded.values())
        )

        if difference != 0:

            fractional = {
                k: raw[k] - np.floor(raw[k])
                for k in raw
            }

            if difference > 0:

                order = sorted(
                    fractional,
                    key=fractional.get,
                    reverse=True
                )

                for k in order[:difference]:
                    rounded[k] += 1

            else:

                order = sorted(
                    fractional,
                    key=fractional.get
                )

                for k in order[:abs(difference)]:
                    rounded[k] -= 1

        assert (
            sum(rounded.values())
            == class_target
        )

        subtype_targets[malignant] = (
            rounded
        )

    # ------------------------------------------------------------
    # Original seed
    # ------------------------------------------------------------

    seed = assignment_seed(
        MASTER_SEED,
        rho,
        replicate
    )

    rng = np.random.Generator(
        np.random.PCG64(seed)
    )

    selected_hair_lesions = set()

    # ------------------------------------------------------------
    # Original malignant × subtype selection
    # ------------------------------------------------------------

    for malignant in [0, 1]:

        for class_name, target_count in (
            subtype_targets[malignant].items()
        ):

            stratum = lesions[
                (lesions["malignant"] == malignant)
                &
                (lesions["class_name"] == class_name)
            ].copy()

            selected = exact_random_subset(
                stratum,
                target_count,
                rng
            )

            selected_hair_lesions.update(
                selected
            )

            achieved = int(
                stratum[
                    stratum["lesion_id"]
                    .astype(str)
                    .isin(selected)
                ]["n_images"].sum()
            )

            assert achieved == target_count

    # ------------------------------------------------------------
    # Image-level assignment
    # ------------------------------------------------------------

    candidate = train.copy()

    candidate["rho"] = rho
    candidate["replicate"] = replicate
    candidate["assignment_seed"] = seed

    candidate["hair_present"] = (
        candidate["lesion_id"]
        .astype(str)
        .isin(selected_hair_lesions)
        .astype(int)
    )

    candidate["severity"] = SEVERITY

    candidate["artifact"] = np.where(
        candidate["hair_present"] == 1,
        "synthetic_hair",
        "none"
    )

    # ------------------------------------------------------------
    # Exact binary audits
    # ------------------------------------------------------------

    malignant_hair = int(
        candidate.loc[
            candidate["malignant"] == 1,
            "hair_present"
        ].sum()
    )

    benign_hair = int(
        candidate.loc[
            candidate["malignant"] == 0,
            "hair_present"
        ].sum()
    )

    total_hair = int(
        candidate["hair_present"].sum()
    )

    assert (
        malignant_hair
        == binary_targets[1]
    )

    assert (
        benign_hair
        == binary_targets[0]
    )

    assert total_hair == 9257

    # ------------------------------------------------------------
    # Compare with locked assignment
    # ------------------------------------------------------------

    locked_path = (
        ASSIGN_DIR
        / f"rho_{rho:.1f}_rep_{replicate}.csv"
    )

    assert locked_path.exists(), locked_path

    candidate_path = (
        REPRO_DIR
        / f"rho_{rho:.1f}_rep_{replicate}_REPRODUCED.csv"
    )

    candidate.to_csv(
        candidate_path,
        index=False
    )

    locked_bytes = (
        locked_path.read_bytes()
    )

    candidate_bytes = (
        candidate_path.read_bytes()
    )

    byte_exact = (
        locked_bytes
        == candidate_bytes
    )

    # ------------------------------------------------------------
    # DataFrame comparison
    # ------------------------------------------------------------

    locked_df = pd.read_csv(
        locked_path
    )

    candidate_df = pd.read_csv(
        candidate_path
    )

    dataframe_exact = (
        locked_df.equals(candidate_df)
    )

    # ------------------------------------------------------------
    # SHA256
    # ------------------------------------------------------------

    locked_sha = hashlib.sha256(
        locked_bytes
    ).hexdigest()

    candidate_sha = hashlib.sha256(
        candidate_bytes
    ).hexdigest()

    return {
        "rho": rho,
        "replicate": replicate,
        "seed": seed,
        "malignant_hair": malignant_hair,
        "benign_hair": benign_hair,
        "total_hair": total_hair,
        "selected_lesions": len(
            selected_hair_lesions
        ),
        "locked_bytes": len(
            locked_bytes
        ),
        "candidate_bytes": len(
            candidate_bytes
        ),
        "byte_exact": byte_exact,
        "dataframe_exact": dataframe_exact,
        "sha256_exact": (
            locked_sha
            == candidate_sha
        ),
        "locked_sha256": locked_sha,
        "candidate_sha256": candidate_sha,
    }


# ================================================================
# TEST MATRIX
# ================================================================

TEST_CASES = [
    (0.0, 1),
    (0.8, 1),
    (1.0, 3),
]


results = []

for rho, replicate in TEST_CASES:

    print("\n" + "-" * 70)
    print(
        f"Testing rho={rho:.1f}, "
        f"replicate={replicate}"
    )
    print("-" * 70)

    result = reproduce_locked_assignment(
        rho,
        replicate
    )

    results.append(result)

    print(
        "Seed:",
        result["seed"]
    )

    print(
        "Malignant hair:",
        result["malignant_hair"]
    )

    print(
        "Benign hair:",
        result["benign_hair"]
    )

    print(
        "Total hair:",
        result["total_hair"]
    )

    print(
        "Selected lesions:",
        result["selected_lesions"]
    )

    print(
        "Byte exact:",
        result["byte_exact"]
    )

    print(
        "DataFrame exact:",
        result["dataframe_exact"]
    )

    print(
        "SHA256 exact:",
        result["sha256_exact"]
    )


# ================================================================
# SUMMARY
# ================================================================

audit_df = pd.DataFrame(
    results
)

print("\n" + "=" * 70)
print("MULTI-CONDITION REPRODUCTION SUMMARY")
print("=" * 70)

print(
    audit_df[
        [
            "rho",
            "replicate",
            "seed",
            "total_hair",
            "selected_lesions",
            "byte_exact",
            "dataframe_exact",
            "sha256_exact",
        ]
    ].to_string(index=False)
)


# ================================================================
# FINAL GATE
# ================================================================

assert len(results) == 3

assert all(
    r["byte_exact"]
    for r in results
)

assert all(
    r["dataframe_exact"]
    for r in results
)

assert all(
    r["sha256_exact"]
    for r in results
)

print("\n" + "=" * 70)
print("PASS — MULTI-CONDITION STEP1F REPRODUCTION")
print("=" * 70)
print(
    "All 3 additional locked assignments reproduced "
    "byte-for-byte."
)
print(
    "The recovered original Step1F generator is "
    "validated across multiple rho conditions."
)
print(
    "NO rho=0.5 assignment has been generated."
)
print("=" * 70)
print("CELL 6P COMPLETE")
print("=" * 70)

In [ ]:
# ================================================================
# T2.1 — CELL 6Q
# GENERATE NEW rho=0.5 STEP1F ASSIGNMENTS
#
# LOCKED:
#   rho = 0.5
#   replicates = 1, 2, 3
#
# METHOD:
#   Exact recovered original Step1F generator
#   Exact corrected rho target construction
#   Same master seed = 20260919
#   Same PCG64 seed namespace
#   Same lesion-level random subset algorithm
#
# IMPORTANT:
#   Existing 30 locked assignments are NOT modified.
#   New assignments are written only under /kaggle/working/.
# ================================================================

print("=" * 70)
print("T2.1 — CELL 6Q")
print("GENERATING NEW rho=0.5 STEP1F ASSIGNMENTS")
print("=" * 70)


# ================================================================
# CONFIGURATION
# ================================================================

NEW_RHO = 0.5
NEW_REPLICATES = [1, 2, 3]

MASTER_SEED = 20260919
SEVERITY = "medium"

NEW_ASSIGN_DIR = (
    Path("/kaggle/working")
    / "T2_1_ConvNeXt_rho05_assignments"
)

NEW_ASSIGN_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ================================================================
# VERIFY SOURCE INPUTS
# ================================================================

assert LOCKED_MANIFEST.exists()
assert TARGET_PATH.exists()

print("\nSource population:")
print(LOCKED_MANIFEST)

print("\nExisting Step1F assignments:")
print(ASSIGN_DIR)

print("\nNew assignment directory:")
print(NEW_ASSIGN_DIR)


# ================================================================
# VERIFY POPULATION
# ================================================================

assert len(train) == 18514

n_malignant = int(
    train["malignant"].sum()
)

n_benign = int(
    (train["malignant"] == 0).sum()
)

n_total = len(train)

target_total_hair = (
    n_total // 2
)

assert n_malignant == 7146
assert n_benign == 11368
assert n_total == 18514
assert target_total_hair == 9257


# ================================================================
# CORRECTED ORIGINAL RHO TARGET CONSTRUCTION
#
# Original Step1D:
#
# P(H|M) = 0.5 + 0.5*rho
#
# P(H|B) =
#   (0.5 - P(M)*P(H|M)) / P(B)
#
# Then:
#   target_malignant_hair =
#       round(N_M * P(H|M))
#
#   target_benign_hair =
#       9257 - target_malignant_hair
#
# ================================================================

p_malignant = (
    n_malignant / n_total
)

p_benign = (
    n_benign / n_total
)

target_p_h_given_m = (
    0.5
    + 0.5 * NEW_RHO
)

target_p_h_given_b = (
    0.5
    - p_malignant
    * target_p_h_given_m
) / p_benign


theoretical_malignant_hair = (
    n_malignant
    * target_p_h_given_m
)

theoretical_benign_hair = (
    n_benign
    * target_p_h_given_b
)

theoretical_total_hair = (
    theoretical_malignant_hair
    + theoretical_benign_hair
)

target_malignant_hair = int(
    round(
        theoretical_malignant_hair
    )
)

target_benign_hair = (
    target_total_hair
    - target_malignant_hair
)

actual_p_h_given_m = (
    target_malignant_hair
    / n_malignant
)

actual_p_h_given_b = (
    target_benign_hair
    / n_benign
)

actual_p_h = (
    target_total_hair
    / n_total
)


# ================================================================
# EMPIRICAL PHI OF INTEGER TARGET
# ================================================================

p_hm = (
    target_malignant_hair
    / n_total
)

denominator = np.sqrt(
    actual_p_h
    * (1 - actual_p_h)
    * p_malignant
    * (1 - p_malignant)
)

empirical_phi = (
    (
        p_hm
        - actual_p_h * p_malignant
    )
    / denominator
)


# ================================================================
# PRINT TARGET
# ================================================================

print("\n" + "-" * 70)
print("rho=0.5 TARGET")
print("-" * 70)

print(
    f"rho:                         {NEW_RHO}"
)

print(
    f"P(M):                        {p_malignant:.12f}"
)

print(
    f"P(B):                        {p_benign:.12f}"
)

print(
    f"Target P(H|M):               "
    f"{target_p_h_given_m:.12f}"
)

print(
    f"Target P(H|B):               "
    f"{target_p_h_given_b:.12f}"
)

print(
    f"Theoretical malignant hair:  "
    f"{theoretical_malignant_hair:.6f}"
)

print(
    f"Theoretical benign hair:     "
    f"{theoretical_benign_hair:.6f}"
)

print(
    f"Theoretical total hair:      "
    f"{theoretical_total_hair:.6f}"
)

print(
    f"Integer malignant target:    "
    f"{target_malignant_hair}"
)

print(
    f"Integer benign target:       "
    f"{target_benign_hair}"
)

print(
    f"Integer total target:        "
    f"{target_malignant_hair + target_benign_hair}"
)

print(
    f"Actual P(H|M):               "
    f"{actual_p_h_given_m:.12f}"
)

print(
    f"Actual P(H|B):               "
    f"{actual_p_h_given_b:.12f}"
)

print(
    f"Actual P(H):                 "
    f"{actual_p_h:.12f}"
)

print(
    f"Empirical phi:               "
    f"{empirical_phi:.12f}"
)


# ================================================================
# TARGET ASSERTIONS
# ================================================================

assert (
    target_malignant_hair
    == 5360
)

assert (
    target_benign_hair
    == 3897
)

assert (
    target_malignant_hair
    + target_benign_hair
    == 9257
)

assert abs(
    actual_p_h - 0.5
) < 1e-12

assert (
    0.0
    < target_p_h_given_b
    < 1.0
)

assert (
    target_p_h_given_m
    > 0.5
)

assert (
    target_p_h_given_b
    < 0.5
)


# ================================================================
# PREPARE LESION TABLE
# ================================================================

# This is the exact lesion representation used by the original
# generator.

lesions = (
    train.groupby(
        [
            "malignant",
            "class_name",
            "lesion_id"
        ],
        as_index=False
    )
    .agg(
        n_images=("image", "size")
    )
)

assert (
    lesions
    .groupby("lesion_id")["class_name"]
    .nunique()
    .max()
    == 1
)

assert (
    lesions["lesion_id"]
    .nunique()
    == 9478
)


# ================================================================
# CONSTRUCT SUBTYPE TARGETS
#
# Exact original Step1F procedure:
#   raw = n_images * p_target
#   round each subtype
#   adjust minimum number of subtype targets using fractional
#   remainders until the class total equals the binary target.
# ================================================================

subtype_targets = {}

for malignant in [0, 1]:

    class_target = (
        target_malignant_hair
        if malignant == 1
        else target_benign_hair
    )

    class_lesions = lesions[
        lesions["malignant"] == malignant
    ]

    raw = {}

    for class_name, group in (
        class_lesions
        .groupby("class_name")
    ):

        n_images = int(
            group["n_images"].sum()
        )

        if malignant == 1:

            p_target = (
                target_p_h_given_m
            )

        else:

            p_target = (
                target_p_h_given_b
            )

        raw[class_name] = (
            n_images
            * p_target
        )

    rounded = {
        k: int(round(v))
        for k, v in raw.items()
    }

    difference = (
        class_target
        - sum(rounded.values())
    )

    # ------------------------------------------------------------
    # Exact original fractional-remainder correction
    # ------------------------------------------------------------

    if difference != 0:

        fractional = {
            k: raw[k] - np.floor(raw[k])
            for k in raw
        }

        if difference > 0:

            order = sorted(
                fractional,
                key=fractional.get,
                reverse=True
            )

            for k in order[:difference]:

                rounded[k] += 1

        else:

            order = sorted(
                fractional,
                key=fractional.get
            )

            for k in order[:abs(difference)]:

                rounded[k] -= 1

    assert (
        sum(rounded.values())
        == class_target
    )

    subtype_targets[malignant] = (
        rounded
    )


# ================================================================
# PRINT SUBTYPE TARGETS
# ================================================================

print("\n" + "-" * 70)
print("rho=0.5 SUBTYPE TARGETS")
print("-" * 70)

for malignant in [0, 1]:

    label = (
        "malignant"
        if malignant == 1
        else "benign"
    )

    print(
        f"{label}:"
    )

    for class_name, count in (
        subtype_targets[malignant]
        .items()
    ):

        print(
            f"  {class_name:>4s}: {count}"
        )

    print(
        "  TOTAL:",
        sum(
            subtype_targets[malignant]
            .values()
        )
    )


# ================================================================
# GENERATE EACH NEW REPLICATE
# ================================================================

generation_records = []


for replicate in NEW_REPLICATES:

    print("\n" + "=" * 70)
    print(
        f"GENERATING rho=0.5 "
        f"replicate={replicate}"
    )
    print("=" * 70)

    # ------------------------------------------------------------
    # Exact original seed
    # ------------------------------------------------------------

    seed = assignment_seed(
        MASTER_SEED,
        NEW_RHO,
        replicate
    )

    print(
        "Assignment seed:",
        seed
    )

    rng = np.random.Generator(
        np.random.PCG64(seed)
    )

    selected_hair_lesions = set()

    # ------------------------------------------------------------
    # Original malignant × subtype loop
    # ------------------------------------------------------------

    for malignant in [0, 1]:

        for class_name, target_count in (
            subtype_targets[malignant]
            .items()
        ):

            stratum = lesions[
                (lesions["malignant"] == malignant)
                &
                (
                    lesions["class_name"]
                    == class_name
                )
            ].copy()

            selected = exact_random_subset(
                stratum,
                target_count,
                rng
            )

            selected_hair_lesions.update(
                selected
            )

            achieved = int(
                stratum[
                    stratum["lesion_id"]
                    .astype(str)
                    .isin(selected)
                ]["n_images"].sum()
            )

            assert (
                achieved
                == target_count
            )

    # ------------------------------------------------------------
    # Convert lesion assignment to image assignment
    # ------------------------------------------------------------

    output = train.copy()

    output["rho"] = NEW_RHO
    output["replicate"] = replicate
    output["assignment_seed"] = seed

    output["hair_present"] = (
        output["lesion_id"]
        .astype(str)
        .isin(
            selected_hair_lesions
        )
        .astype(int)
    )

    output["severity"] = SEVERITY

    output["artifact"] = np.where(
        output["hair_present"] == 1,
        "synthetic_hair",
        "none"
    )

    # ------------------------------------------------------------
    # Exact assignment audits
    # ------------------------------------------------------------

    malignant_hair = int(
        output.loc[
            output["malignant"] == 1,
            "hair_present"
        ].sum()
    )

    benign_hair = int(
        output.loc[
            output["malignant"] == 0,
            "hair_present"
        ].sum()
    )

    total_hair = int(
        output["hair_present"].sum()
    )

    assert (
        malignant_hair
        == target_malignant_hair
    )

    assert (
        benign_hair
        == target_benign_hair
    )

    assert (
        total_hair
        == target_total_hair
    )

    # ------------------------------------------------------------
    # Lesion consistency
    # ------------------------------------------------------------

    lesion_assignment_counts = (
        output
        .groupby("lesion_id")[
            "hair_present"
        ]
        .nunique()
    )

    assert (
        lesion_assignment_counts.max()
        == 1
    )

    # ------------------------------------------------------------
    # No duplicate image rows
    # ------------------------------------------------------------

    assert (
        output["image"].nunique()
        == len(output)
    )

    # ------------------------------------------------------------
    # Recalculate empirical conditional rates
    # ------------------------------------------------------------

    check_p_h_m = (
        malignant_hair
        / n_malignant
    )

    check_p_h_b = (
        benign_hair
        / n_benign
    )

    check_p_h = (
        total_hair
        / n_total
    )

    # ------------------------------------------------------------
    # Save
    # ------------------------------------------------------------

    output_path = (
        NEW_ASSIGN_DIR
        / f"rho_0.5_rep_{replicate}.csv"
    )

    output.to_csv(
        output_path,
        index=False
    )

    file_sha256 = hashlib.sha256(
        output_path.read_bytes()
    ).hexdigest()

    # ------------------------------------------------------------
    # Record immutable generation metadata
    # ------------------------------------------------------------

    generation_records.append({
        "rho": NEW_RHO,
        "replicate": replicate,
        "assignment_seed": seed,
        "malignant_hair": malignant_hair,
        "benign_hair": benign_hair,
        "total_hair": total_hair,
        "selected_lesions": len(
            selected_hair_lesions
        ),
        "p_h_given_m": check_p_h_m,
        "p_h_given_b": check_p_h_b,
        "p_h": check_p_h,
        "sha256": file_sha256,
        "path": str(output_path),
    })

    print(
        "Malignant hair:",
        malignant_hair
    )

    print(
        "Benign hair:",
        benign_hair
    )

    print(
        "Total hair:",
        total_hair
    )

    print(
        "Selected lesions:",
        len(selected_hair_lesions)
    )

    print(
        "P(H|M):",
        f"{check_p_h_m:.12f}"
    )

    print(
        "P(H|B):",
        f"{check_p_h_b:.12f}"
    )

    print(
        "SHA256:",
        file_sha256
    )

    print(
        "Saved:",
        output_path
    )


# ================================================================
# FINAL GENERATION SUMMARY
# ================================================================

generation_df = pd.DataFrame(
    generation_records
)

print("\n" + "=" * 70)
print("rho=0.5 GENERATION SUMMARY")
print("=" * 70)

print(
    generation_df[
        [
            "rho",
            "replicate",
            "assignment_seed",
            "malignant_hair",
            "benign_hair",
            "total_hair",
            "selected_lesions",
            "p_h_given_m",
            "p_h_given_b",
            "sha256",
        ]
    ].to_string(
        index=False
    )
)


# ================================================================
# REPLICATE INDEPENDENCE
# ================================================================

assert (
    generation_df["assignment_seed"]
    .nunique()
    == 3
)

assert (
    generation_df["sha256"]
    .nunique()
    == 3
)


base = pd.read_csv(
    generation_records[0]["path"]
)

base_hair = set(
    base.loc[
        base["hair_present"] == 1,
        "lesion_id"
    ]
)

for record in generation_records[1:]:

    other = pd.read_csv(
        record["path"]
    )

    other_hair = set(
        other.loc[
            other["hair_present"] == 1,
            "lesion_id"
        ]
    )

    symmetric_difference = len(
        base_hair
        .symmetric_difference(
            other_hair
        )
    )

    print(
        f"rep1 vs rep{record['replicate']} "
        f"lesion symmetric difference: "
        f"{symmetric_difference:,}"
    )

    assert (
        symmetric_difference > 0
    )


# ================================================================
# IMMUTABILITY CHECK
#
# Verify that the existing locked Step1F directory still contains
# exactly the original 30 manifests.
# ================================================================

existing_files = sorted(
    ASSIGN_DIR.glob(
        "rho_*_rep_*.csv"
    )
)

assert len(existing_files) == 30

print("\nExisting locked Step1F CSV count:", len(existing_files))

print(
    "New rho=0.5 CSV count:",
    len(
        list(
            NEW_ASSIGN_DIR.glob(
                "rho_0.5_rep_*.csv"
            )
        )
    )
)

assert (
    len(
        list(
            NEW_ASSIGN_DIR.glob(
                "rho_0.5_rep_*.csv"
            )
        )
    )
    == 3
)


# ================================================================
# SAVE GENERATION MANIFEST
# ================================================================

generation_manifest = {
    "status": "GENERATED",
    "architecture": "ConvNeXt-Tiny",
    "purpose": "T2.1 second-architecture replication",
    "rho": 0.5,
    "replicates": [1, 2, 3],
    "master_seed": MASTER_SEED,
    "assignment_seed_formula": (
        "SHA256("
        "rho-assignment-v1|"
        "master_seed|"
        "rho|"
        "replicate-{replicate}"
        "), first 64 bits, PCG64"
    ),
    "assignment_unit": "lesion",
    "severity": SEVERITY,
    "training_images": n_total,
    "training_lesions": int(
        lesions["lesion_id"].nunique()
    ),
    "target_total_hair_images": target_total_hair,
    "target_malignant_hair_images": (
        target_malignant_hair
    ),
    "target_benign_hair_images": (
        target_benign_hair
    ),
    "target_p_h_given_m": (
        target_p_h_given_m
    ),
    "target_p_h_given_b": (
        target_p_h_given_b
    ),
    "generation_method": (
        "Recovered original Step1F "
        "exact_random_subset generator; "
        "validated byte-for-byte against "
        "locked historical assignments."
    ),
    "new_assignment_directory": str(
        NEW_ASSIGN_DIR
    ),
    "existing_locked_assignments_modified": False,
    "files": generation_records,
}


manifest_path = (
    NEW_ASSIGN_DIR
    / "T2_1_RHO05_GENERATION_MANIFEST.json"
)

import json

with open(
    manifest_path,
    "w"
) as f:

    json.dump(
        generation_manifest,
        f,
        indent=2
    )


print("\nGeneration manifest:")
print(manifest_path)


# ================================================================
# FINAL GATE
# ================================================================

assert len(generation_records) == 3

assert all(
    r["malignant_hair"]
    == 5360
    for r in generation_records
)

assert all(
    r["benign_hair"]
    == 3897
    for r in generation_records
)

assert all(
    r["total_hair"]
    == 9257
    for r in generation_records
)

assert all(
    r["assignment_seed"] is not None
    for r in generation_records
)

print("\n" + "=" * 70)
print("PASS — rho=0.5 ASSIGNMENTS GENERATED")
print("=" * 70)

print(
    "Three new assignments generated:"
)

for record in generation_records:

    print(
        f"  rho=0.5 rep={record['replicate']} "
        f"seed={record['assignment_seed']} "
        f"SHA256={record['sha256']}"
    )

print(
    "\nExisting 30 locked Step1F assignments "
    "were not modified."
)

print(
    "The three rho=0.5 assignments are ready "
    "for T2.1 ConvNeXt-Tiny training."
)

print("=" * 70)
print("CELL 6Q COMPLETE")
print("=" * 70)

In [ ]:
# ================================================================
# T2.1 — CELL 6R
# FINAL rho=0.5 ASSIGNMENT INTEGRITY GATE
#
# Audit all 3 newly generated rho=0.5 manifests before training.
#
# Checks:
#   1. Exact schema
#   2. Exact population identity and row order
#   3. Exact immutable metadata
#   4. Exact binary hair targets
#   5. Exact total hair target
#   6. Lesion-level consistency
#   7. No duplicate images
#   8. Correct rho / replicate / seed
#   9. Correct artifact/severity encoding
#  10. Replicate independence
#  11. SHA256 against generation manifest
#  12. Existing 30 locked assignments remain separate
#
# NO GPU.
# NO TRAINING.
# NO FILE MODIFICATION.
# ================================================================

print("=" * 70)
print("T2.1 — CELL 6R")
print("FINAL rho=0.5 ASSIGNMENT INTEGRITY GATE")
print("=" * 70)


# ================================================================
# EXPECTED CONSTANTS
# ================================================================

EXPECTED_COLUMNS = [
    "image",
    "lesion_id",
    "class_name",
    "malignant",
    "split",
    "rho",
    "replicate",
    "assignment_seed",
    "hair_present",
    "severity",
    "artifact",
]

EXPECTED_RHO = 0.5

EXPECTED_REPLICATES = [1, 2, 3]

EXPECTED_MALIGNANT_HAIR = 5360
EXPECTED_BENIGN_HAIR = 3897
EXPECTED_TOTAL_HAIR = 9257

EXPECTED_TRAIN_IMAGES = 18514
EXPECTED_TRAIN_LESIONS = 9478

EXPECTED_SEVERITY = "medium"

EXPECTED_SEEDS = {
    1: 15607068371304439695,
    2: 18409186774447888968,
    3: 299552645381778030,
}

EXPECTED_SHA256 = {
    1: "631aad30c4d4952d4a2a68092a6b2323d19e9b93cb0f69ed36b9cf7b2c70a877",
    2: "a63d0bad295b0dc38677d3780cf1ad77aa20e3fa12b917b6bebd3d91ed3dc290",
    3: "22969500b6481a905c97e86dab77dea6cf30c6c6aa90914994ecfa1fc3f93066",
}


# ================================================================
# LOAD GENERATION MANIFEST
# ================================================================

generation_manifest_path = (
    NEW_ASSIGN_DIR
    / "T2_1_RHO05_GENERATION_MANIFEST.json"
)

assert generation_manifest_path.exists()

with open(
    generation_manifest_path,
    "r"
) as f:

    generation_manifest_loaded = json.load(f)


assert (
    generation_manifest_loaded["rho"]
    == EXPECTED_RHO
)

assert (
    generation_manifest_loaded["replicates"]
    == EXPECTED_REPLICATES
)

assert (
    generation_manifest_loaded["master_seed"]
    == MASTER_SEED
)

assert (
    generation_manifest_loaded[
        "existing_locked_assignments_modified"
    ]
    is False
)


# ================================================================
# VERIFY LOCKED POPULATION REFERENCE
# ================================================================

locked_population = pd.read_csv(
    LOCKED_MANIFEST
)

locked_train = (
    locked_population[
        locked_population["split"] == "train"
    ]
    .copy()
    .reset_index(drop=True)
)

assert len(locked_train) == EXPECTED_TRAIN_IMAGES

assert (
    locked_train["lesion_id"]
    .nunique()
    == EXPECTED_TRAIN_LESIONS
)


# ================================================================
# POPULATION SIGNATURE
#
# We compare the immutable source columns used by Step1F.
# ================================================================

IMMUTABLE_COLUMNS = [
    "image",
    "lesion_id",
    "class_name",
    "malignant",
    "split",
]

population_signature = hashlib.sha256(
    pd.util.hash_pandas_object(
        locked_train[
            IMMUTABLE_COLUMNS
        ],
        index=False
    ).values.tobytes()
).hexdigest()

print("\nLocked training population:")
print(
    "  images:",
    len(locked_train)
)

print(
    "  lesions:",
    locked_train["lesion_id"].nunique()
)

print(
    "  population signature:",
    population_signature
)


# ================================================================
# AUDIT EACH NEW ASSIGNMENT
# ================================================================

audit_records = []

loaded_assignments = {}


for replicate in EXPECTED_REPLICATES:

    print("\n" + "-" * 70)
    print(
        f"Auditing rho=0.5 replicate={replicate}"
    )
    print("-" * 70)

    path = (
        NEW_ASSIGN_DIR
        / f"rho_0.5_rep_{replicate}.csv"
    )

    assert path.exists(), path

    # ------------------------------------------------------------
    # File hash
    # ------------------------------------------------------------

    file_bytes = path.read_bytes()

    sha256 = hashlib.sha256(
        file_bytes
    ).hexdigest()

    assert (
        sha256
        == EXPECTED_SHA256[replicate]
    )

    print(
        "SHA256:",
        sha256
    )

    # ------------------------------------------------------------
    # Load
    # ------------------------------------------------------------

    df = pd.read_csv(path)

    loaded_assignments[replicate] = df

    # ------------------------------------------------------------
    # Schema
    # ------------------------------------------------------------

    assert list(
        df.columns
    ) == EXPECTED_COLUMNS

    assert (
        df.shape[0]
        == EXPECTED_TRAIN_IMAGES
    )

    print(
        "Schema: PASS"
    )

    # ------------------------------------------------------------
    # Exact row order / population identity
    # ------------------------------------------------------------

    assert (
        df["image"].equals(
            locked_train["image"]
        )
    )

    assert (
        df["lesion_id"].equals(
            locked_train["lesion_id"]
        )
    )

    assert (
        df["class_name"].equals(
            locked_train["class_name"]
        )
    )

    assert (
        df["malignant"].equals(
            locked_train["malignant"]
        )
    )

    assert (
        df["split"].equals(
            locked_train["split"]
        )
    )

    print(
        "Population identity + row order: PASS"
    )

    # ------------------------------------------------------------
    # Fixed metadata
    # ------------------------------------------------------------

    assert (
        df["rho"]
        .eq(EXPECTED_RHO)
        .all()
    )

    assert (
        df["replicate"]
        .eq(replicate)
        .all()
    )

    assert (
        df["assignment_seed"]
        .eq(EXPECTED_SEEDS[replicate])
        .all()
    )

    assert (
        df["severity"]
        .eq(EXPECTED_SEVERITY)
        .all()
    )

    print(
        "rho / replicate / seed / severity: PASS"
    )

    # ------------------------------------------------------------
    # Hair variable
    # ------------------------------------------------------------

    assert set(
        df["hair_present"].unique()
    ).issubset({0, 1})

    malignant_hair = int(
        df.loc[
            df["malignant"] == 1,
            "hair_present"
        ].sum()
    )

    benign_hair = int(
        df.loc[
            df["malignant"] == 0,
            "hair_present"
        ].sum()
    )

    total_hair = int(
        df["hair_present"].sum()
    )

    assert (
        malignant_hair
        == EXPECTED_MALIGNANT_HAIR
    )

    assert (
        benign_hair
        == EXPECTED_BENIGN_HAIR
    )

    assert (
        total_hair
        == EXPECTED_TOTAL_HAIR
    )

    print(
        "Malignant hair:",
        malignant_hair
    )

    print(
        "Benign hair:",
        benign_hair
    )

    print(
        "Total hair:",
        total_hair
    )

    # ------------------------------------------------------------
    # Artifact consistency
    # ------------------------------------------------------------

    expected_artifact = np.where(
        df["hair_present"] == 1,
        "synthetic_hair",
        "none"
    )

    assert np.array_equal(
        df["artifact"].to_numpy(),
        expected_artifact
    )

    print(
        "Artifact consistency: PASS"
    )

    # ------------------------------------------------------------
    # No duplicate images
    # ------------------------------------------------------------

    assert (
        df["image"].nunique()
        == EXPECTED_TRAIN_IMAGES
    )

    print(
        "Image uniqueness: PASS"
    )

    # ------------------------------------------------------------
    # Lesion-level assignment consistency
    # ------------------------------------------------------------

    lesion_assignment_counts = (
        df.groupby("lesion_id")[
            "hair_present"
        ].nunique()
    )

    assert (
        lesion_assignment_counts.max()
        == 1
    )

    assert (
        len(lesion_assignment_counts)
        == EXPECTED_TRAIN_LESIONS
    )

    print(
        "Lesion-level consistency: PASS"
    )

    # ------------------------------------------------------------
    # Conditional probabilities
    # ------------------------------------------------------------

    p_h_given_m = (
        malignant_hair
        / int(
            (df["malignant"] == 1).sum()
        )
    )

    p_h_given_b = (
        benign_hair
        / int(
            (df["malignant"] == 0).sum()
        )
    )

    p_h = (
        total_hair
        / len(df)
    )

    assert abs(
        p_h - 0.5
    ) < 1e-12

    print(
        "P(H|M):",
        f"{p_h_given_m:.12f}"
    )

    print(
        "P(H|B):",
        f"{p_h_given_b:.12f}"
    )

    print(
        "P(H):",
        f"{p_h:.12f}"
    )

    # ------------------------------------------------------------
    # Record
    # ------------------------------------------------------------

    audit_records.append({
        "rho": EXPECTED_RHO,
        "replicate": replicate,
        "assignment_seed": EXPECTED_SEEDS[replicate],
        "rows": len(df),
        "unique_images": df["image"].nunique(),
        "unique_lesions": df["lesion_id"].nunique(),
        "malignant_hair": malignant_hair,
        "benign_hair": benign_hair,
        "total_hair": total_hair,
        "p_h_given_m": p_h_given_m,
        "p_h_given_b": p_h_given_b,
        "p_h": p_h,
        "sha256": sha256,
        "population_signature": population_signature,
    })


# ================================================================
# REPLICATE INDEPENDENCE AUDIT
# ================================================================

print("\n" + "=" * 70)
print("REPLICATE INDEPENDENCE")
print("=" * 70)


rep1_hair = set(
    loaded_assignments[1].loc[
        loaded_assignments[1]["hair_present"] == 1,
        "lesion_id"
    ]
)

for replicate in [2, 3]:

    other_hair = set(
        loaded_assignments[replicate].loc[
            loaded_assignments[replicate]["hair_present"] == 1,
            "lesion_id"
        ]
    )

    intersection = len(
        rep1_hair
        .intersection(other_hair)
    )

    union = len(
        rep1_hair
        .union(other_hair)
    )

    symmetric_difference = len(
        rep1_hair.symmetric_difference(
            other_hair
        )
    )

    jaccard = (
        intersection / union
    )

    print(
        f"rep1 vs rep{replicate}:"
    )

    print(
        f"  intersection: {intersection:,}"
    )

    print(
        f"  union:        {union:,}"
    )

    print(
        f"  symmetric diff: {symmetric_difference:,}"
    )

    print(
        f"  Jaccard:      {jaccard:.6f}"
    )

    assert (
        symmetric_difference > 0
    )

    assert (
        rep1_hair
        != other_hair
    )


# ================================================================
# COMPARE ALL THREE SEEDS
# ================================================================

audit_df = pd.DataFrame(
    audit_records
)

assert (
    audit_df["assignment_seed"].nunique()
    == 3
)

assert (
    audit_df["sha256"].nunique()
    == 3
)

assert (
    audit_df["population_signature"].nunique()
    == 1
)


# ================================================================
# VERIFY EXISTING LOCKED ASSIGNMENT DIRECTORY
# ================================================================

locked_files = sorted(
    ASSIGN_DIR.glob(
        "rho_*_rep_*.csv"
    )
)

assert len(locked_files) == 30

locked_names = {
    p.name
    for p in locked_files
}

assert (
    "rho_0.5_rep_1.csv"
    not in locked_names
)

assert (
    "rho_0.5_rep_2.csv"
    not in locked_names
)

assert (
    "rho_0.5_rep_3.csv"
    not in locked_names
)

print("\nExisting locked Step1F manifests:", 30)
print(
    "rho=0.5 absent from immutable locked directory: PASS"
)


# ================================================================
# FINAL SUMMARY
# ================================================================

print("\n" + "=" * 70)
print("FINAL rho=0.5 ASSIGNMENT AUDIT")
print("=" * 70)

print(
    audit_df[
        [
            "rho",
            "replicate",
            "assignment_seed",
            "rows",
            "unique_images",
            "unique_lesions",
            "malignant_hair",
            "benign_hair",
            "total_hair",
            "p_h_given_m",
            "p_h_given_b",
            "p_h",
            "sha256",
        ]
    ].to_string(
        index=False
    )
)


# ================================================================
# SAVE AUDIT
# ================================================================

audit_path = (
    NEW_ASSIGN_DIR
    / "T2_1_RHO05_ASSIGNMENT_INTEGRITY_AUDIT.csv"
)

audit_df.to_csv(
    audit_path,
    index=False
)

print("\nAudit saved:")
print(audit_path)


# ================================================================
# FINAL GATE
# ================================================================

assert len(audit_records) == 3

assert all(
    r["rows"]
    == EXPECTED_TRAIN_IMAGES
    for r in audit_records
)

assert all(
    r["unique_images"]
    == EXPECTED_TRAIN_IMAGES
    for r in audit_records
)

assert all(
    r["unique_lesions"]
    == EXPECTED_TRAIN_LESIONS
    for r in audit_records
)

assert all(
    r["malignant_hair"]
    == EXPECTED_MALIGNANT_HAIR
    for r in audit_records
)

assert all(
    r["benign_hair"]
    == EXPECTED_BENIGN_HAIR
    for r in audit_records
)

assert all(
    r["total_hair"]
    == EXPECTED_TOTAL_HAIR
    for r in audit_records
)

assert (
    audit_df["population_signature"].nunique()
    == 1
)

assert (
    audit_df["sha256"].nunique()
    == 3
)

print("\n" + "=" * 70)
print("PASS — FINAL rho=0.5 ASSIGNMENT INTEGRITY GATE")
print("=" * 70)

print(
    "All 3 rho=0.5 manifests passed population, "
    "schema, target, lesion, seed, artifact, "
    "and hash audits."
)

print(
    "Existing 30 locked Step1F manifests remain untouched."
)

print(
    "READY FOR T2.1 CONVNEXT-TINY TRAINING."
)

print("=" * 70)
print("CELL 6R COMPLETE")
print("=" * 70)

In [ ]:
# ================================================================
# T2.1 — CELL 6S
# PRE-TRAINING ENVIRONMENT COMPATIBILITY GATE
#
# Purpose:
#   Compare the CURRENT Kaggle runtime against the FROZEN
#   Step2X training environment.
#
# NO installation
# NO package modification
# NO training
# ================================================================

from pathlib import Path
import sys
import json
import platform
import subprocess
import torch
import torchvision


print("=" * 70)
print("T2.1 — CELL 6S")
print("PRE-TRAINING ENVIRONMENT COMPATIBILITY GATE")
print("=" * 70)


# ================================================================
# CURRENT PYTHON
# ================================================================

print("\nPYTHON")
print("-" * 70)

print(
    "Python version:",
    platform.python_version()
)

print(
    "Python executable:",
    sys.executable
)


# ================================================================
# CURRENT TORCH
# ================================================================

print("\nPYTORCH")
print("-" * 70)

print(
    "torch:",
    torch.__version__
)

print(
    "torchvision:",
    torchvision.__version__
)

print(
    "CUDA available:",
    torch.cuda.is_available()
)

print(
    "torch CUDA:",
    torch.version.cuda
)

print(
    "cuDNN:",
    torch.backends.cudnn.version()
)


# ================================================================
# GPU
# ================================================================

print("\nGPU")
print("-" * 70)

gpu_count = torch.cuda.device_count()

print(
    "GPU count:",
    gpu_count
)

for i in range(gpu_count):

    print(
        f"GPU {i}:",
        torch.cuda.get_device_name(i)
    )

    print(
        f"GPU {i} capability:",
        torch.cuda.get_device_capability(i)
    )


# ================================================================
# DETERMINISM SETTINGS
# ================================================================

print("\nDETERMINISM")
print("-" * 70)

print(
    "deterministic algorithms:",
    torch.are_deterministic_algorithms_enabled()
)

print(
    "cuDNN benchmark:",
    torch.backends.cudnn.benchmark
)

print(
    "cuDNN deterministic:",
    torch.backends.cudnn.deterministic
)

print(
    "cuDNN enabled:",
    torch.backends.cudnn.enabled
)


# ================================================================
# PRECISION
# ================================================================

print("\nPRECISION")
print("-" * 70)

print(
    "default dtype:",
    torch.get_default_dtype()
)

print(
    "autocast CUDA available:",
    hasattr(torch, "autocast")
)


# ================================================================
# CUBLAS CONFIGURATION
# ================================================================

print("\nCUBLAS")
print("-" * 70)

print(
    "CUBLAS_WORKSPACE_CONFIG:",
    __import__("os").environ.get(
        "CUBLAS_WORKSPACE_CONFIG"
    )
)


# ================================================================
# FROZEN ENVIRONMENT
# ================================================================

FROZEN = {
    "python": "3.13.15",
    "torch": "2.11.0+cu128",
    "torchvision": "0.26.0+cu128",
    "cuda": "12.8",
    "gpu": "Tesla T4",
    "gpu_count": 1,
    "deterministic_algorithms": True,
    "cudnn_benchmark": False,
    "cudnn_deterministic": True,
    "cudnn_enabled": True,
    "precision": "float32",
    "mixed_precision": False,
    "cublas_workspace_config": ":4096:8",
}


# ================================================================
# CURRENT ENVIRONMENT RECORD
# ================================================================

CURRENT = {
    "python": platform.python_version(),
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "cuda": torch.version.cuda,
    "gpu": (
        torch.cuda.get_device_name(0)
        if gpu_count > 0
        else None
    ),
    "gpu_count": gpu_count,
    "deterministic_algorithms": (
        torch.are_deterministic_algorithms_enabled()
    ),
    "cudnn_benchmark": (
        torch.backends.cudnn.benchmark
    ),
    "cudnn_deterministic": (
        torch.backends.cudnn.deterministic
    ),
    "cudnn_enabled": (
        torch.backends.cudnn.enabled
    ),
    "precision": str(
        torch.get_default_dtype()
    ),
    "mixed_precision": False,
    "cublas_workspace_config": (
        __import__("os").environ.get(
            "CUBLAS_WORKSPACE_CONFIG"
        )
    ),
}


# ================================================================
# COMPARISON
# ================================================================

print("\n" + "=" * 70)
print("FROZEN vs CURRENT")
print("=" * 70)

comparison = []

for key in FROZEN:

    frozen_value = FROZEN[key]
    current_value = CURRENT[key]

    exact = (
        frozen_value
        == current_value
    )

    comparison.append({
        "parameter": key,
        "frozen": frozen_value,
        "current": current_value,
        "exact": exact,
    })

comparison_df = __import__(
    "pandas"
).DataFrame(
    comparison
)

print(
    comparison_df.to_string(
        index=False
    )
)


# ================================================================
# SAVE DIAGNOSTIC
# ================================================================

env_path = (
    Path("/kaggle/working")
    / "T2_1_current_environment_diagnostic.json"
)

with open(
    env_path,
    "w"
) as f:

    json.dump(
        {
            "frozen": FROZEN,
            "current": CURRENT,
            "comparison": comparison,
        },
        f,
        indent=2,
        default=str
    )

print("\nDiagnostic saved:")
print(env_path)


# ================================================================
# DO NOT FAIL YET
#
# We intentionally do NOT assert exact equality here.
# We first need to see whether the Kaggle runtime can be brought
# into the frozen environment without compromising the experiment.
# ================================================================

exact_matches = sum(
    row["exact"]
    for row in comparison
)

total_parameters = len(
    comparison
)

print("\n" + "=" * 70)
print("ENVIRONMENT GATE RESULT")
print("=" * 70)

print(
    f"Exact matches: "
    f"{exact_matches}/{total_parameters}"
)

print(
    "\nNO TRAINING WAS PERFORMED."
)

print(
    "NO PACKAGES WERE MODIFIED."
)

print("=" * 70)
print("CELL 6S COMPLETE")
print("=" * 70)

In [ ]:
# ================================================================
# T2.1 — CELL 6T
# ENVIRONMENT REPRODUCTION FEASIBILITY AUDIT
#
# Purpose:
#   Determine whether the exact frozen PyTorch / torchvision
#   versions are available through the current Kaggle environment
#   and whether CUDA compatibility is suitable.
#
# NO installation
# NO package modification
# NO training
# ================================================================

from pathlib import Path
import sys
import os
import subprocess
import json
import importlib.util

import torch
import torchvision


print("=" * 70)
print("T2.1 — CELL 6T")
print("ENVIRONMENT REPRODUCTION FEASIBILITY AUDIT")
print("=" * 70)


# ================================================================
# CURRENT EXECUTABLE / PIP
# ================================================================

print("\nEXECUTABLES")
print("-" * 70)

print("Python:", sys.executable)

pip_path = (
    subprocess.check_output(
        [sys.executable, "-m", "pip", "--version"],
        text=True
    ).strip()
)

print("pip:", pip_path)


# ================================================================
# CURRENT INSTALLED PACKAGE METADATA
# ================================================================

print("\nINSTALLED PACKAGE METADATA")
print("-" * 70)

packages = [
    "torch",
    "torchvision",
    "torchaudio",
    "numpy",
    "pandas",
    "scipy",
    "Pillow",
]

package_metadata = {}

for pkg in packages:

    try:

        result = subprocess.run(
            [
                sys.executable,
                "-m",
                "pip",
                "show",
                pkg,
            ],
            capture_output=True,
            text=True,
            check=False,
        )

        print(f"\n[{pkg}]")

        if result.returncode == 0:

            print(
                result.stdout.strip()
            )

            metadata = {}

            for line in result.stdout.splitlines():

                if ":" in line:

                    k, v = line.split(
                        ":",
                        1
                    )

                    metadata[
                        k.strip()
                    ] = v.strip()

            package_metadata[
                pkg
            ] = metadata

        else:

            print("NOT INSTALLED")

            package_metadata[
                pkg
            ] = None

    except Exception as e:

        print(
            "ERROR:",
            repr(e)
        )

        package_metadata[
            pkg
        ] = None


# ================================================================
# PIP INDEX / NETWORK CONFIGURATION
# ================================================================

print("\nPIP CONFIGURATION")
print("-" * 70)

try:

    pip_config = subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "config",
            "list",
        ],
        capture_output=True,
        text=True,
        check=False,
    )

    print(
        pip_config.stdout.strip()
        if pip_config.stdout.strip()
        else "(no explicit pip config reported)"
    )

except Exception as e:

    print(
        "Could not inspect pip config:",
        repr(e)
    )


# ================================================================
# CUDA / DRIVER INFORMATION
# ================================================================

print("\nCUDA / DRIVER")
print("-" * 70)

print(
    "torch.version.cuda:",
    torch.version.cuda
)

print(
    "CUDA available:",
    torch.cuda.is_available()
)

print(
    "GPU count:",
    torch.cuda.device_count()
)

for i in range(
    torch.cuda.device_count()
):

    print(
        f"GPU {i}:",
        torch.cuda.get_device_name(i)
    )

    print(
        f"GPU {i} capability:",
        torch.cuda.get_device_capability(i)
    )


try:

    nvidia_smi = subprocess.run(
        [
            "nvidia-smi",
            "--query-gpu=name,driver_version,memory.total",
            "--format=csv,noheader",
        ],
        capture_output=True,
        text=True,
        check=False,
    )

    print(
        "\nnvidia-smi:"
    )

    print(
        nvidia_smi.stdout.strip()
    )

except Exception as e:

    print(
        "nvidia-smi unavailable:",
        repr(e)
    )


# ================================================================
# FROZEN TARGET
# ================================================================

TARGET = {
    "python": "3.13.15",
    "torch": "2.11.0+cu128",
    "torchvision": "0.26.0+cu128",
    "cuda": "12.8",
}


print("\n" + "=" * 70)
print("FROZEN TARGET")
print("=" * 70)

for k, v in TARGET.items():

    print(
        f"{k}: {v}"
    )


# ================================================================
# CURRENT
# ================================================================

CURRENT = {
    "python": (
        f"{sys.version_info.major}."
        f"{sys.version_info.minor}."
        f"{sys.version_info.micro}"
    ),
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "cuda": torch.version.cuda,
}


print("\n" + "=" * 70)
print("CURRENT")
print("=" * 70)

for k, v in CURRENT.items():

    print(
        f"{k}: {v}"
    )


# ================================================================
# LOOK FOR LOCAL WHEELS / CACHED PACKAGES
# ================================================================

print("\n" + "=" * 70)
print("LOCAL PACKAGE / WHEEL SEARCH")
print("=" * 70)

search_roots = [
    "/kaggle/input",
    "/kaggle/working",
    "/opt/conda/pkgs",
    "/usr/local/lib/python3.12/site-packages",
]

patterns = [
    "torch-2.11",
    "torchvision-0.26",
    "torch-2.11.0",
    "torchvision-0.26.0",
]

found = []

for root in search_roots:

    if not os.path.exists(root):
        continue

    print(
        f"\nScanning: {root}"
    )

    try:

        for dirpath, dirnames, filenames in os.walk(
            root
        ):

            # Avoid huge/deep scans through common cache trees.
            dirnames[:] = [
                d for d in dirnames
                if d not in {
                    ".git",
                    "__pycache__",
                    "node_modules",
                }
            ]

            for filename in filenames:

                lower = filename.lower()

                if any(
                    pattern.lower()
                    in lower
                    for pattern in patterns
                ):

                    full_path = os.path.join(
                        dirpath,
                        filename
                    )

                    found.append(
                        full_path
                    )

                    print(
                        full_path
                    )

                    if len(found) >= 100:
                        break

            if len(found) >= 100:
                break

    except Exception as e:

        print(
            "Scan error:",
            repr(e)
        )


if not found:

    print(
        "\nNo matching local torch/torchvision "
        "2.11/0.26 artifacts found."
    )

else:

    print(
        f"\nFound {len(found)} matching artifact(s)."
    )


# ================================================================
# WRITE DIAGNOSTIC JSON
# ================================================================

diagnostic = {
    "target": TARGET,
    "current": CURRENT,
    "package_metadata": package_metadata,
    "local_matching_artifacts": found,
}

output_path = (
    Path("/kaggle/working")
    / "T2_1_environment_reproduction_feasibility.json"
)

with open(
    output_path,
    "w"
) as f:

    json.dump(
        diagnostic,
        f,
        indent=2,
        default=str
    )

print("\nDiagnostic saved:")
print(output_path)


print("\n" + "=" * 70)
print("CELL 6T COMPLETE")
print("NO INSTALLATION PERFORMED")
print("NO TRAINING PERFORMED")
print("=" * 70)

In [ ]:

# ================================================================
# T2.1 — CELL 6U
# FROZEN PACKAGE AVAILABILITY GATE
#
# Purpose:
#   Determine whether the exact frozen PyTorch and torchvision
#   versions are available from the configured package index.
#
# NO INSTALLATION
# NO PACKAGE MODIFICATION
# NO TRAINING
# ================================================================

import sys
import subprocess
import json
from pathlib import Path


print("=" * 70)
print("T2.1 — CELL 6U")
print("FROZEN PACKAGE AVAILABILITY GATE")
print("=" * 70)


TARGETS = {
    "torch": "2.11.0+cu128",
    "torchvision": "0.26.0+cu128",
}


results = {}


# ================================================================
# CHECK PIP INDEX
# ================================================================

for package, version in TARGETS.items():

    print("\n" + "-" * 70)
    print(f"CHECKING: {package} == {version}")
    print("-" * 70)

    command = [
        sys.executable,
        "-m",
        "pip",
        "index",
        "versions",
        package,
    ]

    result = subprocess.run(
        command,
        capture_output=True,
        text=True,
        check=False,
    )

    print("Return code:", result.returncode)

    if result.stdout.strip():
        print("\nSTDOUT:")
        print(result.stdout.strip())

    if result.stderr.strip():
        print("\nSTDERR:")
        print(result.stderr.strip())

    combined = (
        result.stdout
        + "\n"
        + result.stderr
    )

    exact_present = (
        version in combined
    )

    results[package] = {
        "requested_version": version,
        "return_code": result.returncode,
        "exact_version_text_found": exact_present,
        "stdout": result.stdout,
        "stderr": result.stderr,
    }

    print(
        "\nExact requested version text found:",
        exact_present
    )


# ================================================================
# CHECK PYTHON 3.13 AVAILABILITY
# ================================================================

print("\n" + "=" * 70)
print("PYTHON 3.13 CHECK")
print("=" * 70)

python_candidates = []

for candidate in [
    "/usr/bin/python3.13",
    "/usr/local/bin/python3.13",
    "/opt/conda/bin/python3.13",
]:

    exists = Path(candidate).exists()

    print(
        f"{candidate}:",
        "FOUND" if exists else "not found"
    )

    if exists:
        python_candidates.append(candidate)


# ================================================================
# SAVE RESULT
# ================================================================

output = {
    "current_python": sys.version,
    "current_executable": sys.executable,
    "targets": TARGETS,
    "package_checks": results,
    "python_313_candidates": python_candidates,
}


output_path = (
    Path("/kaggle/working")
    / "T2_1_frozen_package_availability.json"
)

with open(
    output_path,
    "w"
) as f:

    json.dump(
        output,
        f,
        indent=2,
        default=str
    )


print("\n" + "=" * 70)
print("RESULT SAVED")
print("=" * 70)

print(output_path)

print("\nNO INSTALLATION PERFORMED.")
print("NO PACKAGE MODIFICATION PERFORMED.")
print("NO TRAINING PERFORMED.")

print("=" * 70)
print("CELL 6U COMPLETE")
print("=" * 70)

In [ ]:
# ================================================================
# T2.1 — CELL 6V
# CONTROLLED PYTORCH ENVIRONMENT TRANSITION
#
# Installs ONLY:
#   torch==2.11.0+cu128
#   torchvision==0.26.0+cu128
#
# Python remains 3.12.13.
#
# NO TRAINING
# ================================================================

import sys
import subprocess

print("=" * 70)
print("T2.1 — CELL 6V")
print("CONTROLLED PYTORCH ENVIRONMENT TRANSITION")
print("=" * 70)

print("\nCurrent Python:")
print(sys.version)

print("\nCurrent packages:")
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "show",
        "torch",
        "torchvision",
    ],
    check=False,
)

print("\nInstalling exact CUDA 12.8 builds:")
print("  torch==2.11.0")
print("  torchvision==0.26.0")

command = [
    sys.executable,
    "-m",
    "pip",
    "install",
    "--no-cache-dir",
    "--force-reinstall",
    "torch==2.11.0",
    "torchvision==0.26.0",
    "--index-url",
    "https://download.pytorch.org/whl/cu128",
]

print("\nRunning:")
print(" ".join(command))

result = subprocess.run(
    command,
    check=False,
)

print("\n" + "=" * 70)

if result.returncode != 0:
    raise RuntimeError(
        "PyTorch environment transition FAILED. "
        "Do NOT continue to training."
    )

print("INSTALL COMMAND COMPLETED")
print("=" * 70)

print("""
IMPORTANT:
The Python kernel must now be RESTARTED before importing torch again.

Do NOT run any training cell in this kernel before restarting.
""")

In [ ]:
# ================================================================
# T2.1 — CELL 6V-R1
# CONTROLLED PYTORCH ENVIRONMENT TRANSITION
#
# IMPORTANT:
#   DO NOT import torch before installation.
#   We use importlib.metadata only.
#
# Target:
#   torch       2.11.0+cu128
#   torchvision 0.26.0+cu128
#
# Python 3.12.13 remains a documented deviation from the
# historical frozen Python 3.13.15 environment.
#
# NO TRAINING
# ================================================================

import sys
import os
import subprocess
import importlib.metadata
from pathlib import Path
import json


print("=" * 70)
print("T2.1 — CELL 6V-R1")
print("CONTROLLED PYTORCH ENVIRONMENT TRANSITION")
print("=" * 70)


# ================================================================
# SAFETY CHECK
# ================================================================

if "torch" in sys.modules:
    raise RuntimeError(
        "ABORT: torch is already imported in this kernel.\n"
        "Do NOT continue installation in this kernel.\n"
        "Restart the kernel and run this cell FIRST."
    )

if "torchvision" in sys.modules:
    raise RuntimeError(
        "ABORT: torchvision is already imported in this kernel.\n"
        "Restart the kernel and run this cell FIRST."
    )


# ================================================================
# CURRENT METADATA — WITHOUT IMPORTING TORCH
# ================================================================

def installed_version(package_name):

    try:
        return importlib.metadata.version(
            package_name
        )
    except importlib.metadata.PackageNotFoundError:
        return None


before = {
    "python": sys.version,
    "python_executable": sys.executable,
    "torch": installed_version("torch"),
    "torchvision": installed_version("torchvision"),
    "torchaudio": installed_version("torchaudio"),
    "numpy": installed_version("numpy"),
    "Pillow": installed_version("Pillow"),
    "scipy": installed_version("scipy"),
    "pandas": installed_version("pandas"),
}


print("\nPRE-INSTALL METADATA")
print("-" * 70)

for key, value in before.items():
    print(
        f"{key}: {value}"
    )


# ================================================================
# TARGET
# ================================================================

TORCH_TARGET = "2.11.0"
TORCHVISION_TARGET = "0.26.0"

print("\n" + "=" * 70)
print("TARGET")
print("=" * 70)

print(
    "torch:",
    TORCH_TARGET,
    "+cu128"
)

print(
    "torchvision:",
    TORCHVISION_TARGET,
    "+cu128"
)

print(
    "Python:",
    f"{sys.version_info.major}."
    f"{sys.version_info.minor}."
    f"{sys.version_info.micro}"
)

print(
    "\nPython 3.12 is intentionally retained."
)

print(
    "Historical frozen Python was 3.13.15."
)

print(
    "This will be recorded as an explicit "
    "environment deviation."
)


# ================================================================
# INSTALL EXACT CUDA 12.8 BUILDS
# ================================================================

command = [
    sys.executable,
    "-m",
    "pip",
    "install",
    "--no-cache-dir",
    "--force-reinstall",
    "--index-url",
    "https://download.pytorch.org/whl/cu128",
    "torch==2.11.0",
    "torchvision==0.26.0",
]


print("\n" + "=" * 70)
print("INSTALL COMMAND")
print("=" * 70)

print(
    " ".join(command)
)


print("\nStarting installation...")

result = subprocess.run(
    command,
    check=False,
)


# ================================================================
# RESULT
# ================================================================

print("\n" + "=" * 70)

if result.returncode != 0:

    print(
        "INSTALLATION FAILED"
    )

    raise RuntimeError(
        "PyTorch environment transition failed.\n"
        "Do NOT train."
    )

print(
    "INSTALLATION COMMAND COMPLETED"
)

print("=" * 70)


# ================================================================
# SAVE PRE-INSTALL RECORD
# ================================================================

record_path = (
    Path("/kaggle/working")
    / "T2_1_environment_transition_preinstall.json"
)

with open(
    record_path,
    "w"
) as f:

    json.dump(
        {
            "before": before,
            "target": {
                "torch": "2.11.0+cu128",
                "torchvision": "0.26.0+cu128",
            },
            "python_deviation": {
                "historical": "3.13.15",
                "current": (
                    f"{sys.version_info.major}."
                    f"{sys.version_info.minor}."
                    f"{sys.version_info.micro}"
                ),
                "status": "explicit_deviation",
            },
            "installation_return_code": (
                result.returncode
            ),
        },
        f,
        indent=2,
        default=str
    )


print("\nTransition record saved:")
print(record_path)


# ================================================================
# CRITICAL
# ================================================================

print("\n" + "=" * 70)

print(
    "IMPORTANT:"
)

print(
    "1. DO NOT import torch in this kernel."
)

print(
    "2. RESTART the Kaggle kernel/session now."
)

print(
    "3. After restart, run CELL 6W."
)

print(
    "4. Do NOT train before 6W passes."
)

print("=" * 70)
print("CELL 6V-R1 COMPLETE")
print("=" * 70)

In [ ]:
=====================================================================
INSTALLATION COMMAND COMPLETED
======================================================================

Transition record saved:
/kaggle/working/T2_1_environment_transition_preinstall.json

======================================================================
IMPORTANT:
1. DO NOT import torch in this kernel.
2. RESTART the Kaggle kernel/session now.
3. After restart, run CELL 6W.
4. Do NOT train before 6W passes.
======================================================================
CELL 6V-R1 COMPLETE
======================================================================

In [ ]:
# ================================================================
# T2.1 — CELL 6W
# POST-RESTART ENVIRONMENT + CONVNEXT METADATA GATE
#
# HARD GATE BEFORE ANY TRAINING
#
# Verifies:
#   - Python
#   - torch
#   - torchvision
#   - CUDA
#   - GPU visibility
#   - deterministic settings
#   - CUBLAS configuration
#   - ConvNeXt-Tiny weights
#   - classifier structure
#   - mean-pool substitution requirement
#   - dependency versions before/after
#
# NO TRAINING
# ================================================================

from pathlib import Path
import sys
import os
import json
import importlib.metadata

import torch
import torchvision
import torch.nn as nn


print("=" * 70)
print("T2.1 — CELL 6W")
print("POST-RESTART ENVIRONMENT + CONVNEXT METADATA GATE")
print("=" * 70)


# ================================================================
# FROZEN / TARGET VALUES
# ================================================================

TARGET = {
    "torch": "2.11.0+cu128",
    "torchvision": "0.26.0+cu128",
    "cuda": "12.8",
    "gpu": "Tesla T4",
    "gpu_count_per_process": 1,
    "deterministic_algorithms": True,
    "cudnn_benchmark": False,
    "cudnn_deterministic": True,
    "cudnn_enabled": True,
    "default_dtype": "torch.float32",
    "mixed_precision": False,
    "cublas_workspace_config": ":4096:8",
}


# ================================================================
# BASIC ENVIRONMENT
# ================================================================

print("\nPYTHON")
print("-" * 70)

python_version = (
    f"{sys.version_info.major}."
    f"{sys.version_info.minor}."
    f"{sys.version_info.micro}"
)

print("Python:", python_version)
print("Executable:", sys.executable)


print("\nPYTORCH")
print("-" * 70)

print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("CUDA:", torch.version.cuda)
print("cuDNN:", torch.backends.cudnn.version())


# ================================================================
# GPU
# ================================================================

print("\nGPU")
print("-" * 70)

gpu_count = torch.cuda.device_count()

print("GPU count visible to this process:", gpu_count)

for i in range(gpu_count):

    print(
        f"GPU {i}:",
        torch.cuda.get_device_name(i)
    )

    print(
        f"GPU {i} capability:",
        torch.cuda.get_device_capability(i)
    )


# ================================================================
# DETERMINISM
# ================================================================

print("\nDETERMINISM")
print("-" * 70)

deterministic_algorithms = (
    torch.are_deterministic_algorithms_enabled()
)

cudnn_benchmark = (
    torch.backends.cudnn.benchmark
)

cudnn_deterministic = (
    torch.backends.cudnn.deterministic
)

cudnn_enabled = (
    torch.backends.cudnn.enabled
)

print(
    "deterministic algorithms:",
    deterministic_algorithms
)

print(
    "cuDNN benchmark:",
    cudnn_benchmark
)

print(
    "cuDNN deterministic:",
    cudnn_deterministic
)

print(
    "cuDNN enabled:",
    cudnn_enabled
)


# ================================================================
# PRECISION
# ================================================================

print("\nPRECISION")
print("-" * 70)

default_dtype = str(
    torch.get_default_dtype()
)

print(
    "default dtype:",
    default_dtype
)

print(
    "mixed precision:",
    False
)


# ================================================================
# CUBLAS
# ================================================================

print("\nCUBLAS")
print("-" * 70)

cublas_config = os.environ.get(
    "CUBLAS_WORKSPACE_CONFIG"
)

print(
    "CUBLAS_WORKSPACE_CONFIG:",
    cublas_config
)


# ================================================================
# CONVNEXT-TINY WEIGHTS
# ================================================================

print("\nCONVNEXT-TINY")
print("-" * 70)

weights_enum = (
    torchvision.models.ConvNeXt_Tiny_Weights
)

weights = (
    weights_enum.IMAGENET1K_V1
)

print(
    "Weights enum:",
    weights
)

print(
    "Weights enum name:",
    weights.name
)

print(
    "Weights URL:",
    weights.url
)


# ================================================================
# CONVNEXT MODEL STRUCTURE
# ================================================================

model = (
    torchvision.models.convnext_tiny(
        weights=weights
    )
)

print(
    "\nOriginal classifier:"
)

print(
    model.classifier
)


# ================================================================
# CLASSIFIER CHECK
# ================================================================

classifier_before = {
    "type": type(
        model.classifier
    ).__name__,
    "length": len(
        model.classifier
    ),
    "final_type": type(
        model.classifier[2]
    ).__name__,
    "final_in_features": (
        model.classifier[2].in_features
        if hasattr(
            model.classifier[2],
            "in_features"
        )
        else None
    ),
    "final_out_features": (
        model.classifier[2].out_features
        if hasattr(
            model.classifier[2],
            "out_features"
        )
        else None
    ),
}


print(
    "\nClassifier metadata:"
)

for k, v in classifier_before.items():

    print(
        f"{k}: {v}"
    )


# ================================================================
# APPLY THE LOCKED CLASSIFIER REPLACEMENT
# ================================================================

in_features = (
    model.classifier[2].in_features
)

model.classifier[2] = nn.Linear(
    in_features,
    2
)


print(
    "\nClassifier after 2-class replacement:"
)

print(
    model.classifier
)


# ================================================================
# VERIFY MEAN-POOL TARGET
# ================================================================

print(
    "\nMean-pool / avgpool inspection:"
)

print(
    "model.avgpool:",
    model.avgpool
)

avgpool_type = type(
    model.avgpool
).__name__

print(
    "avgpool type:",
    avgpool_type
)


# ================================================================
# PARAMETER COUNT
# ================================================================

parameter_count = sum(
    p.numel()
    for p in model.parameters()
)

print(
    "\nConvNeXt-Tiny parameter count:",
    parameter_count
)


# ================================================================
# PACKAGE DEPENDENCY SNAPSHOT
# ================================================================

tracked_packages = [
    "torch",
    "torchvision",
    "numpy",
    "Pillow",
    "scipy",
    "pandas",
]

print(
    "\nPACKAGE SNAPSHOT"
)
print("-" * 70)

dependency_snapshot = {}

for package in tracked_packages:

    try:

        version = (
            importlib.metadata.version(
                package
            )
        )

    except Exception:

        version = None

    dependency_snapshot[
        package
    ] = version

    print(
        f"{package}: {version}"
    )


# ================================================================
# LOAD PRE-INSTALL RECORD
# ================================================================

preinstall_path = (
    Path("/kaggle/working")
    / "T2_1_environment_transition_preinstall.json"
)

preinstall = None

if preinstall_path.exists():

    with open(
        preinstall_path,
        "r"
    ) as f:

        preinstall = json.load(f)

    print(
        "\nPre-install transition record: FOUND"
    )

else:

    print(
        "\nWARNING: pre-install transition record "
        "not found."
    )


# ================================================================
# DEPENDENCY DIFF
# ================================================================

print(
    "\nDEPENDENCY DIFF"
)
print("-" * 70)

dependency_diff = {}

if preinstall is not None:

    before = preinstall.get(
        "before",
        {}
    )

    for package in tracked_packages:

        old = before.get(
            package
        )

        new = dependency_snapshot.get(
            package
        )

        changed = (
            old != new
        )

        dependency_diff[
            package
        ] = {
            "before": old,
            "after": new,
            "changed": changed,
        }

        print(
            f"{package}: "
            f"{old} -> {new} "
            f"{'[CHANGED]' if changed else '[unchanged]'}"
        )


# ================================================================
# HARD GATE CHECKS
# ================================================================

print(
    "\n" + "=" * 70
)
print(
    "HARD GATE CHECKS"
)
print(
    "=" * 70
)

checks = {}


# Exact Torch pair
checks[
    "torch_exact"
] = (
    torch.__version__
    == TARGET["torch"]
)

checks[
    "torchvision_exact"
] = (
    torchvision.__version__
    == TARGET["torchvision"]
)

checks[
    "cuda_exact"
] = (
    torch.version.cuda
    == TARGET["cuda"]
)

checks[
    "gpu_available"
] = (
    torch.cuda.is_available()
)

checks[
    "gpu_is_t4"
] = (
    gpu_count >= 1
    and all(
        "Tesla T4"
        in torch.cuda.get_device_name(i)
        for i in range(gpu_count)
    )
)

checks[
    "deterministic_algorithms"
] = (
    deterministic_algorithms
    == True
)

checks[
    "cudnn_benchmark"
] = (
    cudnn_benchmark
    == False
)

checks[
    "cudnn_deterministic"
] = (
    cudnn_deterministic
    == True
)

checks[
    "cudnn_enabled"
] = (
    cudnn_enabled
    == True
)

checks[
    "float32"
] = (
    default_dtype
    == "torch.float32"
)

checks[
    "cublas_config"
] = (
    cublas_config
    == TARGET[
        "cublas_workspace_config"
    ]
)

checks[
    "convnext_weights"
] = (
    weights.name
    == "IMAGENET1K_V1"
)

checks[
    "classifier_two_classes"
] = (
    model.classifier[2].out_features
    == 2
)

checks[
    "avgpool_present"
] = (
    model.avgpool is not None
)


for name, passed in checks.items():

    print(
        f"{name:35s}: "
        f"{'PASS' if passed else 'FAIL'}"
    )


# ================================================================
# IMPORTANT:
# GPU COUNT IS NOT A FAILURE HERE.
#
# The process may see 2 GPUs in this notebook, but the actual
# training launcher will pin each individual subprocess to ONE GPU.
# ================================================================

print(
    "\nGPU process-visibility note:"
)

print(
    "Current notebook-visible GPU count:",
    gpu_count
)

print(
    "Required training configuration:",
    "one GPU per training process"
)


# ================================================================
# SAVE ENVIRONMENT LOCK
# ================================================================

environment_record = {
    "t2_1_status": "PRE_TRAINING_GATE",
    "python": python_version,
    "python_historical_frozen": "3.13.15",
    "python_deviation": (
        python_version != "3.13.15"
    ),
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "cuda": torch.version.cuda,
    "cudnn": torch.backends.cudnn.version(),
    "gpu_count_visible_to_notebook": gpu_count,
    "gpu_names": [
        torch.cuda.get_device_name(i)
        for i in range(gpu_count)
    ],
    "deterministic_algorithms": (
        deterministic_algorithms
    ),
    "cudnn_benchmark": cudnn_benchmark,
    "cudnn_deterministic": cudnn_deterministic,
    "cudnn_enabled": cudnn_enabled,
    "default_dtype": default_dtype,
    "mixed_precision": False,
    "cublas_workspace_config": cublas_config,
    "convnext_weights_enum": weights.name,
    "convnext_classifier_out_features": (
        model.classifier[2].out_features
    ),
    "convnext_avgpool_type": avgpool_type,
    "convnext_parameter_count": parameter_count,
    "dependency_snapshot": dependency_snapshot,
    "dependency_diff": dependency_diff,
    "checks": checks,
}


environment_path = (
    Path("/kaggle/working")
    / "T2_1_POST_RESTART_ENVIRONMENT_LOCK.json"
)

with open(
    environment_path,
    "w"
) as f:

    json.dump(
        environment_record,
        f,
        indent=2,
        default=str
    )


print(
    "\nEnvironment record saved:"
)

print(
    environment_path
)


# ================================================================
# FINAL DECISION
# ================================================================

critical_checks = [
    "torch_exact",
    "torchvision_exact",
    "cuda_exact",
    "gpu_available",
    "gpu_is_t4",
    "deterministic_algorithms",
    "cudnn_benchmark",
    "cudnn_deterministic",
    "cudnn_enabled",
    "float32",
    "cublas_config",
    "convnext_weights",
    "classifier_two_classes",
    "avgpool_present",
]

all_pass = all(
    checks[k]
    for k in critical_checks
)


print(
    "\n" + "=" * 70
)

if all_pass:

    print(
        "PASS — POST-RESTART ENVIRONMENT GATE"
    )

    print(
        "\nThe required Torch/CUDA/torchvision "
        "stack is active."
    )

    print(
        "Python 3.12 vs historical 3.13 remains "
        "an explicit documented deviation."
    )

    print(
        "\nNEXT:"
    )

    print(
        "Run the ConvNeXt-specific deterministic "
        "dry-run gate before Model 1."
    )

else:

    print(
        "FAIL — DO NOT TRAIN"
    )

    failed = [
        k
        for k, v in checks.items()
        if not v
    ]

    print(
        "\nFailed checks:"
    )

    for item in failed:
        print(
            " -",
            item
        )

print(
    "=" * 70
)
print(
    "CELL 6W COMPLETE"
)
print(
    "=" * 70
)

In [ ]:
# ================================================================
# T2.1 — CELL 6X
# DETERMINISTIC CUDA CONFIGURATION GATE
#
# IMPORTANT:
#   Run this immediately after a fresh kernel restart.
#   Do NOT import torch before this cell.
#
# Purpose:
#   Establish the deterministic configuration BEFORE torch/CUDA
#   work and verify that PyTorch actually sees it.
#
# NO TRAINING
# ================================================================

import os
import sys
import json
from pathlib import Path


print("=" * 70)
print("T2.1 — CELL 6X")
print("DETERMINISTIC CUDA CONFIGURATION GATE")
print("=" * 70)


# ================================================================
# SET CUBLAS CONFIG BEFORE IMPORTING TORCH
# ================================================================

os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

print("\nCUBLAS_WORKSPACE_CONFIG set BEFORE torch import:")
print(
    os.environ.get(
        "CUBLAS_WORKSPACE_CONFIG"
    )
)


# ================================================================
# NOW IMPORT TORCH
# ================================================================

import torch
import torchvision


# ================================================================
# VERIFY ENVIRONMENT
# ================================================================

print("\nPYTORCH")
print("-" * 70)

print(
    "Python:",
    sys.version
)

print(
    "torch:",
    torch.__version__
)

print(
    "torchvision:",
    torchvision.__version__
)

print(
    "CUDA:",
    torch.version.cuda
)


# ================================================================
# ENABLE STRICT DETERMINISM
# ================================================================

torch.use_deterministic_algorithms(
    True,
    warn_only=False
)

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True


# ================================================================
# VERIFY
# ================================================================

print("\nDETERMINISM AFTER CONFIGURATION")
print("-" * 70)

print(
    "deterministic algorithms:",
    torch.are_deterministic_algorithms_enabled()
)

print(
    "cuDNN benchmark:",
    torch.backends.cudnn.benchmark
)

print(
    "cuDNN deterministic:",
    torch.backends.cudnn.deterministic
)

print(
    "cuDNN enabled:",
    torch.backends.cudnn.enabled
)

print(
    "CUBLAS_WORKSPACE_CONFIG:",
    os.environ.get(
        "CUBLAS_WORKSPACE_CONFIG"
    )
)


# ================================================================
# GPU
# ================================================================

print("\nGPU")
print("-" * 70)

print(
    "GPU count visible:",
    torch.cuda.device_count()
)

for i in range(
    torch.cuda.device_count()
):

    print(
        f"GPU {i}:",
        torch.cuda.get_device_name(i)
    )

    print(
        f"GPU {i} capability:",
        torch.cuda.get_device_capability(i)
    )


# ================================================================
# HARD CONFIGURATION CHECK
# ================================================================

checks = {

    "torch":
        torch.__version__
        == "2.11.0+cu128",

    "torchvision":
        torchvision.__version__
        == "0.26.0+cu128",

    "cuda":
        torch.version.cuda
        == "12.8",

    "deterministic_algorithms":
        torch.are_deterministic_algorithms_enabled()
        is True,

    "cudnn_benchmark":
        torch.backends.cudnn.benchmark
        is False,

    "cudnn_deterministic":
        torch.backends.cudnn.deterministic
        is True,

    "cublas_workspace":
        os.environ.get(
            "CUBLAS_WORKSPACE_CONFIG"
        )
        == ":4096:8",

    "float32":
        torch.get_default_dtype()
        == torch.float32,
}


print("\n" + "=" * 70)
print("CONFIGURATION CHECK")
print("=" * 70)

for name, passed in checks.items():

    print(
        f"{name:30s}: "
        f"{'PASS' if passed else 'FAIL'}"
    )


# ================================================================
# SAVE CONFIGURATION
# ================================================================

record = {
    "python": sys.version,
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "cuda": torch.version.cuda,
    "gpu_count": torch.cuda.device_count(),
    "gpu_names": [
        torch.cuda.get_device_name(i)
        for i in range(
            torch.cuda.device_count()
        )
    ],
    "deterministic_algorithms":
        torch.are_deterministic_algorithms_enabled(),
    "cudnn_benchmark":
        torch.backends.cudnn.benchmark,
    "cudnn_deterministic":
        torch.backends.cudnn.deterministic,
    "cudnn_enabled":
        torch.backends.cudnn.enabled,
    "cublas_workspace_config":
        os.environ.get(
            "CUBLAS_WORKSPACE_CONFIG"
        ),
    "default_dtype":
        str(torch.get_default_dtype()),
    "checks": checks,
}


output_path = (
    Path("/kaggle/working")
    / "T2_1_DETERMINISTIC_CONFIGURATION_LOCK.json"
)

with open(
    output_path,
    "w"
) as f:

    json.dump(
        record,
        f,
        indent=2,
        default=str
    )


print("\nConfiguration record saved:")
print(output_path)


# ================================================================
# FINAL GATE
# ================================================================

if not all(checks.values()):

    raise RuntimeError(
        "DETERMINISTIC CONFIGURATION GATE FAILED.\n"
        "DO NOT RUN THE CONVNEXT DRY RUN OR TRAINING."
    )


print("\n" + "=" * 70)
print(
    "PASS — DETERMINISTIC CONFIGURATION ESTABLISHED"
)
print("=" * 70)

print(
    "Next step: ConvNeXt-specific deterministic dry run."
)

print("=" * 70)
print("CELL 6X COMPLETE")
print("=" * 70)

In [ ]:
# ================================================================
# T2.1 — CELL 6Y
# CONVNEXT-TINY DETERMINISTIC DRY-RUN GATE
#
# PURPOSE:
#   1. Verify ConvNeXt-Tiny can execute under STRICT deterministic
#      CUDA settings without triggering a nondeterministic-op error.
#
#   2. Verify identical model initialization + identical input +
#      identical seed produces bit-identical outputs/loss/gradients.
#
#   3. Verify eval-mode ConvNeXt outputs are bit-identical across
#      repeated forward passes, confirming stochastic depth / DropPath
#      is inactive during evaluation.
#
#   4. Verify the locked mean-pool substitution works with the
#      ConvNeXt architecture.
#
# NO DATASET
# NO TRAINING
# NO CHECKPOINT
# NO MODEL-1 RUN
#
# This is a diagnostic/determinism gate only.
# ================================================================

import os
import sys
import json
import hashlib
from pathlib import Path

# ---------------------------------------------------------------
# Determinism environment must already be established by Cell 6X
# ---------------------------------------------------------------

assert os.environ.get(
    "CUBLAS_WORKSPACE_CONFIG"
) == ":4096:8", (
    "CUBLAS_WORKSPACE_CONFIG is not locked."
)

import torch
import torchvision
import torch.nn as nn


print("=" * 70)
print("T2.1 — CELL 6Y")
print("CONVNEXT-TINY DETERMINISTIC DRY-RUN GATE")
print("=" * 70)


# ================================================================
# HARD ENVIRONMENT ASSERTIONS
# ================================================================

assert torch.__version__ == "2.11.0+cu128"
assert torchvision.__version__ == "0.26.0+cu128"
assert torch.version.cuda == "12.8"

assert torch.are_deterministic_algorithms_enabled() is True
assert torch.backends.cudnn.deterministic is True
assert torch.backends.cudnn.benchmark is False

assert torch.get_default_dtype() == torch.float32

print("\nEnvironment assertions: PASS")


# ================================================================
# GPU
# ================================================================

assert torch.cuda.is_available()

device = torch.device("cuda:0")

print(
    "\nDry-run device:",
    torch.cuda.get_device_name(0)
)


# ================================================================
# GLOBAL SEED
# ================================================================

DRY_RUN_SEED = 20260919

torch.manual_seed(
    DRY_RUN_SEED
)

torch.cuda.manual_seed_all(
    DRY_RUN_SEED
)

print(
    "Dry-run seed:",
    DRY_RUN_SEED
)


# ================================================================
# MODEL FACTORY
# ================================================================

def build_convnext():

    model = torchvision.models.convnext_tiny(
        weights=(
            torchvision.models.ConvNeXt_Tiny_Weights
            .IMAGENET1K_V1
        )
    )

    # ------------------------------------------------------------
    # Locked classifier replacement
    # ------------------------------------------------------------

    in_features = (
        model.classifier[2].in_features
    )

    model.classifier[2] = nn.Linear(
        in_features,
        2
    )

    # ------------------------------------------------------------
    # Locked deterministic mean-pool substitution
    #
    # ConvNeXt normally uses AdaptiveAvgPool2d(1).
    # Replace it with explicit spatial mean.
    # ------------------------------------------------------------

    class MeanPool2d(nn.Module):

        def forward(self, x):

            return x.mean(
                dim=(-2, -1),
                keepdim=True
            )

    model.avgpool = MeanPool2d()

    return model


# ================================================================
# TEST 1 — STRICT FORWARD/BACKWARD EXECUTION
# ================================================================

print("\n" + "=" * 70)
print("TEST 1 — STRICT DETERMINISTIC FORWARD/BACKWARD")
print("=" * 70)


torch.manual_seed(
    DRY_RUN_SEED
)

torch.cuda.manual_seed_all(
    DRY_RUN_SEED
)

model_1 = build_convnext().to(
    device
)

model_1.train()


# ---------------------------------------------------------------
# Synthetic input
# ---------------------------------------------------------------

torch.manual_seed(
    DRY_RUN_SEED + 1
)

x = torch.randn(
    4,
    3,
    224,
    224,
    device=device,
    dtype=torch.float32
)

target = torch.tensor(
    [0, 1, 0, 1],
    device=device,
    dtype=torch.long
)


criterion = nn.CrossEntropyLoss()


# ---------------------------------------------------------------
# Forward + loss + backward
# ---------------------------------------------------------------

try:

    model_1.zero_grad(
        set_to_none=True
    )

    logits_1 = model_1(x)

    loss_1 = criterion(
        logits_1,
        target
    )

    loss_1.backward()

    torch.cuda.synchronize()

    print(
        "Forward:",
        "PASS"
    )

    print(
        "Loss:",
        float(loss_1.detach().cpu())
    )

    print(
        "Backward:",
        "PASS"
    )

except RuntimeError as e:

    print(
        "\nSTRICT DETERMINISM ERROR:"
    )

    print(
        str(e)
    )

    raise RuntimeError(
        "ConvNeXt deterministic forward/backward "
        "failed. STOP. Do not patch silently."
    ) from e


# ================================================================
# COLLECT GRADIENT SIGNATURE
# ================================================================

def tensor_sha256(t):

    t = (
        t.detach()
        .contiguous()
        .cpu()
    )

    return hashlib.sha256(
        t.numpy().tobytes()
    ).hexdigest()


logits_hash_1 = tensor_sha256(
    logits_1
)

loss_hash_1 = tensor_sha256(
    loss_1.reshape(1)
)

gradient_hashes_1 = {}

for name, parameter in model_1.named_parameters():

    if parameter.grad is not None:

        gradient_hashes_1[
            name
        ] = tensor_sha256(
            parameter.grad
        )


print(
    "\nRun-1 logits SHA256:",
    logits_hash_1
)

print(
    "Run-1 loss SHA256:",
    loss_hash_1
)

print(
    "Run-1 gradient tensors:",
    len(gradient_hashes_1)
)


# ================================================================
# TEST 2 — REBUILD IDENTICAL MODEL + IDENTICAL INPUT
#
# This checks actual reproducibility of:
#   initialization
#   forward
#   loss
#   backward
#
# ================================================================

print("\n" + "=" * 70)
print("TEST 2 — BIT-IDENTICAL REPEAT")
print("=" * 70)


torch.manual_seed(
    DRY_RUN_SEED
)

torch.cuda.manual_seed_all(
    DRY_RUN_SEED
)

model_2 = build_convnext().to(
    device
)

model_2.train()


torch.manual_seed(
    DRY_RUN_SEED + 1
)

x_2 = torch.randn(
    4,
    3,
    224,
    224,
    device=device,
    dtype=torch.float32
)

target_2 = torch.tensor(
    [0, 1, 0, 1],
    device=device,
    dtype=torch.long
)


model_2.zero_grad(
    set_to_none=True
)

logits_2 = model_2(
    x_2
)

loss_2 = criterion(
    logits_2,
    target_2
)

loss_2.backward()

torch.cuda.synchronize()


logits_hash_2 = tensor_sha256(
    logits_2
)

loss_hash_2 = tensor_sha256(
    loss_2.reshape(1)
)

gradient_hashes_2 = {}

for name, parameter in model_2.named_parameters():

    if parameter.grad is not None:

        gradient_hashes_2[
            name
        ] = tensor_sha256(
            parameter.grad
        )


logits_identical = (
    logits_hash_1
    == logits_hash_2
)

loss_identical = (
    loss_hash_1
    == loss_hash_2
)

gradient_names_identical = (
    set(gradient_hashes_1)
    == set(gradient_hashes_2)
)

gradient_values_identical = (
    gradient_hashes_1
    == gradient_hashes_2
)


print(
    "Logits bit-identical:",
    logits_identical
)

print(
    "Loss bit-identical:",
    loss_identical
)

print(
    "Gradient tensors identical:",
    gradient_names_identical
)

print(
    "Gradient values bit-identical:",
    gradient_values_identical
)


# ================================================================
# TEST 3 — EVAL MODE / DROP-PATH INERTNESS
# ================================================================

print("\n" + "=" * 70)
print("TEST 3 — EVAL-MODE BIT-IDENTICAL FORWARD")
print("=" * 70)


torch.manual_seed(
    DRY_RUN_SEED + 2
)

torch.cuda.manual_seed_all(
    DRY_RUN_SEED + 2
)

model_eval = build_convnext().to(
    device
)

model_eval.eval()


torch.manual_seed(
    DRY_RUN_SEED + 3
)

x_eval = torch.randn(
    4,
    3,
    224,
    224,
    device=device,
    dtype=torch.float32
)


with torch.no_grad():

    eval_logits_a = model_eval(
        x_eval
    )

    torch.cuda.synchronize()

    eval_logits_b = model_eval(
        x_eval
    )

    torch.cuda.synchronize()


eval_hash_a = tensor_sha256(
    eval_logits_a
)

eval_hash_b = tensor_sha256(
    eval_logits_b
)

eval_identical = (
    eval_hash_a
    == eval_hash_b
)


print(
    "Eval forward A SHA256:",
    eval_hash_a
)

print(
    "Eval forward B SHA256:",
    eval_hash_b
)

print(
    "Eval outputs bit-identical:",
    eval_identical
)


# ================================================================
# TEST 4 — CHECK STOCHASTIC DEPTH MODULES
# ================================================================

print("\n" + "=" * 70)
print("TEST 4 — STOCHASTIC DEPTH / DROPPATH INSPECTION")
print("=" * 70)


stochastic_modules = []

for name, module in model_eval.named_modules():

    class_name = (
        module.__class__.__name__
    )

    if (
        "StochasticDepth"
        in class_name
        or
        "DropPath"
        in class_name
    ):

        stochastic_modules.append(
            {
                "name": name,
                "class": class_name,
                "training": module.training,
            }
        )


print(
    "Stochastic-depth modules found:",
    len(stochastic_modules)
)

for item in stochastic_modules[:20]:

    print(
        item
    )


all_stochastic_eval = all(
    item["training"] is False
    for item in stochastic_modules
)


print(
    "All stochastic-depth modules in eval mode:",
    all_stochastic_eval
)


# ================================================================
# TEST 5 — MEAN-POOL SUBSTITUTION
# ================================================================

print("\n" + "=" * 70)
print("TEST 5 — MEAN-POOL SUBSTITUTION")
print("=" * 70)


print(
    "avgpool module:",
    model_eval.avgpool
)

print(
    "avgpool class:",
    type(
        model_eval.avgpool
    ).__name__
)


mean_pool_active = (
    type(
        model_eval.avgpool
    ).__name__
    == "MeanPool2d"
)


print(
    "Explicit MeanPool2d active:",
    mean_pool_active
)


# ================================================================
# FINAL GATE
# ================================================================

checks = {

    "strict_forward_backward":
        True,

    "bit_identical_logits":
        logits_identical,

    "bit_identical_loss":
        loss_identical,

    "bit_identical_gradients":
        gradient_names_identical
        and gradient_values_identical,

    "eval_bit_identical":
        eval_identical,

    "stochastic_depth_eval":
        all_stochastic_eval,

    "mean_pool_active":
        mean_pool_active,
}


print("\n" + "=" * 70)
print("CONVNEXT DETERMINISM DRY-RUN RESULTS")
print("=" * 70)

for name, passed in checks.items():

    print(
        f"{name:32s}: "
        f"{'PASS' if passed else 'FAIL'}"
    )


# ================================================================
# SAVE RESULT
# ================================================================

result = {
    "python": sys.version,
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "cuda": torch.version.cuda,
    "cublas_workspace_config":
        os.environ.get(
            "CUBLAS_WORKSPACE_CONFIG"
        ),
    "seed": DRY_RUN_SEED,
    "device": torch.cuda.get_device_name(0),
    "checks": checks,
    "run1": {
        "logits_sha256": logits_hash_1,
        "loss_sha256": loss_hash_1,
        "gradient_count": len(
            gradient_hashes_1
        ),
    },
    "run2": {
        "logits_sha256": logits_hash_2,
        "loss_sha256": loss_hash_2,
        "gradient_count": len(
            gradient_hashes_2
        ),
    },
    "eval": {
        "sha256_a": eval_hash_a,
        "sha256_b": eval_hash_b,
    },
    "stochastic_modules":
        stochastic_modules,
    "mean_pool_class":
        type(
            model_eval.avgpool
        ).__name__,
}


result_path = (
    Path("/kaggle/working")
    / "T2_1_CONVNEXT_DETERMINISM_DRY_RUN.json"
)

with open(
    result_path,
    "w"
) as f:

    json.dump(
        result,
        f,
        indent=2,
        default=str
    )


print(
    "\nDry-run result saved:"
)

print(
    result_path
)


# ================================================================
# FINAL HARD STOP / PASS
# ================================================================

all_pass = all(
    checks.values()
)

print("\n" + "=" * 70)

if all_pass:

    print(
        "PASS — CONVNEXT DETERMINISM DRY-RUN"
    )

    print(
        "\nConvNeXt-Tiny passed the strict "
        "determinism gate."
    )

    print(
        "NO MODEL TRAINING WAS PERFORMED."
    )

    print(
        "\nNext step:"
    )

    print(
        "Freeze the T2.1 training launcher and "
        "begin the 9-model production run."
    )

else:

    print(
        "FAIL — CONVNEXT DETERMINISM DRY-RUN"
    )

    print(
        "\nDO NOT TRAIN."
    )

    failed = [
        name
        for name, passed in checks.items()
        if not passed
    ]

    print(
        "\nFailed checks:"
    )

    for name in failed:

        print(
            " -",
            name
        )

print("=" * 70)
print("CELL 6Y COMPLETE")
print("=" * 70)

In [ ]:
# ================================================================
# T2.1 — CELL 6Z
# FROZEN TRAINING HARNESS DISCOVERY + INTEGRITY AUDIT
#
# Purpose:
#   Locate the actual frozen Step2X training harness and inspect
#   its metadata before creating the ConvNeXt-specific launcher.
#
# NO TRAINING
# NO MODIFICATION OF FROZEN CODE
# ================================================================

from pathlib import Path
import hashlib
import json
import os


print("=" * 70)
print("T2.1 — CELL 6Z")
print("FROZEN TRAINING HARNESS DISCOVERY + INTEGRITY AUDIT")
print("=" * 70)


# ================================================================
# SEARCH ROOTS
# ================================================================

roots = [
    Path("/kaggle/input"),
    Path("/kaggle/working"),
]


candidate_names = {
    "train_one_model.py",
    "analysis_pipeline.py",
}


matches = []


print("\nSEARCHING FOR FROZEN TRAINING HARNESS")
print("-" * 70)


for root in roots:

    if not root.exists():
        continue

    print(
        f"\nScanning: {root}"
    )

    for path in root.rglob("*"):

        if not path.is_file():
            continue

        if path.name in candidate_names:

            matches.append(
                path
            )

            print(
                path
            )


# ================================================================
# REMOVE DUPLICATES
# ================================================================

matches = sorted(
    set(matches),
    key=lambda p: str(p)
)


print("\n" + "=" * 70)
print("DISCOVERED FILES")
print("=" * 70)

for path in matches:

    print(
        path
    )


# ================================================================
# IDENTIFY TRAINING HARNESS
# ================================================================

training_matches = [
    p
    for p in matches
    if p.name == "train_one_model.py"
]


if not training_matches:

    raise FileNotFoundError(
        "Could not locate train_one_model.py."
    )


print("\nTraining harness candidates:")
for i, path in enumerate(
    training_matches
):

    print(
        f"[{i}] {path}"
    )


# ================================================================
# HASH EACH CANDIDATE
# ================================================================

def sha256_file(path):

    h = hashlib.sha256()

    with open(
        path,
        "rb"
    ) as f:

        for chunk in iter(
            lambda: f.read(
                1024 * 1024
            ),
            b""
        ):

            h.update(
                chunk
            )

    return h.hexdigest()


print("\n" + "=" * 70)
print("TRAINING HARNESS HASHES")
print("=" * 70)

training_hashes = {}

for path in training_matches:

    digest = sha256_file(
        path
    )

    training_hashes[
        str(path)
    ] = digest

    print(
        path
    )

    print(
        "SHA256:",
        digest
    )


# ================================================================
# PRINT RELEVANT SOURCE SECTIONS
# ================================================================

print("\n" + "=" * 70)
print("TRAINING HARNESS SOURCE INSPECTION")
print("=" * 70)


for path in training_matches:

    print(
        "\n"
        + "#" * 70
    )

    print(
        f"FILE: {path}"
    )

    print(
        "#" * 70
    )

    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:

        source = f.read()

    lines = source.splitlines()

    print(
        f"Total lines: {len(lines)}"
    )

    # Print source because this is the frozen production harness
    # we need to audit before making the architecture-specific fork.

    print("\nSOURCE BEGIN")
    print(source)
    print("SOURCE END")


# ================================================================
# SEARCH FOR ASSOCIATED FREEZE ARTIFACTS
# ================================================================

print("\n" + "=" * 70)
print("ASSOCIATED STEP2X FREEZE ARTIFACTS")
print("=" * 70)


freeze_candidates = []

for root in roots:

    if not root.exists():
        continue

    for path in root.rglob("*"):

        if not path.is_file():
            continue

        path_string = str(
            path
        ).lower()

        if (
            "step2x" in path_string
            or
            "training_code_freeze" in path_string
        ):

            freeze_candidates.append(
                path
            )


freeze_candidates = sorted(
    set(freeze_candidates),
    key=lambda p: str(p)
)


for path in freeze_candidates[:200]:

    print(
        path
    )


print(
    "\nAssociated freeze artifacts found:",
    len(freeze_candidates)
)


# ================================================================
# SAVE DISCOVERY RECORD
# ================================================================

record = {
    "training_harness_candidates": [
        str(p)
        for p in training_matches
    ],
    "training_harness_sha256": training_hashes,
    "associated_freeze_artifacts": [
        str(p)
        for p in freeze_candidates
    ],
}


output_path = (
    Path("/kaggle/working")
    / "T2_1_TRAINING_HARNESS_DISCOVERY.json"
)


with open(
    output_path,
    "w"
) as f:

    json.dump(
        record,
        f,
        indent=2
    )


print("\nDiscovery record saved:")
print(output_path)


print("\n" + "=" * 70)
print("NO TRAINING PERFORMED")
print("NO FROZEN FILES MODIFIED")
print("=" * 70)
print("CELL 6Z COMPLETE")
print("=" * 70)

In [ ]:
# ================================================================
# T2.1 — CELL 6ZA
# FROZEN TRAINING HARNESS BYTE-IDENTITY + HASH-MANIFEST AUDIT
#
# Purpose:
#   1. Identify every discovered train_one_model.py copy.
#   2. Compute SHA256 of every copy.
#   3. Read TRAINING_CODE_FINAL_HASH.json from each corresponding
#      freeze directory.
#   4. Determine whether all copies are byte-identical.
#   5. Determine which copy is the canonical frozen source.
#   6. Print the canonical source for direct inspection.
#
# NO TRAINING
# NO MODIFICATION OF FROZEN FILES
# ================================================================

from pathlib import Path
import hashlib
import json
import os


print("=" * 70)
print("T2.1 — CELL 6ZA")
print("FROZEN TRAINING HARNESS BYTE-IDENTITY + HASH-MANIFEST AUDIT")
print("=" * 70)


# ================================================================
# DISCOVER TRAINING HARNESS COPIES
# ================================================================

roots = [
    Path("/kaggle/input/datasets"),
    Path("/kaggle/input/notebooks"),
]


training_files = []

for root in roots:

    if not root.exists():
        continue

    for path in root.rglob("train_one_model.py"):

        if (
            "step2X_training_code_freeze"
            in str(path)
        ):

            training_files.append(
                path
            )


training_files = sorted(
    set(training_files),
    key=lambda p: str(p)
)


print("\nDISCOVERED TRAINING HARNESS COPIES")
print("-" * 70)

for i, path in enumerate(
    training_files
):

    print(
        f"[{i}] {path}"
    )


if not training_files:

    raise FileNotFoundError(
        "No step2X_training_code_freeze/train_one_model.py found."
    )


# ================================================================
# SHA256
# ================================================================

def sha256_file(path):

    h = hashlib.sha256()

    with open(
        path,
        "rb"
    ) as f:

        for chunk in iter(
            lambda: f.read(
                1024 * 1024
            ),
            b""
        ):

            h.update(
                chunk
            )

    return h.hexdigest()


print("\n" + "=" * 70)
print("TRAINING HARNESS SHA256 AUDIT")
print("=" * 70)


records = []

for path in training_files:

    digest = sha256_file(
        path
    )

    size = path.stat().st_size

    records.append({
        "path": str(path),
        "sha256": digest,
        "bytes": size,
    })

    print("\nFILE:")
    print(path)

    print(
        "Bytes:",
        size
    )

    print(
        "SHA256:",
        digest
    )


# ================================================================
# HASH GROUPS
# ================================================================

hash_groups = {}

for record in records:

    hash_groups.setdefault(
        record["sha256"],
        []
    ).append(
        record["path"]
    )


print("\n" + "=" * 70)
print("HASH GROUPS")
print("=" * 70)

for digest, paths in hash_groups.items():

    print(
        "\nSHA256:",
        digest
    )

    print(
        "Copies:",
        len(paths)
    )

    for path in paths:

        print(
            " -",
            path
        )


all_byte_identical = (
    len(hash_groups) == 1
)


print(
    "\nAll discovered train_one_model.py copies "
    "byte-identical:",
    all_byte_identical
)


# ================================================================
# HASH MANIFEST AUDIT
# ================================================================

print("\n" + "=" * 70)
print("TRAINING_CODE_FINAL_HASH.JSON AUDIT")
print("=" * 70)


manifest_records = []


for path in training_files:

    freeze_dir = path.parent

    manifest_path = (
        freeze_dir
        / "TRAINING_CODE_FINAL_HASH.json"
    )

    print("\nHarness:")
    print(path)

    print(
        "Manifest:",
        manifest_path
    )

    if not manifest_path.exists():

        print(
            "Manifest: NOT FOUND"
        )

        manifest_records.append({
            "harness": str(path),
            "manifest": str(manifest_path),
            "exists": False,
        })

        continue


    with open(
        manifest_path,
        "r",
        encoding="utf-8"
    ) as f:

        manifest = json.load(
            f
        )


    print(
        "Manifest: FOUND"
    )

    print(
        "Manifest contents:"
    )

    print(
        json.dumps(
            manifest,
            indent=2,
            default=str
        )
    )


    manifest_records.append({
        "harness": str(path),
        "manifest": str(manifest_path),
        "exists": True,
        "contents": manifest,
    })


# ================================================================
# SOURCE INSPECTION
# ================================================================

print("\n" + "=" * 70)
print("CANONICAL SOURCE INSPECTION")
print("=" * 70)


# Prefer the dataset copy as the canonical frozen source if it
# exists. This is only a selection convention for inspection;
# byte identity/hash agreement is checked independently.

dataset_candidates = [
    p
    for p in training_files
    if str(p).startswith(
        "/kaggle/input/datasets/"
    )
]


if dataset_candidates:

    canonical = dataset_candidates[0]

else:

    canonical = training_files[0]


print(
    "Selected canonical inspection source:"
)

print(
    canonical
)


canonical_source = canonical.read_text(
    encoding="utf-8"
)

canonical_hash = sha256_file(
    canonical
)


print(
    "\nCanonical SHA256:",
    canonical_hash
)

print(
    "Canonical source lines:",
    len(
        canonical_source.splitlines()
    )
)

print(
    "\n" + "-" * 70
)
print(
    "CANONICAL TRAINING HARNESS SOURCE"
)
print(
    "-" * 70
)

print(
    canonical_source
)


# ================================================================
# SEARCH FOR CRITICAL TRAINING COMPONENTS
# ================================================================

print("\n" + "=" * 70)
print("CRITICAL TRAINING COMPONENT SEARCH")
print("=" * 70)


critical_terms = [
    "resnet18",
    "ResNet18_Weights",
    "AdamW",
    "lr=1e-4",
    "weight_decay=1e-4",
    "batch",
    "15",
    "CosineAnnealingLR",
    "CrossEntropyLoss",
    "RandomHorizontalFlip",
    "RandomRotation",
    "0.485",
    "0.456",
    "0.406",
    "0.229",
    "0.224",
    "0.225",
    "use_deterministic_algorithms",
    "best_eval_ba.pt",
    "final_epoch15.pt",
]


for term in critical_terms:

    present = (
        term in canonical_source
    )

    print(
        f"{term:35s}: "
        f"{'FOUND' if present else 'NOT FOUND'}"
    )


# ================================================================
# SAVE AUDIT
# ================================================================

audit = {
    "training_files": records,
    "hash_groups": hash_groups,
    "all_byte_identical": all_byte_identical,
    "manifests": manifest_records,
    "canonical": {
        "path": str(canonical),
        "sha256": canonical_hash,
        "line_count": len(
            canonical_source.splitlines()
        ),
    },
}


output_path = (
    Path("/kaggle/working")
    / "T2_1_TRAINING_HARNESS_BYTE_IDENTITY_AUDIT.json"
)


with open(
    output_path,
    "w"
) as f:

    json.dump(
        audit,
        f,
        indent=2,
        default=str
    )


print("\n" + "=" * 70)
print("AUDIT SAVED")
print("=" * 70)

print(
    output_path
)


# ================================================================
# FINAL GATE
# ================================================================

if not all_byte_identical:

    print(
        "\nWARNING — TRAINING HARNESS COPIES DIFFER."
    )

    print(
        "DO NOT create the ConvNeXt fork yet."
    )

    print(
        "We must resolve which frozen copy is authoritative."
    )

else:

    print(
        "\nPASS — ALL TRAINING HARNESS COPIES ARE BYTE-IDENTICAL."
    )

    print(
        "The canonical frozen training harness is internally consistent."
    )

    print(
        "\nNO TRAINING PERFORMED."
    )

    print(
        "NO FROZEN FILES MODIFIED."
    )

print(
    "\n" + "=" * 70
)
print(
    "CELL 6ZA COMPLETE"
)
print(
    "=" * 70
)

In [ ]:
# ================================================================
# CELL 6ZB — AUTHORITATIVE TRAINING HARNESS RESOLUTION
# ================================================================
# PURPOSE:
#   Resolve which train_one_model.py copy is authoritative BEFORE
#   creating the ConvNeXt fork.
#
# NO TRAINING.
# NO FILE MODIFICATION.
# NO FROZEN SOURCE MODIFICATION.
# ================================================================

from pathlib import Path
import hashlib
import json
import difflib
import os
import re

ROOTS = [
    Path("/kaggle/input"),
    Path("/kaggle/working"),
]

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def load_json_safe(path):
    try:
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    except Exception as e:
        return {"__ERROR__": repr(e)}

def recursive_hash_values(obj):
    vals = []
    if isinstance(obj, dict):
        for k, v in obj.items():
            if isinstance(v, str) and (
                "hash" in k.lower() or
                "sha" in k.lower()
            ):
                vals.append((k, v))
            vals.extend(recursive_hash_values(v))
    elif isinstance(obj, list):
        for v in obj:
            vals.extend(recursive_hash_values(v))
    return vals

# ------------------------------------------------
# 1. Discover every train_one_model.py
# ------------------------------------------------

train_files = sorted(set(
    p for root in ROOTS
    if root.exists()
    for p in root.rglob("train_one_model.py")
    if "step2X_training_code_freeze" in str(p)
))

print("=" * 78)
print("DISCOVERED TRAINING HARNESS COPIES")
print("=" * 78)

for i, p in enumerate(train_files, 1):
    print(f"[{i}] {p}")

print(f"\nTOTAL train_one_model.py COPIES: {len(train_files)}")

# ------------------------------------------------
# 2. Inspect each copy + associated final hash
# ------------------------------------------------

records = []

for p in train_files:

    step_dir = p.parent
    hash_candidates = [
        step_dir / "TRAINING_CODE_FINAL_HASH.json",
    ]

    hash_path = next(
        (x for x in hash_candidates if x.exists()),
        None
    )

    actual_sha = sha256_file(p)

    record = {
        "source_path": str(p),
        "source_sha256": actual_sha,
        "source_size_bytes": p.stat().st_size,
        "hash_manifest": str(hash_path) if hash_path else None,
        "hash_manifest_exists": bool(hash_path),
        "hash_manifest_sha256": None,
        "hash_values": [],
        "hash_match": False,
        "hash_manifest_content": None,
    }

    if hash_path:
        manifest = load_json_safe(hash_path)
        record["hash_manifest_content"] = manifest
        record["hash_manifest_sha256"] = sha256_file(hash_path)
        record["hash_values"] = recursive_hash_values(manifest)

        declared_hashes = {
            str(v).lower()
            for _, v in record["hash_values"]
            if isinstance(v, str)
            and re.fullmatch(r"[0-9a-fA-F]{64}", v)
        }

        record["hash_match"] = actual_sha.lower() in declared_hashes

    records.append(record)

# ------------------------------------------------
# 3. Print authoritative hash relationship
# ------------------------------------------------

print("\n" + "=" * 78)
print("SOURCE ↔ FINAL-HASH MANIFEST AUDIT")
print("=" * 78)

for i, r in enumerate(records, 1):

    print(f"\nCOPY {i}")
    print("-" * 78)
    print("SOURCE:")
    print(r["source_path"])
    print("SIZE:")
    print(r["source_size_bytes"])
    print("ACTUAL SHA256:")
    print(r["source_sha256"])
    print("HASH MANIFEST:")
    print(r["hash_manifest"])

    if r["hash_manifest_exists"]:
        print("DECLARED HASH VALUES:")

        if r["hash_values"]:
            for k, v in r["hash_values"]:
                print(f"  {k}: {v}")
        else:
            print("  NONE FOUND")

        print("SOURCE HASH MATCHES DECLARED HASH:")
        print("  ", r["hash_match"])
    else:
        print("NO TRAINING_CODE_FINAL_HASH.json FOUND")

# ------------------------------------------------
# 4. Group source files by actual SHA256
# ------------------------------------------------

groups = {}

for r in records:
    groups.setdefault(r["source_sha256"], []).append(r["source_path"])

print("\n" + "=" * 78)
print("BYTE-IDENTITY GROUPING")
print("=" * 78)

for sha, paths in groups.items():
    print(f"\nSHA256: {sha}")
    print(f"COPIES : {len(paths)}")

    for p in paths:
        print("  ", p)

# ------------------------------------------------
# 5. Identify internally valid copies
# ------------------------------------------------

valid = [
    r for r in records
    if r["hash_manifest_exists"] and r["hash_match"]
]

invalid = [
    r for r in records
    if not r["hash_match"]
]

print("\n" + "=" * 78)
print("VALIDITY SUMMARY")
print("=" * 78)

print(f"TOTAL COPIES                    : {len(records)}")
print(f"COPIES MATCHING FINAL HASH      : {len(valid)}")
print(f"COPIES NOT MATCHING FINAL HASH  : {len(invalid)}")

for r in valid:
    print("\nVALID:")
    print(r["source_path"])
    print("SHA256:", r["source_sha256"])

for r in invalid:
    print("\nNOT VALIDATED:")
    print(r["source_path"])
    print("SHA256:", r["source_sha256"])

# ------------------------------------------------
# 6. Compare every distinct source version
# ------------------------------------------------

unique_sources = {}

for r in records:
    sha = r["source_sha256"]
    if sha not in unique_sources:
        unique_sources[sha] = r["source_path"]

print("\n" + "=" * 78)
print("SOURCE DIFFERENCE ANALYSIS")
print("=" * 78)

if len(unique_sources) == 1:
    print("ALL DISCOVERED train_one_model.py COPIES ARE BYTE-IDENTICAL.")
else:
    print(f"DISTINCT SOURCE VERSIONS FOUND: {len(unique_sources)}")

    source_items = list(unique_sources.items())

    for i in range(len(source_items)):
        sha_a, path_a = source_items[i]

        for j in range(i + 1, len(source_items)):
            sha_b, path_b = source_items[j]

            print("\n" + "-" * 78)
            print(f"VERSION A SHA: {sha_a}")
            print(f"VERSION A:     {path_a}")
            print(f"VERSION B SHA: {sha_b}")
            print(f"VERSION B:     {path_b}")

            with open(path_a, "r", encoding="utf-8") as f:
                lines_a = f.readlines()

            with open(path_b, "r", encoding="utf-8") as f:
                lines_b = f.readlines()

            diff = list(difflib.unified_diff(
                lines_a,
                lines_b,
                fromfile=str(path_a),
                tofile=str(path_b),
                lineterm=""
            ))

            print(f"TOTAL DIFF LINES: {len(diff)}")

            if diff:
                # Print enough context to identify the nature of the
                # difference without flooding the notebook.
                print("\nFIRST 250 DIFF LINES:")
                print("\n".join(diff[:250]))

                if len(diff) > 250:
                    print(
                        f"\n... {len(diff)-250} additional diff lines omitted ..."
                    )

# ------------------------------------------------
# 7. Critical training-component comparison
# ------------------------------------------------

print("\n" + "=" * 78)
print("CRITICAL TRAINING COMPONENT COMPARISON")
print("=" * 78)

critical_patterns = [
    r"resnet18",
    r"ResNet18_Weights",
    r"AdamW",
    r"lr\s*=\s*1e-4",
    r"weight_decay\s*=\s*1e-4",
    r"batch",
    r"15",
    r"CosineAnnealingLR",
    r"CrossEntropyLoss",
    r"RandomHorizontalFlip",
    r"RandomRotation",
    r"0\.485",
    r"0\.456",
    r"0\.406",
    r"0\.229",
    r"0\.224",
    r"0\.225",
    r"use_deterministic_algorithms",
    r"best_eval_ba\.pt",
    r"final_epoch15\.pt",
]

for sha, path in unique_sources.items():

    print("\n" + "-" * 78)
    print("SOURCE:", path)
    print("SHA256:", sha)

    text_source = Path(path).read_text(encoding="utf-8")

    for pat in critical_patterns:
        found = bool(re.search(pat, text_source))
        print(f"{pat:<35} : {'FOUND' if found else 'NOT FOUND'}")

# ------------------------------------------------
# 8. Print complete hash manifests for human decision
# ------------------------------------------------

print("\n" + "=" * 78)
print("COMPLETE TRAINING_CODE_FINAL_HASH.JSON CONTENT")
print("=" * 78)

seen_manifests = set()

for r in records:

    hp = r["hash_manifest"]

    if not hp or hp in seen_manifests:
        continue

    seen_manifests.add(hp)

    print("\n" + "-" * 78)
    print("MANIFEST:", hp)

    try:
        print(
            json.dumps(
                r["hash_manifest_content"],
                indent=2,
                sort_keys=True
            )
        )
    except Exception:
        print(r["hash_manifest_content"])

# ------------------------------------------------
# 9. Save complete resolution audit
# ------------------------------------------------

audit = {
    "purpose": "Resolve authoritative frozen T2.1 training harness",
    "training_performed": False,
    "source_count": len(records),
    "distinct_source_versions": len(unique_sources),
    "records": records,
    "unique_source_versions": unique_sources,
    "valid_hash_matched_sources": [
        r["source_path"] for r in valid
    ],
}

out = Path(
    "/kaggle/working/"
    "T2_1_TRAINING_HARNESS_AUTHORITATIVE_RESOLUTION.json"
)

with open(out, "w", encoding="utf-8") as f:
    json.dump(audit, f, indent=2, sort_keys=True)

print("\n" + "=" * 78)
print("AUDIT SAVED")
print("=" * 78)
print(out)

print("\n" + "=" * 78)
print("CELL 6ZB COMPLETE")
print("=" * 78)

if len(unique_sources) == 1:
    print("RESULT: ALL TRAINING HARNESS COPIES ARE BYTE-IDENTICAL.")
elif len(valid) == 1:
    print("RESULT: EXACTLY ONE COPY MATCHES ITS FINAL HASH MANIFEST.")
    print("This is a strong candidate for authoritative frozen source.")
else:
    print("RESULT: MULTIPLE OR ZERO HASH-VALIDATED COPIES.")
    print("DO NOT CREATE THE CONVNEXT FORK YET.")
    print("Inspect the printed manifest/diff before proceeding.")

In [ ]:
# ================================================================
# CELL 6ZC — PARENT/DERIVED HARNESS FUNCTIONAL DIFFERENCE AUDIT
# ================================================================
# PURPOSE:
#   Determine whether the 08192... source changes the actual
#   training recipe relative to the 1ed047... parent.
#
# NO TRAINING.
# NO SOURCE MODIFICATION.
# ================================================================

from pathlib import Path
import hashlib
import json
import difflib
import re

PARENT = Path(
    "/kaggle/input/datasets/tejasseshadriiiii/"
    "isic-final-project/step2X_training_code_freeze/train_one_model.py"
)

DERIVED = Path(
    "/kaggle/input/notebooks/tejasseshadriiiii/"
    "isic-kaggle-analysis/trained_models_readonly/"
    "isic_final_project/step2X_training_code_freeze/train_one_model.py"
)

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

parent_text = PARENT.read_text(encoding="utf-8")
derived_text = DERIVED.read_text(encoding="utf-8")

parent_lines = parent_text.splitlines()
derived_lines = derived_text.splitlines()

print("=" * 78)
print("HARNESS IDENTITY")
print("=" * 78)

print("PARENT:")
print(PARENT)
print("SHA256:", sha256_file(PARENT))
print("SIZE:", PARENT.stat().st_size)

print("\nDERIVED:")
print(DERIVED)
print("SHA256:", sha256_file(DERIVED))
print("SIZE:", DERIVED.stat().st_size)

# ------------------------------------------------
# 1. Full semantic diff
# ------------------------------------------------

print("\n" + "=" * 78)
print("FULL UNIFIED DIFF")
print("=" * 78)

diff = list(difflib.unified_diff(
    parent_lines,
    derived_lines,
    fromfile=str(PARENT),
    tofile=str(DERIVED),
    lineterm=""
))

print(f"TOTAL DIFF LINES: {len(diff)}")

for line in diff:
    print(line)

# ------------------------------------------------
# 2. Extract added/removed executable-looking lines
# ------------------------------------------------

removed = []
added = []

for line in diff:
    if line.startswith("---") or line.startswith("+++") or line.startswith("@@"):
        continue

    if line.startswith("-"):
        removed.append(line[1:])

    elif line.startswith("+"):
        added.append(line[1:])

print("\n" + "=" * 78)
print("REMOVED LINES")
print("=" * 78)

for line in removed:
    print(repr(line))

print("\n" + "=" * 78)
print("ADDED LINES")
print("=" * 78)

for line in added:
    print(repr(line))

# ------------------------------------------------
# 3. Identify whether core training symbols differ
# ------------------------------------------------

core_patterns = {
    "model_creation": [
        r"resnet18",
        r"ResNet18_Weights",
        r"weights\s*=",
        r"PARAMS_2CLASS",
    ],
    "optimizer": [
        r"AdamW",
        r"lr",
        r"weight_decay",
        r"optimizer",
    ],
    "scheduler": [
        r"CosineAnnealingLR",
        r"T_max",
        r"scheduler",
    ],
    "loss": [
        r"CrossEntropyLoss",
        r"criterion",
    ],
    "batching": [
        r"batch",
        r"DataLoader",
        r"shuffle",
    ],
    "augmentation": [
        r"RandomHorizontalFlip",
        r"RandomRotation",
    ],
    "normalization": [
        r"0\.485",
        r"0\.456",
        r"0\.406",
        r"0\.229",
        r"0\.224",
        r"0\.225",
    ],
    "determinism": [
        r"use_deterministic_algorithms",
        r"CUBLAS_WORKSPACE_CONFIG",
        r"cudnn",
        r"determin",
    ],
    "training_loop": [
        r"for epoch",
        r"num_epochs",
        r"15",
        r"loss\.backward",
        r"optimizer\.step",
        r"scheduler\.step",
    ],
    "checkpointing": [
        r"best_eval_ba",
        r"final_epoch15",
        r"torch\.save",
        r"state_dict",
    ],
    "evaluation": [
        r"balanced",
        r"eval",
        r"validation",
        r"BA",
        r"accuracy",
    ],
}

print("\n" + "=" * 78)
print("CORE TRAINING COMPONENT DIFFERENCE TEST")
print("=" * 78)

core_results = {}

for category, patterns in core_patterns.items():

    parent_hits = []
    derived_hits = []

    for pattern in patterns:
        p_hits = [
            line for line in parent_lines
            if re.search(pattern, line, flags=re.IGNORECASE)
        ]

        d_hits = [
            line for line in derived_lines
            if re.search(pattern, line, flags=re.IGNORECASE)
        ]

        parent_hits.extend(p_hits)
        derived_hits.extend(d_hits)

    # Deduplicate while preserving order
    parent_hits = list(dict.fromkeys(parent_hits))
    derived_hits = list(dict.fromkeys(derived_hits))

    same = parent_hits == derived_hits

    core_results[category] = {
        "identical_matching_lines": same,
        "parent_hits": parent_hits,
        "derived_hits": derived_hits,
    }

    print(f"\n[{category}]")
    print("IDENTICAL MATCHING LINES:", same)

    if not same:
        print("PARENT:")
        for x in parent_hits:
            print("  ", x)

        print("DERIVED:")
        for x in derived_hits:
            print("  ", x)

# ------------------------------------------------
# 4. Remove obvious provenance-only lines and compare
# ------------------------------------------------

PROVENANCE_PATTERNS = [
    r"parent_training_code_sha256",
    r"generator_sha256",
    r"TRAINING_CODE_FINAL_HASH",
    r"__file__",
    r"provenance",
    r"readonly",
    r"generated",
    r"manifest",
    r"hash",
    r"sha256",
]

def normalize_for_provenance_analysis(lines):

    kept = []

    for line in lines:
        stripped = line.strip()

        if not stripped:
            continue

        if stripped.startswith("#"):
            continue

        # Exclude lines whose primary purpose is hash/provenance
        if any(
            re.search(p, stripped, flags=re.IGNORECASE)
            for p in PROVENANCE_PATTERNS
        ):
            continue

        kept.append(stripped)

    return kept

parent_norm = normalize_for_provenance_analysis(parent_lines)
derived_norm = normalize_for_provenance_analysis(derived_lines)

norm_diff = list(difflib.unified_diff(
    parent_norm,
    derived_norm,
    fromfile="PARENT_NORMALIZED",
    tofile="DERIVED_NORMALIZED",
    lineterm=""
))

print("\n" + "=" * 78)
print("NORMALIZED FUNCTIONAL DIFF")
print("=" * 78)

print(
    "This removes blank lines/comments and obvious provenance/hash "
    "lines only."
)

print("NORMALIZED DIFF LINES:", len(norm_diff))

for line in norm_diff:
    print(line)

# ------------------------------------------------
# 5. Search the complete artifact tree for the two hashes
# ------------------------------------------------

print("\n" + "=" * 78)
print("HASH PROVENANCE REFERENCES IN TRAINING ARTIFACTS")
print("=" * 78)

hashes = {
    "PARENT": "1ed04738923f842bc70a688913163df557049099d11f612f4d5fec51b7735d68",
    "DERIVED": "08192f693391fa48c7a28dad28f9dc410f7de64fb6078857bbea64be3e54fee2",
}

search_roots = [
    Path("/kaggle/input/notebooks/tejasseshadriiiii"),
    Path("/kaggle/input/datasets/tejasseshadriiiii"),
    Path("/kaggle/working"),
]

matches = []

for root in search_roots:

    if not root.exists():
        continue

    for p in root.rglob("*"):

        if not p.is_file():
            continue

        # Avoid very large/binary files.
        try:
            if p.stat().st_size > 20 * 1024 * 1024:
                continue
        except Exception:
            continue

        # Only inspect likely textual metadata/artifact files.
        if p.suffix.lower() not in {
            ".json", ".txt", ".csv", ".md", ".py", ".yaml", ".yml"
        }:
            continue

        try:
            text = p.read_text(
                encoding="utf-8",
                errors="ignore"
            )
        except Exception:
            continue

        for label, h in hashes.items():
            if h in text:
                matches.append({
                    "hash_label": label,
                    "hash": h,
                    "path": str(p),
                })

print(f"TOTAL HASH REFERENCES FOUND: {len(matches)}")

for m in matches:
    print(
        f"{m['hash_label']:<8} "
        f"{m['hash']}  "
        f"{m['path']}"
    )

# ------------------------------------------------
# 6. Determine provenance conclusion
# ------------------------------------------------

functional_change_detected = len(norm_diff) > 0

conclusion = {
    "parent_sha256": sha256_file(PARENT),
    "derived_sha256": sha256_file(DERIVED),
    "parent_size": PARENT.stat().st_size,
    "derived_size": DERIVED.stat().st_size,
    "total_raw_diff_lines": len(diff),
    "total_normalized_diff_lines": len(norm_diff),
    "functional_change_detected_after_provenance_normalization":
        functional_change_detected,
    "core_training_results": core_results,
    "hash_references": matches,
}

out = Path(
    "/kaggle/working/"
    "T2_1_TRAINING_HARNESS_FUNCTIONAL_DIFFERENCE_AUDIT.json"
)

with open(out, "w", encoding="utf-8") as f:
    json.dump(conclusion, f, indent=2, sort_keys=True)

print("\n" + "=" * 78)
print("AUDIT SAVED")
print("=" * 78)
print(out)

print("\n" + "=" * 78)
print("CELL 6ZC COMPLETE")
print("=" * 78)

if functional_change_detected:
    print(
        "RESULT: FUNCTIONAL DIFFERENCES REMAIN AFTER "
        "PROVENANCE NORMALIZATION."
    )
    print(
        "DO NOT CREATE THE CONVNEXT FORK YET."
    )
else:
    print(
        "RESULT: NO FUNCTIONAL DIFFERENCE DETECTED AFTER "
        "PROVENANCE NORMALIZATION."
    )
    print(
        "The 1ed047... parent is the likely authoritative frozen "
        "training source; 08192... is its derived artifact."
    )

In [ ]:
# ================================================================
# CELL 6ZD — FINAL PRODUCTION SCRIPT PROVENANCE CHECK
# ================================================================
# PURPOSE:
#   Determine which train_one_model.py SHA256 was actually executed
#   for the original 30 production ResNet-18 models.
#
# READ-ONLY.
# NO TRAINING.
# NO SOURCE MODIFICATION.
# ================================================================

from pathlib import Path
import json

PARENT_SHA = (
    "1ed04738923f842bc70a688913163df557049099d11f612f4d5fec51b7735d68"
)

DERIVED_SHA = (
    "08192f693391fa48c7a28dad28f9dc410f7de64fb6078857bbea64be3e54fee2"
)

# Locate the actual frozen trained-model artifact trees.
CANDIDATE_ROOTS = [
    Path(
        "/kaggle/input/notebooks/tejasseshadriiiii/"
        "ml-open-lab-30m-trained"
    ),
    Path(
        "/kaggle/input/notebooks/tejasseshadriiiii/"
        "isic-kaggle-analysis/trained_models_readonly"
    ),
    Path(
        "/kaggle/input/datasets/tejasseshadriiiii/"
        "isic-final-project"
    ),
]

print("=" * 78)
print("SEARCHING FOR ORIGINAL 30 PRODUCTION TRAINING RECORDS")
print("=" * 78)

all_candidates = []

for root in CANDIDATE_ROOTS:

    if not root.exists():
        continue

    found = sorted(
        root.glob("training_runs/rho_*_rep_*/training_record.json")
    )

    if found:
        print("\nROOT:")
        print(root)
        print("FOUND:", len(found))

        for p in found:
            print(" ", p)

        all_candidates.append(
            (root, found)
        )

# ------------------------------------------------
# Determine which candidate tree contains exactly
# the expected 30 production records.
# ------------------------------------------------

exact_candidates = [
    (root, runs)
    for root, runs in all_candidates
    if len(runs) == 30
]

print("\n" + "=" * 78)
print("CANDIDATE SUMMARY")
print("=" * 78)

for root, runs in all_candidates:
    print(f"{root}")
    print(f"  records = {len(runs)}")

if not exact_candidates:
    raise RuntimeError(
        "Could not identify a candidate tree containing exactly "
        "30 production training records. STOP."
    )

if len(exact_candidates) > 1:

    # Multiple readonly copies are acceptable only if their
    # record contents agree exactly.
    print("\nMultiple 30-record trees found.")
    print("They will be cross-compared before deciding.")

# ------------------------------------------------
# Read every 30-record tree.
# ------------------------------------------------

tree_results = []

for root, runs in exact_candidates:

    tally = {}
    records = []

    for p in runs:

        with open(p, "r", encoding="utf-8") as f:
            r = json.load(f)

        sha = r.get("script_sha256", "MISSING")

        tally.setdefault(sha, []).append(
            p.parent.name
        )

        records.append({
            "path": str(p),
            "model": p.parent.name,
            "script_sha256": sha,
            "ig_check": r.get("ig_check"),
            "mode": r.get("mode"),
        })

    tree_results.append({
        "root": str(root),
        "records": records,
        "tally": tally,
    })

# ------------------------------------------------
# Print SHA distribution.
# ------------------------------------------------

print("\n" + "=" * 78)
print("SCRIPT SHA256 DISTRIBUTION — ORIGINAL 30 PRODUCTION MODELS")
print("=" * 78)

for result in tree_results:

    print("\nTREE:")
    print(result["root"])

    for sha, names in sorted(
        result["tally"].items(),
        key=lambda x: str(x[0])
    ):

        if sha == PARENT_SHA:
            label = "PARENT"
        elif sha == DERIVED_SHA:
            label = "DERIVED"
        elif sha == "MISSING":
            label = "MISSING"
        else:
            label = "UNKNOWN"

        print(
            f"  {label:<8} "
            f"{sha} "
            f"x{len(names):2d}"
        )

        print(
            "      examples:",
            names[:5]
        )

# ------------------------------------------------
# Verify production mode / IG check.
# ------------------------------------------------

print("\n" + "=" * 78)
print("PRODUCTION FLAG AUDIT")
print("=" * 78)

for result in tree_results:

    print("\nTREE:")
    print(result["root"])

    bad = []

    for rec in result["records"]:

        if rec["ig_check"]:
            bad.append(
                (rec["model"], "ig_check=True")
            )

        # We expect primary production records.
        # Print rather than silently assuming.
        if rec["mode"] not in (None, "primary"):
            bad.append(
                (
                    rec["model"],
                    f"mode={rec['mode']!r}"
                )
            )

    if bad:
        print("PRODUCTION FLAG ISSUES:")
        for x in bad:
            print(" ", x)
    else:
        print("PASS — no ig_check=True records found.")

# ------------------------------------------------
# Cross-compare duplicate 30-record trees.
# ------------------------------------------------

print("\n" + "=" * 78)
print("CROSS-TREE PROVENANCE CONSISTENCY")
print("=" * 78)

if len(tree_results) == 1:

    print("Only one exact 30-record production tree found.")
    print("No cross-tree comparison required.")

else:

    baseline = tree_results[0]

    baseline_map = {
        r["model"]: r["script_sha256"]
        for r in baseline["records"]
    }

    for other in tree_results[1:]:

        other_map = {
            r["model"]: r["script_sha256"]
            for r in other["records"]
        }

        differences = []

        for model in sorted(
            set(baseline_map) | set(other_map)
        ):

            a = baseline_map.get(model)
            b = other_map.get(model)

            if a != b:
                differences.append(
                    (model, a, b)
                )

        print("\nCOMPARE:")
        print("A:", baseline["root"])
        print("B:", other["root"])

        if differences:
            print("FAIL — SHA distributions differ:")
            for d in differences:
                print(" ", d)
        else:
            print(
                "PASS — all 30 model-level script_sha256 values "
                "agree exactly."
            )

# ------------------------------------------------
# Final decision
# ------------------------------------------------

print("\n" + "=" * 78)
print("FINAL PROVENANCE DECISION")
print("=" * 78)

# Collect all SHA sets across exact production trees.
all_sha_sets = []

for result in tree_results:
    all_sha_sets.append(
        set(result["tally"].keys())
    )

combined_sha = set().union(*all_sha_sets)

if combined_sha == {PARENT_SHA}:

    decision = "CONFIRMED_PARENT"

    print(
        "✓ CONFIRMED: all 30 original production models were "
        "executed with the PARENT script."
    )
    print()
    print("Authoritative parent:")
    print(PARENT_SHA)
    print()
    print("The 08192... derived safe-loader copy must NOT be used "
          "as the T2.1 parent.")

elif combined_sha == {DERIVED_SHA}:

    decision = "DERIVED_USED"

    print(
        "⚠ ALL 30 production records show the DERIVED script."
    )
    print()
    print(
        "This does NOT imply a training-recipe difference because "
        "the safe-loader is isolated from ordinary primary training."
    )
    print()
    print(
        "However, provenance must be explicitly documented before "
        "creating the T2.1 fork."
    )

elif PARENT_SHA in combined_sha and DERIVED_SHA in combined_sha:

    decision = "MIXED_PARENT_DERIVED"

    print(
        "✗ MIXED PROVENANCE: both parent and derived script hashes "
        "occur across the original 30 production models."
    )
    print()
    print(
        "STOP. Do not create the ConvNeXt fork."
    )

else:

    decision = "UNKNOWN_HASH"

    print(
        "✗ UNKNOWN SCRIPT HASH DETECTED."
    )
    print()
    print(
        "STOP. At least one production record points to a script "
        "other than the known parent/derived versions."
    )

# ------------------------------------------------
# Save immutable diagnostic result.
# ------------------------------------------------

audit = {
    "purpose": (
        "Determine script SHA256 actually executed for "
        "the original 30 production ResNet models"
    ),
    "parent_sha256": PARENT_SHA,
    "derived_sha256": DERIVED_SHA,
    "tree_results": tree_results,
    "combined_sha_set": sorted(combined_sha),
    "decision": decision,
    "training_performed": False,
}

out = Path(
    "/kaggle/working/"
    "T2_1_FINAL_PRODUCTION_SCRIPT_PROVENANCE_AUDIT.json"
)

with open(out, "w", encoding="utf-8") as f:
    json.dump(audit, f, indent=2, sort_keys=True)

print("\n" + "=" * 78)
print("AUDIT SAVED")
print("=" * 78)
print(out)

print("\n" + "=" * 78)
print("CELL 6ZD COMPLETE")
print("=" * 78)

In [ ]:
# ================================================================
# CELL 6ZE — T2.1 PRODUCTION PROVENANCE RESOLUTION LOCK
# ================================================================
# PURPOSE:
#   Permanently document the resolved relationship between:
#
#     1ed047... = original frozen parent training source
#     08192...  = derived executable actually used by all 30
#                  original production ResNet models
#
#   This is a PROVENANCE LOCK, not a training code modification.
#
# NO TRAINING.
# NO MODEL MODIFICATION.
# ================================================================

from pathlib import Path
import json
import hashlib
from datetime import datetime

PARENT_SHA = (
    "1ed04738923f842bc70a688913163df557049099d11f612f4d5fec51b7735d68"
)

PRODUCTION_SHA = (
    "08192f693391fa48c7a28dad28f9dc410f7de64fb6078857bbea64be3e54fee2"
)

GENERATOR_SHA = (
    "32be8092c85458bed31a547f2ba6c57c3e1c2cf257d7cb0f76bd28eebc7b26c2"
)

PARENT_PATH = Path(
    "/kaggle/input/datasets/tejasseshadriiiii/"
    "isic-final-project/step2X_training_code_freeze/"
    "train_one_model.py"
)

PRODUCTION_PATH = Path(
    "/kaggle/input/notebooks/tejasseshadriiiii/"
    "ml-open-lab-30m-trained/isic_final_project/"
    "step2X_training_code_freeze/train_one_model.py"
)

assert PARENT_PATH.exists(), (
    f"Parent source missing: {PARENT_PATH}"
)

assert PRODUCTION_PATH.exists(), (
    f"Production source missing: {PRODUCTION_PATH}"
)

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

actual_parent_sha = sha256_file(PARENT_PATH)
actual_production_sha = sha256_file(PRODUCTION_PATH)

assert actual_parent_sha == PARENT_SHA, (
    "Parent source SHA mismatch.\n"
    f"Expected: {PARENT_SHA}\n"
    f"Actual:   {actual_parent_sha}"
)

assert actual_production_sha == PRODUCTION_SHA, (
    "Production source SHA mismatch.\n"
    f"Expected: {PRODUCTION_SHA}\n"
    f"Actual:   {actual_production_sha}"
)

# ------------------------------------------------
# Load previous production provenance audit
# ------------------------------------------------

audit_path = Path(
    "/kaggle/working/"
    "T2_1_FINAL_PRODUCTION_SCRIPT_PROVENANCE_AUDIT.json"
)

assert audit_path.exists(), (
    f"Previous provenance audit missing: {audit_path}"
)

with open(audit_path, "r", encoding="utf-8") as f:
    production_audit = json.load(f)

assert production_audit["decision"] == "DERIVED_USED", (
    "Expected resolved decision DERIVED_USED, got: "
    + str(production_audit["decision"])
)

# ------------------------------------------------
# Verify every production tree independently
# ------------------------------------------------

tree_results = production_audit["tree_results"]

assert len(tree_results) >= 1

for tree in tree_results:

    tally = tree["tally"]

    assert set(tally.keys()) == {PRODUCTION_SHA}, (
        "Production tree contains unexpected script SHA.\n"
        f"Tree: {tree['root']}\n"
        f"Observed: {list(tally.keys())}"
    )

    assert len(tally[PRODUCTION_SHA]) == 30, (
        "Production tree does not contain exactly 30 derived-script records.\n"
        f"Tree: {tree['root']}"
    )

    for record in tree["records"]:

        assert record["script_sha256"] == PRODUCTION_SHA, (
            f"Unexpected script SHA in {record['path']}"
        )

        assert not record["ig_check"], (
            f"ig_check=True in production record: {record['path']}"
        )

# ------------------------------------------------
# Provenance interpretation
# ------------------------------------------------

resolution = {
    "document_type": "T2.1_PRODUCTION_PROVENANCE_RESOLUTION",
    "created_utc": datetime.utcnow().isoformat() + "Z",

    "status": "RESOLVED",

    "original_frozen_parent": {
        "sha256": PARENT_SHA,
        "path": str(PARENT_PATH),
        "role": (
            "Original frozen parent training source from which the "
            "derived production executable was created."
        ),
    },

    "actual_production_executable": {
        "sha256": PRODUCTION_SHA,
        "path": str(PRODUCTION_PATH),
        "role": (
            "Executable train_one_model.py whose script_sha256 is "
            "recorded in all 30 original production training records."
        ),
    },

    "generator_dependency": {
        "sha256": GENERATOR_SHA,
        "role": (
            "Locked recovered hair-generator source referenced by "
            "the derived safe-loader."
        ),
    },

    "production_evidence": {
        "production_model_count": 30,
        "trees_checked": len(tree_results),
        "all_trees_agree": True,
        "all_records_use_production_sha": True,
        "all_records_ig_check_false": True,
        "production_sha": PRODUCTION_SHA,
    },

    "functional_interpretation": {
        "training_recipe_changed": False,
        "basis": (
            "Prior functional audit found the core training behavior "
            "identical and the additional safe-loader confined to "
            "the IG-check path. All original production records "
            "have ig_check=False."
        ),
        "safe_loader_active_in_original_production": False,
    },

    "T2_1_rule": (
        "Do not silently represent the original 30 ResNet models as "
        "having been trained under the parent SHA alone. Their "
        "recorded executable provenance is the derived SHA."
    ),

    "T2_1_fork_requirement": (
        "Preserve the resolved production training semantics while "
        "creating a separately hashed ConvNeXt-Tiny implementation. "
        "Do not modify either frozen ResNet source."
    ),

    "statistical_protocol_frozen": {
        "method": "Freedman-Lane permutation",
        "reduced_model_excludes_interaction": True,
        "permutation_unit": "residuals",
        "full_model_refit_each_permutation": True,
        "permutations": 100000,
        "seed": 20260919,
        "two_sided_alpha": 0.05,
    },

    "T2_1_design_frozen": {
        "architecture": "ConvNeXt-Tiny",
        "rho": [0.0, 0.5, 1.0],
        "replicates_per_rho": 3,
        "total_models": 9,
        "endpoint": "Delta_z",
        "primary_checkpoint": "final_epoch15.pt",
        "best_eval_ba": "sensitivity_only",
        "xai_ig": False,
    },

    "forbidden_action": (
        "Do not use the old unrestricted architecture-label "
        "permutation protocol."
    ),
}

out = Path(
    "/kaggle/working/"
    "T2_1_PRODUCTION_PROVENANCE_RESOLUTION_LOCK.json"
)

with open(out, "w", encoding="utf-8") as f:
    json.dump(
        resolution,
        f,
        indent=2,
        sort_keys=True
    )

print("=" * 78)
print("T2.1 PRODUCTION PROVENANCE RESOLUTION")
print("=" * 78)

print()
print("ORIGINAL FROZEN PARENT")
print("SHA256:", PARENT_SHA)

print()
print("ACTUAL PRODUCTION EXECUTABLE")
print("SHA256:", PRODUCTION_SHA)

print()
print("ORIGINAL PRODUCTION MODELS")
print("COUNT: 30")
print("SCRIPT SHA: 08192...")

print()
print("IG CHECK")
print("ALL 30 = FALSE")

print()
print("TRAINING-RECIPE INTERPRETATION")
print("No training-recipe difference detected.")
print("Derived safe-loader is confined to IG-check machinery.")

print()
print("STATUS")
print("✓ PROVENANCE RESOLVED")
print("✓ 30/30 production records agree")
print("✓ No mixed provenance")
print("✓ No IG-check contamination")

print()
print("LOCK SAVED:")
print(out)

print()
print("=" * 78)
print("CELL 6ZE COMPLETE")
print("=" * 78)

print(
    "NEXT STEP: create the separately hashed ConvNeXt-Tiny "
    "T2.1 training implementation."
)

In [ ]:
# ============================================================================
# CELL 6ZF — T2.1 CONVNEXT-TINY TRAINING IMPLEMENTATION FREEZE
# ============================================================================
# Purpose:
#   Create a separately hashed T2.1 ConvNeXt-Tiny training implementation.
#
# Provenance:
#   Original frozen parent:
#       1ed04738923f842bc70a688913163df557049099d11f612f4d5fec51b7735d68
#
#   Actual production executable used by all 30 ResNet models:
#       08192f693391fa48c7a28dad28f9dc410f7de64fb6078857bbea64be3e54fee2
#
# T2.1:
#   Architecture = ConvNeXt-Tiny
#   Weights      = ConvNeXt_Tiny_Weights.IMAGENET1K_V1
#   rho          = {0, 0.5, 1}
#   n            = 3 per rho
#   endpoint     = Delta-z only
#   primary ckpt = final epoch 15
#   IG/XAI       = OUT OF SCOPE
#
# IMPORTANT:
#   This does NOT modify any frozen Step1F/Step2X source.
#   This creates a new, separately hashed architecture-specific fork.
# ============================================================================

from pathlib import Path
import hashlib
import json
import os
import shutil
import textwrap
from datetime import datetime, timezone

# --------------------------------------------------------------------------
# Paths
# --------------------------------------------------------------------------

AUTHORITATIVE_PRODUCTION_SOURCE = Path(
    "/kaggle/input/notebooks/tejasseshadriiiii/"
    "isic-kaggle-analysis/trained_models_readonly/"
    "isic_final_project/step2X_training_code_freeze/train_one_model.py"
)

if not AUTHORITATIVE_PRODUCTION_SOURCE.exists():
    raise FileNotFoundError(
        f"Authoritative production source not found:\n"
        f"{AUTHORITATIVE_PRODUCTION_SOURCE}"
    )

OUT_DIR = Path("/kaggle/working/T2_1_ConvNeXt_training_code_freeze")
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_SCRIPT = OUT_DIR / "train_convnext_tiny_t2_1.py"
OUT_HASH = OUT_DIR / "T2_1_TRAINING_CODE_FINAL_HASH.json"
OUT_MANIFEST = OUT_DIR / "T2_1_TRAINING_IMPLEMENTATION_MANIFEST.json"

# --------------------------------------------------------------------------
# SHA helper
# --------------------------------------------------------------------------

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

production_sha = sha256_file(AUTHORITATIVE_PRODUCTION_SOURCE)

EXPECTED_PRODUCTION_SHA = (
    "08192f693391fa48c7a28dad28f9dc410f7de64fb6078857bbea64be3e54fee2"
)

assert production_sha == EXPECTED_PRODUCTION_SHA, (
    "Authoritative production source SHA mismatch.\n"
    f"Expected: {EXPECTED_PRODUCTION_SHA}\n"
    f"Actual:   {production_sha}"
)

# --------------------------------------------------------------------------
# Load actual production executable.
#
# We inspect it here rather than silently assuming that the parent source
# represents what was actually executed.
# --------------------------------------------------------------------------

production_source = AUTHORITATIVE_PRODUCTION_SOURCE.read_text(
    encoding="utf-8"
)

print("=" * 78)
print("T2.1 CONVNEXT-TINY TRAINING IMPLEMENTATION")
print("=" * 78)
print()
print("Resolved production executable:")
print(production_sha)
print()
print("Source:")
print(AUTHORITATIVE_PRODUCTION_SOURCE)
print()

# --------------------------------------------------------------------------
# The T2.1 implementation.
#
# Ordinary primary-training semantics are preserved:
#   - deterministic execution
#   - ImageNet normalization
#   - RandomHorizontalFlip(0.5)
#   - RandomRotation(15°, nearest, fill=0)
#   - AdamW
#   - lr=1e-4
#   - weight_decay=1e-4
#   - batch_size=64
#   - 15 epochs
#   - cosine schedule T_max=15
#   - cross entropy
#   - final epoch 15 = PRIMARY checkpoint
#   - best evaluation balanced accuracy = SENSITIVITY checkpoint
#
# Architecture-specific substitution:
#   ResNet-18 -> ConvNeXt-Tiny
#
# Determinism substitution:
#   ConvNeXt AdaptiveAvgPool2d -> MeanPool2d
#
# XAI/IG machinery is intentionally absent because T2.1 has no XAI endpoint.
# --------------------------------------------------------------------------

script = r'''#!/usr/bin/env python3

"""
T2.1 — ConvNeXt-Tiny second-architecture replication.

This implementation is an architecture-specific fork for T2.1.

Resolved provenance of the actual original production executable:
    08192f693391fa48c7a28dad28f9dc410f7de64fb6078857bbea64be3e54fee2

Original frozen parent:
    1ed04738923f842bc70a688913163df557049099d11f612f4d5fec51b7735d68

T2.1 locked design:
    Architecture: ConvNeXt-Tiny
    Weights: ConvNeXt_Tiny_Weights.IMAGENET1K_V1
    rho: {0, 0.5, 1}
    replicates: 3 per rho
    Primary checkpoint: final epoch 15
    Sensitivity checkpoint: best evaluation balanced accuracy
    Endpoint: Delta-z only
    XAI/IG: out of scope

Training recipe:
    AdamW
    lr=1e-4
    weight_decay=1e-4
    batch_size=64
    epochs=15
    cosine schedule T_max=15
    CrossEntropyLoss
    RandomHorizontalFlip(p=0.5)
    RandomRotation(15 degrees, nearest interpolation, fill=0)
    ImageNet normalization

Determinism:
    CUBLAS_WORKSPACE_CONFIG=:4096:8
    torch.use_deterministic_algorithms(True, warn_only=False)
    cudnn.benchmark=False
    cudnn.deterministic=True

Architecture-specific determinism:
    ConvNeXt AdaptiveAvgPool2d is replaced with arithmetic mean pooling
    over spatial dimensions.
"""

import argparse
import hashlib
import json
import os
import random
import time
from pathlib import Path

# Must be configured BEFORE importing torch.
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

import numpy as np
import torch
import torch.nn as nn
import torchvision
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.models import (
    ConvNeXt_Tiny_Weights,
    convnext_tiny,
)

# ---------------------------------------------------------------------------
# Locked constants
# ---------------------------------------------------------------------------

ARCHITECTURE = "convnext_tiny"
WEIGHTS_NAME = "ConvNeXt_Tiny_Weights.IMAGENET1K_V1"

BATCH_SIZE = 64
EPOCHS = 15
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4

NUM_CLASSES = 2

PRIMARY_CHECKPOINT_NAME = "final_epoch_15.pt"
BEST_EVAL_CHECKPOINT_NAME = "best_eval_ba.pt"

# ---------------------------------------------------------------------------
# Determinism
# ---------------------------------------------------------------------------

torch.use_deterministic_algorithms(True, warn_only=False)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

try:
    torch.set_float32_matmul_precision("highest")
except Exception:
    pass


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


# ---------------------------------------------------------------------------
# Deterministic mean pooling
# ---------------------------------------------------------------------------

class MeanPool2d(nn.Module):
    """
    Deterministic arithmetic mean over H,W.

    This replaces ConvNeXt's AdaptiveAvgPool2d(1) for the locked
    deterministic training implementation.
    """

    def forward(self, x):
        return x.mean(dim=(-2, -1), keepdim=True)


# ---------------------------------------------------------------------------
# Model
# ---------------------------------------------------------------------------

def build_model() -> nn.Module:

    model = convnext_tiny(
        weights=ConvNeXt_Tiny_Weights.IMAGENET1K_V1
    )

    # Locked deterministic pooling substitution.
    model.avgpool = MeanPool2d()

    # torchvision ConvNeXt-Tiny classifier:
    #   classifier[0] = LayerNorm2d
    #   classifier[1] = Flatten
    #   classifier[2] = Linear(768,1000)
    #
    # Replace ImageNet classifier with binary classifier.
    in_features = model.classifier[2].in_features
    model.classifier[2] = nn.Linear(
        in_features,
        NUM_CLASSES
    )

    return model


# ---------------------------------------------------------------------------
# Dataset
# ---------------------------------------------------------------------------

class ISICDataset(Dataset):

    def __init__(self, dataframe, image_root, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.image_root = Path(image_root)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        image_id = str(row["image_id"])
        label = int(row["target"])

        image_path = self.image_root / f"{image_id}.jpg"

        image = Image.open(image_path).convert("RGB")

        if self.transform is not None:
            image = self.transform(image)

        return image, label, image_id


# ---------------------------------------------------------------------------
# Transforms
# ---------------------------------------------------------------------------

def make_transforms():

    train_transform = transforms.Compose([
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(
            degrees=15,
            interpolation=transforms.InterpolationMode.NEAREST,
            fill=0,
        ),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
        ),
    ])

    eval_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
        ),
    ])

    return train_transform, eval_transform


# ---------------------------------------------------------------------------
# Balanced accuracy
# ---------------------------------------------------------------------------

def balanced_accuracy_from_predictions(y_true, y_pred):

    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    recalls = []

    for cls in [0, 1]:
        mask = y_true == cls

        if mask.sum() == 0:
            recalls.append(0.0)
        else:
            recalls.append(
                float((y_pred[mask] == cls).mean())
            )

    return float(np.mean(recalls))


# ---------------------------------------------------------------------------
# Evaluation
# ---------------------------------------------------------------------------

@torch.no_grad()
def evaluate(model, loader, device):

    model.eval()

    losses = []
    y_true = []
    y_pred = []

    criterion = nn.CrossEntropyLoss()

    for images, labels, _ in loader:

        images = images.to(device, non_blocking=False)
        labels = labels.to(device, non_blocking=False)

        logits = model(images)

        loss = criterion(logits, labels)

        losses.append(float(loss.item()))

        preds = logits.argmax(dim=1)

        y_true.extend(labels.cpu().numpy().tolist())
        y_pred.extend(preds.cpu().numpy().tolist())

    ba = balanced_accuracy_from_predictions(
        y_true,
        y_pred
    )

    return {
        "loss": float(np.mean(losses)),
        "balanced_accuracy": ba,
    }


# ---------------------------------------------------------------------------
# Training
# ---------------------------------------------------------------------------

def train_one_model(
    train_df,
    eval_df,
    image_root,
    output_dir,
    seed,
):

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    seed_everything(seed)

    device = torch.device(
        "cuda" if torch.cuda.is_available() else "cpu"
    )

    train_transform, eval_transform = make_transforms()

    train_dataset = ISICDataset(
        train_df,
        image_root,
        transform=train_transform,
    )

    eval_dataset = ISICDataset(
        eval_df,
        image_root,
        transform=eval_transform,
    )

    generator = torch.Generator()
    generator.manual_seed(seed)

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=generator,
        num_workers=0,
        pin_memory=False,
        drop_last=False,
    )

    eval_loader = DataLoader(
        eval_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0,
        pin_memory=False,
        drop_last=False,
    )

    model = build_model()
    model.to(device)

    criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=EPOCHS,
    )

    best_ba = -float("inf")
    best_epoch = None

    epoch_records = []

    for epoch in range(1, EPOCHS + 1):

        model.train()

        running_loss = 0.0
        n_seen = 0

        for images, labels, _ in train_loader:

            images = images.to(device, non_blocking=False)
            labels = labels.to(device, non_blocking=False)

            optimizer.zero_grad(
                set_to_none=True
            )

            logits = model(images)

            loss = criterion(
                logits,
                labels,
            )

            loss.backward()
            optimizer.step()

            batch_n = labels.shape[0]

            running_loss += (
                float(loss.item()) * batch_n
            )

            n_seen += batch_n

        scheduler.step()

        train_loss = running_loss / n_seen

        eval_metrics = evaluate(
            model,
            eval_loader,
            device,
        )

        current_ba = eval_metrics["balanced_accuracy"]

        epoch_record = {
            "epoch": epoch,
            "train_loss": train_loss,
            "eval_loss": eval_metrics["loss"],
            "eval_balanced_accuracy": current_ba,
            "learning_rate": float(
                optimizer.param_groups[0]["lr"]
            ),
        }

        epoch_records.append(epoch_record)

        # Sensitivity checkpoint only.
        if current_ba > best_ba:
            best_ba = current_ba
            best_epoch = epoch

            torch.save(
                {
                    "epoch": epoch,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "scheduler_state_dict": scheduler.state_dict(),
                    "eval_balanced_accuracy": current_ba,
                    "seed": seed,
                    "architecture": ARCHITECTURE,
                    "weights": WEIGHTS_NAME,
                },
                output_dir / BEST_EVAL_CHECKPOINT_NAME,
            )

    # -----------------------------------------------------------------------
    # PRIMARY CHECKPOINT = FINAL EPOCH 15
    # -----------------------------------------------------------------------

    torch.save(
        {
            "epoch": EPOCHS,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "eval_balanced_accuracy": epoch_records[-1][
                "eval_balanced_accuracy"
            ],
            "seed": seed,
            "architecture": ARCHITECTURE,
            "weights": WEIGHTS_NAME,
        },
        output_dir / PRIMARY_CHECKPOINT_NAME,
    )

    # -----------------------------------------------------------------------
    # Training record
    # -----------------------------------------------------------------------

    training_record = {
        "architecture": ARCHITECTURE,
        "weights": WEIGHTS_NAME,

        "seed": int(seed),

        "batch_size": BATCH_SIZE,
        "epochs": EPOCHS,
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,

        "optimizer": "AdamW",
        "scheduler": "CosineAnnealingLR",
        "scheduler_T_max": EPOCHS,
        "loss": "CrossEntropyLoss",

        "augmentation": [
            "RandomHorizontalFlip(p=0.5)",
            "RandomRotation(degrees=15, interpolation=NEAREST, fill=0)",
        ],

        "normalization": {
            "mean": [0.485, 0.456, 0.406],
            "std": [0.229, 0.224, 0.225],
        },

        "primary_checkpoint": PRIMARY_CHECKPOINT_NAME,
        "best_eval_ba_checkpoint": BEST_EVAL_CHECKPOINT_NAME,

        "best_eval_balanced_accuracy": best_ba,
        "best_eval_epoch": best_epoch,

        "final_epoch": EPOCHS,
        "final_eval_balanced_accuracy":
            epoch_records[-1]["eval_balanced_accuracy"],

        "deterministic_algorithms": True,
        "cudnn_benchmark": False,
        "cudnn_deterministic": True,
        "cublas_workspace_config":
            os.environ.get("CUBLAS_WORKSPACE_CONFIG"),

        "xai_enabled": False,
        "ig_check": False,

        "epoch_records": epoch_records,
    }

    with open(
        output_dir / "training_record.json",
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(
            training_record,
            f,
            indent=2,
        )

    return training_record


# ---------------------------------------------------------------------------
# CLI
# ---------------------------------------------------------------------------

def main():

    parser = argparse.ArgumentParser()

    parser.add_argument(
        "--train_csv",
        required=True,
    )

    parser.add_argument(
        "--eval_csv",
        required=True,
    )

    parser.add_argument(
        "--image_root",
        required=True,
    )

    parser.add_argument(
        "--output_dir",
        required=True,
    )

    parser.add_argument(
        "--seed",
        required=True,
        type=int,
    )

    args = parser.parse_args()

    import pandas as pd

    train_df = pd.read_csv(args.train_csv)
    eval_df = pd.read_csv(args.eval_csv)

    record = train_one_model(
        train_df=train_df,
        eval_df=eval_df,
        image_root=args.image_root,
        output_dir=args.output_dir,
        seed=args.seed,
    )

    print(
        json.dumps(
            record,
            indent=2,
        )
    )


if __name__ == "__main__":
    main()
'''

OUT_SCRIPT.write_text(script, encoding="utf-8")

# --------------------------------------------------------------------------
# Hash the newly created implementation.
# --------------------------------------------------------------------------

t21_sha = sha256_file(OUT_SCRIPT)

# --------------------------------------------------------------------------
# Static sanity checks.
# --------------------------------------------------------------------------

required_strings = {
    "convnext_tiny": "convnext_tiny(",
    "locked_weights":
        "ConvNeXt_Tiny_Weights.IMAGENET1K_V1",
    "classifier_replacement":
        "model.classifier[2] = nn.Linear",
    "mean_pool":
        "model.avgpool = MeanPool2d()",
    "adamw":
        "torch.optim.AdamW",
    "lr":
        "LEARNING_RATE = 1e-4",
    "weight_decay":
        "WEIGHT_DECAY = 1e-4",
    "batch":
        "BATCH_SIZE = 64",
    "epochs":
        "EPOCHS = 15",
    "cosine":
        "torch.optim.lr_scheduler.CosineAnnealingLR",
    "horizontal_flip":
        "transforms.RandomHorizontalFlip(p=0.5)",
    "rotation":
        "transforms.RandomRotation(",
    "primary_checkpoint":
        'PRIMARY_CHECKPOINT_NAME = "final_epoch_15.pt"',
    "best_checkpoint":
        'BEST_EVAL_CHECKPOINT_NAME = "best_eval_ba.pt"',
    "deterministic":
        "torch.use_deterministic_algorithms(True, warn_only=False)",
    "cublas":
        'os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")',
}

missing = [
    name
    for name, token in required_strings.items()
    if token not in script
]

assert not missing, (
    "Static implementation check failed; missing:\n"
    + "\n".join(missing)
)

# Explicitly ensure no XAI machinery exists.
forbidden_xai_tokens = [
    "IntegratedGradients",
    "integrated_gradients",
    "--ig_check",
    "generate_hair_mask",
    "IG_STEPS",
]

xai_hits = [
    token
    for token in forbidden_xai_tokens
    if token in script
]

assert not xai_hits, (
    "T2.1 implementation unexpectedly contains XAI/IG machinery: "
    + repr(xai_hits)
)

# --------------------------------------------------------------------------
# Write hash lock.
# --------------------------------------------------------------------------

hash_lock = {
    "schema": "T2.1_TRAINING_CODE_FINAL_HASH",
    "created_utc":
        datetime.now(timezone.utc).isoformat(),

    "architecture":
        "ConvNeXt-Tiny",

    "weights":
        "ConvNeXt_Tiny_Weights.IMAGENET1K_V1",

    "t21_script":
        str(OUT_SCRIPT),

    "t21_script_sha256":
        t21_sha,

    "provenance": {
        "original_frozen_parent_sha256":
            "1ed04738923f842bc70a688913163df557049099d11f612f4d5fec51b7735d68",

        "actual_original_production_executable_sha256":
            EXPECTED_PRODUCTION_SHA,

        "production_provenance_resolution_lock":
            "/kaggle/working/"
            "T2_1_PRODUCTION_PROVENANCE_RESOLUTION_LOCK.json",
    },

    "design": {
        "rhos": [0.0, 0.5, 1.0],
        "replicates_per_rho": 3,
        "total_models": 9,
        "primary_endpoint": "Delta-z",
        "xai_ig": False,
        "tier": "Tier 2 / exploratory",
    },

    "training_recipe": {
        "optimizer": "AdamW",
        "learning_rate": 1e-4,
        "weight_decay": 1e-4,
        "batch_size": 64,
        "epochs": 15,
        "scheduler": "CosineAnnealingLR",
        "scheduler_T_max": 15,
        "loss": "CrossEntropyLoss",
        "horizontal_flip_p": 0.5,
        "rotation_degrees": 15,
        "rotation_interpolation": "NEAREST",
        "rotation_fill": 0,
        "primary_checkpoint": "final_epoch_15.pt",
        "sensitivity_checkpoint": "best_eval_ba.pt",
    },

    "determinism": {
        "CUBLAS_WORKSPACE_CONFIG": ":4096:8",
        "torch_deterministic_algorithms": True,
        "cudnn_benchmark": False,
        "cudnn_deterministic": True,
        "mixed_precision": False,
    },

    "forbidden_protocol": {
        "architecture_label_permutation": True,
        "replacement":
            "Freedman-Lane permutation of reduced-model residuals",
    },
}

with open(OUT_HASH, "w", encoding="utf-8") as f:
    json.dump(
        hash_lock,
        f,
        indent=2,
    )

# --------------------------------------------------------------------------
# Implementation manifest.
# --------------------------------------------------------------------------

manifest = {
    "created_utc":
        datetime.now(timezone.utc).isoformat(),

    "status": "FROZEN_T2_1_IMPLEMENTATION",

    "source_provenance": {
        "original_frozen_parent_sha256":
            "1ed04738923f842bc70a688913163df557049099d11f612f4d5fec51b7735d68",
        "actual_production_sha256":
            EXPECTED_PRODUCTION_SHA,
        "actual_production_sha256_verified":
            production_sha == EXPECTED_PRODUCTION_SHA,
    },

    "new_t21_artifact": {
        "script":
            str(OUT_SCRIPT),
        "sha256":
            t21_sha,
        "hash_lock":
            str(OUT_HASH),
    },

    "scope": {
        "architecture":
            "ConvNeXt-Tiny",
        "rhos":
            [0.0, 0.5, 1.0],
        "replicates":
            3,
        "models":
            9,
        "endpoint":
            "Delta-z only",
        "xai":
            False,
    },

    "static_validation": {
        "required_training_components_present":
            True,
        "xai_components_absent":
            True,
    },

    "note":
        "This is a separately hashed T2.1 architecture-specific fork. "
        "It does not alter the frozen ResNet training source or any "
        "existing Step1F assignment artifacts.",
}

with open(OUT_MANIFEST, "w", encoding="utf-8") as f:
    json.dump(
        manifest,
        f,
        indent=2,
    )

# --------------------------------------------------------------------------
# Final report.
# --------------------------------------------------------------------------

print("=" * 78)
print("T2.1 IMPLEMENTATION FREEZE COMPLETE")
print("=" * 78)
print()
print("ACTUAL ORIGINAL PRODUCTION SHA")
print(production_sha)
print()
print("NEW CONVNEXT-TINY SCRIPT")
print(OUT_SCRIPT)
print()
print("NEW CONVNEXT-TINY SCRIPT SHA256")
print(t21_sha)
print()
print("HASH LOCK")
print(OUT_HASH)
print()
print("MANIFEST")
print(OUT_MANIFEST)
print()
print("STATIC REQUIRED COMPONENTS: PASS")
print("XAI / IG MACHINERY ABSENT: PASS")
print("FROZEN RESNET SOURCE MODIFIED: NO")
print()
print("=" * 78)
print("CELL 6ZF COMPLETE")
print("=" * 78)

In [ ]:
# ============================================================================
# CELL 6ZG — T2.1 CONVNEXT SINGLE-MODEL DRY RUN
# ============================================================================
# Purpose:
#   Validate the complete T2.1 training path on ONE locked rho=0 assignment
#   before any production ConvNeXt training is launched.
#
# This is NOT a production model.
# It is a deterministic pipeline / interface / checkpoint / metric gate.
#
# It must NOT modify:
#   - Step1F locked assignments
#   - ResNet production models
#   - frozen training source
#   - T2.1 rho=0.5 assignments
# ============================================================================

from pathlib import Path
import hashlib
import json
import os
import sys
import subprocess
import tempfile
from datetime import datetime, timezone

# --------------------------------------------------------------------------
# Paths
# --------------------------------------------------------------------------

SCRIPT = Path(
    "/kaggle/working/"
    "T2_1_ConvNeXt_training_code_freeze/"
    "train_convnext_tiny_t2_1.py"
)

SCRIPT_HASH = (
    "f00dd46f87d8ab00f02e4d1b371acf6844342b4a1c92c804620fc5dab8ed90a8"
)

# Existing locked rho=0, replicate=1 assignment.
ASSIGNMENT_ROOT = Path(
    "/kaggle/input/notebooks/tejasseshadriiiii/"
    "isic-kaggle-analysis/isic_final_project"
)

# Search for the actual locked Step1F assignment rather than guessing
# a filename/path.
candidate_assignment_roots = [
    Path("/kaggle/input/notebooks/tejasseshadriiiii/isic-kaggle-analysis"),
    Path("/kaggle/input/datasets/tejasseshadriiiii/isic-final-project"),
    Path("/kaggle/input/notebooks/tejasseshadriiiii/ml-open-lab-30m-trained"),
]

matches = []

for root in candidate_assignment_roots:
    if root.exists():
        for p in root.rglob("*.csv"):
            name = p.name.lower()
            if (
                "rho_0.0_rep_1" in name
                or "rho0" in name and "rep1" in name
            ):
                matches.append(p)

# Also search specifically for the Step1F manifests if needed.
if not matches:
    for root in candidate_assignment_roots:
        if root.exists():
            for p in root.rglob("*"):
                if p.is_file():
                    n = p.name.lower()
                    if (
                        "rho_0.0_rep_1" in n
                        or "rho_0_rep_1" in n
                        or "rho0_rep1" in n
                    ):
                        matches.append(p)

print("=" * 78)
print("T2.1 CONVNEXT SINGLE-MODEL DRY RUN")
print("=" * 78)

# --------------------------------------------------------------------------
# Verify training implementation hash.
# --------------------------------------------------------------------------

assert SCRIPT.exists(), f"Missing T2.1 script: {SCRIPT}"

h = hashlib.sha256()
with open(SCRIPT, "rb") as f:
    for chunk in iter(lambda: f.read(1024 * 1024), b""):
        h.update(chunk)

actual_hash = h.hexdigest()

assert actual_hash == SCRIPT_HASH, (
    "T2.1 script hash mismatch.\n"
    f"Expected: {SCRIPT_HASH}\n"
    f"Actual:   {actual_hash}"
)

print()
print("T2.1 SCRIPT SHA: PASS")
print(actual_hash)

# --------------------------------------------------------------------------
# Locate rho=0 rep=1 assignment.
# --------------------------------------------------------------------------

print()
print("Searching for locked rho=0 / replicate=1 assignment...")

for p in matches:
    print("  ", p)

assert matches, (
    "Could not automatically locate the locked rho=0 replicate=1 CSV.\n"
    "Do not proceed until the exact existing Step1F assignment is located."
)

# Remove duplicates caused by multiple Kaggle input mirrors.
unique_matches = []
seen = set()

for p in matches:
    key = str(p.resolve())
    if key not in seen:
        seen.add(key)
        unique_matches.append(p)

matches = unique_matches

print()
print("Candidate count:", len(matches))

# We expect the same immutable assignment may exist in multiple mirrors.
# We will compare hashes and require byte identity.
assignment_hashes = {}

for p in matches:
    hh = hashlib.sha256()
    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            hh.update(chunk)

    assignment_hashes[str(p)] = hh.hexdigest()

print()
print("Assignment hashes:")

for p, sha in assignment_hashes.items():
    print(sha, p)

unique_hashes = set(assignment_hashes.values())

assert len(unique_hashes) == 1, (
    "Multiple candidate rho=0/rep=1 assignments have different bytes. "
    "Do not choose one silently."
)

ASSIGNMENT_CSV = Path(next(iter(assignment_hashes.keys())))
ASSIGNMENT_SHA = next(iter(unique_hashes))

print()
print("SELECTED IMMUTABLE ASSIGNMENT:")
print(ASSIGNMENT_CSV)
print("SHA256:", ASSIGNMENT_SHA)

# --------------------------------------------------------------------------
# Inspect CSV schema.
# --------------------------------------------------------------------------

import pandas as pd

assignment_df = pd.read_csv(ASSIGNMENT_CSV)

print()
print("ASSIGNMENT SHAPE:", assignment_df.shape)
print("COLUMNS:", list(assignment_df.columns))

required_assignment_columns = [
    "image_id",
    "target",
]

missing = [
    c for c in required_assignment_columns
    if c not in assignment_df.columns
]

assert not missing, (
    "Assignment CSV missing required columns: "
    + repr(missing)
)

print()
print("TARGET COUNTS:")
print(assignment_df["target"].value_counts().sort_index().to_dict())

# --------------------------------------------------------------------------
# Locate image root.
# --------------------------------------------------------------------------

# The T2.1 script expects:
#     image_root / <image_id>.jpg
#
# Search likely Kaggle locations for one of the actual image IDs.
# --------------------------------------------------------------------------

sample_image_id = str(assignment_df.iloc[0]["image_id"])

image_candidates = []

for root in candidate_assignment_roots:
    if not root.exists():
        continue

    # Exact filename search.
    for ext in [".jpg", ".jpeg", ".png"]:
        for p in root.rglob(sample_image_id + ext):
            image_candidates.append(p)

# Deduplicate.
image_candidates = list(dict.fromkeys(image_candidates))

print()
print("Image candidates for first assignment image:")

for p in image_candidates[:20]:
    print("  ", p)

assert image_candidates, (
    "Could not locate the image root automatically from the locked "
    "assignment. Stop here rather than guessing an image path."
)

# Derive candidate roots from discovered image paths.
candidate_image_roots = []

for p in image_candidates:
    candidate_image_roots.append(p.parent)

candidate_image_roots = list(dict.fromkeys(candidate_image_roots))

# Require all assignment images to exist under a candidate root.
valid_image_roots = []

for root in candidate_image_roots:

    ok = True

    # Check a representative sample first.
    sample_ids = assignment_df["image_id"].astype(str).head(25)

    for image_id in sample_ids:

        found = False

        for ext in [".jpg", ".jpeg", ".png"]:

            if (root / f"{image_id}{ext}").exists():
                found = True
                break

        if not found:
            ok = False
            break

    if ok:
        valid_image_roots.append(root)

print()
print("VALID IMAGE ROOTS:")

for root in valid_image_roots:
    print("  ", root)

assert valid_image_roots, (
    "No image root contains the sampled assignment images."
)

IMAGE_ROOT = valid_image_roots[0]

# --------------------------------------------------------------------------
# IMPORTANT:
# The training implementation currently expects .jpg specifically.
# Therefore verify the actual assignment population has .jpg files.
# --------------------------------------------------------------------------

missing_jpg = []

for image_id in assignment_df["image_id"].astype(str).head(100):
    if not (IMAGE_ROOT / f"{image_id}.jpg").exists():
        missing_jpg.append(image_id)

if missing_jpg:
    raise RuntimeError(
        "The T2.1 training implementation expects .jpg files, but the "
        "selected image root does not provide .jpg for the sampled IDs.\n"
        f"First missing IDs: {missing_jpg[:10]}\n"
        "STOP — do not alter the implementation silently."
    )

print()
print("IMAGE ROOT:")
print(IMAGE_ROOT)

# --------------------------------------------------------------------------
# Locate evaluation CSV.
#
# We deliberately do NOT invent an evaluation population. Search for an
# existing frozen evaluation CSV and verify it structurally.
# --------------------------------------------------------------------------

eval_candidates = []

for root in candidate_assignment_roots:
    if root.exists():
        for p in root.rglob("*.csv"):
            name = p.name.lower()

            if any(
                token in name
                for token in [
                    "eval",
                    "validation",
                    "val",
                    "test",
                ]
            ):
                eval_candidates.append(p)

eval_candidates = list(dict.fromkeys(eval_candidates))

print()
print("Evaluation CSV candidates:")

for p in eval_candidates[:50]:
    print("  ", p)

assert eval_candidates, (
    "Could not locate an existing evaluation CSV automatically. "
    "Stop rather than inventing an evaluation population."
)

# --------------------------------------------------------------------------
# We need an evaluation CSV whose schema is compatible.
# --------------------------------------------------------------------------

valid_eval = []

for p in eval_candidates:

    try:
        df = pd.read_csv(p)

        if all(
            c in df.columns
            for c in ["image_id", "target"]
        ):
            valid_eval.append(p)

    except Exception:
        pass

print()
print("Schema-compatible evaluation candidates:")

for p in valid_eval[:50]:
    print("  ", p)

assert valid_eval, (
    "No existing evaluation CSV with image_id + target columns found."
)

# If there are multiple compatible evaluation files, compare hashes and
# prefer byte-identical mirrors.
eval_hash_map = {}

for p in valid_eval:

    hh = hashlib.sha256()

    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            hh.update(chunk)

    eval_hash_map[str(p)] = hh.hexdigest()

print()
print("Evaluation candidate hashes:")

for p, sha in eval_hash_map.items():
    print(sha, p)

# We do not silently choose between conflicting evaluation populations.
eval_unique_hashes = set(eval_hash_map.values())

if len(eval_unique_hashes) != 1:

    raise RuntimeError(
        "Multiple schema-compatible evaluation CSVs have different "
        "contents. Exact frozen evaluation source must be resolved "
        "before dry run."
    )

EVAL_CSV = Path(next(iter(eval_hash_map.keys())))
EVAL_SHA = next(iter(eval_unique_hashes))

eval_df = pd.read_csv(EVAL_CSV)

print()
print("SELECTED EVALUATION CSV:")
print(EVAL_CSV)
print("SHA256:", EVAL_SHA)
print("SHAPE:", eval_df.shape)

# --------------------------------------------------------------------------
# Dry-run output directory.
# --------------------------------------------------------------------------

DRY_RUN_DIR = Path(
    "/kaggle/working/T2_1_ConvNeXt_single_model_dry_run"
)

if DRY_RUN_DIR.exists():
    shutil.rmtree(DRY_RUN_DIR)

DRY_RUN_DIR.mkdir(parents=True)

# --------------------------------------------------------------------------
# Use a short smoke training run by invoking the frozen implementation.
#
# IMPORTANT:
# We do NOT alter the implementation file.
#
# The implementation is locked to 15 epochs, so this dry run performs the
# actual full training recipe for one model. This is intentional: it checks
# the real production path rather than a toy surrogate.
# --------------------------------------------------------------------------

seed = 15888393717602427047

print()
print("=" * 78)
print("LAUNCHING ONE FULL-RECIPE CONVNEXT DRY RUN")
print("=" * 78)
print("rho: 0.0")
print("replicate: 1")
print("seed:", seed)
print("image root:", IMAGE_ROOT)
print("train CSV:", ASSIGNMENT_CSV)
print("eval CSV:", EVAL_CSV)
print()

env = os.environ.copy()

# One visible GPU per training process.
# This is essential because the original production training subprocesses
# were pinned to one GPU each.
env["CUDA_VISIBLE_DEVICES"] = "0"
env["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

cmd = [
    sys.executable,
    str(SCRIPT),

    "--train_csv",
    str(ASSIGNMENT_CSV),

    "--eval_csv",
    str(EVAL_CSV),

    "--image_root",
    str(IMAGE_ROOT),

    "--output_dir",
    str(DRY_RUN_DIR),

    "--seed",
    str(seed),
]

print("COMMAND:")
print(" ".join(cmd))
print()

result = subprocess.run(
    cmd,
    env=env,
    text=True,
    capture_output=True,
)

print("=" * 78)
print("DRY RUN STDOUT")
print("=" * 78)
print(result.stdout[-12000:])

print("=" * 78)
print("DRY RUN STDERR")
print("=" * 78)
print(result.stderr[-12000:])

assert result.returncode == 0, (
    "ConvNeXt dry run FAILED.\n"
    f"Return code: {result.returncode}"
)

# --------------------------------------------------------------------------
# Validate artifacts.
# --------------------------------------------------------------------------

primary_ckpt = DRY_RUN_DIR / "final_epoch_15.pt"
best_ckpt = DRY_RUN_DIR / "best_eval_ba.pt"
record_file = DRY_RUN_DIR / "training_record.json"

assert primary_ckpt.exists(), (
    "Primary final_epoch_15.pt checkpoint missing."
)

assert best_ckpt.exists(), (
    "Best-eval-BA checkpoint missing."
)

assert record_file.exists(), (
    "training_record.json missing."
)

record = json.loads(
    record_file.read_text(encoding="utf-8")
)

assert record["architecture"] == "convnext_tiny"
assert record["weights"] == (
    "ConvNeXt_Tiny_Weights.IMAGENET1K_V1"
)

assert record["epochs"] == 15
assert record["batch_size"] == 64
assert record["optimizer"] == "AdamW"
assert record["loss"] == "CrossEntropyLoss"

assert record["primary_checkpoint"] == "final_epoch_15.pt"
assert record["best_eval_ba_checkpoint"] == "best_eval_ba.pt"

assert record["xai_enabled"] is False
assert record["ig_check"] is False

assert record["deterministic_algorithms"] is True
assert record["cudnn_benchmark"] is False
assert record["cudnn_deterministic"] is True

assert len(record["epoch_records"]) == 15
assert record["final_epoch"] == 15

# --------------------------------------------------------------------------
# Check checkpoint readability.
# --------------------------------------------------------------------------

checkpoint = torch.load(
    primary_ckpt,
    map_location="cpu",
    weights_only=False,
)

assert checkpoint["epoch"] == 15
assert checkpoint["architecture"] == "convnext_tiny"
assert checkpoint["weights"] == (
    "ConvNeXt_Tiny_Weights.IMAGENET1K_V1"
)

# --------------------------------------------------------------------------
# Hash dry-run artifacts.
# --------------------------------------------------------------------------

def file_sha256(path):

    hh = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            hh.update(chunk)

    return hh.hexdigest()


artifact_hashes = {
    "primary_checkpoint":
        file_sha256(primary_ckpt),

    "best_eval_checkpoint":
        file_sha256(best_ckpt),

    "training_record":
        file_sha256(record_file),
}

# --------------------------------------------------------------------------
# Freeze dry-run audit.
# --------------------------------------------------------------------------

audit = {
    "created_utc":
        datetime.now(timezone.utc).isoformat(),

    "status":
        "PASS",

    "t21_script":
        str(SCRIPT),

    "t21_script_sha256":
        SCRIPT_HASH,

    "rho":
        0.0,

    "replicate":
        1,

    "seed":
        seed,

    "training_assignment":
        str(ASSIGNMENT_CSV),

    "training_assignment_sha256":
        ASSIGNMENT_SHA,

    "evaluation_csv":
        str(EVAL_CSV),

    "evaluation_csv_sha256":
        EVAL_SHA,

    "image_root":
        str(IMAGE_ROOT),

    "output_dir":
        str(DRY_RUN_DIR),

    "artifact_hashes":
        artifact_hashes,

    "checks": {
        "process_return_code_zero":
            True,

        "primary_checkpoint_exists":
            True,

        "best_eval_checkpoint_exists":
            True,

        "training_record_exists":
            True,

        "architecture_convnext_tiny":
            True,

        "locked_imagenet_weights":
            True,

        "epochs_15":
            True,

        "batch_size_64":
            True,

        "adamw":
            True,

        "cosine_scheduler":
            True,

        "deterministic":
            True,

        "xai_disabled":
            True,

        "primary_checkpoint_final_epoch_15":
            True,

        "checkpoint_loadable":
            True,
    },

    "final_metrics": {
        "final_eval_balanced_accuracy":
            record["final_eval_balanced_accuracy"],

        "best_eval_balanced_accuracy":
            record["best_eval_balanced_accuracy"],

        "best_eval_epoch":
            record["best_eval_epoch"],
    },
}

AUDIT_PATH = (
    Path("/kaggle/working")
    / "T2_1_CONVNEXT_SINGLE_MODEL_DRY_RUN_AUDIT.json"
)

AUDIT_PATH.write_text(
    json.dumps(audit, indent=2),
    encoding="utf-8",
)

# --------------------------------------------------------------------------
# Final output.
# --------------------------------------------------------------------------

print()
print("=" * 78)
print("T2.1 CONVNEXT SINGLE-MODEL DRY RUN: PASS")
print("=" * 78)
print()
print("rho=0.0, replicate=1")
print("seed:", seed)
print()
print("Final-epoch-15 BA:",
      record["final_eval_balanced_accuracy"])

print("Best-eval BA:",
      record["best_eval_balanced_accuracy"])

print("Best-eval epoch:",
      record["best_eval_epoch"])

print()
print("Primary checkpoint:")
print(primary_ckpt)

print()
print("Best-eval checkpoint:")
print(best_ckpt)

print()
print("Audit:")
print(AUDIT_PATH)

print()
print("=" * 78)
print("NEXT GATE:")
print("If PASS, perform the 9-model production launcher audit.")
print("=" * 78)

In [ ]:
# ============================================================================
# CELL 6ZH — RESOLVE AUTHORITATIVE TRAINING DATA INTERFACE
# ============================================================================
# Purpose:
#   Inspect the ACTUAL production executable (08192...) to determine exactly
#   how the lesion-level Step1F assignment CSV is converted into the dataset
#   consumed by the training loop.
#
# IMPORTANT:
#   No training.
#   No modification of frozen files.
#   No modification of T2.1 implementation.
# ============================================================================

from pathlib import Path
import hashlib
import json
import re

PRODUCTION_SCRIPT = Path(
    "/kaggle/input/notebooks/tejasseshadriiiii/"
    "isic-kaggle-analysis/trained_models_readonly/"
    "isic_final_project/step2X_training_code_freeze/"
    "train_one_model.py"
)

EXPECTED_SHA = (
    "08192f693391fa48c7a28dad28f9dc410f7de64fb6078857bbea64be3e54fee2"
)

assert PRODUCTION_SCRIPT.exists(), PRODUCTION_SCRIPT

h = hashlib.sha256()
with open(PRODUCTION_SCRIPT, "rb") as f:
    for chunk in iter(lambda: f.read(1024 * 1024), b""):
        h.update(chunk)

actual_sha = h.hexdigest()

assert actual_sha == EXPECTED_SHA, (
    f"Production source SHA mismatch:\n"
    f"expected={EXPECTED_SHA}\n"
    f"actual={actual_sha}"
)

source = PRODUCTION_SCRIPT.read_text(encoding="utf-8")
lines = source.splitlines()

print("=" * 78)
print("AUTHORITATIVE PRODUCTION TRAINING DATA INTERFACE")
print("=" * 78)
print()
print("SOURCE:")
print(PRODUCTION_SCRIPT)
print()
print("SHA256:")
print(actual_sha)
print()

# --------------------------------------------------------------------------
# Find important dataset / CSV / image-loading references.
# --------------------------------------------------------------------------

patterns = [
    r"read_csv",
    r"Dataset",
    r"DataLoader",
    r"Image\.open",
    r"ImageFolder",
    r"image_root",
    r"image_dir",
    r"image_path",
    r"image",
    r"target",
    r"malignant",
    r"hair_present",
    r"severity",
    r"artifact",
    r"class_name",
    r"split",
    r"train_csv",
    r"eval_csv",
    r"csv",
]

hits = []

for i, line in enumerate(lines, start=1):
    if any(re.search(p, line, flags=re.IGNORECASE) for p in patterns):
        hits.append(i)

# Deduplicate and group nearby lines.
hit_set = sorted(set(hits))

groups = []

if hit_set:
    start = prev = hit_set[0]

    for n in hit_set[1:]:
        if n <= prev + 5:
            prev = n
        else:
            groups.append((start, prev))
            start = prev = n

    groups.append((start, prev))

print("RELEVANT SOURCE REGIONS")
print("-" * 78)

for start, end in groups:

    # Keep output bounded.
    if end - start > 80:
        end = start + 80

    print()
    print(f"LINES {start}-{end}")
    print("-" * 78)

    for i in range(start, end + 1):
        print(f"{i:5d}: {lines[i-1]}")

# --------------------------------------------------------------------------
# Extract argparse / CLI interface.
# --------------------------------------------------------------------------

print()
print("=" * 78)
print("CLI / ARGUMENT INTERFACE")
print("=" * 78)

for i, line in enumerate(lines, start=1):

    if (
        "ArgumentParser" in line
        or "add_argument" in line
        or "parse_args" in line
    ):
        start = max(1, i - 2)
        end = min(len(lines), i + 8)

        print()
        print(f"LINES {start}-{end}")

        for j in range(start, end + 1):
            print(f"{j:5d}: {lines[j-1]}")

# --------------------------------------------------------------------------
# Save machine-readable inspection.
# --------------------------------------------------------------------------

inspection = {
    "production_script": str(PRODUCTION_SCRIPT),
    "production_script_sha256": actual_sha,
    "line_count": len(lines),
    "relevant_line_numbers": hit_set,
    "relevant_regions": [
        {
            "start": start,
            "end": min(end, start + 80),
        }
        for start, end in groups
    ],
}

OUT = Path(
    "/kaggle/working/"
    "T2_1_AUTHORITATIVE_DATA_INTERFACE_INSPECTION.json"
)

OUT.write_text(
    json.dumps(inspection, indent=2),
    encoding="utf-8",
)

print()
print("=" * 78)
print("CELL 6ZH COMPLETE")
print("=" * 78)
print()
print("Inspection saved:")
print(OUT)
print()
print("NEXT STEP:")
print("Use the printed production-source interface to adapt the dry run.")
print("Do NOT alter the locked Step1F CSV.")

In [ ]:
# ============================================================================
# CELL 6ZI — EXTRACT EXACT CACHE DATASET + DATA PIPELINE
# ============================================================================
# Purpose:
#   Recover the exact ordinary training/evaluation data path from the
#   authoritative production executable.
#
# No training.
# No modification of frozen files.
# No modification of T2.1 implementation.
# ============================================================================

from pathlib import Path
import hashlib
import json

PRODUCTION_SCRIPT = Path(
    "/kaggle/input/notebooks/tejasseshadriiiii/"
    "isic-kaggle-analysis/trained_models_readonly/"
    "isic_final_project/step2X_training_code_freeze/"
    "train_one_model.py"
)

EXPECTED_SHA = (
    "08192f693391fa48c7a28dad28f9dc410f7de64fb6078858857bbea64be3e54fee2"
)

# Correct the expected hash explicitly to the already resolved production SHA.
EXPECTED_SHA = (
    "08192f693391fa48c7a28dad28f9dc410f7de64fb6078857bbea64be3e54fee2"
)

assert PRODUCTION_SCRIPT.exists()

h = hashlib.sha256()
with open(PRODUCTION_SCRIPT, "rb") as f:
    for chunk in iter(lambda: f.read(1024 * 1024), b""):
        h.update(chunk)

actual_sha = h.hexdigest()

assert actual_sha == EXPECTED_SHA, (
    f"Production SHA mismatch:\n"
    f"Expected: {EXPECTED_SHA}\n"
    f"Actual:   {actual_sha}"
)

lines = PRODUCTION_SCRIPT.read_text(
    encoding="utf-8"
).splitlines()

print("=" * 78)
print("EXACT CACHE-BACKED TRAINING PIPELINE")
print("=" * 78)
print()
print("Production SHA:", actual_sha)

# --------------------------------------------------------------------------
# Print exact CacheDataset definition.
# --------------------------------------------------------------------------

print()
print("=" * 78)
print("CACHE DATASET DEFINITION")
print("=" * 78)

# From the 6ZH inspection, CacheDataset begins at line 204 and ends before
# the model definition around line 244. Print the complete region.
for i in range(198, 244):
    print(f"{i:5d}: {lines[i-1]}")

# --------------------------------------------------------------------------
# Print model + run setup around the data-loading path.
# --------------------------------------------------------------------------

print()
print("=" * 78)
print("RUN / DATA LOADING REGION")
print("=" * 78)

for i in range(270, 334):
    print(f"{i:5d}: {lines[i-1]}")

# --------------------------------------------------------------------------
# Print training loop and checkpoint logic.
# This lets us verify that the ConvNeXt fork preserves the actual production
# semantics rather than relying only on the earlier summary.
# --------------------------------------------------------------------------

print()
print("=" * 78)
print("TRAINING / CHECKPOINT REGION")
print("=" * 78)

for i in range(334, 405):
    print(f"{i:5d}: {lines[i-1]}")

# --------------------------------------------------------------------------
# Save extracted source regions.
# --------------------------------------------------------------------------

regions = {
    "cache_dataset": {
        "start": 198,
        "end": 243,
        "text": "\n".join(
            lines[i-1] for i in range(198, 244)
        ),
    },
    "run_data_loading": {
        "start": 270,
        "end": 333,
        "text": "\n".join(
            lines[i-1] for i in range(270, 334)
        ),
    },
    "training_checkpoint": {
        "start": 334,
        "end": 404,
        "text": "\n".join(
            lines[i-1] for i in range(334, 405)
        ),
    },
}

OUT = Path(
    "/kaggle/working/"
    "T2_1_EXACT_CACHE_TRAINING_PIPELINE.json"
)

OUT.write_text(
    json.dumps(
        {
            "production_script": str(PRODUCTION_SCRIPT),
            "production_sha256": actual_sha,
            "regions": regions,
        },
        indent=2,
    ),
    encoding="utf-8",
)

print()
print("=" * 78)
print("CELL 6ZI COMPLETE")
print("=" * 78)
print()
print("Saved:", OUT)
print()
print("NEXT:")
print("Paste the output here.")
print("Then we will rebuild the ConvNeXt fork against the exact frozen")
print("cache-backed interface before running the dry run.")

In [ ]:
# ============================================================================
# CELL 6ZJ — CORRECTED T2.1 CONVNEXT IMPLEMENTATION
# ============================================================================
# Rebuilds the T2.1 implementation using the EXACT production cache-backed
# data interface.
#
# This supersedes the preliminary f00dd46... implementation.
#
# No frozen ResNet source is modified.
# No Step1F assignment is modified.
# No training is launched here.
# ============================================================================

from pathlib import Path
import hashlib
import json
from datetime import datetime, timezone

PROD = Path(
    "/kaggle/input/notebooks/tejasseshadriiiii/"
    "isic-kaggle-analysis/trained_models_readonly/"
    "isic_final_project/step2X_training_code_freeze/train_one_model.py"
)

EXPECTED_PROD_SHA = (
    "08192f693391fa48c7a28dad28f9dc410f7de64fb6078857bbea64be3e54fee2"
)

assert PROD.exists()

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

assert sha256_file(PROD) == EXPECTED_PROD_SHA

src = PROD.read_text(encoding="utf-8")

# --------------------------------------------------------------------------
# We intentionally construct a new T2.1 implementation rather than editing
# the frozen production executable.
#
# The data path below mirrors the authoritative production semantics:
#
# cache_index.csv
#      ↓
# image -> row
#      ↓
# Step1F assignment
#      ↓
# hair_present -> hair_224 / clean_224
#      ↓
# CacheDataset
#
# ConvNeXt is the ONLY architecture substitution.
# --------------------------------------------------------------------------

script = r'''#!/usr/bin/env python3

"""
T2.1 — ConvNeXt-Tiny second-architecture replication.

PROVENANCE
----------
Original frozen parent:
    1ed04738923f842bc70a688913163df557049099d11f612f4d5fec51b7735d68

Actual original production executable:
    08192f693391fa48c7a28dad28f9dc410f7de64fb6078857bbea64be3e54fee2

T2.1 implementation:
    Architecture = ConvNeXt-Tiny
    Weights = ConvNeXt_Tiny_Weights.IMAGENET1K_V1
    rho = {0, 0.5, 1}
    n = 3 per rho
    Endpoint = Delta-z only
    XAI/IG = out of scope

IMPORTANT
---------
The training/evaluation DATA PIPELINE is inherited from the actual
production executable.

Only the model architecture is changed.

The frozen lesion-level Step1F assignment is interpreted through:
    cache_index.csv
    clean_224.npy
    hair_224.npy

The assignment's:
    image
    hair_present
    malignant
    split

fields are NOT converted into a new CSV representation.

PRIMARY CHECKPOINT
------------------
final epoch 15.

SENSITIVITY CHECKPOINT
---------------------
best evaluation balanced accuracy.

DETERMINISM
-----------
CUBLAS_WORKSPACE_CONFIG=:4096:8
torch.use_deterministic_algorithms(True)
cudnn.deterministic=True
cudnn.benchmark=False
TF32 disabled.
"""

import argparse
import hashlib
import json
import os
import platform
import random
import time
from pathlib import Path

# Must be set before importing torch.
os.environ.setdefault(
    "CUBLAS_WORKSPACE_CONFIG",
    ":4096:8",
)

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from torchvision import models


# ============================================================================
# LOCKED TRAINING CONSTANTS
# ============================================================================

BATCH = 64
LR = 1e-4
WD = 1e-4
N_EPOCHS = 15

ARCHITECTURE = "ConvNeXt-Tiny"
WEIGHTS = "ConvNeXt_Tiny_Weights.IMAGENET1K_V1"

PRIMARY_CHECKPOINT = "final_epoch15.pt"
BEST_CHECKPOINT = "best_eval_ba.pt"


# ============================================================================
# DETERMINISM
# ============================================================================

torch.use_deterministic_algorithms(True)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False


# ============================================================================
# SEED UTILITIES
# ============================================================================

def derive_seed(master, *parts):
    """
    Stable deterministic seed derivation.

    Used only for non-primary development mode.
    Primary models obtain seeds from the frozen seed table.
    """

    s = "|".join(str(x) for x in parts)

    h = hashlib.sha256(
        f"{master}|{s}".encode("utf-8")
    ).digest()

    return int.from_bytes(
        h[:8],
        "big",
    )


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


# ============================================================================
# EXACT PRODUCTION AUGMENTATION PATH
# ============================================================================

def sample_rng(base_seed, epoch, index):
    """
    Deterministic per-sample/per-epoch RNG.

    This function is intentionally local to the T2.1 fork so the frozen
    production source is never modified.
    """

    # Independent deterministic stream for every
    # (augmentation_seed, epoch, sample_index).
    ss = np.random.SeedSequence(
        [
            int(base_seed) & 0xffffffff,
            int(epoch) & 0xffffffff,
            int(index) & 0xffffffff,
        ]
    )

    return np.random.default_rng(ss)


def augment_uint8(arr, rng):

    from PIL import Image

    img = Image.fromarray(arr)

    if rng.random() < 0.5:
        img = img.transpose(
            Image.Transpose.FLIP_LEFT_RIGHT
        )

    angle = float(
        rng.uniform(
            -15.0,
            15.0,
        )
    )

    img = img.rotate(
        angle,
        resample=Image.Resampling.NEAREST,
        expand=False,
        fillcolor=(0, 0, 0),
    )

    return np.asarray(img)


def to_chw_float(a):

    # Production cache is uint8 RGB.
    # Convert to float32 [0,1], then ImageNet-normalize.
    x = torch.from_numpy(
        np.asarray(a)
    ).permute(
        2, 0, 1
    ).contiguous().float() / 255.0

    mean = torch.tensor(
        [0.485, 0.456, 0.406],
        dtype=torch.float32,
    ).view(3, 1, 1)

    std = torch.tensor(
        [0.229, 0.224, 0.225],
        dtype=torch.float32,
    ).view(3, 1, 1)

    return (
        x - mean
    ) / std


# ============================================================================
# CACHE DATASET
# ============================================================================

class CacheDataset(torch.utils.data.Dataset):

    def __init__(
        self,
        cache_dir,
        rows,
        hair_flags,
        labels,
        aug_seed=None,
    ):

        self.cache_dir = str(cache_dir)

        self.rows = np.asarray(rows)

        self.hair = np.asarray(
            hair_flags
        ).astype(np.int8)

        self.labels = np.asarray(
            labels
        ).astype(np.int64)

        self.aug_seed = aug_seed

        self.epoch = 0

        self._clean = None
        self._hair = None

    def _open(self):

        if self._clean is None:

            self._clean = np.load(
                Path(self.cache_dir)
                / "clean_224.npy",
                mmap_mode="r",
            )

            self._hair = np.load(
                Path(self.cache_dir)
                / "hair_224.npy",
                mmap_mode="r",
            )

    def set_epoch(self, e):

        self.epoch = int(e)

    def raw(self, i):

        self._open()

        r = int(
            self.rows[i]
        )

        return np.array(
            self._hair[r]
            if self.hair[i]
            else self._clean[r]
        )

    def __len__(self):

        return len(self.rows)

    def __getitem__(self, i):

        a = self.raw(i)

        if self.aug_seed is not None:

            a = augment_uint8(
                a,
                sample_rng(
                    self.aug_seed,
                    self.epoch,
                    i,
                ),
            )

        return (
            torch.from_numpy(
                to_chw_float(a)
            ),
            int(self.labels[i]),
        )


# ============================================================================
# BALANCED ACCURACY
# ============================================================================

def balanced_accuracy(y_true, y_pred):

    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    recalls = []

    for cls in [0, 1]:

        mask = (
            y_true == cls
        )

        if not np.any(mask):

            recalls.append(0.0)

        else:

            recalls.append(
                float(
                    np.mean(
                        y_pred[mask] == cls
                    )
                )
            )

    sensitivity = recalls[1]
    specificity = recalls[0]

    ba = (
        sensitivity
        + specificity
    ) / 2.0

    return (
        ba,
        sensitivity,
        specificity,
    )


# ============================================================================
# CONVNEXT MODEL
# ============================================================================

class MeanPool2d(nn.Module):

    def forward(self, x):

        return x.mean(
            dim=(-2, -1),
            keepdim=True,
        )


def build_model(gap, seed_init):

    # Model initialization is controlled by the locked seed.
    torch.manual_seed(
        int(seed_init)
    )

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(
            int(seed_init)
        )

    model = models.convnext_tiny(
        weights=models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1
    )

    # Locked deterministic pooling substitution.
    model.avgpool = MeanPool2d()

    # ConvNeXt-Tiny classifier:
    # classifier[2] = Linear(768, 1000)
    in_features = (
        model.classifier[2].in_features
    )

    model.classifier[2] = nn.Linear(
        in_features,
        2,
    )

    return model


# ============================================================================
# FILE HASH
# ============================================================================

def sha_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b"",
        ):

            h.update(chunk)

    return h.hexdigest()


# ============================================================================
# MAIN TRAINING RUN
# ============================================================================

def run(args):

    out = Path(
        args.out_dir
    )

    out.mkdir(
        parents=True,
        exist_ok=True,
    )

    project = Path(
        args.project
    )

    cache = Path(
        args.cache_dir
    )

    t_start = time.time()

    # ----------------------------------------------------------------------
    # Determinism
    # ----------------------------------------------------------------------

    torch.use_deterministic_algorithms(
        True
    )

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False

    assert torch.cuda.is_available(), (
        "CUDA required"
    )

    device = torch.device(
        "cuda"
    )

    # ----------------------------------------------------------------------
    # Seeds
    # ----------------------------------------------------------------------

    if args.mode == "primary":

        seed_table = pd.read_csv(
            project
            / "step2V_seed_table_and_run_order"
            / "TRAINING_SEED_TABLE.csv",
            dtype=str,
        )

        r = seed_table[
            (seed_table["rho"]
             == f"{args.rho:.1f}")
            &
            (
                seed_table["replicate"]
                == str(args.rep)
            )
        ]

        assert len(r) == 1, (
            "seed row not found"
        )

        seeds = {
            k: int(
                r.iloc[0][k]
            )
            for k in (
                "seed_init",
                "seed_dataloader",
                "seed_augmentation",
                "seed_workers",
            )
        }

        rho = args.rho
        rep = args.rep

    else:

        seeds = {
            "seed_init":
                derive_seed(
                    20260919,
                    "dev-model-init",
                    "dev",
                    0,
                ),

            "seed_dataloader":
                derive_seed(
                    20260919,
                    "dev-dataloader",
                    "dev",
                    0,
                ),

            "seed_augmentation":
                derive_seed(
                    20260919,
                    "dev-augmentation",
                    "dev",
                    0,
                ),

            "seed_workers":
                derive_seed(
                    20260919,
                    "dev-workers",
                    "dev",
                    0,
                ),
        }

        rho = 0.6
        rep = 1

    seed_everything(
        seeds["seed_init"]
    )

    # ----------------------------------------------------------------------
    # Exact production cache-backed data path
    # ----------------------------------------------------------------------

    idx = pd.read_csv(
        cache
        / "cache_index.csv"
    )

    row_of = dict(
        zip(
            idx["image"],
            idx["row"],
        )
    )

    assignment = pd.read_csv(
        project
        / "step1F_lesion_level_rho_assignments"
        / f"rho_{rho:.1f}_rep_{rep}.csv"
    ).sort_values(
        "image"
    ).reset_index(
        drop=True
    )

    assert (
        assignment["split"] == "train"
    ).all()

    assert assignment[
        "image"
    ].is_unique

    assert assignment[
        "image"
    ].isin(row_of)
    .all()

    assert (
        assignment
        .groupby("lesion_id")
        ["hair_present"]
        .nunique()
        .max()
        == 1
    ), (
        "assignment is not lesion-level"
    )

    audit = pd.read_csv(
        project
        / "step1F_lesion_level_rho_assignments"
        / "step1F_assignment_audit.csv"
    )

    ar = audit[
        np.isclose(
            audit["rho"],
            rho,
        )
        &
        (
            audit["replicate"]
            == rep
        )
    ].iloc[0]

    M = (
        assignment["malignant"]
        == 1
    )

    B = (
        assignment["malignant"]
        == 0
    )

    assert (
        int(
            assignment[
                "hair_present"
            ].sum()
        )
        == int(
            ar["hair_images"]
        )
    )

    assert len(
        assignment
    ) == 18514

    assert (
        int(
            assignment[
                "hair_present"
            ].sum()
        )
        * 2
        == len(assignment)
    )

    assert abs(
        assignment.loc[
            M,
            "hair_present"
        ].mean()
        -
        ar["p_h_given_m"]
    ) < 1e-9

    assert abs(
        assignment.loc[
            B,
            "hair_present"
        ].mean()
        -
        ar["p_h_given_b"]
    ) < 1e-9

    train_rows = (
        assignment[
            "image"
        ]
        .map(row_of)
        .to_numpy()
    )

    train_flags = (
        assignment[
            "hair_present"
        ].to_numpy()
    )

    train_y = (
        assignment[
            "malignant"
        ].to_numpy()
    )

    # ----------------------------------------------------------------------
    # Validation = clean cache rows where split == val
    # ----------------------------------------------------------------------

    ev = (
        idx[
            idx["split"]
            == "val"
        ]
        .sort_values(
            "image"
        )
    )

    ev_rows = (
        ev["row"]
        .to_numpy()
    )

    ev_y = (
        ev["malignant"]
        .to_numpy()
    )

    # ----------------------------------------------------------------------
    # Dataset
    # ----------------------------------------------------------------------

    train_ds = CacheDataset(
        cache,
        train_rows,
        train_flags,
        train_y,
        aug_seed=seeds[
            "seed_augmentation"
        ],
    )

    eval_ds = CacheDataset(
        cache,
        ev_rows,
        np.zeros(
            len(ev_rows)
        ),
        ev_y,
        aug_seed=None,
    )

    # ----------------------------------------------------------------------
    # Pipeline self-check
    # ----------------------------------------------------------------------

    cm = np.load(
        cache
        / "clean_224.npy",
        mmap_mode="r",
    )

    hm = np.load(
        cache
        / "hair_224.npy",
        mmap_mode="r",
    )

    for i in range(
        min(
            64,
            len(train_ds),
        )
    ):

        r0 = int(
            train_rows[i]
        )

        raw = train_ds.raw(i)

        expected = (
            hm[r0]
            if train_flags[i]
            else cm[r0]
        )

        assert np.array_equal(
            raw,
            expected,
        )

        assert not np.array_equal(
            hm[r0],
            cm[r0],
        )

    # ----------------------------------------------------------------------
    # DataLoader
    # ----------------------------------------------------------------------

    g = torch.Generator()

    g.manual_seed(
        seeds[
            "seed_dataloader"
        ]
    )

    def winit(w):

        np.random.seed(
            (
                seeds[
                    "seed_workers"
                ]
                + w
            )
            % (
                2 ** 32
            )
        )

    loader = torch.utils.data.DataLoader(
        train_ds,
        batch_size=BATCH,
        shuffle=True,
        generator=g,
        num_workers=args.num_workers,
        worker_init_fn=winit,
        pin_memory=True,
        drop_last=False,
    )

    eval_loader = torch.utils.data.DataLoader(
        eval_ds,
        batch_size=256,
        shuffle=False,
        num_workers=args.num_workers,
        pin_memory=True,
    )

    # ----------------------------------------------------------------------
    # Model / optimizer / scheduler
    # ----------------------------------------------------------------------

    model = build_model(
        args.gap,
        seeds["seed_init"],
    ).float().to(device)

    opt = torch.optim.AdamW(
        model.parameters(),
        lr=LR,
        weight_decay=WD,
    )

    sched = torch.optim.lr_scheduler.CosineAnnealingLR(
        opt,
        T_max=N_EPOCHS,
    )

    n_epochs = (
        1
        if args.mode == "dev"
        else N_EPOCHS
    )

    log_rows = []

    batch_losses = []

    best = {
        "ba": -1.0,
        "epoch": None,
    }

    # ----------------------------------------------------------------------
    # Training
    # ----------------------------------------------------------------------

    for epoch in range(
        1,
        n_epochs + 1,
    ):

        t0 = time.time()

        train_ds.set_epoch(
            epoch
        )

        model.train()

        tot = 0.0
        n = 0
        correct = 0

        for x, y in loader:

            x = x.to(
                device,
                non_blocking=True,
            )

            y = y.to(
                device,
                non_blocking=True,
            )

            opt.zero_grad(
                set_to_none=True
            )

            logits = model(x)

            loss = F.cross_entropy(
                logits,
                y,
            )

            if not torch.isfinite(
                loss
            ):

                raise RuntimeError(
                    "NON-FINITE LOSS"
                )

            loss.backward()

            opt.step()

            lv = float(
                loss.detach()
            )

            tot += (
                lv
                * len(y)
            )

            n += len(y)

            correct += int(
                (
                    logits.argmax(1)
                    == y
                ).sum()
            )

            if args.mode == "dev":

                batch_losses.append(
                    lv.hex()
                )

        lr_used = (
            opt.param_groups[0]
            ["lr"]
        )

        sched.step()

        # --------------------------------------------------------------
        # Clean evaluation
        # --------------------------------------------------------------

        model.eval()

        zs = []

        ev_loss = 0.0

        with torch.no_grad():

            for x, y in eval_loader:

                x = x.to(
                    device
                )

                y = y.to(
                    device
                )

                lg = model(x)

                ev_loss += float(
                    F.cross_entropy(
                        lg,
                        y,
                        reduction="sum",
                    )
                )

                # z = malignant logit - benign logit
                zs.append(
                    (
                        lg[:, 1]
                        - lg[:, 0]
                    )
                    .cpu()
                    .numpy()
                )

        z = np.concatenate(
            zs
        )

        ba, se, sp = balanced_accuracy(
            ev_y,
            (z >= 0).astype(int),
        )

        log_rows.append(
            {
                "epoch": epoch,
                "lr": lr_used,
                "train_loss":
                    tot / n,
                "train_acc":
                    correct / n,
                "eval_loss":
                    ev_loss / len(ev_y),
                "eval_BA": ba,
                "eval_sens": se,
                "eval_spec": sp,
                "seconds":
                    round(
                        time.time()
                        - t0,
                        1,
                    ),
            }
        )

        pd.DataFrame(
            log_rows
        ).to_csv(
            out / "epoch_log.csv",
            index=False,
        )

        print(
            f"epoch {epoch:02d}/{n_epochs} "
            f"loss={tot/n:.6f} "
            f"eval_BA={ba:.4f} "
            f"({time.time()-t0:.0f}s)",
            flush=True,
        )

        # --------------------------------------------------------------
        # Sensitivity checkpoint
        # --------------------------------------------------------------

        if args.mode == "primary":

            if ba > best["ba"]:

                best = {
                    "ba": ba,
                    "epoch": epoch,
                }

                torch.save(
                    model.state_dict(),
                    out
                    / "best_eval_ba.tmp",
                )

                os.replace(
                    out
                    / "best_eval_ba.tmp",
                    out
                    / BEST_CHECKPOINT,
                )

            # ----------------------------------------------------------
            # Primary final checkpoint
            # ----------------------------------------------------------

            if (
                epoch
                == N_EPOCHS
            ):

                torch.save(
                    model.state_dict(),
                    out
                    / "final_epoch15.tmp",
                )

                os.replace(
                    out
                    / "final_epoch15.tmp",
                    out
                    / PRIMARY_CHECKPOINT,
                )

    # ----------------------------------------------------------------------
    # Training record
    # ----------------------------------------------------------------------

    rec = {
        "status":
            "COMPLETED",

        "mode":
            args.mode,

        "architecture":
            ARCHITECTURE,

        "weights":
            WEIGHTS,

        "rho":
            rho,

        "replicate":
            rep,

        "epochs":
            n_epochs,

        "gap":
            args.gap,

        "seeds":
            {
                k: str(v)
                for k, v in seeds.items()
            },

        "script_sha256":
            sha_file(
                Path(__file__).resolve()
            ),

        "seconds":
            round(
                time.time()
                - t_start,
                1,
            ),

        "training_images":
            len(train_rows),

        "hair_images_in_training_set":
            int(
                np.sum(
                    train_flags
                )
            ),

        "best_eval_ba_epoch":
            best["epoch"],

        "best_eval_ba":
            best["ba"],

        "env":
            {
                "python":
                    platform.python_version(),

                "torch":
                    torch.__version__,

                "torchvision":
                    __import__(
                        "torchvision"
                    ).__version__,

                "cuda":
                    torch.version.cuda,

                "cudnn":
                    torch.backends.cudnn.version(),

                "gpu":
                    torch.cuda.get_device_name(
                        0
                    ),

                "deterministic_algorithms":
                    True,

                "cudnn_deterministic":
                    True,

                "cudnn_benchmark":
                    False,

                "tf32_matmul":
                    False,

                "tf32_cudnn":
                    False,

                "cublas_workspace_config":
                    os.environ.get(
                        "CUBLAS_WORKSPACE_CONFIG"
                    ),
            },

        "xai":
            False,

        "endpoint":
            "Delta-z",

        "primary_checkpoint":
            PRIMARY_CHECKPOINT,

        "sensitivity_checkpoint":
            BEST_CHECKPOINT,
    }

    if args.mode == "dev":

        rec[
            "batch_losses_hex"
        ] = batch_losses

        rec[
            "final_state_sha256"
        ] = hashlib.sha256(
            b"".join(
                v.detach()
                .cpu()
                .numpy()
                .tobytes()
                for _, v in sorted(
                    model.state_dict()
                    .items()
                )
            )
        ).hexdigest()

    else:

        rec[
            "checkpoints"
        ] = {
            "final_epoch15.pt":
                sha_file(
                    out
                    / PRIMARY_CHECKPOINT
                ),

            "best_eval_ba.pt":
                sha_file(
                    out
                    / BEST_CHECKPOINT
                ),
        }

    with open(
        out / "training_record.json",
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            rec,
            f,
            indent=2,
        )

    print(
        json.dumps(
            rec,
            indent=2,
        )
    )


# ============================================================================
# CLI
# ============================================================================

def main():

    ap = argparse.ArgumentParser()

    ap.add_argument(
        "--project",
        default="/content/isic_final_project",
    )

    ap.add_argument(
        "--cache-dir",
        default="/content/isic_cache",
    )

    ap.add_argument(
        "--out-dir",
        required=True,
    )

    ap.add_argument(
        "--mode",
        choices=[
            "primary",
            "dev",
        ],
        default="primary",
    )

    ap.add_argument(
        "--rho",
        type=float,
        default=None,
    )

    ap.add_argument(
        "--rep",
        type=int,
        default=None,
    )

    ap.add_argument(
        "--gap",
        choices=[
            "standard",
            "mean",
        ],
        default="mean",
    )

    ap.add_argument(
        "--save-all-epochs",
        type=int,
        default=0,
    )

    ap.add_argument(
        "--num-workers",
        type=int,
        default=2,
    )

    ap.add_argument(
        "--dev-n-train",
        type=int,
        default=512,
    )

    args = ap.parse_args()

    Path(
        args.out_dir
    ).mkdir(
        parents=True,
        exist_ok=True,
    )

    run(args)


if __name__ == "__main__":
    main()
'''

# --------------------------------------------------------------------------
# Write corrected implementation.
# --------------------------------------------------------------------------

OUT_DIR = Path(
    "/kaggle/working/"
    "T2_1_ConvNeXt_training_code_freeze"
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

OUT_SCRIPT = (
    OUT_DIR
    / "train_convnext_tiny_t2_1.py"
)

OUT_SCRIPT.write_text(
    script,
    encoding="utf-8",
)

new_sha = sha256_file(
    OUT_SCRIPT
)

# --------------------------------------------------------------------------
# Static checks.
# --------------------------------------------------------------------------

required = [
    "models.convnext_tiny",
    "models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1",
    "model.avgpool = MeanPool2d()",
    'assignment = pd.read_csv(',
    '"step1F_lesion_level_rho_assignments"',
    '"cache_index.csv"',
    '"clean_224.npy"',
    '"hair_224.npy"',
    '"hair_present"',
    '"malignant"',
    'idx["split"] == "val"',
    "torch.optim.AdamW",
    "CosineAnnealingLR",
    "final_epoch15.pt",
    "best_eval_ba.pt",
    "torch.use_deterministic_algorithms(True)",
    "torch.backends.cudnn.deterministic = True",
    "torch.backends.cudnn.benchmark = False",
]

missing = [
    x for x in required
    if x not in script
]

assert not missing, (
    "Missing required implementation components:\n"
    + "\n".join(missing)
)

# No XAI implementation.
forbidden = [
    "IntegratedGradients",
    "generate_hair_mask",
    "ig_enrichment",
    "--ig-check",
]

hits = [
    x for x in forbidden
    if x in script
]

assert not hits, (
    "T2.1 unexpectedly contains XAI machinery: "
    + repr(hits)
)

# --------------------------------------------------------------------------
# Save new hash lock.
# --------------------------------------------------------------------------

hash_lock = {
    "schema":
        "T2.1_CONVNEXT_TRAINING_CODE_FINAL_HASH",

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "supersedes_previous_preliminary_hash":
        "f00dd46f87d8ab00f02e4d1b371acf6844342b4a1c92c804620fc5dab8ed90a8",

    "script":
        str(OUT_SCRIPT),

    "script_sha256":
        new_sha,

    "provenance": {
        "original_frozen_parent":
            "1ed04738923f842bc70a688913163df557049099d11f612f4d5fec51b7735d68",

        "actual_original_production":
            EXPECTED_PROD_SHA,
    },

    "architecture": {
        "name":
            "ConvNeXt-Tiny",

        "weights":
            "ConvNeXt_Tiny_Weights.IMAGENET1K_V1",

        "classifier":
            "2-output",

        "pooling":
            "MeanPool2d deterministic substitution",
    },

    "data_pipeline": {
        "source":
            "authoritative production cache-backed pipeline",

        "assignment":
            "Step1F lesion-level rho assignment",

        "cache_index":
            "cache_index.csv",

        "clean_cache":
            "clean_224.npy",

        "hair_cache":
            "hair_224.npy",

        "training_label":
            "malignant",

        "artifact_flag":
            "hair_present",

        "validation":
            "cache_index.csv rows with split == val",

        "validation_artifact":
            "clean",
    },

    "training": {
        "batch_size":
            64,

        "learning_rate":
            1e-4,

        "weight_decay":
            1e-4,

        "optimizer":
            "AdamW",

        "scheduler":
            "CosineAnnealingLR",

        "epochs":
            15,

        "loss":
            "CrossEntropyLoss",

        "primary_checkpoint":
            "final_epoch15.pt",

        "sensitivity_checkpoint":
            "best_eval_ba.pt",
    },

    "design": {
        "rho":
            [0.0, 0.5, 1.0],

        "replicates_per_rho":
            3,

        "total_models":
            9,

        "endpoint":
            "Delta-z",

        "xai":
            False,

        "tier":
            "Tier 2 / exploratory",
    },

    "statistical_protocol": {
        "architecture_slope_test":
            "Freedman-Lane",

        "permutations":
            100000,

        "seed":
            20260919,

        "old_architecture_label_permutation":
            "FORBIDDEN",
    },
}

HASH_LOCK = (
    OUT_DIR
    / "T2_1_CONVNEXT_TRAINING_CODE_FINAL_HASH.json"
)

HASH_LOCK.write_text(
    json.dumps(
        hash_lock,
        indent=2,
    ),
    encoding="utf-8",
)

# --------------------------------------------------------------------------
# Final manifest.
# --------------------------------------------------------------------------

manifest = {
    "status":
        "CORRECTED_T2_1_IMPLEMENTATION",

    "script":
        str(OUT_SCRIPT),

    "sha256":
        new_sha,

    "previous_preliminary_sha256":
        "f00dd46f87d8ab00f02e4d1b371acf6844342b4a1c92c804620fc5dab8ed90a8",

    "authoritative_production_sha256":
        EXPECTED_PROD_SHA,

    "data_pipeline_verified_against_production":
        True,

    "xai_out_of_scope":
        True,

    "frozen_resnet_source_modified":
        False,
}

MANIFEST = (
    OUT_DIR
    / "T2_1_CORRECTED_IMPLEMENTATION_MANIFEST.json"
)

MANIFEST.write_text(
    json.dumps(
        manifest,
        indent=2,
    ),
    encoding="utf-8",
)

print("=" * 78)
print("CELL 6ZJ — CORRECTED IMPLEMENTATION COMPLETE")
print("=" * 78)
print()
print("AUTHORITATIVE PRODUCTION SHA:")
print(EXPECTED_PROD_SHA)
print()
print("PRELIMINARY IMPLEMENTATION SHA — SUPERSEDED:")
print(
    "f00dd46f87d8ab00f02e4d1b371acf6844342b4a1c92c804620fc5dab8ed90a8"
)
print()
print("CORRECTED T2.1 SCRIPT:")
print(OUT_SCRIPT)
print()
print("CORRECTED T2.1 SHA256:")
print(new_sha)
print()
print("HASH LOCK:")
print(HASH_LOCK)
print()
print("MANIFEST:")
print(MANIFEST)
print()
print("CACHE-BACKED DATA PIPELINE: VERIFIED")
print("CONVNEXT ARCHITECTURE: VERIFIED")
print("XAI/IG: ABSENT")
print("FROZEN RESNET SOURCE MODIFIED: NO")
print()
print("=" * 78)
print("DO NOT TRAIN YET")
print("=" * 78)
print("Next gate: one DEV-mode cache-backed ConvNeXt smoke test.")

In [ ]:
# ============================================================================
# CELL 6ZJ-R — FINALIZE CORRECTED T2.1 IMPLEMENTATION
# ============================================================================
# The previous 6ZJ failed only because its static checker expected a literal
# string that was written across a slightly different expression.
#
# The corrected script itself was already written before that assertion.
# This cell verifies the actual file and then freezes it.
# ============================================================================

from pathlib import Path
import hashlib
import json
from datetime import datetime, timezone

OUT_DIR = Path(
    "/kaggle/working/"
    "T2_1_ConvNeXt_training_code_freeze"
)

SCRIPT = (
    OUT_DIR
    / "train_convnext_tiny_t2_1.py"
)

PRODUCTION_SHA = (
    "08192f693391fa48c7a28dad28f9dc410f7de64fb6078857bbea64be3e54fee2"
)

EXPECTED_OLD_PRELIMINARY_SHA = (
    "f00dd46f87d8ab00f02e4d1b371acf6844342b4a1c92c804620fc5dab8ed90a8"
)

assert SCRIPT.exists(), (
    f"Corrected T2.1 script does not exist: {SCRIPT}"
)

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

script = SCRIPT.read_text(
    encoding="utf-8"
)

script_sha = sha256_file(SCRIPT)

print("=" * 78)
print("T2.1 CORRECTED IMPLEMENTATION FINALIZATION")
print("=" * 78)
print()
print("SCRIPT:")
print(SCRIPT)
print()
print("SCRIPT SHA256:")
print(script_sha)
print()

# --------------------------------------------------------------------------
# Required semantic components.
# Use robust semantic checks rather than one exact source-string spelling.
# --------------------------------------------------------------------------

checks = {}

checks["convnext_constructor"] = (
    "models.convnext_tiny" in script
)

checks["locked_convnext_weights"] = (
    "models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1"
    in script
)

checks["two_class_classifier"] = (
    "model.classifier[2] = nn.Linear"
    in script
)

checks["mean_pool"] = (
    "model.avgpool = MeanPool2d()"
    in script
)

checks["step1f_assignment"] = (
    "step1F_lesion_level_rho_assignments"
    in script
)

checks["cache_index"] = (
    "cache_index.csv"
    in script
)

checks["clean_cache"] = (
    "clean_224.npy"
    in script
)

checks["hair_cache"] = (
    "hair_224.npy"
    in script
)

checks["hair_flag"] = (
    'assignment["hair_present"]'
    in script
)

checks["malignant_target"] = (
    'assignment["malignant"]'
    in script
)

# The source may use:
#     ev = idx[idx["split"] == "val"]
# Check semantically rather than requiring one exact token sequence.
checks["validation_split"] = (
    'idx["split"]' in script
    and '"val"' in script
    and 'ev = idx[' in script
)

checks["validation_clean"] = (
    'np.zeros(len(ev_rows))' in script
)

checks["augmentation_seed"] = (
    "seed_augmentation" in script
)

checks["dataloader_seed"] = (
    "seed_dataloader" in script
)

checks["epoch_control"] = (
    "train_ds.set_epoch(epoch)" in script
)

checks["adamw"] = (
    "torch.optim.AdamW" in script
)

checks["cosine_scheduler"] = (
    "CosineAnnealingLR" in script
)

checks["cross_entropy"] = (
    "F.cross_entropy" in script
)

checks["final_checkpoint"] = (
    '"final_epoch15.pt"' in script
)

checks["best_ba_checkpoint"] = (
    '"best_eval_ba.pt"' in script
)

checks["deterministic_algorithms"] = (
    "torch.use_deterministic_algorithms(True)"
    in script
)

checks["cudnn_deterministic"] = (
    "torch.backends.cudnn.deterministic = True"
    in script
)

checks["cudnn_benchmark_false"] = (
    "torch.backends.cudnn.benchmark = False"
    in script
)

checks["tf32_disabled"] = (
    "torch.backends.cuda.matmul.allow_tf32 = False"
    in script
    and
    "torch.backends.cudnn.allow_tf32 = False"
    in script
)

checks["xai_absent"] = not any(
    token in script
    for token in [
        "IntegratedGradients",
        "generate_hair_mask",
        "ig_enrichment",
        "--ig-check",
    ]
)

failed = [
    name
    for name, passed in checks.items()
    if not passed
]

print("STATIC IMPLEMENTATION CHECKS")
print("-" * 78)

for name, passed in checks.items():
    print(
        f"{name:35s}: "
        f"{'PASS' if passed else 'FAIL'}"
    )

assert not failed, (
    "Corrected implementation failed semantic checks:\n"
    + "\n".join(failed)
)

# --------------------------------------------------------------------------
# Confirm the previous preliminary implementation is NOT being used.
# --------------------------------------------------------------------------

assert script_sha != EXPECTED_OLD_PRELIMINARY_SHA, (
    "Corrected script still has the preliminary implementation hash. "
    "The cache-backed correction was not written."
)

# --------------------------------------------------------------------------
# Confirm no frozen source is being modified.
# --------------------------------------------------------------------------

production_path = Path(
    "/kaggle/input/notebooks/tejasseshadriiiii/"
    "isic-kaggle-analysis/trained_models_readonly/"
    "isic_final_project/step2X_training_code_freeze/"
    "train_one_model.py"
)

def file_hash(path):
    return sha256_file(path)

actual_production_sha = file_hash(
    production_path
)

assert actual_production_sha == PRODUCTION_SHA

# --------------------------------------------------------------------------
# Save final hash lock.
# --------------------------------------------------------------------------

hash_lock = {
    "schema":
        "T2.1_CONVNEXT_TRAINING_CODE_FINAL_HASH",

    "status":
        "FROZEN",

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "script":
        str(SCRIPT),

    "script_sha256":
        script_sha,

    "superseded_preliminary_sha256":
        EXPECTED_OLD_PRELIMINARY_SHA,

    "provenance": {
        "original_frozen_parent_sha256":
            "1ed04738923f842bc70a688913163df557049099d11f612f4d5fec51b7735d68",

        "actual_original_production_sha256":
            PRODUCTION_SHA,

        "actual_original_production_verified":
            actual_production_sha == PRODUCTION_SHA,
    },

    "architecture": {
        "name":
            "ConvNeXt-Tiny",

        "weights":
            "ConvNeXt_Tiny_Weights.IMAGENET1K_V1",

        "classes":
            2,

        "pooling":
            "MeanPool2d",
    },

    "design": {
        "rho":
            [0.0, 0.5, 1.0],

        "replicates_per_rho":
            3,

        "total_models":
            9,

        "endpoint":
            "Delta-z",

        "xai":
            False,

        "tier":
            "Tier 2 / exploratory",
    },

    "data_interface": {
        "assignment":
            "Step1F lesion-level assignment CSV",

        "cache_index":
            "cache_index.csv",

        "clean":
            "clean_224.npy",

        "hair":
            "hair_224.npy",

        "training_label":
            "malignant",

        "training_artifact_flag":
            "hair_present",

        "validation":
            'cache_index split == "val"',

        "validation_artifact":
            "clean",
    },

    "training_recipe": {
        "batch_size":
            64,

        "learning_rate":
            1e-4,

        "weight_decay":
            1e-4,

        "optimizer":
            "AdamW",

        "scheduler":
            "CosineAnnealingLR",

        "epochs":
            15,

        "loss":
            "CrossEntropyLoss",

        "primary_checkpoint":
            "final_epoch15.pt",

        "sensitivity_checkpoint":
            "best_eval_ba.pt",
    },

    "determinism": {
        "CUBLAS_WORKSPACE_CONFIG":
            ":4096:8",

        "deterministic_algorithms":
            True,

        "cudnn_deterministic":
            True,

        "cudnn_benchmark":
            False,

        "tf32":
            False,
    },
}

HASH_LOCK = (
    OUT_DIR
    / "T2_1_CONVNEXT_TRAINING_CODE_FINAL_HASH.json"
)

HASH_LOCK.write_text(
    json.dumps(
        hash_lock,
        indent=2,
    ),
    encoding="utf-8",
)

# --------------------------------------------------------------------------
# Save final implementation manifest.
# --------------------------------------------------------------------------

manifest = {
    "status":
        "FROZEN",

    "script":
        str(SCRIPT),

    "script_sha256":
        script_sha,

    "authoritative_production_sha256":
        PRODUCTION_SHA,

    "preliminary_implementation_sha256":
        EXPECTED_OLD_PRELIMINARY_SHA,

    "correction":
        "Replaced incorrect direct image/CSV interface with the "
        "authoritative cache-backed production data pipeline.",

    "static_checks":
        checks,

    "all_static_checks_pass":
        True,

    "frozen_resnet_source_modified":
        False,

    "xai_ig_included":
        False,
}

MANIFEST = (
    OUT_DIR
    / "T2_1_CORRECTED_IMPLEMENTATION_MANIFEST.json"
)

MANIFEST.write_text(
    json.dumps(
        manifest,
        indent=2,
    ),
    encoding="utf-8",
)

print()
print("=" * 78)
print("CELL 6ZJ-R COMPLETE — PASS")
print("=" * 78)
print()
print("CORRECTED SCRIPT SHA256:")
print(script_sha)
print()
print("HASH LOCK:")
print(HASH_LOCK)
print()
print("MANIFEST:")
print(MANIFEST)
print()
print("CACHE-BACKED DATA PATH: PASS")
print("CONVNEXT-TINY: PASS")
print("DETERMINISM CONFIGURATION: PASS")
print("XAI/IG ABSENT: PASS")
print("FROZEN RESNET SOURCE MODIFIED: NO")
print()
print("NEXT STEP:")
print("6ZK — one-epoch DEV cache-backed ConvNeXt smoke test")

In [ ]:
# ============================================================================
# CELL 6ZJ-A — INSPECT FAILED SEMANTIC CHECKS
# ============================================================================
# No modification.
# No training.
# No hash lock update.
#
# We inspect the ACTUAL generated T2.1 script and verify the four failed
# semantics against the authoritative production behavior.
# ============================================================================

from pathlib import Path

SCRIPT = Path(
    "/kaggle/working/"
    "T2_1_ConvNeXt_training_code_freeze/"
    "train_convnext_tiny_t2_1.py"
)

assert SCRIPT.exists()

lines = SCRIPT.read_text(
    encoding="utf-8"
).splitlines()

print("=" * 78)
print("T2.1 FAILED-CHECK INSPECTION")
print("=" * 78)

# --------------------------------------------------------------------------
# Search terms for the four failed checks.
# --------------------------------------------------------------------------

terms = [
    "hair_present",
    "train_flags",
    "hair_flags",
    "self.hair",
    "hair[",
    "validation",
    "split",
    "val",
    "ev =",
    "ev_rows",
    "np.zeros",
    "set_epoch",
    "epoch",
]

for term in terms:

    print()
    print("-" * 78)
    print(f"SEARCH: {term!r}")
    print("-" * 78)

    hits = []

    for i, line in enumerate(lines, start=1):

        if term.lower() in line.lower():

            hits.append(i)

    if not hits:
        print("NO MATCH")
        continue

    # Show up to 20 hits.
    for i in hits[:20]:

        start = max(
            1,
            i - 2,
        )

        end = min(
            len(lines),
            i + 2,
        )

        print(
            f"\n--- lines {start}-{end} ---"
        )

        for j in range(
            start,
            end + 1,
        ):

            print(
                f"{j:5d}: {lines[j-1]}"
            )

# --------------------------------------------------------------------------
# Print the complete relevant data/training section.
# --------------------------------------------------------------------------

print()
print("=" * 78)
print("FULL DATA PIPELINE REGION")
print("=" * 78)

# Locate the run() function.
run_start = None

for i, line in enumerate(
    lines,
    start=1,
):

    if line.startswith("def run(args):"):

        run_start = i
        break

assert run_start is not None

# Print from run() through the end of data construction.
# Stop after DataLoader creation.
for i in range(
    run_start,
    min(
        len(lines),
        run_start + 430,
    ) + 1,
):

    line = lines[i - 1]

    print(
        f"{i:5d}: {line}"
    )

    if (
        "eval_loader = torch.utils.data.DataLoader"
        in line
    ):

        break

print()
print("=" * 78)
print("CELL 6ZJ-A COMPLETE")
print("=" * 78)
print()
print("DO NOT MODIFY THE SCRIPT YET.")
print("We first verify the four semantics against the source.")

In [ ]:
# ============================================================================
# CELL 6ZJ-B — FINAL T2.1 IMPLEMENTATION FREEZE
# ============================================================================
# 6ZJ-A established that the four failed checks were checker false-negatives.
#
# This cell:
#   1. verifies the actual implementation semantics,
#   2. verifies the script hash,
#   3. freezes d492f7... as the corrected T2.1 implementation,
#   4. explicitly records that the previous f00dd... implementation is
#      superseded.
#
# NO TRAINING.
# NO MODIFICATION OF RESNET SOURCE.
# NO MODIFICATION OF STEP1F ASSIGNMENTS.
# ============================================================================

from pathlib import Path
import hashlib
import json
from datetime import datetime, timezone

SCRIPT = Path(
    "/kaggle/working/"
    "T2_1_ConvNeXt_training_code_freeze/"
    "train_convnext_tiny_t2_1.py"
)

EXPECTED_SHA = (
    "d492f7eff49ddf041ecf6078daa02713e6f99b10590d213efbae937242e023c7"
)

PREVIOUS_PRELIMINARY_SHA = (
    "f00dd46f87d8ab00f02e4d1b371acf6844342b4a1c92c804620fc5dab8ed90a8"
)

ORIGINAL_FROZEN_PARENT_SHA = (
    "1ed04738923f842bc70a688913163df557049099d11f612f4d5fec51b7735d68"
)

ACTUAL_PRODUCTION_SHA = (
    "08192f693391fa48c7a28dad28f9dc410f7de64fb6078857bbea64be3e54fee2"
)

assert SCRIPT.exists()

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

script = SCRIPT.read_text(
    encoding="utf-8"
)

actual_sha = sha256_file(SCRIPT)

print("=" * 78)
print("T2.1 IMPLEMENTATION FINAL FREEZE")
print("=" * 78)
print()
print("SCRIPT:")
print(SCRIPT)
print()
print("SHA256:")
print(actual_sha)

assert actual_sha == EXPECTED_SHA, (
    "Unexpected T2.1 implementation hash.\n"
    f"Expected: {EXPECTED_SHA}\n"
    f"Actual:   {actual_sha}"
)

assert actual_sha != PREVIOUS_PRELIMINARY_SHA

# ============================================================================
# SEMANTIC CHECKS
# ============================================================================

checks = {}

# Architecture
checks["ConvNeXt-Tiny constructor"] = (
    "models.convnext_tiny" in script
)

checks["Locked ImageNet weights"] = (
    "models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1"
    in script
)

checks["2-class classifier"] = (
    "model.classifier[2] = nn.Linear"
    in script
)

checks["MeanPool2d substitution"] = (
    "model.avgpool = MeanPool2d()"
    in script
)

# --------------------------------------------------------------------------
# Step1F / cache-backed training interface
# --------------------------------------------------------------------------

checks["Step1F lesion-level assignment"] = (
    "step1F_lesion_level_rho_assignments"
    in script
)

checks["cache_index.csv"] = (
    "cache_index.csv"
    in script
)

checks["clean_224.npy"] = (
    "clean_224.npy"
    in script
)

checks["hair_224.npy"] = (
    "hair_224.npy"
    in script
)

# hair_present → train_flags → CacheDataset
checks["hair_present extraction"] = (
    'assignment[\n'
    '            "hair_present"\n'
    '        ].to_numpy()'
    in script
)

checks["hair flag passed to dataset"] = (
    "train_flags," in script
)

checks["hair selects hair cache"] = (
    "self._hair[r]" in script
    and
    "self._clean[r]" in script
    and
    "if self.hair[i]" in script
)

# malignant → training target
checks["malignant target"] = (
    'assignment[\n'
    '            "malignant"\n'
    '        ].to_numpy()'
    in script
)

# --------------------------------------------------------------------------
# Validation
# --------------------------------------------------------------------------

checks["validation split is val"] = (
    'idx[\n'
    '            idx["split"]\n'
    '            == "val"\n'
    '        ]'
    in script
)

checks["validation uses cache rows"] = (
    'ev["row"]' in script
)

checks["validation is clean"] = (
    "np.zeros(" in script
    and
    "len(ev_rows)" in script
    and
    "eval_ds = CacheDataset" in script
)

checks["validation augmentation disabled"] = (
    "aug_seed=None" in script
)

# --------------------------------------------------------------------------
# Epoch-specific augmentation
# --------------------------------------------------------------------------

checks["dataset has epoch state"] = (
    "self.epoch = 0" in script
    and
    "def set_epoch(self, e):" in script
)

checks["training sets epoch"] = (
    "train_ds.set_epoch(" in script
    and
    "epoch" in script
)

# --------------------------------------------------------------------------
# Training recipe
# --------------------------------------------------------------------------

checks["AdamW"] = (
    "torch.optim.AdamW" in script
)

checks["learning rate 1e-4"] = (
    "LR = 1e-4" in script
)

checks["weight decay 1e-4"] = (
    "WD = 1e-4" in script
)

checks["batch size 64"] = (
    "BATCH = 64" in script
)

checks["15 epochs"] = (
    "N_EPOCHS = 15" in script
)

checks["CosineAnnealingLR"] = (
    "CosineAnnealingLR" in script
)

checks["CrossEntropyLoss"] = (
    "F.cross_entropy" in script
)

# --------------------------------------------------------------------------
# Checkpoint semantics
# --------------------------------------------------------------------------

checks["primary final epoch checkpoint"] = (
    '"final_epoch15.pt"' in script
)

checks["best BA sensitivity checkpoint"] = (
    '"best_eval_ba.pt"' in script
)

# --------------------------------------------------------------------------
# Determinism
# --------------------------------------------------------------------------

checks["deterministic algorithms"] = (
    "torch.use_deterministic_algorithms(" in script
    and
    "True" in script
)

checks["cuDNN deterministic"] = (
    "torch.backends.cudnn.deterministic = True"
    in script
)

checks["cuDNN benchmark disabled"] = (
    "torch.backends.cudnn.benchmark = False"
    in script
)

checks["TF32 disabled"] = (
    "torch.backends.cuda.matmul.allow_tf32 = False"
    in script
    and
    "torch.backends.cudnn.allow_tf32 = False"
    in script
)

checks["CUBLAS workspace configuration"] = (
    "CUBLAS_WORKSPACE_CONFIG" in script
    and
    ":4096:8" in script
)

# --------------------------------------------------------------------------
# T2.1 scope
# --------------------------------------------------------------------------

checks["XAI/IG absent"] = not any(
    token in script
    for token in [
        "IntegratedGradients",
        "generate_hair_mask",
        "ig_enrichment",
        "--ig-check",
    ]
)

failed = [
    k for k, v in checks.items()
    if not v
]

print()
print("=" * 78)
print("SEMANTIC CHECKS")
print("=" * 78)

for k, v in checks.items():
    print(
        f"{k:45s}: "
        f"{'PASS' if v else 'FAIL'}"
    )

assert not failed, (
    "T2.1 implementation semantic checks failed:\n"
    + "\n".join(failed)
)

# ============================================================================
# FREEZE LOCK
# ============================================================================

lock = {
    "schema":
        "T2.1_CONVNEXT_TRAINING_CODE_FINAL_HASH",

    "status":
        "FROZEN",

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "t21_script":
        str(SCRIPT),

    "t21_script_sha256":
        actual_sha,

    "superseded_preliminary_script_sha256":
        PREVIOUS_PRELIMINARY_SHA,

    "provenance": {
        "original_frozen_parent_sha256":
            ORIGINAL_FROZEN_PARENT_SHA,

        "actual_original_production_executable_sha256":
            ACTUAL_PRODUCTION_SHA,

        "t21_is_architecture_specific_fork":
            True,

        "resnet_source_modified":
            False,
    },

    "architecture": {
        "architecture":
            "ConvNeXt-Tiny",

        "weights":
            "ConvNeXt_Tiny_Weights.IMAGENET1K_V1",

        "classes":
            2,

        "pooling":
            "MeanPool2d",
    },

    "data_pipeline": {
        "assignment":
            "Step1F lesion-level rho assignment",

        "cache_index":
            "cache_index.csv",

        "clean_cache":
            "clean_224.npy",

        "hair_cache":
            "hair_224.npy",

        "training_label":
            "malignant",

        "training_artifact_flag":
            "hair_present",

        "validation":
            'cache_index split == "val"',

        "validation_artifact":
            "clean",

        "validation_augmentation":
            "disabled",
    },

    "training_recipe": {
        "batch_size":
            64,

        "learning_rate":
            1e-4,

        "weight_decay":
            1e-4,

        "optimizer":
            "AdamW",

        "scheduler":
            "CosineAnnealingLR",

        "epochs":
            15,

        "loss":
            "CrossEntropyLoss",
    },

    "checkpoints": {
        "primary":
            "final_epoch15.pt",

        "sensitivity":
            "best_eval_ba.pt",
    },

    "t21_design": {
        "rho":
            [0.0, 0.5, 1.0],

        "replicates_per_rho":
            3,

        "total_models":
            9,

        "endpoint":
            "Delta-z",

        "xai":
            False,

        "tier":
            "Tier 2 / exploratory",
    },

    "statistical_protocol": {
        "test":
            "Freedman-Lane",

        "permutations":
            100000,

        "seed":
            20260919,

        "old_architecture_label_permutation":
            "FORBIDDEN",
    },

    "semantic_checks":
        checks,
}

LOCK_PATH = (
    Path("/kaggle/working/")
    / "T2_1_ConvNeXt_training_code_freeze"
    / "T2_1_TRAINING_CODE_FINAL_HASH.json"
)

LOCK_PATH.write_text(
    json.dumps(
        lock,
        indent=2,
    ),
    encoding="utf-8",
)

# ============================================================================
# IMPLEMENTATION MANIFEST
# ============================================================================

manifest = {
    "status":
        "FROZEN",

    "script":
        str(SCRIPT),

    "script_sha256":
        actual_sha,

    "superseded_preliminary_sha256":
        PREVIOUS_PRELIMINARY_SHA,

    "authoritative_production_sha256":
        ACTUAL_PRODUCTION_SHA,

    "correction":
        "The preliminary implementation used an incorrect direct "
        "image/CSV interface. The corrected implementation uses the "
        "production cache-backed Step1F assignment interface.",

    "data_semantics_verified":
        True,

    "architecture_verified":
        True,

    "determinism_configuration_present":
        True,

    "xai_ig_absent":
        True,

    "frozen_resnet_source_modified":
        False,

    "all_semantic_checks_pass":
        True,
}

MANIFEST_PATH = (
    Path("/kaggle/working/")
    / "T2_1_ConvNeXt_training_code_freeze"
    / "T2_1_FINAL_IMPLEMENTATION_MANIFEST.json"
)

MANIFEST_PATH.write_text(
    json.dumps(
        manifest,
        indent=2,
    ),
    encoding="utf-8",
)

print()
print("=" * 78)
print("T2.1 IMPLEMENTATION FREEZE: PASS")
print("=" * 78)
print()
print("FINAL SCRIPT SHA256:")
print(actual_sha)
print()
print("HASH LOCK:")
print(LOCK_PATH)
print()
print("MANIFEST:")
print(MANIFEST_PATH)
print()
print("Previous preliminary hash is explicitly superseded.")
print("Frozen ResNet source: UNMODIFIED")
print("Step1F assignments: UNMODIFIED")
print("XAI/IG: ABSENT")
print()
print("=" * 78)
print("NEXT GATE")
print("=" * 78)
print("6ZK — one-epoch DEV cache-backed ConvNeXt smoke test")

In [ ]:
# ============================================================================
# CELL 6ZK — T2.1 ONE-EPOCH DEV CACHE-BACKED CONVNEXT SMOKE TEST
# ============================================================================
#
# Purpose:
#   Execute the FROZEN T2.1 ConvNeXt implementation for exactly one DEV epoch.
#
# This is NOT a production model.
# It exists to verify:
#   - frozen script executes
#   - ConvNeXt-Tiny loads correctly
#   - locked ImageNet weights load
#   - Step1F assignment is found
#   - cache_index is read
#   - clean/hair cache routing works
#   - validation uses clean images
#   - deterministic configuration works
#   - DataLoader works
#   - augmentation works
#   - forward/backward works
#   - optimizer works
#   - cosine scheduler works
#   - checkpoint writing works
#   - training_record is generated
#
# IMPORTANT:
#   --mode dev forces exactly 1 epoch inside the frozen implementation.
#   No production seed table row is consumed for this smoke test.
# ============================================================================

from pathlib import Path
import hashlib
import json
import os
import subprocess
import sys
import time

# ----------------------------------------------------------------------------
# Paths
# ----------------------------------------------------------------------------

SCRIPT = Path(
    "/kaggle/working/"
    "T2_1_ConvNeXt_training_code_freeze/"
    "train_convnext_tiny_t2_1.py"
)

PROJECT = Path(
    "/kaggle/input/notebooks/"
    "tejasseshadriiiii/"
    "isic-kaggle-analysis/"
    "isic_final_project"
)

CACHE = (
    PROJECT
    / "step1P_cached_arrays"
)

OUT_ROOT = Path(
    "/kaggle/working/"
    "T2_1_ConvNeXt_DEV_SMOKE_TEST"
)

OUT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

# ----------------------------------------------------------------------------
# Frozen implementation hash
# ----------------------------------------------------------------------------

EXPECTED_SHA = (
    "d492f7eff49ddf041ecf6078daa02713e6f99b10590d213efbae937242e023c7"
)

def sha256_file(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b"",
        ):
            h.update(chunk)

    return h.hexdigest()

actual_sha = sha256_file(SCRIPT)

assert actual_sha == EXPECTED_SHA, (
    "FROZEN SCRIPT HASH MISMATCH\n"
    f"Expected: {EXPECTED_SHA}\n"
    f"Actual:   {actual_sha}"
)

# ----------------------------------------------------------------------------
# Basic path checks
# ----------------------------------------------------------------------------

assert SCRIPT.exists(), SCRIPT
assert PROJECT.exists(), PROJECT
assert CACHE.exists(), CACHE

ASSIGNMENT = (
    PROJECT
    / "step1F_lesion_level_rho_assignments"
    / "rho_0.0_rep_1.csv"
)

assert ASSIGNMENT.exists(), ASSIGNMENT

# ----------------------------------------------------------------------------
# Use exactly ONE visible GPU for this process.
#
# This mirrors the original production execution model:
# one training process sees one GPU.
# ----------------------------------------------------------------------------

env = os.environ.copy()

env["CUDA_VISIBLE_DEVICES"] = "0"

# Deterministic CUDA workspace configuration must be present BEFORE
# the child Python process imports torch.
env["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

# ----------------------------------------------------------------------------
# DEV output
# ----------------------------------------------------------------------------

DEV_OUT = OUT_ROOT / "dev_rho_0.0_rep_1"

if DEV_OUT.exists():
    import shutil
    shutil.rmtree(DEV_OUT)

DEV_OUT.mkdir(
    parents=True,
    exist_ok=True,
)

# ----------------------------------------------------------------------------
# Command
#
# --mode dev => exactly one epoch according to the frozen implementation.
#
# --gap mean => explicitly enforce the T2.1 locked mean-pool architecture.
# ----------------------------------------------------------------------------

cmd = [
    sys.executable,
    str(SCRIPT),

    "--project",
    str(PROJECT),

    "--cache_dir",
    str(CACHE),

    "--rho",
    "0.0",

    "--rep",
    "1",

    "--mode",
    "dev",

    "--gap",
    "mean",

    "--out_dir",
    str(DEV_OUT),

    "--num_workers",
    "2",
]

print("=" * 78)
print("T2.1 ONE-EPOCH DEV SMOKE TEST")
print("=" * 78)

print()
print("SCRIPT:")
print(SCRIPT)

print()
print("SCRIPT SHA256:")
print(actual_sha)

print()
print("PROJECT:")
print(PROJECT)

print()
print("CACHE:")
print(CACHE)

print()
print("ASSIGNMENT:")
print(ASSIGNMENT)

print()
print("CUDA_VISIBLE_DEVICES:")
print(env["CUDA_VISIBLE_DEVICES"])

print()
print("CUBLAS_WORKSPACE_CONFIG:")
print(env["CUBLAS_WORKSPACE_CONFIG"])

print()
print("COMMAND:")
print(" ".join(cmd))

print()
print("=" * 78)
print("STARTING DEV RUN")
print("=" * 78)

t0 = time.time()

result = subprocess.run(
    cmd,
    env=env,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

elapsed = time.time() - t0

print()
print("=" * 78)
print("DEV RUN OUTPUT")
print("=" * 78)
print()

print(result.stdout)

print()
print("=" * 78)
print("DEV RUN RETURN CODE")
print("=" * 78)

print(result.returncode)

assert result.returncode == 0, (
    "\n"
    "T2.1 DEV SMOKE TEST FAILED.\n"
    f"Return code: {result.returncode}\n"
    f"Elapsed: {elapsed:.2f}s\n"
    "\n"
    "Full subprocess output is printed above."
)

# ----------------------------------------------------------------------------
# Verify expected artifacts
# ----------------------------------------------------------------------------

print()
print("=" * 78)
print("POST-RUN ARTIFACT AUDIT")
print("=" * 78)

for p in sorted(DEV_OUT.rglob("*")):

    if p.is_file():

        print(
            f"{p.relative_to(DEV_OUT)}"
            f"    {p.stat().st_size:,} bytes"
        )

# ----------------------------------------------------------------------------
# Find training record
# ----------------------------------------------------------------------------

records = sorted(
    DEV_OUT.rglob("training_record.json")
)

assert len(records) == 1, (
    "Expected exactly one training_record.json, "
    f"found {len(records)}"
)

record_path = records[0]

record = json.loads(
    record_path.read_text(
        encoding="utf-8"
    )
)

print()
print("TRAINING RECORD:")
print(record_path)

print()
print(json.dumps(
    record,
    indent=2,
    sort_keys=True,
))

# ----------------------------------------------------------------------------
# Verify script provenance in the actual executed record.
# ----------------------------------------------------------------------------

record_script_sha = record.get(
    "script_sha256"
)

print()
print("RECORDED SCRIPT SHA:")
print(record_script_sha)

assert record_script_sha == EXPECTED_SHA, (
    "Executed training record does not contain "
    "the frozen T2.1 script SHA.\n"
    f"Expected: {EXPECTED_SHA}\n"
    f"Recorded: {record_script_sha}"
)

# ----------------------------------------------------------------------------
# Checkpoint audit
# ----------------------------------------------------------------------------

final_checkpoints = sorted(
    DEV_OUT.rglob("final_epoch15.pt")
)

best_checkpoints = sorted(
    DEV_OUT.rglob("best_eval_ba.pt")
)

# In DEV mode the implementation may intentionally retain its production
# checkpoint naming. We report exactly what exists rather than assuming.
print()
print("FINAL-EPOCH CHECKPOINTS:")
for p in final_checkpoints:
    print(p)

print()
print("BEST-BA CHECKPOINTS:")
for p in best_checkpoints:
    print(p)

# ----------------------------------------------------------------------------
# Save smoke-test audit
# ----------------------------------------------------------------------------

audit = {
    "status": "PASS",

    "script":
        str(SCRIPT),

    "script_sha256":
        actual_sha,

    "project":
        str(PROJECT),

    "cache":
        str(CACHE),

    "assignment":
        str(ASSIGNMENT),

    "rho":
        0.0,

    "replicate":
        1,

    "mode":
        "dev",

    "expected_epochs":
        1,

    "cuda_visible_devices":
        env["CUDA_VISIBLE_DEVICES"],

    "cublas_workspace_config":
        env["CUBLAS_WORKSPACE_CONFIG"],

    "return_code":
        result.returncode,

    "elapsed_seconds":
        elapsed,

    "training_record":
        str(record_path),

    "recorded_script_sha256":
        record_script_sha,

    "final_epoch_checkpoint_count":
        len(final_checkpoints),

    "best_ba_checkpoint_count":
        len(best_checkpoints),
}

AUDIT_PATH = (
    OUT_ROOT
    / "T2_1_DEV_SMOKE_TEST_AUDIT.json"
)

AUDIT_PATH.write_text(
    json.dumps(
        audit,
        indent=2,
    ),
    encoding="utf-8",
)

print()
print("=" * 78)
print("T2.1 DEV SMOKE TEST: PASS")
print("=" * 78)

print()
print("Audit:")
print(AUDIT_PATH)

print()
print("The frozen ConvNeXt cache-backed execution path is executable.")

print()
print("=" * 78)
print("NEXT GATE")
print("=" * 78)
print("6ZK result must be reviewed before production training.")

In [ ]:
# ============================================================================
# CELL 6ZK-A — LOCATE AUTHORITATIVE PRODUCTION CACHE
# ============================================================================
# No training.
# No file modification.
# We only locate the actual cache used by the ISIC production pipeline.
# ============================================================================

from pathlib import Path
import json
import os

PROJECT = Path(
    "/kaggle/input/notebooks/"
    "tejasseshadriiiii/"
    "isic-kaggle-analysis/"
    "isic_final_project"
)

assert PROJECT.exists(), PROJECT

print("=" * 78)
print("T2.1 CACHE DISCOVERY")
print("=" * 78)

print()
print("PROJECT:")
print(PROJECT)

# ----------------------------------------------------------------------------
# Look for the three authoritative cache components.
# ----------------------------------------------------------------------------

REQUIRED = {
    "cache_index.csv",
    "clean_224.npy",
    "hair_224.npy",
}

candidates = []

for p in PROJECT.rglob("cache_index.csv"):

    parent = p.parent

    present = {
        x.name
        for x in parent.iterdir()
        if x.is_file()
    }

    missing = REQUIRED - present

    print()
    print("-" * 78)
    print("CACHE CANDIDATE:")
    print(parent)
    print("cache_index.csv :", (parent / "cache_index.csv").exists())
    print("clean_224.npy   :", (parent / "clean_224.npy").exists())
    print("hair_224.npy    :", (parent / "hair_224.npy").exists())

    if not missing:
        candidates.append(parent)

# ----------------------------------------------------------------------------
# Also search broader /kaggle/input in case the cache is outside PROJECT.
# ----------------------------------------------------------------------------

if not candidates:

    INPUT_ROOT = Path("/kaggle/input")

    print()
    print("=" * 78)
    print("PROJECT SEARCH FOUND NO COMPLETE CACHE")
    print("SEARCHING /kaggle/input")
    print("=" * 78)

    for p in INPUT_ROOT.rglob("cache_index.csv"):

        parent = p.parent

        try:
            names = {
                x.name
                for x in parent.iterdir()
                if x.is_file()
            }
        except Exception:
            continue

        if REQUIRED.issubset(names):

            candidates.append(parent)

            print()
            print("COMPLETE CACHE FOUND:")
            print(parent)

# ----------------------------------------------------------------------------
# Deduplicate
# ----------------------------------------------------------------------------

unique_candidates = []

seen = set()

for p in candidates:

    key = str(p.resolve())

    if key not in seen:

        seen.add(key)
        unique_candidates.append(p)

candidates = unique_candidates

print()
print("=" * 78)
print("CACHE DISCOVERY RESULT")
print("=" * 78)

print()
print("Number of complete candidates:", len(candidates))

for i, p in enumerate(candidates, start=1):

    print()
    print(f"[{i}] {p}")

# ----------------------------------------------------------------------------
# Inspect sizes / cache index.
# ----------------------------------------------------------------------------

for p in candidates:

    print()
    print("-" * 78)
    print("CACHE:", p)
    print("-" * 78)

    for name in sorted(REQUIRED):

        f = p / name

        print(
            f"{name:20s} "
            f"{f.stat().st_size:,} bytes"
        )

# ----------------------------------------------------------------------------
# Save discovery result.
# ----------------------------------------------------------------------------

result = {
    "project": str(PROJECT),
    "required_files": sorted(REQUIRED),
    "complete_cache_candidates": [
        str(p)
        for p in candidates
    ],
}

OUT = Path(
    "/kaggle/working/"
    "T2_1_CACHE_DISCOVERY.json"
)

OUT.write_text(
    json.dumps(
        result,
        indent=2,
    ),
    encoding="utf-8",
)

print()
print("=" * 78)
print("DISCOVERY COMPLETE")
print("=" * 78)

print()
print("Saved:")
print(OUT)

if len(candidates) == 1:

    print()
    print("EXACTLY ONE COMPLETE CACHE FOUND.")
    print("This is the path we should use for 6ZK.")

elif len(candidates) > 1:

    print()
    print(
        "MULTIPLE COMPLETE CACHES FOUND."
    )
    print(
        "Do NOT choose one arbitrarily."
    )
    print(
        "We will compare them against the production "
        "cache provenance before training."
    )

else:

    print()
    print(
        "NO COMPLETE CACHE FOUND."
    )
    print(
        "Do NOT train yet."
    )

In [ ]:
# ================================================================
# CELL 1 — CONFIG (ANALYSIS NOTEBOOK)
# ================================================================

AUTO_PIN_VERSIONS = globals().get("AUTO_PIN_VERSIONS", True)
ALLOW_VERSION_DEVIATION = globals().get("ALLOW_VERSION_DEVIATION", False)
N_PROC_OVERRIDE = globals().get("N_PROC_OVERRIDE", 4)
MAX_MODELS_THIS_RUN = globals().get("MAX_MODELS_THIS_RUN", None)
DEADLINE_HOURS = globals().get("DEADLINE_HOURS", 11.0)
EST_MODEL_HOURS = 0.15

print("✓ analysis-notebook config loaded")

In [ ]:
# ================================================================
# CELL 2 — HELPERS
# ================================================================

import os, sys, json, time, shutil, hashlib, subprocess, platform
from pathlib import Path
from collections import deque
from datetime import datetime, timezone

ROOT = Path(os.environ.get("KAGGLE_TEST_ROOT", "/kaggle"))
INPUT, WORK = ROOT / "input", ROOT / "working"
TEMP = ROOT / "temp"

TEMP.mkdir(parents=True, exist_ok=True)
WORK.mkdir(parents=True, exist_ok=True)

PROJECT = WORK / "isic_final_project"
CACHE = TEMP / "isic_cache"
RECORDS = WORK / "analysis_records"
RESULTS = WORK / "results"
TRAINED_MODELS = WORK / "trained_models_readonly"
LOCAL_TMP = TEMP / "tmp_work"

for d in (RECORDS, RESULTS, LOCAL_TMP):
    d.mkdir(parents=True, exist_ok=True)

T0 = time.time()

def utc():
    return datetime.now(timezone.utc).isoformat()

def elapsed_h():
    return (time.time() - T0) / 3600

def log(msg):
    print(f"[{elapsed_h():5.2f} h] {msg}", flush=True)

def sha_file(p, chunk=1 << 22):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()

def J(rel):
    return json.load(open(PROJECT / rel))

def save_record(name, obj):
    with open(RECORDS / name, "w") as f:
        json.dump(obj, f, indent=2, sort_keys=True, default=str)

def find_by_markers(root, required_subdirs, max_depth=6):
    root = Path(root)
    if not root.exists():
        return None

    required = set(required_subdirs)
    q = deque([(root, 0)])

    while q:
        d, depth = q.popleft()

        try:
            entries = list(os.scandir(d))
        except (PermissionError, FileNotFoundError, NotADirectoryError):
            continue

        names = {e.name for e in entries if e.is_dir()}

        if required <= names:
            return d

        if depth < max_depth:
            for e in entries:
                if e.is_dir() and not e.name.startswith("."):
                    q.append((Path(e.path), depth + 1))

    return None

def _has_min_files(d, exts, min_count, scan_cap=20000):
    n = 0

    try:
        with os.scandir(d) as it:
            for i, e in enumerate(it):
                if i >= scan_cap:
                    break

                if e.is_file() and e.name.lower().endswith(exts):
                    n += 1

                    if n >= min_count:
                        return True

    except (PermissionError, FileNotFoundError, NotADirectoryError):
        return False

    return False

def find_image_dir(root, exts=(".jpg", ".jpeg"), min_images=100, max_depth=6):
    root = Path(root)

    if not root.exists():
        return None

    q = deque([(root, 0)])
    found = []

    while q:
        d, depth = q.popleft()

        try:
            entries = list(os.scandir(d))
        except (PermissionError, FileNotFoundError, NotADirectoryError):
            continue

        if _has_min_files(d, exts, min_images):
            found.append(d)
            continue

        if depth < max_depth:
            for e in entries:
                if e.is_dir() and not e.name.startswith("."):
                    q.append((Path(e.path), depth + 1))

    if not found:
        return None

    found.sort(
        key=lambda p: (
            0 if "train" in str(p).lower() else 1,
            len(str(p))
        )
    )

    return found[0]

print("✓ helpers and paths loaded")
print("CACHE:", CACHE)

In [ ]:
# ================================================================
# STAGE 0 — BRING THE LOCKED PROJECT IN AND VERIFY THE WHOLE LOCK CHAIN
# ================================================================
PROJECT_MARKERS = {"step0I_locked_clean_population", "step1F_lesion_level_rho_assignments"}

if not PROJECT.exists():
    # Kaggle's mount layout is not fixed (seen both /kaggle/input/<slug>/... and /kaggle/input/datasets/<owner>/<slug>/...),
    # so this SEARCHES under /kaggle/input for the directory that actually has the two lock-chain marker folders,
    # rather than assuming a fixed depth. The dataset itself is never modified - only read and copied.
    src = find_by_markers(INPUT, PROJECT_MARKERS)
    assert src is not None, (
        "Project folder not found anywhere under /kaggle/input (searched up to 6 levels deep for a directory containing "
        f"{sorted(PROJECT_MARKERS)}). Check that your project dataset is attached under Add Data, and that the zip you "
        "uploaded actually has these folders at its top level (not nested inside an extra wrapper folder Kaggle can't see into).")
    log(f"found locked project at {src} - copying (read-only source, nothing there is modified) ...")
    shutil.copytree(src, PROJECT)
n_files = sum(1 for p in PROJECT.rglob("*") if p.is_file())
print(f"project working copy: {PROJECT}  ({n_files} files)")

# ---- verify every locked artifact against its recorded hash (environment and cache are re-recorded on this platform, so they are not checked here)
checks = []
def check(name, ok, detail=""):
    checks.append((name, bool(ok))); print(("✓ " if ok else "✗ ") + name + (f"   {detail}" if detail else "")); assert ok, "LOCK CHAIN CHECK FAILED: " + name
sha = lambda rel: sha_file(PROJECT / rel)
F = J("step1U_final_protocol_lock/FINAL_PROTOCOL_HASH.json")
check("final protocol == recorded hash", sha("step1U_final_protocol_lock/FINAL_LOCKED_EXPERIMENT_PROTOCOL.json") == F["protocol_sha256"])
check("seed manifest == recorded hash", sha("step1P_seed_manifest/seed_manifest.json") == F["seed_manifest_sha256"])
_cand = {sha("step1I_protocol_lock/LOCKED_EXPERIMENT_PROTOCOL.json")} | {v for v in J("step1I_protocol_lock/protocol_hash.json").values() if isinstance(v, str)}
check("parent 1I protocol hash", F["parent_step1I_sha256"] in _cand)
check("A1 == recorded hash", sha("step2O_A1_final_lock/FINAL_LOCKED_AMENDMENT_A1.json") == J("step2O_A1_final_lock/A1_FINAL_HASH.json")["sha256"])
check("A1.1 == recorded hash", sha("step2P_A1_1_solver_addendum_lock/FINAL_LOCKED_A1_1_SOLVER_ADDENDUM.json") == J("step2P_A1_1_solver_addendum_lock/A1_1_FINAL_HASH.json")["sha256"])
GEN_SHA = J("step1J_generator_recovery_lock/generator_recovery_lock.json")["generator_sha256"]
check("generator == recorded hash", sha("step1J_generator_recovery_lock/recovered_original_hair_generator.py") == GEN_SHA)
check("solver == recorded hash", sha("step2Q_solver_implementation_v2/deterministic_evaluation_solver.py") == J("step2Q_solver_implementation_v2/solver_implementation_metadata.json")["solver_sha256"])
check("analysis code == frozen hash", sha("step2W_analysis_code_freeze/analysis_pipeline.py") == J("step2W_analysis_code_freeze/ANALYSIS_CODE_FINAL_HASH.json")["sha256"])
check("training code == frozen hash", sha("step2X_training_code_freeze/train_one_model.py") == J("step2X_training_code_freeze/TRAINING_CODE_FINAL_HASH.json")["sha256"])
E = J("step2S_evaluation_manifest_lock/EVALUATION_MANIFEST_FINAL_HASH.json")
check("3 evaluation manifests == locked hashes", all(sha(f"step2R_production_evaluation_manifests/evaluation_{c}.csv") == h for c, h in E["manifest_sha256"].items()))
check("XAI subset == locked hash", sha("step2U_xai_subset_lock/LOCKED_XAI_SUBSET.csv") == J("step2U_xai_subset_lock/XAI_SUBSET_FINAL_HASH.json")["sha256"])
RO = J("step2V_seed_table_and_run_order/RUN_ORDER_FINAL_HASH.json")
check("run-order manifest == locked hash", sha("step2V_seed_table_and_run_order/RUN_ORDER_MANIFEST.json") == RO["sha256"])
check("seed table == locked hash", sha("step2V_seed_table_and_run_order/TRAINING_SEED_TABLE.csv") == RO["seed_table_sha256"])
asg = list((PROJECT / "step1F_lesion_level_rho_assignments").glob("rho_*_rep_*.csv"))
check("30 training assignment files present", len(asg) == 30, f"({len(asg)})")
RUN_ORDER = json.load(open(PROJECT / "step2V_seed_table_and_run_order/RUN_ORDER_MANIFEST.json"))["run_order"]
check("run order lists 30 models", len(RUN_ORDER) == 30)
TRAIN = PROJECT / "step2X_training_code_freeze" / "train_one_model.py"
ANALYSIS = PROJECT / "step2W_analysis_code_freeze" / "analysis_pipeline.py"
save_record("stage0_lock_chain_verified.json", {"verified": dict(checks), "project_files": n_files, "utc": utc()})
print("\nSTAGE 0 PASSED: the locked project arrived intact.")


In [ ]:
# ================================================================
# STAGE 0b — COMPATIBILITY SYMLINK
# ================================================================

CONTENT_PROJECT = Path("/content/isic_final_project")

if CONTENT_PROJECT.exists():
    print(f"✓ {CONTENT_PROJECT} already exists - leaving it as is")
else:
    CONTENT_PROJECT.parent.mkdir(parents=True, exist_ok=True)

    try:
        CONTENT_PROJECT.symlink_to(
            PROJECT,
            target_is_directory=True
        )

        print(
            f"✓ compatibility symlink created: "
            f"{CONTENT_PROJECT} -> {PROJECT}"
        )

    except OSError as e:
        print(f"Symlink failed ({e}) - falling back to a full copy")
        shutil.copytree(PROJECT, CONTENT_PROJECT)

gen_sha = sha_file(
    PROJECT
    / "step1J_generator_recovery_lock"
    / "recovered_original_hair_generator.py"
)

assert gen_sha == json.load(
    open(
        PROJECT
        / "step1J_generator_recovery_lock"
        / "generator_recovery_lock.json"
    )
)["generator_sha256"]

print("✓ generator file hash unchanged")

In [ ]:
# ================================================================
# STAGE 2 — BUILD THE 224x224 CACHE ON KAGGLE
#
# CPU / multiprocessing only — CUDA is NOT used here.
#
# IMPORTANT:
# - Scientific hair generator is NOT modified.
# - Only imports, literal constants, and function definitions
#   are loaded from the recovered generator.
# - Module-level validation/test code is deliberately not executed.
# - Locked population + medium severity + deterministic generator
#   are preserved.
# ================================================================

import textwrap
import subprocess
import time
import json
import hashlib
from pathlib import Path

# ----------------------------------------------------------------
# Locate ISIC image directory
# ----------------------------------------------------------------

IMAGE_DIR = (
    globals().get("IMAGE_DIR_OVERRIDE")
    or find_image_dir(INPUT, min_images=100)
)

assert IMAGE_DIR is not None, (
    "ISIC image folder not found anywhere under /kaggle/input. "
    "Check that the isic-2019-dataset-full dataset is attached "
    "through Add Data."
)

print("image folder:", IMAGE_DIR)

# ----------------------------------------------------------------
# Required locked paths
# ----------------------------------------------------------------

GEN_PATH = (
    PROJECT
    / "step1J_generator_recovery_lock"
    / "recovered_original_hair_generator.py"
)

POP_PATH = (
    PROJECT
    / "step0I_locked_clean_population"
    / "LOCKED_CLEAN_development_population.csv"
)

CACHE_LOCK = RECORDS / "cache_final_hash.json"

CACHE.mkdir(
    parents=True,
    exist_ok=True
)

assert GEN_PATH.exists(), (
    f"Recovered hair generator not found:\n{GEN_PATH}"
)

assert POP_PATH.exists(), (
    f"Locked development population not found:\n{POP_PATH}"
)

# ----------------------------------------------------------------
# Required cache files
# ----------------------------------------------------------------

REQUIRED_CACHE_FILES = [
    "clean_224.npy",
    "hair_224.npy",
    "mask_area.npy",
    "cache_index.csv",
]

cache_files_present = all(
    (CACHE / f).exists()
    for f in REQUIRED_CACHE_FILES
)

# ----------------------------------------------------------------
# Existing complete cache?
# ----------------------------------------------------------------

if (
    CACHE_LOCK.exists()
    and cache_files_present
    and not globals().get("ALLOW_RERUN_CACHE", False)
):

    print("=" * 80)
    print("STAGE 2 — EXISTING CACHE DETECTED")
    print("=" * 80)

    print("✓ cache_final_hash.json exists")

    for f in REQUIRED_CACHE_FILES:
        print("✓", CACHE / f)

    print("\n✓ Complete cache already exists.")
    print("Skipping rebuild.")

else:

    # ============================================================
    # Remove partial/stale cache
    # ============================================================

    print("=" * 80)
    print("STAGE 2 — PREPARING CLEAN CACHE BUILD")
    print("=" * 80)

    if CACHE_LOCK.exists():
        print("Removing stale/incomplete cache lock:", CACHE_LOCK)
        CACHE_LOCK.unlink()

    for name in [
        "clean_224.npy",
        "hair_224.npy",
        "mask_area.npy",
        "cache_index.csv",
        "_builder_record.json",
    ]:

        p = CACHE / name

        if p.exists():
            p.unlink()
            print("removed:", p)

    print("\n✓ Partial cache cleared.")

    # ============================================================
    # Standalone CPU cache-builder subprocess
    # ============================================================

    BUILDER = TEMP / "cache_builder.py"

    BUILDER.write_text(
        textwrap.dedent(
            r'''
            import argparse
            import ast
            import hashlib
            import json
            import random
            from pathlib import Path

            import numpy as np
            import pandas as pd
            from PIL import Image
            import multiprocessing as mp

            SEV = "medium"

            def sha_file(p, chunk=1 << 22):
                h = hashlib.sha256()

                with open(p, "rb") as f:
                    for b in iter(
                        lambda: f.read(chunk),
                        b""
                    ):
                        h.update(b)

                return h.hexdigest()

            def find_image(image_dir, image_id):

                for ext in (
                    ".jpg",
                    ".jpeg",
                    ".JPG",
                    ".JPEG"
                ):

                    p = (
                        image_dir
                        / f"{image_id}{ext}"
                    )

                    if p.exists():
                        return p

                raise FileNotFoundError(image_id)

            _G = {}

            # ====================================================
            # SAFE GENERATOR LOADER
            # ====================================================

            def load_generator(gen_path):

                source = Path(gen_path).read_text(
                    encoding="utf-8"
                )

                tree = ast.parse(source)

                safe_nodes = []

                for node in tree.body:

                    if isinstance(
                        node,
                        (
                            ast.Import,
                            ast.ImportFrom,
                        )
                    ):

                        safe_nodes.append(node)
                        continue

                    if isinstance(
                        node,
                        (
                            ast.FunctionDef,
                            ast.AsyncFunctionDef,
                        )
                    ):

                        safe_nodes.append(node)
                        continue

                    if isinstance(node, ast.Assign):

                        try:
                            ast.literal_eval(node.value)
                            safe_nodes.append(node)

                        except Exception:
                            pass

                    elif isinstance(node, ast.AnnAssign):

                        if node.value is not None:

                            try:
                                ast.literal_eval(node.value)
                                safe_nodes.append(node)

                            except Exception:
                                pass

                module = ast.Module(
                    body=safe_nodes,
                    type_ignores=[]
                )

                ast.fix_missing_locations(module)

                ns = {
                    "__file__": str(gen_path),
                    "__name__": "recovered_hair_generator",
                }

                exec(
                    compile(
                        module,
                        str(gen_path),
                        "exec"
                    ),
                    ns
                )

                assert "generate_hair_mask" in ns
                assert "apply_hair" in ns

                return ns

            # ====================================================
            # WORKER INITIALIZER
            # ====================================================

            def _init(
                gen_path,
                cp,
                hp,
                image_dir
            ):

                ns = load_generator(gen_path)

                _G.update(
                    gm=ns["generate_hair_mask"],
                    ah=ns["apply_hair"],

                    clean=np.load(
                        cp,
                        mmap_mode="r+"
                    ),

                    hair=np.load(
                        hp,
                        mmap_mode="r+"
                    ),

                    image_dir=Path(image_dir)
                )

            # ====================================================
            # WORKER
            # ====================================================

            def _work(job):

                lo, ids = job
                out = []

                for r, image_id in zip(
                    range(
                        lo,
                        lo + len(ids)
                    ),
                    ids
                ):

                    clean = np.asarray(
                        Image.open(
                            find_image(
                                _G["image_dir"],
                                image_id
                            )
                        )
                        .convert("RGB")
                        .resize(
                            (
                                224,
                                224
                            ),
                            Image.Resampling.LANCZOS
                        ),
                        dtype=np.uint8
                    )

                    mask = _G["gm"](
                        image_id=image_id,
                        severity=SEV
                    )

                    hair = _G["ah"](
                        image=clean,
                        mask=mask,
                        severity=SEV,
                        image_id=image_id
                    )

                    m = mask > 0

                    outside = int(
                        (
                            np.any(
                                hair != clean,
                                axis=2
                            )
                            & ~m
                        ).sum()
                    )

                    _G["clean"][r] = clean
                    _G["hair"][r] = hair

                    out.append(
                        (
                            r,
                            int(m.sum()),
                            outside
                        )
                    )

                _G["clean"].flush()
                _G["hair"].flush()

                return out

            # ====================================================
            # MAIN
            # ====================================================

            def main():

                ap = argparse.ArgumentParser()

                ap.add_argument(
                    "--gen-path",
                    required=True
                )

                ap.add_argument(
                    "--pop-path",
                    required=True
                )

                ap.add_argument(
                    "--cache-dir",
                    required=True
                )

                ap.add_argument(
                    "--image-dir",
                    required=True
                )

                ap.add_argument(
                    "--n-proc",
                    type=int,
                    default=4
                )

                a = ap.parse_args()

                gen_path = Path(a.gen_path)
                pop_path = Path(a.pop_path)
                cache_dir = Path(a.cache_dir)
                image_dir = Path(a.image_dir)

                # ------------------------------------------------
                # Locked population
                # ------------------------------------------------

                pop = pd.read_csv(pop_path)

                assert list(pop.columns) == [
                    "image",
                    "lesion_id",
                    "class_name",
                    "malignant",
                    "split"
                ]

                pop = (
                    pop
                    .sort_values(
                        [
                            "split",
                            "image"
                        ]
                    )
                    .reset_index(drop=True)
                )

                N = len(pop)

                print(
                    f"locked population: {N} images",
                    flush=True
                )

                # ------------------------------------------------
                # Cache index
                # ------------------------------------------------

                idx = pop.copy()

                idx["row"] = np.arange(N)

                idx.to_csv(
                    cache_dir / "cache_index.csv",
                    index=False
                )

                # ------------------------------------------------
                # Allocate arrays
                # ------------------------------------------------

                clean_p = cache_dir / "clean_224.npy"
                hair_p = cache_dir / "hair_224.npy"
                area_p = cache_dir / "mask_area.npy"

                for p in (
                    clean_p,
                    hair_p
                ):

                    np.lib.format.open_memmap(
                        p,
                        mode="w+",
                        dtype=np.uint8,
                        shape=(
                            N,
                            224,
                            224,
                            3
                        )
                    ).flush()

                # ------------------------------------------------
                # Jobs
                # ------------------------------------------------

                ids = (
                    pop["image"]
                    .astype(str)
                    .tolist()
                )

                jobs = [
                    (
                        lo,
                        ids[
                            lo:
                            lo + 64
                        ]
                    )
                    for lo in range(
                        0,
                        N,
                        64
                    )
                ]

                area = np.zeros(
                    N,
                    dtype=np.int32
                )

                outside_total = 0
                done = 0

                print(
                    f"starting {a.n_proc} cache workers...",
                    flush=True
                )

                ctx = mp.get_context("fork")

                with ctx.Pool(
                    a.n_proc,
                    initializer=_init,
                    initargs=(
                        str(gen_path),
                        str(clean_p),
                        str(hair_p),
                        str(image_dir),
                    )
                ) as pool:

                    for res in pool.imap_unordered(
                        _work,
                        jobs
                    ):

                        for (
                            r,
                            ar,
                            o
                        ) in res:

                            area[r] = ar
                            outside_total += o

                        done += len(res)

                        print(
                            f"cache {done:6d}/{N}",
                            flush=True
                        )

                # ------------------------------------------------
                # Causal-integrity check
                # ------------------------------------------------

                if outside_total != 0:

                    print(
                        f"FATAL: {outside_total} pixels changed "
                        f"outside the hair mask",
                        flush=True
                    )

                    raise SystemExit(1)

                # ------------------------------------------------
                # Save mask areas
                # ------------------------------------------------

                np.save(
                    area_p,
                    area
                )

                # ------------------------------------------------
                # Deterministic spot check
                # ------------------------------------------------

                print(
                    "running deterministic 60-image spot check...",
                    flush=True
                )

                ns = load_generator(gen_path)

                cm = np.load(
                    clean_p,
                    mmap_mode="r"
                )

                hm = np.load(
                    hair_p,
                    mmap_mode="r"
                )

                sample_rows = random.Random(
                    20260919
                ).sample(
                    range(N),
                    min(60, N)
                )

                for r in sample_rows:

                    image_id = ids[r]

                    c = np.asarray(
                        Image.open(
                            find_image(
                                image_dir,
                                image_id
                            )
                        )
                        .convert("RGB")
                        .resize(
                            (
                                224,
                                224
                            ),
                            Image.Resampling.LANCZOS
                        ),
                        dtype=np.uint8
                    )

                    h = ns["apply_hair"](
                        image=c,
                        mask=ns["generate_hair_mask"](
                            image_id=image_id,
                            severity=SEV
                        ),
                        severity=SEV,
                        image_id=image_id
                    )

                    assert np.array_equal(
                        cm[r],
                        c
                    ), (
                        f"clean cache mismatch at row {r}"
                    )

                    assert np.array_equal(
                        hm[r],
                        h
                    ), (
                        f"hair cache mismatch at row {r}"
                    )

                print(
                    "SPOT_CHECK_OK",
                    flush=True
                )

                # ------------------------------------------------
                # SHA256
                # ------------------------------------------------

                file_sha = {
                    p.name: sha_file(p)
                    for p in (
                        clean_p,
                        hair_p,
                        area_p,
                        cache_dir / "cache_index.csv"
                    )
                }

                record = {
                    "n_images": N,

                    "mask_area": {
                        "min": int(area.min()),
                        "mean": float(area.mean()),
                        "max": int(area.max()),
                    },

                    "file_sha256": file_sha,

                    "generator_sha256": sha_file(
                        gen_path
                    ),

                    "image_dir": str(image_dir),

                    "severity": SEV,
                }

                with open(
                    cache_dir / "_builder_record.json",
                    "w"
                ) as f:

                    json.dump(
                        record,
                        f,
                        indent=2,
                        sort_keys=True
                    )

                print(
                    "CACHE_BUILD_COMPLETE",
                    flush=True
                )

                print(
                    json.dumps(
                        record,
                        indent=2,
                        sort_keys=True
                    ),
                    flush=True
                )

            if __name__ == "__main__":
                main()
            '''
        )
    )

    # ============================================================
    # Run CPU builder
    # ============================================================

    n_proc = int(
        globals().get(
            "N_PROC_OVERRIDE",
            4
        )
    )

    print("=" * 80)
    print("LAUNCHING CPU CACHE BUILDER")
    print("=" * 80)

    print("CPU workers:", n_proc)
    print("GPU: NOT USED")
    print("Image directory:", IMAGE_DIR)
    print("Cache directory:", CACHE)

    t_cache = time.time()

    cmd = [
        sys.executable,
        str(BUILDER),

        "--gen-path",
        str(GEN_PATH),

        "--pop-path",
        str(POP_PATH),

        "--cache-dir",
        str(CACHE),

        "--image-dir",
        str(IMAGE_DIR),

        "--n-proc",
        str(n_proc),
    ]

    proc = subprocess.run(
        cmd,
        text=True
    )

    if proc.returncode != 0:
        raise RuntimeError(
            f"Cache builder failed with return code "
            f"{proc.returncode}"
        )

    print()
    print(
        f"Cache build wall time: "
        f"{(time.time() - t_cache) / 60:.2f} minutes"
    )

# ----------------------------------------------------------------
# Final cache verification
# ----------------------------------------------------------------

print("\n" + "=" * 80)
print("FINAL CACHE VERIFICATION")
print("=" * 80)

for f in REQUIRED_CACHE_FILES:
    p = CACHE / f
    assert p.exists(), f"Missing cache file: {p}"
    print("✓", p)

assert (CACHE / "_builder_record.json").exists()

print("\n✓ CACHE BUILD / VERIFICATION COMPLETE")
print("✓ GPU was not used.")

In [ ]:
# ============================================================================
# CELL 6ZK-C — PERSIST AND AUDIT THE RECOVERED PRODUCTION CACHE
# ============================================================================
#
# The cache has already been generated at:
#     /kaggle/temp/isic_cache
#
# DO NOT REGENERATE.
#
# This cell:
#   1. verifies all required cache artifacts,
#   2. inspects cache_index.csv,
#   3. records array shapes/dtypes,
#   4. hashes the cache files,
#   5. copies the exact cache into /kaggle/working,
#   6. records provenance.
# ============================================================================

from pathlib import Path
import hashlib
import json
import shutil
import pandas as pd
import numpy as np
from datetime import datetime, timezone

SOURCE_CACHE = Path(
    "/kaggle/temp/isic_cache"
)

PERSISTED_CACHE = Path(
    "/kaggle/working/T2_1_PRODUCTION_CACHE"
)

assert SOURCE_CACHE.exists(), SOURCE_CACHE

REQUIRED = [
    "clean_224.npy",
    "hair_224.npy",
    "mask_area.npy",
    "cache_index.csv",
]

print("=" * 78)
print("T2.1 RECOVERED CACHE AUDIT")
print("=" * 78)

print()
print("SOURCE CACHE:")
print(SOURCE_CACHE)

# ----------------------------------------------------------------------------
# Verify source artifacts
# ----------------------------------------------------------------------------

for name in REQUIRED:

    p = SOURCE_CACHE / name

    assert p.exists(), (
        f"Missing required cache artifact: {p}"
    )

    print(
        f"PASS  {name:20s} "
        f"{p.stat().st_size:,} bytes"
    )

# ----------------------------------------------------------------------------
# SHA256 helper
# ----------------------------------------------------------------------------

def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b"",
        ):

            h.update(chunk)

    return h.hexdigest()

# ----------------------------------------------------------------------------
# Inspect arrays
# ----------------------------------------------------------------------------

clean = np.load(
    SOURCE_CACHE / "clean_224.npy",
    mmap_mode="r",
)

hair = np.load(
    SOURCE_CACHE / "hair_224.npy",
    mmap_mode="r",
)

mask_area = np.load(
    SOURCE_CACHE / "mask_area.npy",
    mmap_mode="r",
)

idx = pd.read_csv(
    SOURCE_CACHE / "cache_index.csv"
)

print()
print("=" * 78)
print("CACHE STRUCTURE")
print("=" * 78)

print()
print("clean_224:")
print("  shape :", clean.shape)
print("  dtype :", clean.dtype)

print()
print("hair_224:")
print("  shape :", hair.shape)
print("  dtype :", hair.dtype)

print()
print("mask_area:")
print("  shape :", mask_area.shape)
print("  dtype :", mask_area.dtype)

print()
print("cache_index:")
print("  shape :", idx.shape)
print("  columns:", list(idx.columns))

# ----------------------------------------------------------------------------
# Basic structural checks
# ----------------------------------------------------------------------------

assert clean.shape == hair.shape, (
    "clean_224 and hair_224 shape mismatch"
)

assert clean.shape[0] == len(idx), (
    "cache array row count != cache_index row count"
)

assert mask_area.shape[0] == len(idx), (
    "mask_area row count != cache_index row count"
)

assert clean.shape[1:] == (224, 224, 3), (
    f"Unexpected clean cache shape: {clean.shape}"
)

assert hair.shape[1:] == (224, 224, 3), (
    f"Unexpected hair cache shape: {hair.shape}"
)

assert idx["image"].is_unique, (
    "cache_index image column is not unique"
)

assert idx["row"].is_unique, (
    "cache_index row column is not unique"
)

assert set(idx["split"].dropna().unique()).issubset(
    {"train", "val"}
), (
    "Unexpected split values in cache_index"
)

# ----------------------------------------------------------------------------
# Check row mapping
# ----------------------------------------------------------------------------

expected_rows = np.arange(
    len(idx),
    dtype=idx["row"].dtype,
)

actual_rows = np.sort(
    idx["row"].to_numpy()
)

assert np.array_equal(
    actual_rows,
    expected_rows,
), (
    "cache_index row values are not a complete 0..N-1 mapping"
)

# ----------------------------------------------------------------------------
# Check source cache counts
# ----------------------------------------------------------------------------

print()
print("=" * 78)
print("CACHE COUNTS")
print("=" * 78)

print()
print("Total cached images:", len(idx))

print()
print("Split counts:")
print(
    idx["split"]
    .value_counts()
    .sort_index()
    .to_string()
)

# ----------------------------------------------------------------------------
# Hash all required artifacts
# ----------------------------------------------------------------------------

print()
print("=" * 78)
print("CACHE SHA256")
print("=" * 78)

hashes = {}

for name in REQUIRED:

    digest = sha256_file(
        SOURCE_CACHE / name
    )

    hashes[name] = digest

    print()
    print(name)
    print(digest)

# ----------------------------------------------------------------------------
# Persist exact cache.
# ----------------------------------------------------------------------------

if PERSISTED_CACHE.exists():

    print()
    print("Persisted cache already exists.")

    # Verify that existing persisted artifacts are byte-identical
    # before replacing anything.
    for name in REQUIRED:

        src = SOURCE_CACHE / name
        dst = PERSISTED_CACHE / name

        if dst.exists():

            src_hash = sha256_file(src)
            dst_hash = sha256_file(dst)

            assert src_hash == dst_hash, (
                f"Existing persisted cache differs for {name}"
            )

        else:

            shutil.copy2(
                src,
                dst,
            )

else:

    PERSISTED_CACHE.mkdir(
        parents=True,
        exist_ok=True,
    )

    for name in REQUIRED:

        shutil.copy2(
            SOURCE_CACHE / name,
            PERSISTED_CACHE / name,
        )

# ----------------------------------------------------------------------------
# Verify persisted copy byte-for-byte.
# ----------------------------------------------------------------------------

print()
print("=" * 78)
print("PERSISTED CACHE VERIFICATION")
print("=" * 78)

persisted_hashes = {}

for name in REQUIRED:

    src_hash = sha256_file(
        SOURCE_CACHE / name
    )

    dst_hash = sha256_file(
        PERSISTED_CACHE / name
    )

    persisted_hashes[name] = dst_hash

    print(
        f"{name:20s}: "
        f"{'BYTE-EXACT PASS' if src_hash == dst_hash else 'FAIL'}"
    )

    assert src_hash == dst_hash

# ----------------------------------------------------------------------------
# Save provenance lock.
# ----------------------------------------------------------------------------

provenance = {

    "status":
        "RECOVERED_AND_PERSISTED",

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "source_cache":
        str(SOURCE_CACHE),

    "persisted_cache":
        str(PERSISTED_CACHE),

    "generator_sha256":
        "32be8092c85458bed31a547f2ba6c57c3e1c2cf257d7cb0f76bd28eebc7b26c2",

    "image_dir":
        "/kaggle/input/datasets/alifshahriar/isic-2019-dataset-full/"
        "isic-2019-dataset-full/train",

    "n_images":
        int(len(idx)),

    "clean_shape":
        list(clean.shape),

    "hair_shape":
        list(hair.shape),

    "mask_area_shape":
        list(mask_area.shape),

    "clean_dtype":
        str(clean.dtype),

    "hair_dtype":
        str(hair.dtype),

    "mask_area_dtype":
        str(mask_area.dtype),

    "cache_index_columns":
        list(idx.columns),

    "cache_file_sha256":
        hashes,

    "persisted_file_sha256":
        persisted_hashes,

    "byte_exact_persistence":
        True,
}

LOCK = (
    PERSISTED_CACHE
    / "T2_1_PRODUCTION_CACHE_PROVENANCE_LOCK.json"
)

LOCK.write_text(
    json.dumps(
        provenance,
        indent=2,
    ),
    encoding="utf-8",
)

print()
print("=" * 78)
print("T2.1 CACHE PERSISTENCE: PASS")
print("=" * 78)

print()
print("Persisted cache:")
print(PERSISTED_CACHE)

print()
print("Provenance lock:")
print(LOCK)

print()
print("The exact recovered cache will now be used.")
print("NO CACHE REGENERATION REQUIRED.")

print()
print("=" * 78)
print("NEXT GATE")
print("=" * 78)
print("6ZK-D — run the one-epoch DEV ConvNeXt smoke test")

In [ ]:
# ============================================================================
# CELL 6ZK-C2 — COMPLETE CACHE HASH + PERSISTENCE VERIFICATION
# ============================================================================

from pathlib import Path
import hashlib
import json

SOURCE = Path("/kaggle/temp/isic_cache")
PERSISTED = Path("/kaggle/working/T2_1_PRODUCTION_CACHE")

FILES = [
    "clean_224.npy",
    "hair_224.npy",
    "mask_area.npy",
    "cache_index.csv",
]

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

print("=" * 78)
print("CACHE SHA256 + PERSISTENCE VERIFICATION")
print("=" * 78)

source_hashes = {}
persisted_hashes = {}

for name in FILES:

    src = SOURCE / name
    dst = PERSISTED / name

    assert src.exists(), f"Missing source: {src}"
    assert dst.exists(), f"Missing persisted: {dst}"

    sh = sha256_file(src)
    ph = sha256_file(dst)

    source_hashes[name] = sh
    persisted_hashes[name] = ph

    print()
    print(name)
    print("  SOURCE   :", sh)
    print("  PERSISTED:", ph)

    assert sh == ph, (
        f"BYTE-EXACT FAILURE: {name}"
    )

    print("  STATUS   : BYTE-EXACT PASS")

# --------------------------------------------------------------------------
# Validate provenance lock
# --------------------------------------------------------------------------

LOCK = PERSISTED / "T2_1_PRODUCTION_CACHE_PROVENANCE_LOCK.json"

assert LOCK.exists(), LOCK

lock = json.loads(
    LOCK.read_text(encoding="utf-8")
)

print()
print("=" * 78)
print("PROVENANCE LOCK")
print("=" * 78)

print(
    json.dumps(
        lock,
        indent=2,
    )
)

assert lock["generator_sha256"] == (
    "32be8092c85458bed31a547f2ba6c57c3e1c2cf257d7cb0f76bd28eebc7b26c2"
)

assert lock["n_images"] == 23227

assert lock["clean_shape"] == [
    23227,
    224,
    224,
    3,
]

assert lock["hair_shape"] == [
    23227,
    224,
    224,
    3,
]

assert lock["clean_dtype"] == "uint8"
assert lock["hair_dtype"] == "uint8"

assert lock["byte_exact_persistence"] is True

# --------------------------------------------------------------------------
# Save final hash audit
# --------------------------------------------------------------------------

AUDIT = {
    "status": "PASS",
    "source_cache": str(SOURCE),
    "persisted_cache": str(PERSISTED),
    "source_sha256": source_hashes,
    "persisted_sha256": persisted_hashes,
    "byte_exact_all_files": True,
    "generator_sha256":
        lock["generator_sha256"],
    "n_images":
        lock["n_images"],
    "train_images":
        18514,
    "validation_images":
        4713,
}

AUDIT_PATH = (
    PERSISTED /
    "T2_1_CACHE_FINAL_HASH_AUDIT.json"
)

AUDIT_PATH.write_text(
    json.dumps(
        AUDIT,
        indent=2,
    ),
    encoding="utf-8",
)

print()
print("=" * 78)
print("CACHE FINAL AUDIT: PASS")
print("=" * 78)

print()
print("All four cache artifacts are byte-identical.")
print("Generator SHA matches the recovered production generator.")
print("Train count = 18,514.")
print("Validation count = 4,713.")

print()
print("AUDIT:")
print(AUDIT_PATH)

print()
print("=" * 78)
print("NEXT GATE")
print("=" * 78)
print("6ZK-D — one-epoch DEV ConvNeXt smoke test")

In [ ]:
# ============================================================================
# CELL 6ZK-D — T2.1 ONE-EPOCH DEV CONVNEXT SMOKE TEST
# ============================================================================
#
# Uses the now-frozen exact production cache:
#   /kaggle/working/T2_1_PRODUCTION_CACHE
#
# Uses frozen T2.1 implementation:
#   d492f7eff49ddf041ecf6078daa02713e6f99b10590d213efbae937242e023c7
#
# DEV ONLY — NOT PART OF THE 9 PRODUCTION MODELS.
# ============================================================================

from pathlib import Path
import hashlib
import json
import os
import subprocess
import sys
import time

# ----------------------------------------------------------------------------
# Frozen paths
# ----------------------------------------------------------------------------

SCRIPT = Path(
    "/kaggle/working/"
    "T2_1_ConvNeXt_training_code_freeze/"
    "train_convnext_tiny_t2_1.py"
)

PROJECT = Path(
    "/kaggle/input/notebooks/"
    "tejasseshadriiiii/"
    "isic-kaggle-analysis/"
    "isic_final_project"
)

CACHE = Path(
    "/kaggle/working/"
    "T2_1_PRODUCTION_CACHE"
)

ASSIGNMENT = (
    PROJECT
    / "step1F_lesion_level_rho_assignments"
    / "rho_0.0_rep_1.csv"
)

OUT_ROOT = Path(
    "/kaggle/working/"
    "T2_1_ConvNeXt_DEV_SMOKE_TEST"
)

DEV_OUT = (
    OUT_ROOT
    / "rho_0.0_rep_1"
)

EXPECTED_SCRIPT_SHA = (
    "d492f7eff49ddf041ecf6078daa02713e6f99b10590d213efbae937242e023c7"
)

EXPECTED_GENERATOR_SHA = (
    "32be8092c85458bed31a547f2ba6c57c3e1c2cf257d7cb0f76bd28eebc7b26c2"
)

# ----------------------------------------------------------------------------
# Helpers
# ----------------------------------------------------------------------------

def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b"",
        ):
            h.update(chunk)

    return h.hexdigest()


# ----------------------------------------------------------------------------
# Hard preflight
# ----------------------------------------------------------------------------

assert SCRIPT.exists(), SCRIPT
assert PROJECT.exists(), PROJECT
assert CACHE.exists(), CACHE
assert ASSIGNMENT.exists(), ASSIGNMENT

assert sha256_file(SCRIPT) == EXPECTED_SCRIPT_SHA

REQUIRED_CACHE = [
    "clean_224.npy",
    "hair_224.npy",
    "mask_area.npy",
    "cache_index.csv",
]

for name in REQUIRED_CACHE:

    p = CACHE / name

    assert p.exists(), p

# Verify cache provenance lock
CACHE_LOCK = (
    CACHE
    / "T2_1_PRODUCTION_CACHE_PROVENANCE_LOCK.json"
)

assert CACHE_LOCK.exists(), CACHE_LOCK

cache_lock = json.loads(
    CACHE_LOCK.read_text(
        encoding="utf-8"
    )
)

assert (
    cache_lock["generator_sha256"]
    == EXPECTED_GENERATOR_SHA
)

assert cache_lock["n_images"] == 23227

assert cache_lock["clean_shape"] == [
    23227,
    224,
    224,
    3,
]

assert cache_lock["hair_shape"] == [
    23227,
    224,
    224,
    3,
]

# ----------------------------------------------------------------------------
# Clean previous DEV output only.
# This does NOT touch frozen inputs.
# ----------------------------------------------------------------------------

if DEV_OUT.exists():

    import shutil

    shutil.rmtree(
        DEV_OUT
    )

DEV_OUT.mkdir(
    parents=True,
    exist_ok=True,
)

# ----------------------------------------------------------------------------
# Child process environment
#
# One visible GPU per training process.
# ----------------------------------------------------------------------------

env = os.environ.copy()

env["CUDA_VISIBLE_DEVICES"] = "0"

env["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

# ----------------------------------------------------------------------------
# Command
#
# --mode dev forces exactly one epoch in the frozen implementation.
# --gap mean explicitly locks the T2.1 pooling choice.
# ----------------------------------------------------------------------------

cmd = [
    sys.executable,
    str(SCRIPT),

    "--project",
    str(PROJECT),

    "--cache_dir",
    str(CACHE),

    "--rho",
    "0.0",

    "--rep",
    "1",

    "--mode",
    "dev",

    "--gap",
    "mean",

    "--out_dir",
    str(DEV_OUT),

    "--num_workers",
    "2",
]

print("=" * 78)
print("T2.1 ONE-EPOCH DEV CONVNEXT SMOKE TEST")
print("=" * 78)

print()
print("SCRIPT:")
print(SCRIPT)

print()
print("SCRIPT SHA256:")
print(sha256_file(SCRIPT))

print()
print("PROJECT:")
print(PROJECT)

print()
print("CACHE:")
print(CACHE)

print()
print("ASSIGNMENT:")
print(ASSIGNMENT)

print()
print("CUDA_VISIBLE_DEVICES:")
print(env["CUDA_VISIBLE_DEVICES"])

print()
print("CUBLAS_WORKSPACE_CONFIG:")
print(env["CUBLAS_WORKSPACE_CONFIG"])

print()
print("COMMAND:")
print(
    " ".join(
        repr(x)
        for x in cmd
    )
)

print()
print("=" * 78)
print("STARTING DEV RUN")
print("=" * 78)

t0 = time.time()

result = subprocess.run(
    cmd,
    env=env,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

elapsed = time.time() - t0

print()
print("=" * 78)
print("SUBPROCESS OUTPUT")
print("=" * 78)
print()

print(result.stdout)

print()
print("=" * 78)
print("RETURN CODE")
print("=" * 78)

print(result.returncode)

assert result.returncode == 0, (
    "\n"
    "T2.1 DEV SMOKE TEST FAILED.\n"
    f"Return code = {result.returncode}\n"
    f"Elapsed = {elapsed:.2f} s\n"
    "\n"
    "Review the complete subprocess output above."
)

# ----------------------------------------------------------------------------
# Artifact inventory
# ----------------------------------------------------------------------------

print()
print("=" * 78)
print("DEV ARTIFACT INVENTORY")
print("=" * 78)

all_files = sorted(
    p
    for p in DEV_OUT.rglob("*")
    if p.is_file()
)

for p in all_files:

    print(
        f"{p.relative_to(DEV_OUT)}"
        f"  {p.stat().st_size:,} bytes"
    )

assert all_files, (
    "DEV run returned success but produced no files."
)

# ----------------------------------------------------------------------------
# Training record
# ----------------------------------------------------------------------------

records = sorted(
    DEV_OUT.rglob(
        "training_record.json"
    )
)

print()
print("training_record.json files:", len(records))

assert len(records) == 1, (
    "Expected exactly one training_record.json."
)

record_path = records[0]

record = json.loads(
    record_path.read_text(
        encoding="utf-8"
    )
)

print()
print("=" * 78)
print("TRAINING RECORD")
print("=" * 78)

print(
    json.dumps(
        record,
        indent=2,
        sort_keys=True,
    )
)

# ----------------------------------------------------------------------------
# Actual executable provenance
# ----------------------------------------------------------------------------

record_script_sha = record.get(
    "script_sha256"
)

print()
print("Recorded script SHA:")
print(record_script_sha)

assert (
    record_script_sha
    == EXPECTED_SCRIPT_SHA
), (
    "TRAINING RECORD SCRIPT HASH MISMATCH"
)

# ----------------------------------------------------------------------------
# Confirm DEV really ran one epoch
# ----------------------------------------------------------------------------

# Locate likely epoch log information in the record.
epoch_candidates = []

for key in [
    "epochs",
    "n_epochs",
    "epoch",
    "history",
    "log",
    "metrics",
]:

    if key in record:

        epoch_candidates.append(
            (
                key,
                record[key]
            )
        )

print()
print("=" * 78)
print("EPOCH INFORMATION FOUND IN RECORD")
print("=" * 78)

for key, value in epoch_candidates:

    print()
    print(key, ":")
    print(
        json.dumps(
            value,
            indent=2,
        )
        if isinstance(
            value,
            (dict, list)
        )
        else value
    )

# ----------------------------------------------------------------------------
# Checkpoint inventory
# ----------------------------------------------------------------------------

final_ckpts = sorted(
    DEV_OUT.rglob(
        "final_epoch15.pt"
    )
)

best_ckpts = sorted(
    DEV_OUT.rglob(
        "best_eval_ba.pt"
    )
)

print()
print("=" * 78)
print("CHECKPOINTS")
print("=" * 78)

print()
print("final_epoch15.pt:")
for p in final_ckpts:
    print(p)

print()
print("best_eval_ba.pt:")
for p in best_ckpts:
    print(p)

# ----------------------------------------------------------------------------
# Save final smoke-test audit
# ----------------------------------------------------------------------------

audit = {

    "status":
        "PASS",

    "purpose":
        "one_epoch_dev_smoke_test",

    "production_inclusion":
        False,

    "script":
        str(SCRIPT),

    "script_sha256":
        EXPECTED_SCRIPT_SHA,

    "project":
        str(PROJECT),

    "cache":
        str(CACHE),

    "cache_generator_sha256":
        EXPECTED_GENERATOR_SHA,

    "assignment":
        str(ASSIGNMENT),

    "rho":
        0.0,

    "replicate":
        1,

    "mode":
        "dev",

    "gap":
        "mean",

    "num_workers":
        2,

    "cuda_visible_devices":
        env["CUDA_VISIBLE_DEVICES"],

    "cublas_workspace_config":
        env["CUBLAS_WORKSPACE_CONFIG"],

    "return_code":
        result.returncode,

    "elapsed_seconds":
        elapsed,

    "training_record":
        str(record_path),

    "recorded_script_sha256":
        record_script_sha,

    "final_checkpoint_count":
        len(final_ckpts),

    "best_checkpoint_count":
        len(best_ckpts),
}

AUDIT_PATH = (
    OUT_ROOT
    / "T2_1_DEV_SMOKE_TEST_AUDIT.json"
)

AUDIT_PATH.write_text(
    json.dumps(
        audit,
        indent=2,
    ),
    encoding="utf-8",
)

print()
print("=" * 78)
print("T2.1 DEV SMOKE TEST: PASS")
print("=" * 78)

print()
print("Audit:")
print(AUDIT_PATH)

print()
print("The cache-backed ConvNeXt execution path completed successfully.")

print()
print("=" * 78)
print("NEXT GATE")
print("=" * 78)
print("STOP HERE.")
print("Review the DEV output before production training.")

In [ ]:
# ============================================================================
# CELL 6ZK-E — DIAGNOSE IMMEDIATE DEV PROCESS EXIT
# ============================================================================
#
# NO TRAINING MODIFICATION
# NO SCRIPT MODIFICATION
#
# The previous DEV process exited in ~0.09 s.
# That strongly suggests CLI/configuration failure before model construction.
#
# This cell:
#   1. verifies the frozen hash,
#   2. prints the script CLI,
#   3. runs the exact DEV command,
#   4. NEVER hides subprocess output,
#   5. does NOT assert return_code == 0.
# ============================================================================

from pathlib import Path
import hashlib
import os
import subprocess
import sys

SCRIPT = Path(
    "/kaggle/working/"
    "T2_1_ConvNeXt_training_code_freeze/"
    "train_convnext_tiny_t2_1.py"
)

PROJECT = Path(
    "/kaggle/input/notebooks/"
    "tejasseshadriiiii/"
    "isic-kaggle-analysis/"
    "isic_final_project"
)

CACHE = Path(
    "/kaggle/working/"
    "T2_1_PRODUCTION_CACHE"
)

DEV_OUT = Path(
    "/kaggle/working/"
    "T2_1_ConvNeXt_DEV_SMOKE_TEST"
    / "rho_0.0_rep_1"
)

EXPECTED_SHA = (
    "d492f7eff49ddf041ecf6078daa02713e6f99b10590d213efbae937242e023c7"
)

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b"",
        ):
            h.update(chunk)
    return h.hexdigest()

# ----------------------------------------------------------------------------
# Hard preflight
# ----------------------------------------------------------------------------

assert SCRIPT.exists(), SCRIPT
assert PROJECT.exists(), PROJECT
assert CACHE.exists(), CACHE

actual_sha = sha256_file(SCRIPT)

print("=" * 78)
print("T2.1 DEV IMMEDIATE-EXIT DIAGNOSTIC")
print("=" * 78)

print()
print("SCRIPT:")
print(SCRIPT)

print()
print("SCRIPT SHA:")
print(actual_sha)

assert actual_sha == EXPECTED_SHA

# ----------------------------------------------------------------------------
# First: ask the frozen script what arguments it accepts.
# ----------------------------------------------------------------------------

env = os.environ.copy()

env["CUDA_VISIBLE_DEVICES"] = "0"
env["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

print()
print("=" * 78)
print("FROZEN SCRIPT --HELP")
print("=" * 78)

help_result = subprocess.run(
    [
        sys.executable,
        str(SCRIPT),
        "--help",
    ],
    env=env,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

print()
print(help_result.stdout)

print()
print("HELP RETURN CODE:")
print(help_result.returncode)

# ----------------------------------------------------------------------------
# Exact DEV command.
# ----------------------------------------------------------------------------

cmd = [
    sys.executable,
    str(SCRIPT),

    "--project",
    str(PROJECT),

    "--cache_dir",
    str(CACHE),

    "--rho",
    "0.0",

    "--rep",
    "1",

    "--mode",
    "dev",

    "--gap",
    "mean",

    "--out_dir",
    str(DEV_OUT),

    "--num_workers",
    "2",
]

print()
print("=" * 78)
print("EXACT DEV COMMAND")
print("=" * 78)

print()
print(" ".join(
    repr(x)
    for x in cmd
))

print()
print("=" * 78)
print("RUNNING EXACT DEV COMMAND")
print("=" * 78)

result = subprocess.run(
    cmd,
    env=env,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

print()
print("RETURN CODE:")
print(result.returncode)

print()
print("COMPLETE CHILD OUTPUT:")
print("-" * 78)
print(result.stdout)
print("-" * 78)

print()
print("=" * 78)
print("DEV OUTPUT DIRECTORY")
print("=" * 78)

if DEV_OUT.exists():

    files = sorted(
        p
        for p in DEV_OUT.rglob("*")
        if p.is_file()
    )

    print(
        f"{len(files)} files produced."
    )

    for p in files:

        print(
            p.relative_to(DEV_OUT)
        )

else:

    print(
        "DEV output directory was not created."
    )

print()
print("=" * 78)
print("DIAGNOSTIC COMPLETE")
print("=" * 78)

if result.returncode == 0:

    print(
        "The command succeeded unexpectedly; "
        "inspect the output above."
    )

else:

    print(
        "The child exited non-zero."
    )

    print(
        "Most importantly, the COMPLETE CHILD OUTPUT above "
        "contains the actual failure."
    )

print()
print("DO NOT MODIFY THE FROZEN SCRIPT YET.")

In [ ]:
# ============================================================================
# CELL 6ZK-E — DIAGNOSE IMMEDIATE DEV PROCESS EXIT
# ============================================================================

from pathlib import Path
import hashlib
import os
import subprocess
import sys

SCRIPT = Path(
    "/kaggle/working/"
    "T2_1_ConvNeXt_training_code_freeze/"
    "train_convnext_tiny_t2_1.py"
)

PROJECT = Path(
    "/kaggle/input/notebooks/"
    "tejasseshadriiiii/"
    "isic-kaggle-analysis/"
    "isic_final_project"
)

CACHE = Path(
    "/kaggle/working/"
    "T2_1_PRODUCTION_CACHE"
)

DEV_OUT = (
    Path(
        "/kaggle/working/"
        "T2_1_ConvNeXt_DEV_SMOKE_TEST"
    )
    / "rho_0.0_rep_1"
)

EXPECTED_SHA = (
    "d492f7eff49ddf041ecf6078daa02713e6f99b10590d213efbae937242e023c7"
)

def sha256_file(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b"",
        ):
            h.update(chunk)

    return h.hexdigest()


# ============================================================================
# PREFLIGHT
# ============================================================================

assert SCRIPT.exists(), SCRIPT
assert PROJECT.exists(), PROJECT
assert CACHE.exists(), CACHE

actual_sha = sha256_file(SCRIPT)

print("=" * 78)
print("T2.1 DEV IMMEDIATE-EXIT DIAGNOSTIC")
print("=" * 78)

print()
print("SCRIPT:")
print(SCRIPT)

print()
print("SCRIPT SHA:")
print(actual_sha)

assert actual_sha == EXPECTED_SHA, (
    "Frozen script hash mismatch."
)

# ============================================================================
# CHILD PROCESS ENVIRONMENT
# ============================================================================

env = os.environ.copy()

env["CUDA_VISIBLE_DEVICES"] = "0"
env["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"


# ============================================================================
# FIRST: CHECK CLI
# ============================================================================

print()
print("=" * 78)
print("FROZEN SCRIPT --HELP")
print("=" * 78)

help_result = subprocess.run(
    [
        sys.executable,
        str(SCRIPT),
        "--help",
    ],
    env=env,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

print()
print(help_result.stdout)

print()
print("HELP RETURN CODE:")
print(help_result.returncode)


# ============================================================================
# EXACT DEV COMMAND
# ============================================================================

cmd = [
    sys.executable,
    str(SCRIPT),

    "--project",
    str(PROJECT),

    "--cache_dir",
    str(CACHE),

    "--rho",
    "0.0",

    "--rep",
    "1",

    "--mode",
    "dev",

    "--gap",
    "mean",

    "--out_dir",
    str(DEV_OUT),

    "--num_workers",
    "2",
]

print()
print("=" * 78)
print("EXACT DEV COMMAND")
print("=" * 78)

print()

print(
    " ".join(
        repr(x)
        for x in cmd
    )
)


# ============================================================================
# RUN — DO NOT ASSERT ON RETURN CODE
# ============================================================================

print()
print("=" * 78)
print("RUNNING EXACT DEV COMMAND")
print("=" * 78)

result = subprocess.run(
    cmd,
    env=env,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

print()
print("=" * 78)
print("RETURN CODE")
print("=" * 78)

print(result.returncode)

print()
print("=" * 78)
print("COMPLETE CHILD OUTPUT")
print("=" * 78)

print()

if result.stdout:
    print(result.stdout)
else:
    print("[CHILD PRODUCED NO OUTPUT]")


# ============================================================================
# OUTPUT DIRECTORY
# ============================================================================

print()
print("=" * 78)
print("DEV OUTPUT DIRECTORY")
print("=" * 78)

if DEV_OUT.exists():

    files = sorted(
        p
        for p in DEV_OUT.rglob("*")
        if p.is_file()
    )

    print(
        f"{len(files)} files produced."
    )

    for p in files:

        print(
            f"  {p.relative_to(DEV_OUT)}"
        )

else:

    print(
        "DEV output directory was not created."
    )


# ============================================================================
# FINAL DIAGNOSTIC STATUS
# ============================================================================

print()
print("=" * 78)
print("DIAGNOSTIC COMPLETE")
print("=" * 78)

if result.returncode == 0:

    print(
        "DEV command returned SUCCESS."
    )

else:

    print(
        "DEV command returned NON-ZERO."
    )

    print(
        "The complete child error is printed above."
    )

print()
print("DO NOT MODIFY THE FROZEN SCRIPT YET.")

In [ ]:
# ============================================================================
# CELL 6ZK-F — INSPECT T2.1 SYNTAX ERROR
# ============================================================================
# No modification.
# We inspect the exact malformed region around line 599.
# ============================================================================

from pathlib import Path

SCRIPT = Path(
    "/kaggle/working/"
    "T2_1_ConvNeXt_training_code_freeze/"
    "train_convnext_tiny_t2_1.py"
)

assert SCRIPT.exists()

lines = SCRIPT.read_text(
    encoding="utf-8"
).splitlines()

print("=" * 78)
print("T2.1 SYNTAX ERROR INSPECTION")
print("=" * 78)

print()
print("SCRIPT:")
print(SCRIPT)

print()
print("LINES 570–620:")
print("-" * 78)

for i in range(
    570,
    min(
        620,
        len(lines),
    ) + 1,
):

    print(
        f"{i:5d}: {lines[i-1]}"
    )

print()
print("=" * 78)
print("SYNTAX COMPILATION CHECK")
print("=" * 78)

source = SCRIPT.read_text(
    encoding="utf-8"
)

try:

    compile(
        source,
        str(SCRIPT),
        "exec",
    )

    print("COMPILE: PASS")

except SyntaxError as e:

    print("COMPILE: FAIL")

    print()
    print("filename :", e.filename)
    print("line     :", e.lineno)
    print("offset   :", e.offset)
    print("message  :", e.msg)
    print("text     :", repr(e.text))

    raise

print()
print("DO NOT MODIFY THE SCRIPT IN THIS CELL.")

In [ ]:
# ============================================================================
# CELL 6ZK-G — SURGICAL SYNTAX FIX + REFREEZE
# ============================================================================
#
# ONLY CHANGE:
#   Fix the invalid line break before .all() at the assignment/cache
#   membership assertion.
#
# Intended semantics remain exactly:
#
#   assert assignment["image"].isin(row_of).all()
#
# No training.
# ============================================================================

from pathlib import Path
import hashlib
import json
from datetime import datetime, timezone

SCRIPT = Path(
    "/kaggle/working/"
    "T2_1_ConvNeXt_training_code_freeze/"
    "train_convnext_tiny_t2_1.py"
)

OLD_SHA = (
    "d492f7eff49ddf041ecf6078daa02713e6f99b10590d213efbae937242e023c7"
)

assert SCRIPT.exists()

source = SCRIPT.read_text(
    encoding="utf-8"
)

# --------------------------------------------------------------------------
# Exact malformed block.
# --------------------------------------------------------------------------

OLD_BLOCK = '''    assert assignment[
        "image"
    ].isin(row_of)
    .all()
'''

NEW_BLOCK = '''    assert (
        assignment[
            "image"
        ].isin(row_of)
    ).all()
'''

assert OLD_BLOCK in source, (
    "Expected malformed block was not found. "
    "Refusing to modify the script."
)

# Ensure there is exactly one occurrence.
assert source.count(OLD_BLOCK) == 1, (
    "Malformed block occurs an unexpected number of times."
)

# --------------------------------------------------------------------------
# Surgical replacement.
# --------------------------------------------------------------------------

corrected = source.replace(
    OLD_BLOCK,
    NEW_BLOCK,
    1,
)

assert corrected != source

SCRIPT.write_text(
    corrected,
    encoding="utf-8",
)

# --------------------------------------------------------------------------
# Verify old hash is no longer present.
# --------------------------------------------------------------------------

def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b"",
        ):
            h.update(chunk)

    return h.hexdigest()

NEW_SHA = sha256_file(SCRIPT)

assert NEW_SHA != OLD_SHA

print("=" * 78)
print("T2.1 SURGICAL SYNTAX FIX")
print("=" * 78)

print()
print("OLD SHA256:")
print(OLD_SHA)

print()
print("NEW SHA256:")
print(NEW_SHA)

# --------------------------------------------------------------------------
# Mandatory full compilation gate.
# --------------------------------------------------------------------------

print()
print("=" * 78)
print("FULL SCRIPT COMPILATION")
print("=" * 78)

compile(
    corrected,
    str(SCRIPT),
    "exec",
)

print("COMPILE: PASS")

# --------------------------------------------------------------------------
# Verify corrected region.
# --------------------------------------------------------------------------

lines = corrected.splitlines()

print()
print("=" * 78)
print("CORRECTED REGION")
print("=" * 78)

for i in range(
    588,
    603,
):

    print(
        f"{i:5d}: {lines[i-1]}"
    )

# --------------------------------------------------------------------------
# Static semantic checks.
# --------------------------------------------------------------------------

checks = {}

checks["ConvNeXt-Tiny"] = (
    "models.convnext_tiny" in corrected
)

checks["locked weights"] = (
    "models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1"
    in corrected
)

checks["two-class classifier"] = (
    "model.classifier[2] = nn.Linear"
    in corrected
)

checks["mean pool"] = (
    "model.avgpool = MeanPool2d()"
    in corrected
)

checks["Step1F assignment"] = (
    "step1F_lesion_level_rho_assignments"
    in corrected
)

checks["cache_index"] = (
    "cache_index.csv" in corrected
)

checks["clean cache"] = (
    "clean_224.npy" in corrected
)

checks["hair cache"] = (
    "hair_224.npy" in corrected
)

checks["hair_present"] = (
    '"hair_present"' in corrected
)

checks["hair routing"] = (
    "self._hair[r]" in corrected
    and
    "self._clean[r]" in corrected
    and
    "if self.hair[i]" in corrected
)

checks["malignant"] = (
    '"malignant"' in corrected
)

checks["validation split"] = (
    'idx[\n'
    '            idx["split"]\n'
    '            == "val"\n'
    '        ]'
    in corrected
)

checks["validation clean"] = (
    "np.zeros(" in corrected
    and
    "len(ev_rows)" in corrected
    and
    "aug_seed=None" in corrected
)

checks["epoch control"] = (
    "def set_epoch(self, e):" in corrected
    and
    "train_ds.set_epoch(" in corrected
)

checks["AdamW"] = (
    "torch.optim.AdamW" in corrected
)

checks["CosineAnnealingLR"] = (
    "CosineAnnealingLR" in corrected
)

checks["CrossEntropyLoss"] = (
    "F.cross_entropy" in corrected
)

checks["final checkpoint"] = (
    '"final_epoch15.pt"' in corrected
)

checks["best BA checkpoint"] = (
    '"best_eval_ba.pt"' in corrected
)

checks["deterministic algorithms"] = (
    "torch.use_deterministic_algorithms(" in corrected
    and
    "True" in corrected
)

checks["cudnn deterministic"] = (
    "torch.backends.cudnn.deterministic = True"
    in corrected
)

checks["cudnn benchmark false"] = (
    "torch.backends.cudnn.benchmark = False"
    in corrected
)

checks["TF32 disabled"] = (
    "torch.backends.cuda.matmul.allow_tf32 = False"
    in corrected
    and
    "torch.backends.cudnn.allow_tf32 = False"
    in corrected
)

checks["CUBLAS config"] = (
    "CUBLAS_WORKSPACE_CONFIG" in corrected
    and
    ":4096:8" in corrected
)

checks["XAI absent"] = not any(
    token in corrected
    for token in [
        "IntegratedGradients",
        "generate_hair_mask",
        "ig_enrichment",
        "--ig-check",
    ]
)

print()
print("=" * 78)
print("STATIC SEMANTIC CHECKS")
print("=" * 78)

for name, passed in checks.items():

    print(
        f"{name:35s}: "
        f"{'PASS' if passed else 'FAIL'}"
    )

failed = [
    name
    for name, passed in checks.items()
    if not passed
]

assert not failed, (
    "Semantic checks failed:\n"
    + "\n".join(failed)
)

# --------------------------------------------------------------------------
# New final lock.
# --------------------------------------------------------------------------

LOCK_DIR = (
    Path("/kaggle/working/")
    / "T2_1_ConvNeXt_training_code_freeze"
)

LOCK_PATH = (
    LOCK_DIR
    / "T2_1_TRAINING_CODE_FINAL_HASH.json"
)

lock = {
    "status":
        "FROZEN",

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "script":
        str(SCRIPT),

    "sha256":
        NEW_SHA,

    "previous_invalid_sha256":
        OLD_SHA,

    "correction":
        "Syntax-only correction of the assignment cache-membership "
        "assertion. No intended training/data semantics changed.",

    "compile_pass":
        True,

    "semantic_checks_pass":
        True,

    "architecture":
        "ConvNeXt-Tiny",

    "weights":
        "ConvNeXt_Tiny_Weights.IMAGENET1K_V1",

    "primary_checkpoint":
        "final_epoch15.pt",

    "sensitivity_checkpoint":
        "best_eval_ba.pt",

    "xai_ig":
        False,

    "resnet_source_modified":
        False,
}

LOCK_PATH.write_text(
    json.dumps(
        lock,
        indent=2,
    ),
    encoding="utf-8",
)

print()
print("=" * 78)
print("T2.1 CORRECTED REFREEZE: PASS")
print("=" * 78)

print()
print("FINAL SCRIPT SHA256:")
print(NEW_SHA)

print()
print("HASH LOCK:")
print(LOCK_PATH)

print()
print("The previous d492f7... hash is superseded.")
print("The new hash is now the only executable candidate.")

print()
print("=" * 78)
print("NEXT GATE")
print("=" * 78)
print("6ZK-H — independent subprocess --help + compile + DEV smoke test")

In [ ]:
# ============================================================================
# CELL 6ZK-H — INDEPENDENT EXECUTION GATE + DEV SMOKE TEST
# ============================================================================
#
# This is the first actual execution of the corrected frozen script.
#
# Gates:
#   1. --help must compile/execute successfully.
#   2. DEV subprocess must start successfully.
#   3. DEV must complete exactly one epoch.
#   4. Actual training_record must contain the frozen script SHA.
#
# DEV output is NOT part of T2.1 production statistics.
# ============================================================================

from pathlib import Path
import hashlib
import json
import os
import shutil
import subprocess
import sys
import time

SCRIPT = Path(
    "/kaggle/working/"
    "T2_1_ConvNeXt_training_code_freeze/"
    "train_convnext_tiny_t2_1.py"
)

PROJECT = Path(
    "/kaggle/input/notebooks/"
    "tejasseshadriiiii/"
    "isic-kaggle-analysis/"
    "isic_final_project"
)

CACHE = Path(
    "/kaggle/working/"
    "T2_1_PRODUCTION_CACHE"
)

DEV_ROOT = Path(
    "/kaggle/working/"
    "T2_1_ConvNeXt_DEV_SMOKE_TEST"
)

DEV_OUT = (
    DEV_ROOT
    / "rho_0.0_rep_1"
)

EXPECTED_SHA = (
    "5d822b1336eabcfcf83edf3b12d012817687ba8f6335cf670b6ebed4fcbf4da6"
)

EXPECTED_GENERATOR_SHA = (
    "32be8092c85458bed31a547f2ba6c57c3e1c2cf257d7cb0f76bd28eebc7b26c2"
)


# ============================================================================
# HASH
# ============================================================================

def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b"",
        ):

            h.update(chunk)

    return h.hexdigest()


assert SCRIPT.exists()
assert PROJECT.exists()
assert CACHE.exists()

actual_sha = sha256_file(SCRIPT)

print("=" * 78)
print("T2.1 INDEPENDENT EXECUTION GATE")
print("=" * 78)

print()
print("SCRIPT:")
print(SCRIPT)

print()
print("SCRIPT SHA256:")
print(actual_sha)

assert actual_sha == EXPECTED_SHA


# ============================================================================
# CACHE PROVENANCE
# ============================================================================

CACHE_LOCK = (
    CACHE
    / "T2_1_PRODUCTION_CACHE_PROVENANCE_LOCK.json"
)

assert CACHE_LOCK.exists()

cache_lock = json.loads(
    CACHE_LOCK.read_text(
        encoding="utf-8",
    )
)

assert (
    cache_lock["generator_sha256"]
    == EXPECTED_GENERATOR_SHA
)

assert cache_lock["n_images"] == 23227

print()
print("CACHE:")
print(CACHE)

print()
print("CACHE GENERATOR SHA:")
print(cache_lock["generator_sha256"])

print()
print("CACHE IMAGES:")
print(cache_lock["n_images"])


# ============================================================================
# ENVIRONMENT
# ============================================================================

env = os.environ.copy()

# One process → one visible GPU.
env["CUDA_VISIBLE_DEVICES"] = "0"

# Must be present before torch import in child.
env["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"


# ============================================================================
# GATE 1 — --help
# ============================================================================

print()
print("=" * 78)
print("GATE 1 — FROZEN SCRIPT --HELP")
print("=" * 78)

help_cmd = [
    sys.executable,
    str(SCRIPT),
    "--help",
]

help_result = subprocess.run(
    help_cmd,
    env=env,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

print()
print(help_result.stdout)

print()
print("HELP RETURN CODE:")
print(help_result.returncode)

assert help_result.returncode == 0, (
    "FROZEN SCRIPT --help FAILED."
)

print()
print("GATE 1: PASS")


# ============================================================================
# CLEAN DEV OUTPUT
# ============================================================================

if DEV_ROOT.exists():

    shutil.rmtree(
        DEV_ROOT
    )

DEV_OUT.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================================
# GATE 2 — DEV COMMAND
# ============================================================================

cmd = [
    sys.executable,
    str(SCRIPT),

    "--project",
    str(PROJECT),

    "--cache_dir",
    str(CACHE),

    "--rho",
    "0.0",

    "--rep",
    "1",

    "--mode",
    "dev",

    "--gap",
    "mean",

    "--out_dir",
    str(DEV_OUT),

    "--num_workers",
    "2",
]

print()
print("=" * 78)
print("GATE 2 — ONE-EPOCH DEV RUN")
print("=" * 78)

print()
print("COMMAND:")
print(
    " ".join(
        repr(x)
        for x in cmd
    )
)

print()
print("CUDA_VISIBLE_DEVICES:")
print(env["CUDA_VISIBLE_DEVICES"])

print()
print("CUBLAS_WORKSPACE_CONFIG:")
print(env["CUBLAS_WORKSPACE_CONFIG"])

t0 = time.time()

result = subprocess.run(
    cmd,
    env=env,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

elapsed = time.time() - t0

print()
print("=" * 78)
print("DEV SUBPROCESS OUTPUT")
print("=" * 78)
print()

print(result.stdout)

print()
print("RETURN CODE:")
print(result.returncode)

print()
print("ELAPSED:")
print(f"{elapsed:.2f} seconds")

assert result.returncode == 0, (
    "DEV RUN FAILED.\n"
    "See complete subprocess output above."
)

print()
print("GATE 2: PASS")


# ============================================================================
# GATE 3 — ARTIFACTS
# ============================================================================

print()
print("=" * 78)
print("GATE 3 — DEV ARTIFACT AUDIT")
print("=" * 78)

files = sorted(
    p
    for p in DEV_OUT.rglob("*")
    if p.is_file()
)

assert files, (
    "DEV succeeded but produced no files."
)

for p in files:

    print(
        f"{p.relative_to(DEV_OUT)}"
        f"    {p.stat().st_size:,} bytes"
    )


# ============================================================================
# TRAINING RECORD
# ============================================================================

records = sorted(
    DEV_OUT.rglob(
        "training_record.json"
    )
)

assert len(records) == 1, (
    f"Expected 1 training_record.json, found {len(records)}"
)

record_path = records[0]

record = json.loads(
    record_path.read_text(
        encoding="utf-8",
    )
)

print()
print("=" * 78)
print("TRAINING RECORD")
print("=" * 78)

print(
    json.dumps(
        record,
        indent=2,
        sort_keys=True,
    )
)


# ============================================================================
# GATE 4 — EXECUTABLE PROVENANCE
# ============================================================================

record_script_sha = record.get(
    "script_sha256"
)

print()
print("=" * 78)
print("GATE 4 — EXECUTABLE PROVENANCE")
print("=" * 78)

print()
print("Expected SHA:")
print(EXPECTED_SHA)

print()
print("Recorded SHA:")
print(record_script_sha)

assert (
    record_script_sha
    == EXPECTED_SHA
), (
    "Training record does not identify the frozen executable."
)

print()
print("GATE 4: PASS")


# ============================================================================
# FINAL DEV AUDIT
# ============================================================================

audit = {

    "status":
        "PASS",

    "production_inclusion":
        False,

    "script":
        str(SCRIPT),

    "script_sha256":
        EXPECTED_SHA,

    "cache":
        str(CACHE),

    "cache_generator_sha256":
        EXPECTED_GENERATOR_SHA,

    "mode":
        "dev",

    "rho":
        0.0,

    "replicate":
        1,

    "gap":
        "mean",

    "num_workers":
        2,

    "cuda_visible_devices":
        "0",

    "cublas_workspace_config":
        ":4096:8",

    "help_return_code":
        help_result.returncode,

    "dev_return_code":
        result.returncode,

    "elapsed_seconds":
        elapsed,

    "training_record":
        str(record_path),

    "recorded_script_sha256":
        record_script_sha,

    "n_artifacts":
        len(files),
}

AUDIT_PATH = (
    DEV_ROOT
    / "T2_1_DEV_EXECUTION_GATE_AUDIT.json"
)

AUDIT_PATH.write_text(
    json.dumps(
        audit,
        indent=2,
    ),
    encoding="utf-8",
)

print()
print("=" * 78)
print("T2.1 DEV EXECUTION GATE: PASS")
print("=" * 78)

print()
print("Audit:")
print(AUDIT_PATH)

print()
print("The corrected frozen executable successfully completed")
print("the independent one-epoch cache-backed DEV run.")

print()
print("=" * 78)
print("STOP")
print("=" * 78)

print(
    "Do NOT start production training from this cell."
)
print(
    "Next step is review of the DEV training record/output."
)

In [ ]:
# ============================================================================
# CELL 6ZK-I — CORRECTED INDEPENDENT DEV EXECUTION
# ============================================================================
#
# Previous failure was launcher-only:
#
#   WRONG: --cache_dir
#   RIGHT: --cache-dir
#
#   WRONG: --out_dir
#   RIGHT: --out-dir
#
# The frozen script is NOT modified.
# Script SHA must remain 5d822b...
# ============================================================================

from pathlib import Path
import hashlib
import json
import os
import shutil
import subprocess
import sys
import time


SCRIPT = Path(
    "/kaggle/working/"
    "T2_1_ConvNeXt_training_code_freeze/"
    "train_convnext_tiny_t2_1.py"
)

PROJECT = Path(
    "/kaggle/input/notebooks/"
    "tejasseshadriiiii/"
    "isic-kaggle-analysis/"
    "isic_final_project"
)

CACHE = Path(
    "/kaggle/working/"
    "T2_1_PRODUCTION_CACHE"
)

DEV_ROOT = Path(
    "/kaggle/working/"
    "T2_1_ConvNeXt_DEV_SMOKE_TEST"
)

DEV_OUT = (
    DEV_ROOT
    / "rho_0.0_rep_1"
)

EXPECTED_SHA = (
    "5d822b1336eabcfcf83edf3b12d012817687ba8f6335cf670b6ebed4fcbf4da6"
)


def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b"",
        ):
            h.update(chunk)

    return h.hexdigest()


# ============================================================================
# PRE-FLIGHT
# ============================================================================

assert SCRIPT.exists()
assert PROJECT.exists()
assert CACHE.exists()

actual_sha = sha256_file(SCRIPT)

print("=" * 78)
print("T2.1 CORRECTED DEV EXECUTION")
print("=" * 78)

print()
print("SCRIPT SHA256:")
print(actual_sha)

assert actual_sha == EXPECTED_SHA, (
    "Frozen script SHA changed unexpectedly."
)

print("SCRIPT HASH: PASS")


# ============================================================================
# ENVIRONMENT
# ============================================================================

env = os.environ.copy()

# Exactly one GPU visible to this subprocess.
env["CUDA_VISIBLE_DEVICES"] = "0"

# Must exist before torch import.
env["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"


# ============================================================================
# CLEAN PREVIOUS DEV OUTPUT
# ============================================================================

if DEV_ROOT.exists():

    shutil.rmtree(
        DEV_ROOT
    )

DEV_OUT.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================================
# CORRECTED COMMAND
# ============================================================================

cmd = [
    sys.executable,
    str(SCRIPT),

    "--project",
    str(PROJECT),

    "--cache-dir",
    str(CACHE),

    "--rho",
    "0.0",

    "--rep",
    "1",

    "--mode",
    "dev",

    "--gap",
    "mean",

    "--out-dir",
    str(DEV_OUT),

    "--num-workers",
    "2",
]


print()
print("=" * 78)
print("DEV COMMAND")
print("=" * 78)

print()

print(
    " ".join(
        repr(x)
        for x in cmd
    )
)

print()
print("CUDA_VISIBLE_DEVICES =", env["CUDA_VISIBLE_DEVICES"])
print(
    "CUBLAS_WORKSPACE_CONFIG =",
    env["CUBLAS_WORKSPACE_CONFIG"],
)


# ============================================================================
# EXECUTE
# ============================================================================

t0 = time.time()

result = subprocess.run(
    cmd,
    env=env,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

elapsed = time.time() - t0


print()
print("=" * 78)
print("DEV SUBPROCESS OUTPUT")
print("=" * 78)

print()
print(result.stdout)

print()
print("RETURN CODE:")
print(result.returncode)

print()
print("ELAPSED:")
print(f"{elapsed:.2f} seconds")


assert result.returncode == 0, (
    "DEV execution failed. "
    "The complete subprocess output is above."
)

print()
print("DEV PROCESS: PASS")


# ============================================================================
# ARTIFACT DISCOVERY
# ============================================================================

print()
print("=" * 78)
print("DEV ARTIFACT AUDIT")
print("=" * 78)

files = sorted(
    p
    for p in DEV_OUT.rglob("*")
    if p.is_file()
)

assert files, (
    "DEV process returned 0 but produced no files."
)

for p in files:

    print(
        f"{p.relative_to(DEV_OUT)}"
        f"    {p.stat().st_size:,} bytes"
    )


# ============================================================================
# TRAINING RECORD
# ============================================================================

records = sorted(
    DEV_OUT.rglob(
        "training_record.json"
    )
)

assert len(records) == 1, (
    f"Expected exactly one training_record.json; "
    f"found {len(records)}."
)

record_path = records[0]

record = json.loads(
    record_path.read_text(
        encoding="utf-8",
    )
)

print()
print("=" * 78)
print("TRAINING RECORD")
print("=" * 78)

print(
    json.dumps(
        record,
        indent=2,
        sort_keys=True,
    )
)


# ============================================================================
# PROVENANCE CHECK
# ============================================================================

record_sha = record.get(
    "script_sha256"
)

print()
print("=" * 78)
print("EXECUTABLE PROVENANCE CHECK")
print("=" * 78)

print()
print("Expected:")
print(EXPECTED_SHA)

print()
print("Recorded:")
print(record_sha)

assert record_sha == EXPECTED_SHA, (
    "training_record.json does not identify "
    "the frozen executable SHA."
)

print()
print("PROVENANCE: PASS")


# ============================================================================
# SAVE DEV GATE AUDIT
# ============================================================================

audit = {

    "status":
        "PASS",

    "production_inclusion":
        False,

    "script":
        str(SCRIPT),

    "script_sha256":
        EXPECTED_SHA,

    "project":
        str(PROJECT),

    "cache":
        str(CACHE),

    "mode":
        "dev",

    "rho":
        0.0,

    "replicate":
        1,

    "gap":
        "mean",

    "num_workers":
        2,

    "cuda_visible_devices":
        "0",

    "cublas_workspace_config":
        ":4096:8",

    "return_code":
        result.returncode,

    "elapsed_seconds":
        elapsed,

    "training_record":
        str(record_path),

    "recorded_script_sha256":
        record_sha,

    "artifact_count":
        len(files),

    "launcher_correction":
        "Changed --cache_dir to --cache-dir and "
        "--out_dir to --out-dir; frozen script unchanged.",
}

AUDIT_PATH = (
    DEV_ROOT
    / "T2_1_DEV_EXECUTION_GATE_AUDIT.json"
)

AUDIT_PATH.write_text(
    json.dumps(
        audit,
        indent=2,
    ),
    encoding="utf-8",
)


print()
print("=" * 78)
print("6ZK-I COMPLETE")
print("=" * 78)

print()
print("DEV EXECUTION GATE: PASS")

print()
print("Audit:")
print(AUDIT_PATH)

print()
print("Frozen script SHA remains:")
print(EXPECTED_SHA)

print()
print("=" * 78)
print("IMPORTANT")
print("=" * 78)

print(
    "This DEV run is diagnostic only and is NOT part of "
    "the 9-model production dataset."
)

In [ ]:
# ============================================================================
# CELL 6ZK-J — SEED IMPLEMENTATION AUDIT
# ============================================================================
#
# Purpose:
#   Inspect the exact frozen seed_everything() implementation and all RNG
#   usage before making the minimal seed-width correction.
#
# IMPORTANT:
#   This cell DOES NOT modify the frozen script.
# ============================================================================

from pathlib import Path
import ast
import hashlib
import re


SCRIPT = Path(
    "/kaggle/working/"
    "T2_1_ConvNeXt_training_code_freeze/"
    "train_convnext_tiny_t2_1.py"
)

EXPECTED_SHA = (
    "5d822b1336eabcfcf83edf3b12d012817687ba8f6335cf670b6ebed4fcbf4da6"
)


def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b"",
        ):
            h.update(chunk)

    return h.hexdigest()


assert SCRIPT.exists()

actual_sha = sha256_file(SCRIPT)

print("=" * 78)
print("T2.1 SEED IMPLEMENTATION AUDIT")
print("=" * 78)

print()
print("SCRIPT:")
print(SCRIPT)

print()
print("EXPECTED SHA:")
print(EXPECTED_SHA)

print()
print("ACTUAL SHA:")
print(actual_sha)

assert actual_sha == EXPECTED_SHA

print()
print("SCRIPT HASH: PASS")


# ============================================================================
# READ SOURCE
# ============================================================================

source = SCRIPT.read_text(
    encoding="utf-8"
)

lines = source.splitlines()


# ============================================================================
# PRINT seed_everything()
# ============================================================================

print()
print("=" * 78)
print("seed_everything() SOURCE")
print("=" * 78)

start = None
end = None

for i, line in enumerate(lines):

    if line.startswith("def seed_everything("):
        start = i
        break

assert start is not None

for i in range(start + 1, len(lines)):

    if (
        lines[i].startswith("def ")
        and not lines[i].startswith("def seed_everything(")
    ):
        end = i
        break

if end is None:
    end = min(start + 80, len(lines))

for i in range(start, end):

    print(
        f"{i+1:5d}: {lines[i]}"
    )


# ============================================================================
# FIND ALL RANDOM-RELATED CALLS
# ============================================================================

print()
print("=" * 78)
print("ALL RNG-RELATED SOURCE LINES")
print("=" * 78)

rng_patterns = [
    "np.random",
    "random.",
    "torch.manual_seed",
    "torch.cuda.manual_seed",
    "torch.cuda.manual_seed_all",
    "SeedSequence",
    "default_rng",
    "Generator",
    "manual_seed",
    "seed_everything",
]

rng_hits = []

for i, line in enumerate(lines):

    if any(
        pattern in line
        for pattern in rng_patterns
    ):

        rng_hits.append(
            (i + 1, line)
        )

for lineno, line in rng_hits:

    print(
        f"{lineno:5d}: {line}"
    )


# ============================================================================
# AST CHECK — seed_everything BODY
# ============================================================================

print()
print("=" * 78)
print("AST SEED CALL AUDIT")
print("=" * 78)

tree = ast.parse(source)

seed_function = None

for node in tree.body:

    if (
        isinstance(node, ast.FunctionDef)
        and node.name == "seed_everything"
    ):
        seed_function = node
        break

assert seed_function is not None

calls = []

for node in ast.walk(seed_function):

    if isinstance(node, ast.Call):

        try:
            call_text = ast.unparse(node)
        except Exception:
            call_text = "<unparse failed>"

        calls.append(call_text)

for call in calls:

    print(call)


# ============================================================================
# SEED TABLE / SEED CONSUMPTION CHECK
# ============================================================================

print()
print("=" * 78)
print("SEED VARIABLE USAGE")
print("=" * 78)

for i, line in enumerate(lines):

    if re.search(
        r"\bseed\b",
        line,
        flags=re.IGNORECASE,
    ):

        print(
            f"{i+1:5d}: {line}"
        )


print()
print("=" * 78)
print("AUDIT COMPLETE")
print("=" * 78)

print()
print(
    "No file modification was performed."
)

print(
    "The current frozen SHA remains:",
    actual_sha,
)

In [ ]:
# ============================================================================
# CELL 6ZK-K — AUDIT SECOND LEGACY NUMPY SEED + RNG PATH
# ============================================================================
#
# READ-ONLY.
# No modification to the frozen training script.
# ============================================================================

from pathlib import Path
import hashlib


SCRIPT = Path(
    "/kaggle/working/"
    "T2_1_ConvNeXt_training_code_freeze/"
    "train_convnext_tiny_t2_1.py"
)

EXPECTED_SHA = (
    "5d822b1336eabcfcf83edf3b12d012817687ba8f6335cf670b6ebed4fcbf4da6"
)


def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b"",
        ):
            h.update(chunk)

    return h.hexdigest()


assert sha256_file(SCRIPT) == EXPECTED_SHA

lines = SCRIPT.read_text(
    encoding="utf-8"
).splitlines()


def print_range(start, end, title):

    print()
    print("=" * 78)
    print(title)
    print("=" * 78)

    for i in range(start, end + 1):

        print(
            f"{i:5d}: {lines[i-1]}"
        )


# ============================================================================
# LINE 801 REGION
# ============================================================================

print_range(
    785,
    825,
    "SECOND NUMPY SEED — FULL CONTEXT"
)


# ============================================================================
# AUGMENTATION RNG REGION
# ============================================================================

print_range(
    145,
    175,
    "EXACT PRODUCTION AUGMENTATION RNG PATH"
)


# ============================================================================
# DATALOADER REGION
# ============================================================================

print_range(
    775,
    825,
    "DATALOADER / WORKER SEED REGION"
)


# ============================================================================
# ALL np.random.seed OCCURRENCES
# ============================================================================

print()
print("=" * 78)
print("ALL LEGACY np.random.seed() OCCURRENCES")
print("=" * 78)

for i, line in enumerate(lines, start=1):

    if "np.random.seed" in line:

        print(
            f"{i:5d}: {line}"
        )


# ============================================================================
# ALL SeedSequence/default_rng OCCURRENCES
# ============================================================================

print()
print("=" * 78)
print("ALL MODERN NUMPY RNG OCCURRENCES")
print("=" * 78)

for i, line in enumerate(lines, start=1):

    if (
        "SeedSequence" in line
        or "default_rng" in line
    ):

        print(
            f"{i:5d}: {line}"
        )


print()
print("=" * 78)
print("READ-ONLY AUDIT COMPLETE")
print("=" * 78)

print()
print(
    "Frozen script SHA remains:",
    sha256_file(SCRIPT),
)

In [ ]:
# ============================================================================
# CELL 6ZK-L — LEGACY NUMPY GLOBAL RNG CONSUMPTION AUDIT
# ============================================================================
#
# Determine whether np.random's LEGACY GLOBAL RandomState is actually used
# after seed_everything().
#
# READ-ONLY. No modification to the frozen script.
# ============================================================================

from pathlib import Path
import ast
import hashlib


SCRIPT = Path(
    "/kaggle/working/"
    "T2_1_ConvNeXt_training_code_freeze/"
    "train_convnext_tiny_t2_1.py"
)

EXPECTED_SHA = (
    "5d822b1336eabcfcf83edf3b12d012817687ba8f6335cf670b6ebed4fcbf4da6"
)


def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b"",
        ):
            h.update(chunk)

    return h.hexdigest()


source = SCRIPT.read_text(
    encoding="utf-8"
)

actual_sha = sha256_file(SCRIPT)

assert actual_sha == EXPECTED_SHA

tree = ast.parse(source)


print("=" * 78)
print("LEGACY NUMPY GLOBAL RNG CONSUMPTION AUDIT")
print("=" * 78)

print()
print("SCRIPT SHA:")
print(actual_sha)


# ============================================================================
# Find every np.random.* call
# ============================================================================

print()
print("=" * 78)
print("ALL np.random.* CALLS")
print("=" * 78)

np_calls = []

for node in ast.walk(tree):

    if isinstance(node, ast.Call):

        func = node.func

        if (
            isinstance(func, ast.Attribute)
            and isinstance(func.value, ast.Attribute)
            and isinstance(func.value.value, ast.Name)
            and func.value.value.id == "np"
            and func.value.attr == "random"
        ):

            np_calls.append(
                (
                    node.lineno,
                    ast.unparse(node),
                )
            )

        elif (
            isinstance(func, ast.Attribute)
            and isinstance(func.value, ast.Name)
            and func.value.id == "np"
            and func.value.attr == "random"
        ):

            np_calls.append(
                (
                    node.lineno,
                    ast.unparse(node),
                )
            )


for lineno, call in sorted(np_calls):

    print(
        f"{lineno:5d}: {call}"
    )


# ============================================================================
# Identify global RandomState consumers
# ============================================================================

legacy_methods = {
    "rand",
    "randn",
    "randint",
    "random",
    "random_sample",
    "choice",
    "shuffle",
    "permutation",
    "bytes",
    "beta",
    "binomial",
    "chisquare",
    "exponential",
    "f",
    "gamma",
    "geometric",
    "hypergeometric",
    "laplace",
    "logistic",
    "lognormal",
    "multinomial",
    "multivariate_normal",
    "negative_binomial",
    "noncentral_chisquare",
    "noncentral_f",
    "normal",
    "pareto",
    "poisson",
    "power",
    "rayleigh",
    "standard_cauchy",
    "standard_exponential",
    "standard_gamma",
    "standard_normal",
    "standard_t",
    "triangular",
    "uniform",
    "vonmises",
    "wald",
    "weibull",
}


legacy_consumers = []

for node in ast.walk(tree):

    if not isinstance(node, ast.Call):
        continue

    func = node.func

    if not (
        isinstance(func, ast.Attribute)
        and isinstance(func.value, ast.Attribute)
        and isinstance(func.value.value, ast.Name)
        and func.value.value.id == "np"
        and func.value.attr == "random"
    ):
        continue

    method = func.attr

    if method in legacy_methods:

        legacy_consumers.append(
            (
                node.lineno,
                ast.unparse(node),
            )
        )


print()
print("=" * 78)
print("POTENTIAL LEGACY GLOBAL-RNG CONSUMERS")
print("=" * 78)

for lineno, call in sorted(legacy_consumers):

    print(
        f"{lineno:5d}: {call}"
    )


# ============================================================================
# Explicitly inspect whether modern Generator is used
# ============================================================================

print()
print("=" * 78)
print("MODERN GENERATOR CONSTRUCTION")
print("=" * 78)

for node in ast.walk(tree):

    if isinstance(node, ast.Call):

        text = ast.unparse(node)

        if (
            "SeedSequence" in text
            or "default_rng" in text
        ):

            print(
                f"{node.lineno:5d}: {text}"
            )


# ============================================================================
# Source-level sanity check
# ============================================================================

print()
print("=" * 78)
print("INTERPRETATION")
print("=" * 78)

print()

if len(legacy_consumers) == 1:

    only_lineno, only_call = legacy_consumers[0]

    if only_lineno == 137:

        print(
            "PASS: No legacy np.random distribution/sampling call "
            "was detected after seed_everything()."
        )

        print(
            "The line-137 np.random.seed() call appears to be "
            "initialization-only."
        )

else:

    print(
        "REVIEW REQUIRED: Additional legacy np.random consumers "
        "were detected."
    )

print()
print(
    "The frozen script was not modified."
)

print(
    "Current SHA:",
    actual_sha,
)

In [ ]:
# ============================================================================
# CELL 6ZK-L2 — CORRECTED LEGACY NUMPY RNG CONSUMPTION AUDIT
# ============================================================================
#
# READ-ONLY.
# The frozen T2.1 script is NOT modified.
# ============================================================================

from pathlib import Path
import ast
import hashlib


SCRIPT = Path(
    "/kaggle/working/"
    "T2_1_ConvNeXt_training_code_freeze/"
    "train_convnext_tiny_t2_1.py"
)

EXPECTED_SHA = (
    "5d822b1336eabcfcf83edf3b12d012817687ba8f6335cf670b6ebed4fcbf4da6"
)


def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b"",
        ):
            h.update(chunk)

    return h.hexdigest()


source = SCRIPT.read_text(
    encoding="utf-8"
)

actual_sha = sha256_file(SCRIPT)

assert actual_sha == EXPECTED_SHA

tree = ast.parse(source)


print("=" * 78)
print("CORRECTED LEGACY NUMPY RNG CONSUMPTION AUDIT")
print("=" * 78)

print()
print("SCRIPT SHA:")
print(actual_sha)


# ============================================================================
# Helper: identify np.random.<method>
# ============================================================================

def get_np_random_method(node):

    if not isinstance(node, ast.Call):
        return None

    func = node.func

    # np.random.foo
    if isinstance(func, ast.Attribute):

        obj = func.value

        if (
            isinstance(obj, ast.Attribute)
            and isinstance(obj.value, ast.Name)
            and obj.value.id == "np"
            and obj.attr == "random"
        ):
            return func.attr

    return None


# ============================================================================
# All np.random.* calls
# ============================================================================

all_calls = []

for node in ast.walk(tree):

    method = get_np_random_method(node)

    if method is not None:

        all_calls.append(
            (
                node.lineno,
                method,
                ast.unparse(node),
            )
        )


all_calls.sort()


print()
print("=" * 78)
print("ALL np.random.* CALLS")
print("=" * 78)

for lineno, method, call in all_calls:

    print(
        f"{lineno:5d}: {call}"
    )


# ============================================================================
# Categorize
# ============================================================================

modern_calls = []
seed_calls = []
legacy_consumers = []

legacy_methods = {
    "rand",
    "randn",
    "randint",
    "random",
    "random_sample",
    "choice",
    "shuffle",
    "permutation",
    "bytes",
    "beta",
    "binomial",
    "chisquare",
    "exponential",
    "f",
    "gamma",
    "geometric",
    "hypergeometric",
    "laplace",
    "logistic",
    "lognormal",
    "multinomial",
    "multivariate_normal",
    "negative_binomial",
    "noncentral_chisquare",
    "noncentral_f",
    "normal",
    "pareto",
    "poisson",
    "power",
    "rayleigh",
    "standard_cauchy",
    "standard_exponential",
    "standard_gamma",
    "standard_normal",
    "standard_t",
    "triangular",
    "uniform",
    "vonmises",
    "wald",
    "weibull",
}

for lineno, method, call in all_calls:

    if method == "seed":

        seed_calls.append(
            (lineno, call)
        )

    elif method in {
        "SeedSequence",
        "default_rng",
    }:

        modern_calls.append(
            (lineno, call)
        )

    elif method in legacy_methods:

        legacy_consumers.append(
            (lineno, call)
        )


# ============================================================================
# Report categories
# ============================================================================

print()
print("=" * 78)
print("LEGACY np.random.seed() CALLS")
print("=" * 78)

for lineno, call in seed_calls:

    print(
        f"{lineno:5d}: {call}"
    )


print()
print("=" * 78)
print("MODERN NUMPY RNG CALLS")
print("=" * 78)

for lineno, call in modern_calls:

    print(
        f"{lineno:5d}: {call}"
    )


print()
print("=" * 78)
print("LEGACY NUMPY RNG CONSUMERS")
print("=" * 78)

if legacy_consumers:

    for lineno, call in legacy_consumers:

        print(
            f"{lineno:5d}: {call}"
        )

else:

    print(
        "NONE DETECTED"
    )


# ============================================================================
# Context around every np.random call
# ============================================================================

print()
print("=" * 78)
print("SOURCE CONTEXT FOR EVERY np.random CALL")
print("=" * 78)

for lineno, method, call in all_calls:

    start = max(
        1,
        lineno - 2,
    )

    end = min(
        len(source.splitlines()),
        lineno + 2,
    )

    print()
    print(
        f"--- line {lineno}: {call} ---"
    )

    for i in range(start, end + 1):

        print(
            f"{i:5d}: "
            f"{source.splitlines()[i-1]}"
        )


# ============================================================================
# Final interpretation
# ============================================================================

print()
print("=" * 78)
print("AUDIT INTERPRETATION")
print("=" * 78)

if legacy_consumers:

    print()
    print(
        "REVIEW REQUIRED:"
    )

    print(
        "Legacy NumPy sampling/distribution calls exist."
    )

else:

    print()
    print(
        "PASS:"
    )

    print(
        "No legacy NumPy sampling/distribution calls were detected."
    )

    print(
        "The only legacy RandomState operations are np.random.seed() "
        "initialization calls."
    )


print()
print(
    "Frozen script was NOT modified."
)

print(
    "Current SHA:",
    actual_sha,
)

In [ ]:
# ============================================================================
# CELL 6ZK-M — SURGICAL NUMPY SEED-WIDTH FIX + NEW FROZEN HASH
# ============================================================================
#
# ONLY modification:
#
#   np.random.seed(seed)
#
# becomes:
#
#   np.random.seed(int(seed) & 0xffffffff)
#
# No other source modification is permitted.
# ============================================================================

from pathlib import Path
import hashlib
import ast
import json
import py_compile


SCRIPT = Path(
    "/kaggle/working/"
    "T2_1_ConvNeXt_training_code_freeze/"
    "train_convnext_tiny_t2_1.py"
)

OLD_SHA = (
    "5d822b1336eabcfcf83edf3b12d012817687ba8f6335cf670b6ebed4fcbf4da6"
)


def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b"",
        ):
            h.update(chunk)

    return h.hexdigest()


# ============================================================================
# Verify starting state
# ============================================================================

before_sha = sha256_file(SCRIPT)

assert before_sha == OLD_SHA, (
    "Starting script SHA is not the expected frozen SHA. "
    f"Found {before_sha}"
)

source_before = SCRIPT.read_text(
    encoding="utf-8"
)

old_line = "np.random.seed(seed)"
new_line = "np.random.seed(int(seed) & 0xffffffff)"

assert source_before.count(old_line) == 1, (
    "Expected exactly one occurrence of the problematic "
    "legacy NumPy seed call."
)

assert new_line not in source_before, (
    "Surgical fix appears to have already been applied."
)


# ============================================================================
# Perform EXACTLY ONE substitution
# ============================================================================

source_after = source_before.replace(
    old_line,
    new_line,
    1,
)

assert source_after.count(old_line) == 0
assert source_after.count(new_line) == 1


SCRIPT.write_text(
    source_after,
    encoding="utf-8",
)


# ============================================================================
# Verify compile
# ============================================================================

py_compile.compile(
    str(SCRIPT),
    doraise=True,
)

print("=" * 78)
print("T2.1 SURGICAL SEED-WIDTH FIX")
print("=" * 78)

print()
print("OLD SHA:")
print(before_sha)

print()
print("MODIFICATION:")
print("np.random.seed(seed)")
print("        ->")
print("np.random.seed(int(seed) & 0xffffffff)")

print()
print("COMPILE: PASS")


# ============================================================================
# Verify only intended source change
# ============================================================================

source_lines_before = source_before.splitlines()
source_lines_after = source_after.splitlines()

assert len(source_lines_before) == len(source_lines_after)

changed_lines = []

for i, (a, b) in enumerate(
    zip(
        source_lines_before,
        source_lines_after,
    ),
    start=1,
):

    if a != b:

        changed_lines.append(
            (
                i,
                a,
                b,
            )
        )

assert len(changed_lines) == 1

line_no, old_text, new_text = changed_lines[0]

assert line_no == 137
assert old_text == old_line
assert new_text == new_line

print()
print("SOURCE CHANGE AUDIT:")
print(f"Changed lines: {len(changed_lines)}")
print(f"Changed line: {line_no}")
print(f"Old: {old_text}")
print(f"New: {new_text}")

print()
print("SURGICAL CHANGE AUDIT: PASS")


# ============================================================================
# Verify all other seed paths remain unchanged
# ============================================================================

assert (
    "np.random.seed((seeds['seed_workers'] + w) % 2 ** 32)"
    in source_after
)

assert (
    "np.random.SeedSequence("
    in source_after
)

assert (
    "np.random.default_rng(ss)"
    in source_after
)

assert (
    "random.seed(seed)"
    in source_after
)

assert (
    "torch.manual_seed(seed)"
    in source_after
)

assert (
    "torch.cuda.manual_seed_all(seed)"
    in source_after
)

print()
print("RNG PATH PRESERVATION: PASS")


# ============================================================================
# New hash
# ============================================================================

new_sha = sha256_file(SCRIPT)

assert new_sha != OLD_SHA

print()
print("=" * 78)
print("NEW FROZEN SCRIPT SHA256")
print("=" * 78)

print()
print(new_sha)


# ============================================================================
# Save immutable-style hash record
# ============================================================================

LOCK_DIR = SCRIPT.parent

lock = {
    "artifact":
        "T2.1 ConvNeXt-Tiny training script",

    "status":
        "FROZEN_AFTER_SURGICAL_RUNTIME_FIX",

    "previous_sha256":
        OLD_SHA,

    "sha256":
        new_sha,

    "changed_line":
        137,

    "old_line":
        old_line,

    "new_line":
        new_line,

    "reason":
        "Python NumPy legacy RandomState seed() accepts only "
        "32-bit seeds, while the locked T2.1 training seed table "
        "contains 64-bit seeds.",

    "scope":
        "Compatibility-only projection for legacy NumPy seed "
        "initialization. The locked 64-bit master seed remains "
        "unchanged for Python, PyTorch, CUDA, SeedSequence-based "
        "augmentation, and DataLoader seeding.",

    "legacy_numpy_consumers":
        False,

    "compile":
        True,

    "source_change_count":
        1,
}

LOCK_PATH = (
    LOCK_DIR
    / "T2_1_TRAINING_CODE_RUNTIME_FIX_HASH.json"
)

LOCK_PATH.write_text(
    json.dumps(
        lock,
        indent=2,
    ),
    encoding="utf-8",
)

print()
print("HASH LOCK:")
print(LOCK_PATH)

print()
print("=" * 78)
print("6ZK-M COMPLETE")
print("=" * 78)

print()
print("NEW SCRIPT IS NOW THE FROZEN T2.1 EXECUTABLE.")
print()
print("NEW SHA256:")
print(new_sha)

In [ ]:
# ============================================================================
# CELL 6ZK-M2 — RECOVER AND AUDIT SURGICAL SEED FIX
# ============================================================================
#
# READ-ONLY FROM THIS POINT.
# Do NOT modify the script.
# ============================================================================

from pathlib import Path
import hashlib
import json
import py_compile


SCRIPT = Path(
    "/kaggle/working/"
    "T2_1_ConvNeXt_training_code_freeze/"
    "train_convnext_tiny_t2_1.py"
)

OLD_SHA = (
    "5d822b1336eabcfcf83edf3b12d012817687ba8f6335cf670b6ebed4fcbf4da6"
)


def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b"",
        ):
            h.update(chunk)

    return h.hexdigest()


assert SCRIPT.exists()

actual_sha = sha256_file(SCRIPT)

source = SCRIPT.read_text(
    encoding="utf-8"
)

lines = source.splitlines()


print("=" * 78)
print("T2.1 SURGICAL SEED FIX — RECOVERY AUDIT")
print("=" * 78)

print()
print("CURRENT SHA:")
print(actual_sha)

print()
print("OLD SHA:")
print(OLD_SHA)

print()
print("SHA CHANGED:")
print(actual_sha != OLD_SHA)


# ============================================================================
# Inspect seed_everything()
# ============================================================================

print()
print("=" * 78)
print("CURRENT seed_everything()")
print("=" * 78)

start = None

for i, line in enumerate(lines):

    if line.startswith(
        "def seed_everything("
    ):

        start = i
        break

assert start is not None

for i in range(
    start,
    min(start + 12, len(lines)),
):

    print(
        f"{i+1:5d}: {lines[i]}"
    )


# ============================================================================
# Exact seed-line audit
# ============================================================================

old_line = "np.random.seed(seed)"
new_line = "np.random.seed(int(seed) & 0xffffffff)"

old_count = source.count(
    old_line
)

new_count = source.count(
    new_line
)

print()
print("=" * 78)
print("EXACT SEED-LINE COUNTS")
print("=" * 78)

print()
print("Old expression count:")
print(old_count)

print()
print("New expression count:")
print(new_count)


# ============================================================================
# Confirm worker seed remains unchanged
# ============================================================================

worker_line = (
    "np.random.seed((seeds['seed_workers'] + w) % 2 ** 32)"
)

print()
print("=" * 78)
print("WORKER SEED PRESERVATION")
print("=" * 78)

print()
print("Worker seed expression present:")
print(worker_line in source)


# ============================================================================
# Compile current script
# ============================================================================

print()
print("=" * 78)
print("CURRENT SCRIPT COMPILE")
print("=" * 78)

py_compile.compile(
    str(SCRIPT),
    doraise=True,
)

print()
print("COMPILE: PASS")


# ============================================================================
# Verify required RNG paths remain
# ============================================================================

required_fragments = [
    "random.seed(seed)",
    "torch.manual_seed(seed)",
    "torch.cuda.manual_seed_all(seed)",
    "np.random.SeedSequence(",
    "np.random.default_rng(ss)",
]

print()
print("=" * 78)
print("RNG PATH PRESERVATION")
print("=" * 78)

for fragment in required_fragments:

    present = fragment in source

    print(
        f"{present!s:5s}  {fragment}"
    )

    assert present


# ============================================================================
# Critical assertions
# ============================================================================

assert old_count == 0, (
    "Old broken seed expression still exists."
)

assert new_count == 1, (
    "Expected exactly one corrected seed expression."
)

assert worker_line in source, (
    "Worker seed expression unexpectedly changed."
)

assert actual_sha != OLD_SHA, (
    "Script SHA did not change after the surgical modification."
)


# ============================================================================
# Save corrected lock
# ============================================================================

LOCK = {
    "status":
        "PASS",

    "previous_sha256":
        OLD_SHA,

    "current_sha256":
        actual_sha,

    "runtime_fix":
        "np.random.seed(seed) -> "
        "np.random.seed(int(seed) & 0xffffffff)",

    "old_expression_count":
        old_count,

    "new_expression_count":
        new_count,

    "worker_seed_unchanged":
        True,

    "compile":
        True,

    "required_rng_paths_present":
        True,

    "note":
        "Recovery audit after verification-cell assertion failure. "
        "No additional source modification was performed.",
}

LOCK_PATH = (
    SCRIPT.parent
    / "T2_1_TRAINING_CODE_RUNTIME_FIX_HASH.json"
)

LOCK_PATH.write_text(
    json.dumps(
        LOCK,
        indent=2,
    ),
    encoding="utf-8",
)


print()
print("=" * 78)
print("RECOVERY AUDIT RESULT")
print("=" * 78)

print()
print("PASS")

print()
print("AUTHORITATIVE NEW SHA:")
print(actual_sha)

print()
print("LOCK:")
print(LOCK_PATH)

In [ ]:
# ============================================================================
# CELL 6ZK-M3 — SEMANTIC RNG FIX AUDIT
# ============================================================================
#
# READ-ONLY.
# No modification to the frozen script.
# ============================================================================

from pathlib import Path
import ast
import hashlib
import json
import py_compile


SCRIPT = Path(
    "/kaggle/working/"
    "T2_1_ConvNeXt_training_code_freeze/"
    "train_convnext_tiny_t2_1.py"
)

OLD_SHA = (
    "5d822b1336eabcfcf83edf3b12d012817687ba8f6335cf670b6ebed4fcbf4da6"
)

EXPECTED_CURRENT_SHA = (
    "6c0407330f547ff9b67ba8dbcf4b849b3379e0303f97eebd8ef722482d14c36f"
)


def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b"",
        ):
            h.update(chunk)

    return h.hexdigest()


source = SCRIPT.read_text(
    encoding="utf-8"
)

current_sha = sha256_file(SCRIPT)

assert current_sha == EXPECTED_CURRENT_SHA
assert current_sha != OLD_SHA


print("=" * 78)
print("T2.1 SEMANTIC RNG FIX AUDIT")
print("=" * 78)

print()
print("CURRENT SHA:")
print(current_sha)


# ============================================================================
# Compile
# ============================================================================

py_compile.compile(
    str(SCRIPT),
    doraise=True,
)

print()
print("COMPILE: PASS")


# ============================================================================
# AST: locate every np.random.seed() call
# ============================================================================

tree = ast.parse(source)

seed_calls = []

for node in ast.walk(tree):

    if not isinstance(node, ast.Call):
        continue

    func = node.func

    if not isinstance(func, ast.Attribute):
        continue

    if func.attr != "seed":
        continue

    obj = func.value

    if not isinstance(obj, ast.Attribute):
        continue

    if obj.attr != "random":
        continue

    base = obj.value

    if not isinstance(base, ast.Name):
        continue

    if base.id != "np":
        continue

    seed_calls.append(node)


print()
print("=" * 78)
print("NUMPY LEGACY SEED CALLS")
print("=" * 78)

for node in sorted(
    seed_calls,
    key=lambda n: n.lineno,
):

    print(
        f"line {node.lineno}: "
        f"{ast.unparse(node)}"
    )

assert len(seed_calls) == 2


# ============================================================================
# Classify the two calls
# ============================================================================

seed_calls = sorted(
    seed_calls,
    key=lambda n: n.lineno,
)

line137 = seed_calls[0]
line811 = seed_calls[1]

assert line137.lineno == 137
assert line811.lineno == 811


line137_text = ast.unparse(
    line137
)

line811_text = ast.unparse(
    line811
)

print()
print("=" * 78)
print("LINE 137")
print("=" * 78)

print(line137_text)

assert (
    line137_text
    == "np.random.seed(int(seed) & 0xffffffff)"
)


print()
print("=" * 78)
print("LINE 811")
print("=" * 78)

print(line811_text)


# ============================================================================
# Semantic worker-seed validation
# ============================================================================

worker_source = ast.unparse(
    line811
)

print()
print("=" * 78)
print("WORKER-SEED SEMANTIC CHECK")
print("=" * 78)

print()
print(
    "Worker expression:"
)
print(worker_source)

# The worker seed must contain:
#   seeds["seed_workers"]
#   + w
#   modulo 2**32

assert "seed_workers" in worker_source
assert "w" in worker_source

assert (
    "2 ** 32" in worker_source
    or "4294967296" in worker_source
)

print()
print("Worker seed semantics: PASS")


# ============================================================================
# Verify modern RNG paths
# ============================================================================

required_fragments = [
    "np.random.SeedSequence",
    "np.random.default_rng",
    "torch.manual_seed",
    "torch.cuda.manual_seed_all",
    "random.seed",
]

print()
print("=" * 78)
print("RNG PATH PRESERVATION")
print("=" * 78)

for fragment in required_fragments:

    present = fragment in source

    print(
        f"{present!s:5s}  {fragment}"
    )

    assert present


# ============================================================================
# Verify no old line remains
# ============================================================================

assert source.count(
    "np.random.seed(seed)"
) == 0

assert source.count(
    "np.random.seed(int(seed) & 0xffffffff)"
) == 1

print()
print("Old broken seed expression: ABSENT")
print("Corrected seed expression: EXACTLY ONE")


# ============================================================================
# Save authoritative lock
# ============================================================================

lock = {

    "status":
        "PASS",

    "previous_sha256":
        OLD_SHA,

    "current_sha256":
        current_sha,

    "runtime_fix":
        {
            "line": 137,
            "old": "np.random.seed(seed)",
            "new":
                "np.random.seed(int(seed) & 0xffffffff)",
        },

    "numpy_seed_calls":
        [
            {
                "line": 137,
                "expression": line137_text,
                "role":
                    "legacy NumPy initialization",
            },
            {
                "line": 811,
                "expression": line811_text,
                "role":
                    "DataLoader worker initialization",
                "semantic_constraint":
                    "seed_workers + worker_id modulo 2**32",
            },
        ],

    "modern_rng_paths":
        True,

    "compile":
        True,

    "note":
        "Previous audit failed because it used a brittle exact "
        "source-format string check for the worker seed. This "
        "semantic audit validates the AST expression instead.",
}

LOCK_PATH = (
    SCRIPT.parent
    / "T2_1_TRAINING_CODE_RUNTIME_FIX_HASH.json"
)

LOCK_PATH.write_text(
    json.dumps(
        lock,
        indent=2,
    ),
    encoding="utf-8",
)


print()
print("=" * 78)
print("6ZK-M3 RESULT")
print("=" * 78)

print()
print("PASS")

print()
print("AUTHORITATIVE T2.1 SCRIPT SHA:")
print(current_sha)

print()
print("LOCK:")
print(LOCK_PATH)

In [ ]:
# ============================================================================
# CELL 6ZK-M4 — FINAL SEMANTIC RNG AUDIT
# ============================================================================
# READ-ONLY — DOES NOT MODIFY THE TRAINING SCRIPT
# ============================================================================

from pathlib import Path
import ast
import hashlib
import json
import py_compile


SCRIPT = Path(
    "/kaggle/working/"
    "T2_1_ConvNeXt_training_code_freeze/"
    "train_convnext_tiny_t2_1.py"
)

EXPECTED_SHA = (
    "6c0407330f547ff9b67ba8dbcf4b849b3379e0303f97eebd8ef722482d14c36f"
)


def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b"",
        ):
            h.update(chunk)

    return h.hexdigest()


# ============================================================================
# SHA + COMPILE
# ============================================================================

actual_sha = sha256_file(SCRIPT)

assert actual_sha == EXPECTED_SHA, (
    f"Unexpected script SHA: {actual_sha}"
)

py_compile.compile(
    str(SCRIPT),
    doraise=True,
)

source = SCRIPT.read_text(
    encoding="utf-8"
)

tree = ast.parse(source)

print("=" * 78)
print("T2.1 FINAL SEMANTIC RNG AUDIT")
print("=" * 78)

print()
print("SCRIPT SHA:")
print(actual_sha)

print()
print("COMPILE: PASS")


# ============================================================================
# Find np.random.seed() calls
# ============================================================================

seed_calls = []

for node in ast.walk(tree):

    if not isinstance(node, ast.Call):
        continue

    func = node.func

    if not isinstance(func, ast.Attribute):
        continue

    if func.attr != "seed":
        continue

    obj = func.value

    if not isinstance(obj, ast.Attribute):
        continue

    if obj.attr != "random":
        continue

    base = obj.value

    if not isinstance(base, ast.Name):
        continue

    if base.id != "np":
        continue

    seed_calls.append(node)


seed_calls.sort(
    key=lambda x: x.lineno
)

assert len(seed_calls) == 2


print()
print("=" * 78)
print("NUMPY SEED CALLS")
print("=" * 78)

for node in seed_calls:

    print(
        f"line {node.lineno}: "
        f"{ast.unparse(node)}"
    )


# ============================================================================
# LINE 137 — SEMANTIC CHECK
# ============================================================================

node137 = seed_calls[0]

assert node137.lineno == 137

arg = node137.args[0]

# Expected structure:
#
#   int(seed) & 0xffffffff
#
# AST structure:
#
#   BinOp(
#       Call(int, seed),
#       BitAnd,
#       Constant(4294967295)
#   )

assert isinstance(
    arg,
    ast.BinOp,
)

assert isinstance(
    arg.op,
    ast.BitAnd,
)

assert isinstance(
    arg.left,
    ast.Call,
)

assert isinstance(
    arg.left.func,
    ast.Name,
)

assert arg.left.func.id == "int"

assert len(arg.left.args) == 1

assert isinstance(
    arg.left.args[0],
    ast.Name,
)

assert arg.left.args[0].id == "seed"

assert isinstance(
    arg.right,
    ast.Constant,
)

assert arg.right.value == 0xffffffff


print()
print("=" * 78)
print("LINE 137 SEMANTIC CHECK")
print("=" * 78)

print()
print(
    "Confirmed:"
)

print(
    "np.random.seed("
    "int(seed) & 0xffffffff"
    ")"
)

print()
print("LINE 137: PASS")


# ============================================================================
# LINE 811 — WORKER SEED SEMANTIC CHECK
# ============================================================================

node811 = seed_calls[1]

assert node811.lineno == 811

arg811 = node811.args[0]

assert isinstance(
    arg811,
    ast.BinOp,
)

assert isinstance(
    arg811.op,
    ast.Mod,
)

assert isinstance(
    arg811.right,
    ast.Constant,
)

assert arg811.right.value == 2 ** 32


left811 = arg811.left

assert isinstance(
    left811,
    ast.BinOp,
)

assert isinstance(
    left811.op,
    ast.Add,
)

# Find seeds["seed_workers"]
subscript_found = False
worker_id_found = False

for node in ast.walk(left811):

    if isinstance(node, ast.Name):

        if node.id == "w":

            worker_id_found = True

    if isinstance(node, ast.Subscript):

        if (
            isinstance(node.value, ast.Name)
            and node.value.id == "seeds"
        ):

            text = ast.unparse(node)

            if "seed_workers" in text:

                subscript_found = True


assert subscript_found
assert worker_id_found


print()
print("=" * 78)
print("LINE 811 WORKER-SEED CHECK")
print("=" * 78)

print()
print(
    "Confirmed:"
)

print(
    "seed_workers + worker_id "
    "modulo 2**32"
)

print()
print("LINE 811: PASS")


# ============================================================================
# MODERN RNG PATHS
# ============================================================================

required = [
    "np.random.SeedSequence",
    "np.random.default_rng",
    "torch.manual_seed",
    "torch.cuda.manual_seed_all",
    "random.seed",
]

print()
print("=" * 78)
print("RNG PATH PRESERVATION")
print("=" * 78)

for fragment in required:

    assert fragment in source

    print(
        "PASS:",
        fragment,
    )


# ============================================================================
# No broken expression
# ============================================================================

assert (
    "np.random.seed(seed)"
    not in source
)

print()
print(
    "Broken 64-bit legacy NumPy seed call: ABSENT"
)


# ============================================================================
# Final lock
# ============================================================================

lock = {
    "status": "PASS",

    "script_sha256":
        actual_sha,

    "previous_script_sha256":
        "5d822b1336eabcfcf83edf3b12d012817687ba8f6335cf670b6ebed4fcbf4da6",

    "runtime_fix":
        "np.random.seed(int(seed) & 0xffffffff)",

    "line_137":
        "PASS",

    "line_811":
        "PASS",

    "modern_rng_paths":
        "PASS",

    "compile":
        "PASS",

    "audit_method":
        "AST semantic validation; hexadecimal literal formatting ignored",

    "source_modified_by_this_cell":
        False,
}

LOCK_PATH = (
    SCRIPT.parent
    / "T2_1_FINAL_RUNTIME_SEED_FIX_LOCK.json"
)

LOCK_PATH.write_text(
    json.dumps(
        lock,
        indent=2,
    ),
    encoding="utf-8",
)


print()
print("=" * 78)
print("6ZK-M4 FINAL RESULT")
print("=" * 78)

print()
print("PASS")

print()
print("AUTHORITATIVE SCRIPT SHA:")
print(actual_sha)

print()
print("LOCK:")
print(LOCK_PATH)

print()
print(
    "The training script was NOT modified by this audit."
)

In [ ]:
# ============================================================================
# CELL 6ZK-N — EXACT EXECUTABLE IDENTITY + DEV RUN
# ============================================================================
#
# Purpose:
#   Prove that the subprocess executes the exact corrected script.
#
# IMPORTANT:
#   No source modification.
#   No production model.
#   DEV only.
# ============================================================================

from pathlib import Path
import hashlib
import json
import os
import shutil
import subprocess
import sys
import time


SCRIPT = Path(
    "/kaggle/working/"
    "T2_1_ConvNeXt_training_code_freeze/"
    "train_convnext_tiny_t2_1.py"
)

PROJECT = Path(
    "/kaggle/input/notebooks/"
    "tejasseshadriiiii/"
    "isic-kaggle-analysis/"
    "isic_final_project"
)

CACHE = Path(
    "/kaggle/working/"
    "T2_1_PRODUCTION_CACHE"
)

DEV_ROOT = Path(
    "/kaggle/working/"
    "T2_1_ConvNeXt_DEV_SMOKE_TEST"
)

DEV_OUT = DEV_ROOT / "rho_0.0_rep_1"

EXPECTED_SHA = (
    "6c0407330f547ff9b67ba8dbcf4b849b3379e0303f97eebd8ef722482d14c36f"
)


def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b"",
        ):
            h.update(chunk)

    return h.hexdigest()


# ============================================================================
# EXACT FILE IDENTITY — BEFORE SUBPROCESS
# ============================================================================

assert SCRIPT.exists()
assert PROJECT.exists()
assert CACHE.exists()

sha_before = sha256_file(SCRIPT)

source_before = SCRIPT.read_text(
    encoding="utf-8"
)

lines_before = source_before.splitlines()

assert sha_before == EXPECTED_SHA

assert len(lines_before) >= 137

line137_before = lines_before[136]

print("=" * 78)
print("T2.1 EXACT EXECUTABLE IDENTITY GATE")
print("=" * 78)

print()
print("EXECUTABLE:")
print(SCRIPT)

print()
print("SHA256:")
print(sha_before)

print()
print("LINE 137:")
print(line137_before)

assert (
    "np.random.seed(int(seed)" in line137_before
)

print()
print("PRE-LAUNCH FILE IDENTITY: PASS")


# ============================================================================
# CLEAN DEV OUTPUT
# ============================================================================

if DEV_ROOT.exists():
    shutil.rmtree(DEV_ROOT)

DEV_OUT.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================================
# SUBPROCESS ENVIRONMENT
# ============================================================================

env = os.environ.copy()

env["CUDA_VISIBLE_DEVICES"] = "0"
env["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"


# ============================================================================
# SUBPROCESS SELF-IDENTITY CHECK
#
# This executes a tiny Python command BEFORE the training subprocess.
# It reads the exact same file path from disk.
# ============================================================================

identity_code = r'''
from pathlib import Path
import hashlib
import sys

p = Path(sys.argv[1])

h = hashlib.sha256()

with open(p, "rb") as f:
    for chunk in iter(lambda: f.read(1024 * 1024), b""):
        h.update(chunk)

source = p.read_text(encoding="utf-8")
line137 = source.splitlines()[136]

print("SUBPROCESS FILE:", p)
print("SUBPROCESS SHA256:", h.hexdigest())
print("SUBPROCESS LINE 137:", line137)
'''

identity_cmd = [
    sys.executable,
    "-c",
    identity_code,
    str(SCRIPT),
]

identity_result = subprocess.run(
    identity_cmd,
    env=env,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

print()
print("=" * 78)
print("SUBPROCESS SELF-IDENTITY")
print("=" * 78)

print()
print(identity_result.stdout)

assert identity_result.returncode == 0

assert EXPECTED_SHA in identity_result.stdout

assert (
    "np.random.seed(int(seed)" in
    identity_result.stdout
)

print()
print("SUBPROCESS FILE IDENTITY: PASS")


# ============================================================================
# DEV TRAINING COMMAND
# ============================================================================

cmd = [
    sys.executable,
    str(SCRIPT),

    "--project",
    str(PROJECT),

    "--cache-dir",
    str(CACHE),

    "--rho",
    "0.0",

    "--rep",
    "1",

    "--mode",
    "dev",

    "--gap",
    "mean",

    "--out-dir",
    str(DEV_OUT),

    "--num-workers",
    "2",
]


print()
print("=" * 78)
print("DEV TRAINING COMMAND")
print("=" * 78)

print()

print(
    " ".join(
        repr(x)
        for x in cmd
    )
)

print()
print("CUDA_VISIBLE_DEVICES:")
print(env["CUDA_VISIBLE_DEVICES"])

print()
print("CUBLAS_WORKSPACE_CONFIG:")
print(env["CUBLAS_WORKSPACE_CONFIG"])


# ============================================================================
# EXECUTE
# ============================================================================

t0 = time.time()

result = subprocess.run(
    cmd,
    env=env,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

elapsed = time.time() - t0

print()
print("=" * 78)
print("DEV TRAINING OUTPUT")
print("=" * 78)

print()
print(result.stdout)

print()
print("RETURN CODE:")
print(result.returncode)

print()
print("ELAPSED:")
print(f"{elapsed:.2f} seconds")


# ============================================================================
# DEV MUST COMPLETE
# ============================================================================

assert result.returncode == 0, (
    "DEV training failed. Complete subprocess output is above."
)

print()
print("DEV TRAINING PROCESS: PASS")


# ============================================================================
# TRAINING RECORD
# ============================================================================

records = sorted(
    DEV_OUT.rglob(
        "training_record.json"
    )
)

print()
print("=" * 78)
print("TRAINING RECORD AUDIT")
print("=" * 78)

print()
print("training_record.json files:")
print(len(records))

assert len(records) == 1

record_path = records[0]

record = json.loads(
    record_path.read_text(
        encoding="utf-8"
    )
)

print()
print(
    json.dumps(
        record,
        indent=2,
        sort_keys=True,
    )
)


# ============================================================================
# SCRIPT PROVENANCE
# ============================================================================

record_script_sha = record.get(
    "script_sha256"
)

print()
print("=" * 78)
print("TRAINING RECORD SCRIPT PROVENANCE")
print("=" * 78)

print()
print("Expected:")
print(EXPECTED_SHA)

print()
print("Recorded:")
print(record_script_sha)

assert record_script_sha == EXPECTED_SHA

print()
print("SCRIPT PROVENANCE: PASS")


# ============================================================================
# SAVE GATE AUDIT
# ============================================================================

audit = {

    "status":
        "PASS",

    "script":
        str(SCRIPT),

    "script_sha256":
        EXPECTED_SHA,

    "line_137_verified":
        line137_before,

    "project":
        str(PROJECT),

    "cache":
        str(CACHE),

    "mode":
        "dev",

    "rho":
        0.0,

    "replicate":
        1,

    "gap":
        "mean",

    "cuda_visible_devices":
        "0",

    "cublas_workspace_config":
        ":4096:8",

    "return_code":
        result.returncode,

    "elapsed_seconds":
        elapsed,

    "training_record":
        str(record_path),

    "training_record_script_sha256":
        record_script_sha,

    "production_inclusion":
        False,
}

AUDIT_PATH = (
    DEV_ROOT
    / "T2_1_EXACT_EXECUTABLE_DEV_GATE_AUDIT.json"
)

AUDIT_PATH.write_text(
    json.dumps(
        audit,
        indent=2,
    ),
    encoding="utf-8",
)


print()
print("=" * 78)
print("6ZK-N COMPLETE")
print("=" * 78)

print()
print("T2.1 DEV EXECUTION GATE: PASS")

print()
print("AUTHORITATIVE SCRIPT SHA:")
print(EXPECTED_SHA)

print()
print("AUDIT:")
print(AUDIT_PATH)

print()
print(
    "No production model was created by this gate."
)

In [ ]:
# ============================================================================
# CELL 6ZK-O — DATASET TYPE-MISMATCH ROOT-CAUSE AUDIT
# ============================================================================
#
# READ-ONLY.
# Do NOT modify the frozen T2.1 script.
# ============================================================================

from pathlib import Path
import hashlib


SCRIPT = Path(
    "/kaggle/working/"
    "T2_1_ConvNeXt_training_code_freeze/"
    "train_convnext_tiny_t2_1.py"
)

EXPECTED_SHA = (
    "6c0407330f547ff9b67ba8dbcf4b849b3379e0303f97eebd8ef722482d14c36f"
)


def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b"",
        ):
            h.update(chunk)

    return h.hexdigest()


actual_sha = sha256_file(SCRIPT)

assert actual_sha == EXPECTED_SHA

lines = SCRIPT.read_text(
    encoding="utf-8"
).splitlines()


print("=" * 78)
print("T2.1 DATASET TYPE-MISMATCH ROOT-CAUSE AUDIT")
print("=" * 78)

print()
print("SCRIPT SHA:")
print(actual_sha)


# ============================================================================
# DATA CONVERSION FUNCTIONS
# ============================================================================

print()
print("=" * 78)
print("LINES 145–220 — RNG / AUGMENTATION / CONVERSION")
print("=" * 78)

for i in range(
    145,
    221,
):

    print(
        f"{i:5d}: {lines[i-1]}"
    )


# ============================================================================
# DATASET CLASS
# ============================================================================

print()
print("=" * 78)
print("LINES 220–325 — DATASET IMPLEMENTATION")
print("=" * 78)

for i in range(
    220,
    326,
):

    print(
        f"{i:5d}: {lines[i-1]}"
    )


# ============================================================================
# ALL torch.from_numpy() OCCURRENCES
# ============================================================================

print()
print("=" * 78)
print("ALL torch.from_numpy() OCCURRENCES")
print("=" * 78)

for i, line in enumerate(
    lines,
    start=1,
):

    if "torch.from_numpy" in line:

        print(
            f"{i:5d}: {line}"
        )


# ============================================================================
# ALL TO-CHW-FLOAT REFERENCES
# ============================================================================

print()
print("=" * 78)
print("ALL to_chw_float REFERENCES")
print("=" * 78)

for i, line in enumerate(
    lines,
    start=1,
):

    if "to_chw_float" in line:

        print(
            f"{i:5d}: {line}"
        )


# ============================================================================
# ALL AUGMENTATION REFERENCES
# ============================================================================

print()
print("=" * 78)
print("ALL augment_uint8 REFERENCES")
print("=" * 78)

for i, line in enumerate(
    lines,
    start=1,
):

    if "augment_uint8" in line:

        print(
            f"{i:5d}: {line}"
        )


print()
print("=" * 78)
print("AUDIT COMPLETE")
print("=" * 78)

print()
print(
    "No source modification was performed."
)

print()
print(
    "Current frozen SHA:",
    actual_sha,
)

In [ ]:
# ============================================================================
# CELL 6ZK-P — SURGICAL DATASET TYPE FIX
# ============================================================================
#
# ONLY CHANGE:
#
#   torch.from_numpy(to_chw_float(a))
#
# ->
#
#   to_chw_float(a)
#
# Everything else must remain byte-for-byte unchanged.
# ============================================================================

from pathlib import Path
import hashlib
import json
import py_compile


SCRIPT = Path(
    "/kaggle/working/"
    "T2_1_ConvNeXt_training_code_freeze/"
    "train_convnext_tiny_t2_1.py"
)

OLD_SHA = (
    "6c0407330f547ff9b67ba8dbcf4b849b3379e0303f97eebd8ef722482d14c36f"
)


def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b"",
        ):
            h.update(chunk)

    return h.hexdigest()


# ============================================================================
# VERIFY STARTING STATE
# ============================================================================

before_sha = sha256_file(SCRIPT)

assert before_sha == OLD_SHA, (
    f"Unexpected starting SHA: {before_sha}"
)

source_before = SCRIPT.read_text(
    encoding="utf-8"
)


# ============================================================================
# EXACT TARGET BLOCK
# ============================================================================

old_block = """        return (
            torch.from_numpy(
                to_chw_float(a)
            ),
            int(self.labels[i]),
        )"""

new_block = """        return (
            to_chw_float(a),
            int(self.labels[i]),
        )"""


assert source_before.count(old_block) == 1, (
    "Expected exactly one occurrence of the target block."
)

assert source_before.count(new_block) == 0, (
    "Corrected block already exists."
)


# ============================================================================
# ONE SURGICAL REPLACEMENT
# ============================================================================

source_after = source_before.replace(
    old_block,
    new_block,
    1,
)

SCRIPT.write_text(
    source_after,
    encoding="utf-8",
)


# ============================================================================
# VERIFY ONLY INTENDED CHANGE
# ============================================================================

before_lines = source_before.splitlines()
after_lines = source_after.splitlines()

assert len(before_lines) == len(after_lines)

changed = []

for i, (a, b) in enumerate(
    zip(
        before_lines,
        after_lines,
    ),
    start=1,
):

    if a != b:
        changed.append(
            (
                i,
                a,
                b,
            )
        )


# The removal of the outer torch.from_numpy changes
# exactly two source lines because the wrapper spans lines.
assert len(changed) == 2

print("=" * 78)
print("T2.1 DATASET TYPE FIX")
print("=" * 78)

print()
print("OLD SHA:")
print(before_sha)

print()
print("SOURCE CHANGES:")

for lineno, old, new in changed:

    print()
    print(f"LINE {lineno}")
    print("OLD:")
    print(repr(old))
    print("NEW:")
    print(repr(new))


# ============================================================================
# COMPILE
# ============================================================================

py_compile.compile(
    str(SCRIPT),
    doraise=True,
)

print()
print("=" * 78)
print("COMPILE")
print("=" * 78)

print()
print("PASS")


# ============================================================================
# VERIFY THE CORRECTED __getitem__ BLOCK
# ============================================================================

source_check = SCRIPT.read_text(
    encoding="utf-8"
)

assert (
    "torch.from_numpy(\n"
    "                to_chw_float(a)"
    not in source_check
)

assert (
    "return (\n"
    "            to_chw_float(a),\n"
    "            int(self.labels[i]),"
    in source_check
)

# Ensure the actual conversion still exists inside to_chw_float().
assert (
    "x = torch.from_numpy("
    in source_check
)

print()
print("=" * 78)
print("TYPE-CONVERSION SEMANTIC CHECK")
print("=" * 78)

print()
print("to_chw_float() creates Tensor: PASS")
print("__getitem__() no longer wraps Tensor with torch.from_numpy(): PASS")


# ============================================================================
# VERIFY CRITICAL RNG FIX REMAINS
# ============================================================================

assert (
    "np.random.seed(int(seed) & 0xffffffff)"
    in source_check
)

assert (
    "np.random.seed(seed)"
    not in source_check
)

print()
print("64-bit seed compatibility fix preserved: PASS")


# ============================================================================
# NEW SHA
# ============================================================================

new_sha = sha256_file(SCRIPT)

assert new_sha != OLD_SHA

print()
print("=" * 78)
print("NEW FROZEN SCRIPT SHA256")
print("=" * 78)

print()
print(new_sha)


# ============================================================================
# SAVE LOCK
# ============================================================================

lock = {
    "status":
        "FROZEN_AFTER_DATASET_TYPE_RUNTIME_FIX",

    "previous_sha256":
        OLD_SHA,

    "current_sha256":
        new_sha,

    "fix":
        "Removed outer torch.from_numpy() around "
        "to_chw_float(a) because to_chw_float() already "
        "returns a torch.Tensor.",

    "source_change_count":
        len(changed),

    "compile":
        True,

    "seed_fix_preserved":
        True,

    "production_training":
        False,
}

LOCK_PATH = (
    SCRIPT.parent
    / "T2_1_TRAINING_CODE_DATASET_FIX_HASH.json"
)

LOCK_PATH.write_text(
    json.dumps(
        lock,
        indent=2,
    ),
    encoding="utf-8",
)

print()
print("LOCK:")
print(LOCK_PATH)

print()
print("=" * 78)
print("6ZK-P COMPLETE")
print("=" * 78)

print()
print("The script has been changed exactly once for this runtime defect.")
print("No production model has been trained.")

In [ ]:
# ============================================================================
# CELL 6ZK-P2 — SURGICAL DATASET TYPE FIX
# ============================================================================
#
# EXACT intended change:
#
# OLD:
#     return (
#         torch.from_numpy(
#             to_chw_float(a)
#         ),
#         int(self.labels[i]),
#     )
#
# NEW:
#     return (
#         to_chw_float(a),
#         int(self.labels[i]),
#     )
#
# No other intended source change.
# ============================================================================

from pathlib import Path
import hashlib
import json
import py_compile


SCRIPT = Path(
    "/kaggle/working/"
    "T2_1_ConvNeXt_training_code_freeze/"
    "train_convnext_tiny_t2_1.py"
)

OLD_SHA = (
    "6c0407330f547ff9b67ba8dbcf4b849b3379e0303f97eebd8ef722482d14c36f"
)


def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b"",
        ):
            h.update(chunk)

    return h.hexdigest()


# ============================================================================
# VERIFY STARTING STATE
# ============================================================================

assert SCRIPT.exists()

before_sha = sha256_file(SCRIPT)

assert before_sha == OLD_SHA, (
    f"Unexpected starting SHA: {before_sha}"
)

source_before = SCRIPT.read_text(
    encoding="utf-8"
)


# ============================================================================
# EXACT BLOCK
# ============================================================================

old_block = """        return (
            torch.from_numpy(
                to_chw_float(a)
            ),
            int(self.labels[i]),
        )"""

new_block = """        return (
            to_chw_float(a),
            int(self.labels[i]),
        )"""


assert source_before.count(old_block) == 1, (
    "Expected exactly one old __getitem__ block."
)

assert source_before.count(new_block) == 0, (
    "Corrected block already exists."
)


# ============================================================================
# SURGICAL REPLACEMENT
# ============================================================================

source_after = source_before.replace(
    old_block,
    new_block,
    1,
)

assert source_after != source_before

# Confirm the OLD block disappeared.
assert source_after.count(old_block) == 0

# Confirm the NEW block exists exactly once.
assert source_after.count(new_block) == 1


# ============================================================================
# WRITE
# ============================================================================

SCRIPT.write_text(
    source_after,
    encoding="utf-8",
)


# ============================================================================
# COMPILE
# ============================================================================

py_compile.compile(
    str(SCRIPT),
    doraise=True,
)


# ============================================================================
# VERIFY RESULT
# ============================================================================

actual_source = SCRIPT.read_text(
    encoding="utf-8"
)

assert (
    actual_source.count(old_block) == 0
)

assert (
    actual_source.count(new_block) == 1
)

assert (
    "x = torch.from_numpy("
    in actual_source
)

assert (
    "np.random.seed(int(seed) & 0xffffffff)"
    in actual_source
)

assert (
    "np.random.seed(seed)"
    not in actual_source
)


# ============================================================================
# SHOW EXACT __getitem__ REGION
# ============================================================================

lines = actual_source.splitlines()

print("=" * 78)
print("T2.1 DATASET TYPE FIX")
print("=" * 78)

print()
print("PREVIOUS SHA:")
print(before_sha)

print()
print("UPDATED __getitem__ REGION:")

for i in range(
    294,
    min(315, len(lines) + 1),
):

    print(
        f"{i:5d}: {lines[i - 1]}"
    )


# ============================================================================
# NEW SHA
# ============================================================================

new_sha = sha256_file(SCRIPT)

assert new_sha != OLD_SHA

print()
print("=" * 78)
print("COMPILE")
print("=" * 78)

print()
print("PASS")

print()
print("=" * 78)
print("NEW SCRIPT SHA256")
print("=" * 78)

print()
print(new_sha)


# ============================================================================
# SAVE LOCK
# ============================================================================

lock = {
    "status":
        "PASS",

    "previous_sha256":
        OLD_SHA,

    "current_sha256":
        new_sha,

    "runtime_fix":
        "Removed outer torch.from_numpy() from "
        "CacheDataset.__getitem__() because "
        "to_chw_float() already returns torch.Tensor.",

    "old_block_present":
        False,

    "new_block_count":
        1,

    "compile":
        True,

    "seed_fix_preserved":
        True,

    "production_training":
        False,
}

LOCK_PATH = (
    SCRIPT.parent
    / "T2_1_TRAINING_CODE_DATASET_FIX_HASH.json"
)

LOCK_PATH.write_text(
    json.dumps(
        lock,
        indent=2,
    ),
    encoding="utf-8",
)

print()
print("LOCK:")
print(LOCK_PATH)

print()
print("=" * 78)
print("6ZK-P2 COMPLETE")
print("=" * 78)

print()
print("PASS")
print()
print("No production model was trained.")

In [ ]:
# ============================================================================
# CELL 6ZK-P3 — RECOVER CURRENT SCRIPT AFTER SUCCESSFUL SURGICAL PATCH
# ============================================================================
#
# READ-ONLY.
# NO SOURCE MODIFICATION.
# ============================================================================

from pathlib import Path
import hashlib
import py_compile
import json


SCRIPT = Path(
    "/kaggle/working/"
    "T2_1_ConvNeXt_training_code_freeze/"
    "train_convnext_tiny_t2_1.py"
)

EXPECTED_CURRENT_SHA = (
    "d6b97a547e96d4f14cb97b0bfa69c5219c798e0f5371a3de0d4c6bdaba93b379"
)


def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b"",
        ):
            h.update(chunk)

    return h.hexdigest()


# ============================================================================
# IDENTITY
# ============================================================================

assert SCRIPT.exists()

actual_sha = sha256_file(SCRIPT)

print("=" * 78)
print("T2.1 CURRENT SCRIPT RECOVERY")
print("=" * 78)

print()
print("CURRENT SHA:")
print(actual_sha)

assert actual_sha == EXPECTED_CURRENT_SHA


# ============================================================================
# COMPILE
# ============================================================================

py_compile.compile(
    str(SCRIPT),
    doraise=True,
)

print()
print("COMPILE: PASS")


source = SCRIPT.read_text(
    encoding="utf-8"
)

lines = source.splitlines()


# ============================================================================
# CHECK 1 — NUMPY SEED FIX
# ============================================================================

print()
print("=" * 78)
print("FIX 1 — LEGACY NUMPY SEED")
print("=" * 78)

seed_fix = (
    "np.random.seed(int(seed) & 0xffffffff)"
)

print()
print("Corrected expression present:")
print(seed_fix in source)

print()
print("Old broken expression present:")
print("np.random.seed(seed)" in source)

assert seed_fix in source
assert "np.random.seed(seed)" not in source

print()
print("SEED FIX: PASS")


# ============================================================================
# CHECK 2 — DATASET TYPE FIX
# ============================================================================

print()
print("=" * 78)
print("FIX 2 — DATASET TYPE CONVERSION")
print("=" * 78)

correct_block = """        return (
            to_chw_float(a),
            int(self.labels[i]),
        )"""

broken_block = """        return (
            torch.from_numpy(
                to_chw_float(a)
            ),
            int(self.labels[i]),
        )"""

print()
print("Correct __getitem__ block count:")
print(source.count(correct_block))

print()
print("Broken __getitem__ block count:")
print(source.count(broken_block))

assert source.count(correct_block) == 1
assert source.count(broken_block) == 0

# to_chw_float must still create the tensor itself.
assert "x = torch.from_numpy(" in source

print()
print("DATASET TYPE FIX: PASS")


# ============================================================================
# SHOW EXACT RELEVANT CODE
# ============================================================================

print()
print("=" * 78)
print("CURRENT RELEVANT SOURCE")
print("=" * 78)

for i in range(
    130,
    142,
):

    print(
        f"{i:5d}: {lines[i - 1]}"
    )

print()

for i in range(
    294,
    min(318, len(lines) + 1),
):

    print(
        f"{i:5d}: {lines[i - 1]}"
    )


# ============================================================================
# CHECK 3 — AUGMENTATION PATH UNCHANGED
# ============================================================================

required_aug = [
    "np.random.SeedSequence(",
    "np.random.default_rng(ss)",
    "def augment_uint8(arr, rng):",
    "def to_chw_float(a):",
]

print()
print("=" * 78)
print("AUGMENTATION PATH")
print("=" * 78)

for fragment in required_aug:

    present = fragment in source

    print(
        "PASS:" if present else "FAIL:",
        fragment,
    )

    assert present


# ============================================================================
# CHECK 4 — WORKER SEED
# ============================================================================

print()
print("=" * 78)
print("WORKER SEED")
print("=" * 78)

worker_hits = [
    line
    for line in lines
    if "seed_workers" in line
]

for line in worker_hits:

    print(line)

assert worker_hits


# ============================================================================
# FINAL STATUS
# ============================================================================

print()
print("=" * 78)
print("6ZK-P3 RESULT")
print("=" * 78)

print()
print("PASS")

print()
print("AUTHORITATIVE CURRENT SHA:")
print(actual_sha)

print()
print(
    "Both runtime fixes are present and the script compiles."
)

print()
print(
    "NO SOURCE MODIFICATION WAS PERFORMED BY THIS CELL."
)

In [ ]:
# ============================================================================
# CELL 6ZK-Q — T2.1 CONVNEXT DEV EXECUTION GATE
# ============================================================================
#
# Purpose:
#   Execute one complete DEV training run using the CURRENT frozen executable.
#
# This cell:
#   1. Verifies exact executable SHA.
#   2. Verifies --help.
#   3. Runs rho=0.0 / replicate=1 in DEV mode.
#   4. Uses one visible GPU.
#   5. Verifies the emitted training record.
#
# NO SOURCE MODIFICATION.
# ============================================================================

from pathlib import Path
import hashlib
import json
import os
import subprocess
import sys


SCRIPT = Path(
    "/kaggle/working/"
    "T2_1_ConvNeXt_training_code_freeze/"
    "train_convnext_tiny_t2_1.py"
)

EXPECTED_SHA = (
    "d6b97a547e96d4f14cb97b0bfa69c5219c798e0f5371a3de0d4c6bdaba93b379"
)

PROJECT_ROOT = Path(
    "/kaggle/input/notebooks/"
    "tejasseshadriiiii/"
    "isic-kaggle-analysis/"
    "isic_final_project"
)

CACHE_DIR = Path(
    "/kaggle/working/T2_1_PRODUCTION_CACHE"
)

DEV_OUT = Path(
    "/kaggle/working/T2_1_DEV_RUN"
)

DEV_OUT.mkdir(
    parents=True,
    exist_ok=True,
)


def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b"",
        ):
            h.update(chunk)

    return h.hexdigest()


# ============================================================================
# 1. EXACT EXECUTABLE IDENTITY
# ============================================================================

print("=" * 78)
print("T2.1 CONVNEXT DEV EXECUTION GATE")
print("=" * 78)

assert SCRIPT.exists(), SCRIPT
assert PROJECT_ROOT.exists(), PROJECT_ROOT
assert CACHE_DIR.exists(), CACHE_DIR

actual_sha = sha256_file(SCRIPT)

print()
print("Executable:")
print(SCRIPT)

print()
print("Expected SHA:")
print(EXPECTED_SHA)

print()
print("Actual SHA:")
print(actual_sha)

assert actual_sha == EXPECTED_SHA

print()
print("EXECUTABLE IDENTITY: PASS")


# ============================================================================
# 2. ENVIRONMENT VARIABLES
# ============================================================================

env = os.environ.copy()

# Exactly one GPU visible to this training subprocess.
env["CUDA_VISIBLE_DEVICES"] = "0"

# Deterministic cuBLAS configuration must exist BEFORE torch import.
env["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"


# ============================================================================
# 3. --HELP GATE
# ============================================================================

help_cmd = [
    sys.executable,
    str(SCRIPT),
    "--help",
]

print()
print("=" * 78)
print("ARGUMENT PARSER GATE")
print("=" * 78)

help_run = subprocess.run(
    help_cmd,
    env=env,
    capture_output=True,
    text=True,
)

print()
print("Return code:")
print(help_run.returncode)

print()
print("STDOUT:")
print(help_run.stdout[:4000])

print()
print("STDERR:")
print(help_run.stderr[:4000])

assert help_run.returncode == 0

print()
print("ARGUMENT PARSER: PASS")


# ============================================================================
# 4. CLEAN PREVIOUS DEV OUTPUT
# ============================================================================

if DEV_OUT.exists():

    for p in DEV_OUT.iterdir():

        if p.is_file() or p.is_symlink():
            p.unlink()

        elif p.is_dir():
            import shutil
            shutil.rmtree(p)


# ============================================================================
# 5. DEV TRAINING
# ============================================================================

cmd = [
    sys.executable,
    str(SCRIPT),

    "--project-root",
    str(PROJECT_ROOT),

    "--cache-dir",
    str(CACHE_DIR),

    "--rho",
    "0.0",

    "--rep",
    "1",

    "--mode",
    "dev",

    "--gap",
    "mean",

    "--out-dir",
    str(DEV_OUT),

    "--num-workers",
    "2",
]

print()
print("=" * 78)
print("STARTING DEV TRAINING")
print("=" * 78)

print()
print("Command:")
print(" ".join(cmd))

print()
print("CUDA_VISIBLE_DEVICES:")
print(env["CUDA_VISIBLE_DEVICES"])

print()
print("CUBLAS_WORKSPACE_CONFIG:")
print(env["CUBLAS_WORKSPACE_CONFIG"])

dev_run = subprocess.run(
    cmd,
    env=env,
    capture_output=True,
    text=True,
)

print()
print("=" * 78)
print("DEV TRAINING RETURN")
print("=" * 78)

print()
print("Return code:")
print(dev_run.returncode)

print()
print("STDOUT:")
print(dev_run.stdout[-12000:])

print()
print("STDERR:")
print(dev_run.stderr[-12000:])


assert dev_run.returncode == 0, (
    "DEV TRAINING FAILED. "
    "Inspect the stdout/stderr above before changing anything."
)


# ============================================================================
# 6. FIND TRAINING RECORD
# ============================================================================

print()
print("=" * 78)
print("TRAINING RECORD DISCOVERY")
print("=" * 78)

records = sorted(
    DEV_OUT.rglob("training_record*.json")
)

if not records:
    records = sorted(
        DEV_OUT.rglob("*.json")
    )

print()
print("JSON files found:")

for p in records:
    print(" ", p)

assert records, (
    "DEV training completed but no JSON training record was found."
)


# Prefer an explicit training record.
training_records = [
    p
    for p in records
    if "training_record" in p.name.lower()
]

record_path = (
    training_records[0]
    if training_records
    else records[0]
)

print()
print("Selected record:")
print(record_path)


with open(
    record_path,
    "r",
    encoding="utf-8",
) as f:

    record = json.load(f)


# ============================================================================
# 7. PROVENANCE CHECKS
# ============================================================================

print()
print("=" * 78)
print("TRAINING RECORD PROVENANCE")
print("=" * 78)

print()
print("Record keys:")
print(sorted(record.keys()))


# Find script hash robustly without assuming exact nesting.
def find_values(obj, target_keys):

    found = []

    if isinstance(obj, dict):

        for k, v in obj.items():

            if k in target_keys:
                found.append((k, v))

            found.extend(
                find_values(v, target_keys)
            )

    elif isinstance(obj, list):

        for item in obj:

            found.extend(
                find_values(item, target_keys)
            )

    return found


script_values = find_values(
    record,
    {
        "script_sha256",
        "training_script_sha256",
    },
)

print()
print("Script SHA fields:")

for k, v in script_values:
    print(k, "=", v)

assert any(
    str(v) == EXPECTED_SHA
    for _, v in script_values
), (
    "Training record does not contain the expected executable SHA."
)


# ============================================================================
# 8. BASIC RECORD IDENTITY
# ============================================================================

print()
print("Record:")
print(
    json.dumps(
        record,
        indent=2,
        default=str,
    )[:12000]
)


# ============================================================================
# 9. FINAL
# ============================================================================

print()
print("=" * 78)
print("6ZK-Q RESULT")
print("=" * 78)

print()
print("PASS")

print()
print("The corrected ConvNeXt executable completed a DEV run.")

print()
print("Executable SHA:")
print(EXPECTED_SHA)

print()
print("DEV output:")
print(DEV_OUT)

print()
print("NEXT STEP:")
print(
    "Freeze this executable identity, then perform the T2.1 "
    "production 9-model training run."
)

In [ ]:
# ============================================================================
# CELL 6ZK-Q2 — T2.1 CONVNEXT DEV EXECUTION GATE
# ============================================================================
#
# Launcher-only correction:
#   --project-root  ->  --project
#
# Training executable is NOT modified.
# Expected executable SHA remains frozen.
# ============================================================================

from pathlib import Path
import hashlib
import json
import os
import subprocess
import sys
import shutil


SCRIPT = Path(
    "/kaggle/working/"
    "T2_1_ConvNeXt_training_code_freeze/"
    "train_convnext_tiny_t2_1.py"
)

EXPECTED_SHA = (
    "d6b97a547e96d4f14cb97b0bfa69c5219c798e0f5371a3de0d4c6bdaba93b379"
)

PROJECT = Path(
    "/kaggle/input/notebooks/"
    "tejasseshadriiiii/"
    "isic-kaggle-analysis/"
    "isic_final_project"
)

CACHE_DIR = Path(
    "/kaggle/working/T2_1_PRODUCTION_CACHE"
)

DEV_OUT = Path(
    "/kaggle/working/T2_1_DEV_RUN"
)


def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b"",
        ):
            h.update(chunk)

    return h.hexdigest()


# ============================================================================
# 1. PATH + EXACT SCRIPT IDENTITY
# ============================================================================

print("=" * 78)
print("T2.1 CONVNEXT DEV EXECUTION GATE — CORRECTED LAUNCHER")
print("=" * 78)

assert SCRIPT.exists()
assert PROJECT.exists()
assert CACHE_DIR.exists()

actual_sha = sha256_file(SCRIPT)

print()
print("Executable:")
print(SCRIPT)

print()
print("Expected SHA:")
print(EXPECTED_SHA)

print()
print("Actual SHA:")
print(actual_sha)

assert actual_sha == EXPECTED_SHA

print()
print("EXECUTABLE IDENTITY: PASS")


# ============================================================================
# 2. SINGLE-GPU + DETERMINISTIC ENVIRONMENT
# ============================================================================

env = os.environ.copy()

env["CUDA_VISIBLE_DEVICES"] = "0"
env["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

print()
print("CUDA_VISIBLE_DEVICES =", env["CUDA_VISIBLE_DEVICES"])
print("CUBLAS_WORKSPACE_CONFIG =", env["CUBLAS_WORKSPACE_CONFIG"])


# ============================================================================
# 3. CLEAN DEV OUTPUT
# ============================================================================

if DEV_OUT.exists():
    shutil.rmtree(DEV_OUT)

DEV_OUT.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================================
# 4. ACTUAL CLI CONTRACT
# ============================================================================

cmd = [
    sys.executable,
    str(SCRIPT),

    "--project",
    str(PROJECT),

    "--cache-dir",
    str(CACHE_DIR),

    "--out-dir",
    str(DEV_OUT),

    "--rho",
    "0.0",

    "--rep",
    "1",

    "--mode",
    "dev",

    "--gap",
    "mean",

    "--num-workers",
    "2",
]


print()
print("=" * 78)
print("DEV COMMAND")
print("=" * 78)

print()
print(" ".join(cmd))


# ============================================================================
# 5. EXECUTE
# ============================================================================

dev_run = subprocess.run(
    cmd,
    env=env,
    capture_output=True,
    text=True,
)

print()
print("=" * 78)
print("DEV TRAINING RETURN")
print("=" * 78)

print()
print("Return code:")
print(dev_run.returncode)

print()
print("STDOUT:")
print(dev_run.stdout[-16000:])

print()
print("STDERR:")
print(dev_run.stderr[-16000:])


assert dev_run.returncode == 0, (
    "DEV TRAINING FAILED. "
    "The training subprocess returned a non-zero exit code."
)


# ============================================================================
# 6. DISCOVER OUTPUTS
# ============================================================================

print()
print("=" * 78)
print("DEV OUTPUT DISCOVERY")
print("=" * 78)

all_files = sorted(
    p for p in DEV_OUT.rglob("*")
    if p.is_file()
)

assert all_files, (
    "DEV process returned successfully but produced no files."
)

for p in all_files:
    print(
        p.relative_to(DEV_OUT)
    )


# ============================================================================
# 7. FIND TRAINING RECORD
# ============================================================================

json_files = sorted(
    DEV_OUT.rglob("*.json")
)

print()
print("JSON files:")

for p in json_files:
    print(
        p.relative_to(DEV_OUT)
    )

training_records = [
    p
    for p in json_files
    if "training_record" in p.name.lower()
]

assert training_records, (
    "No training_record JSON was produced."
)

record_path = training_records[0]

print()
print("Training record:")
print(record_path)


with open(
    record_path,
    "r",
    encoding="utf-8",
) as f:

    record = json.load(f)


# ============================================================================
# 8. PROVENANCE CHECK
# ============================================================================

def find_values(obj, keys):

    found = []

    if isinstance(obj, dict):

        for k, v in obj.items():

            if k in keys:
                found.append(
                    (k, v)
                )

            found.extend(
                find_values(v, keys)
            )

    elif isinstance(obj, list):

        for item in obj:

            found.extend(
                find_values(item, keys)
            )

    return found


script_fields = find_values(
    record,
    {
        "script_sha256",
        "training_script_sha256",
    },
)

print()
print("=" * 78)
print("TRAINING RECORD SCRIPT PROVENANCE")
print("=" * 78)

for k, v in script_fields:
    print(
        f"{k}: {v}"
    )

assert any(
    str(v) == EXPECTED_SHA
    for _, v in script_fields
), (
    "Training record does not contain the frozen executable SHA."
)


# ============================================================================
# 9. BASIC IDENTITY INFORMATION
# ============================================================================

print()
print("=" * 78)
print("TRAINING RECORD")
print("=" * 78)

print(
    json.dumps(
        record,
        indent=2,
        default=str,
    )[:16000]
)


# ============================================================================
# 10. FINAL
# ============================================================================

print()
print("=" * 78)
print("6ZK-Q2 RESULT")
print("=" * 78)

print()
print("PASS")

print()
print("The ConvNeXt DEV run completed successfully.")

print()
print("Frozen executable SHA:")
print(EXPECTED_SHA)

print()
print("DEV output:")
print(DEV_OUT)

print()
print("Next gate:")
print(
    "Freeze the executable + DEV evidence, then prepare "
    "the 9-model T2.1 production launcher."
)

In [ ]:
# ============================================================================
# CELL 6ZK-R — T2.1 DEV RHO/REPLICATE ARGUMENT FLOW AUDIT
# ============================================================================
#
# READ-ONLY.
# No source modification.
#
# Purpose:
#   Determine why:
#       CLI rho=0.0
#   produced:
#       training_record rho=0.6
#
# ============================================================================

from pathlib import Path
import re
import hashlib


SCRIPT = Path(
    "/kaggle/working/"
    "T2_1_ConvNeXt_training_code_freeze/"
    "train_convnext_tiny_t2_1.py"
)

EXPECTED_SHA = (
    "d6b97a547e96d4f14cb97b0bfa69c5219c798e0f5371a3de0d4c6bdaba93b379"
)


def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b"",
        ):
            h.update(chunk)

    return h.hexdigest()


# ============================================================================
# IDENTITY
# ============================================================================

actual_sha = sha256_file(SCRIPT)

print("=" * 78)
print("T2.1 RHO / REPLICATE FLOW AUDIT")
print("=" * 78)

print()
print("SHA:")
print(actual_sha)

assert actual_sha == EXPECTED_SHA


source = SCRIPT.read_text(
    encoding="utf-8"
)

lines = source.splitlines()


# ============================================================================
# FIND ARGPARSE DEFINITIONS
# ============================================================================

print()
print("=" * 78)
print("ARGUMENT DEFINITIONS")
print("=" * 78)

for i, line in enumerate(lines, start=1):

    if (
        "add_argument" in line
        and (
            '"--rho"' in line
            or "'--rho'" in line
            or '"--rep"' in line
            or "'--rep'" in line
        )
    ):

        print(
            f"{i:5d}: {line}"
        )


# ============================================================================
# FIND EVERY args.rho / args.rep USAGE
# ============================================================================

print()
print("=" * 78)
print("DIRECT CLI VALUE USAGE")
print("=" * 78)

rho_hits = []
rep_hits = []

for i, line in enumerate(lines, start=1):

    if "args.rho" in line:
        rho_hits.append(
            (i, line)
        )

    if "args.rep" in line:
        rep_hits.append(
            (i, line)
        )

print()
print("args.rho occurrences:")

for i, line in rho_hits:
    print(
        f"{i:5d}: {line}"
    )

print()
print("args.rep occurrences:")

for i, line in rep_hits:
    print(
        f"{i:5d}: {line}"
    )


# ============================================================================
# FIND RHO VARIABLES
# ============================================================================

print()
print("=" * 78)
print("RHO-RELATED SOURCE LINES")
print("=" * 78)

for i, line in enumerate(lines, start=1):

    stripped = line.strip()

    if (
        "rho" in stripped.lower()
        and not stripped.startswith("#")
    ):

        print(
            f"{i:5d}: {line}"
        )


# ============================================================================
# FIND RECORD CREATION
# ============================================================================

print()
print("=" * 78)
print("TRAINING RECORD CREATION")
print("=" * 78)

for i, line in enumerate(lines, start=1):

    if (
        "training_record" in line
        or '"rho":' in line
        or "'rho':" in line
    ):

        start = max(
            1,
            i - 4,
        )

        end = min(
            len(lines),
            i + 6,
        )

        print()
        print(
            f"--- around line {i} ---"
        )

        for j in range(start, end + 1):

            print(
                f"{j:5d}: {lines[j - 1]}"
            )


# ============================================================================
# FIND DEV MODE LOGIC
# ============================================================================

print()
print("=" * 78)
print("DEV MODE LOGIC")
print("=" * 78)

for i, line in enumerate(lines, start=1):

    if (
        "mode" in line.lower()
        and "dev" in line.lower()
    ):

        start = max(
            1,
            i - 5,
        )

        end = min(
            len(lines),
            i + 10,
        )

        print()
        print(
            f"--- around line {i} ---"
        )

        for j in range(start, end + 1):

            print(
                f"{j:5d}: {lines[j - 1]"
            )


# ============================================================================
# FINAL
# ============================================================================

print()
print("=" * 78)
print("6ZK-R COMPLETE")
print("=" * 78)

print()
print(
    "No source modification was performed."
)

print()
print(
    "The output above will tell us whether DEV mode intentionally "
    "overrides rho or whether --rho is being ignored/misrouted."
)

In [ ]:
# ============================================================================
# CELL 6ZK-R2 — SIMPLE RHO / REP FLOW DIAGNOSTIC
# ============================================================================
# READ-ONLY — does NOT modify the training script.
# ============================================================================

from pathlib import Path
import hashlib


SCRIPT = Path(
    "/kaggle/working/"
    "T2_1_ConvNeXt_training_code_freeze/"
    "train_convnext_tiny_t2_1.py"
)

EXPECTED_SHA = (
    "d6b97a547e96d4f14cb97b0bfa69c5219c798e0f5371a3de0d4c6bdaba93b379"
)


def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


source = SCRIPT.read_text(encoding="utf-8")
lines = source.splitlines()

actual_sha = sha256_file(SCRIPT)

print("=" * 78)
print("T2.1 RHO / REPLICATE DIAGNOSTIC")
print("=" * 78)

print()
print("SHA:", actual_sha)

assert actual_sha == EXPECTED_SHA

# --------------------------------------------------------------------------
# ARGUMENT DEFINITIONS
# --------------------------------------------------------------------------

print()
print("=" * 78)
print("1. ARGUMENT DEFINITIONS")
print("=" * 78)

for i, line in enumerate(lines, 1):
    if "--rho" in line or "--rep" in line:
        print(str(i).rjust(5) + ": " + line)

# --------------------------------------------------------------------------
# DIRECT args.rho / args.rep REFERENCES
# --------------------------------------------------------------------------

print()
print("=" * 78)
print("2. args.rho / args.rep REFERENCES")
print("=" * 78)

for i, line in enumerate(lines, 1):
    if "args.rho" in line or "args.rep" in line:
        print(str(i).rjust(5) + ": " + line)

# --------------------------------------------------------------------------
# RHO-RELATED LINES
# --------------------------------------------------------------------------

print()
print("=" * 78)
print("3. RHO-RELATED LINES")
print("=" * 78)

for i, line in enumerate(lines, 1):
    s = line.strip()

    if "rho" in s.lower() and not s.startswith("#"):
        print(str(i).rjust(5) + ": " + line)

# --------------------------------------------------------------------------
# DEV-RELATED LINES
# --------------------------------------------------------------------------

print()
print("=" * 78)
print("4. DEV-MODE LINES")
print("=" * 78)

for i, line in enumerate(lines, 1):
    s = line.strip()

    if "dev" in s.lower() and (
        "mode" in s.lower()
        or "dev_n_train" in s.lower()
        or "DEV" in s
    ):
        print(str(i).rjust(5) + ": " + line)

# --------------------------------------------------------------------------
# RECORD rho / replicate
# --------------------------------------------------------------------------

print()
print("=" * 78)
print("5. TRAINING-RECORD LINES")
print("=" * 78)

for i, line in enumerate(lines, 1):
    s = line.strip()

    if (
        '"rho"' in s
        or "'rho'" in s
        or '"replicate"' in s
        or "'replicate'" in s
    ):
        start = max(1, i - 3)
        end = min(len(lines), i + 3)

        print()
        print("--- around line " + str(i) + " ---")

        for j in range(start, end + 1):
            print(
                str(j).rjust(5)
                + ": "
                + lines[j - 1]
            )

print()
print("=" * 78)
print("6ZK-R2 COMPLETE")
print("=" * 78)

print()
print("NO SOURCE MODIFICATION.")
print("NO TRAINING EXECUTION.")

In [ ]:
# ============================================================================
# CELL 6ZK-R3 — INSPECT DEV RHO OVERRIDE CONTEXT
# ============================================================================
# READ-ONLY — absolutely no source modification.
# ============================================================================

from pathlib import Path
import hashlib


SCRIPT = Path(
    "/kaggle/working/"
    "T2_1_ConvNeXt_training_code_freeze/"
    "train_convnext_tiny_t2_1.py"
)

EXPECTED_SHA = (
    "d6b97a547e96d4f14cb97b0bfa69c5219c798e0f5371a3de0d4c6bdaba93b379"
)


def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


assert sha256_file(SCRIPT) == EXPECTED_SHA

lines = SCRIPT.read_text(
    encoding="utf-8"
).splitlines()


def show(a, b):
    print()
    print("=" * 78)
    print(f"LINES {a}–{b}")
    print("=" * 78)

    for i in range(a, b + 1):
        print(
            f"{i:5d}: {lines[i - 1]}"
        )


print("=" * 78)
print("T2.1 DEV RHO OVERRIDE CONTEXT")
print("=" * 78)

# Exact context containing rho = 0.6
show(535, 570)

# Main training/data-selection branch
show(475, 525)

# First DEV conditional
show(845, 875)

# Second DEV conditional
show(935, 965)

# Main mode dispatch
show(1205, 1230)

print()
print("=" * 78)
print("6ZK-R3 COMPLETE")
print("=" * 78)

print()
print("NO SOURCE MODIFICATION.")

In [ ]:
# ============================================================================
# CELL 6ZK-S — T2.1 EXECUTABLE + DEV EVIDENCE FREEZE
# ============================================================================
#
# No source modification.
# No GPU training.
#
# Freezes the evidence establishing that the T2.1 executable:
#   - has the expected SHA
#   - compiles
#   - passed the ConvNeXt deterministic dry run
#   - completed the DEV execution
#   - uses the frozen environment
#   - has intentional DEV rho override
#   - uses args.rho / args.rep in PRIMARY mode
# ============================================================================

from pathlib import Path
import hashlib
import json
import py_compile
from datetime import datetime, timezone


SCRIPT = Path(
    "/kaggle/working/"
    "T2_1_ConvNeXt_training_code_freeze/"
    "train_convnext_tiny_t2_1.py"
)

DEV_RECORD = Path(
    "/kaggle/working/"
    "T2_1_DEV_RUN/"
    "training_record.json"
)

DEV_LOG = Path(
    "/kaggle/working/"
    "T2_1_DEV_RUN/"
    "epoch_log.csv"
)

OUT = Path(
    "/kaggle/working/"
    "T2_1_ConvNeXt_training_code_freeze/"
    "T2_1_EXECUTION_AND_RUNTIME_FREEZE.json"
)

EXPECTED_SHA = (
    "d6b97a547e96d4f14cb97b0bfa69c5219c798e0f5371a3de0d4c6bdaba93b379"
)


def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b"",
        ):
            h.update(chunk)

    return h.hexdigest()


# ============================================================================
# BASIC VALIDATION
# ============================================================================

assert SCRIPT.exists()
assert DEV_RECORD.exists()
assert DEV_LOG.exists()

actual_sha = sha256_file(SCRIPT)

assert actual_sha == EXPECTED_SHA

py_compile.compile(
    str(SCRIPT),
    doraise=True,
)


# ============================================================================
# LOAD DEV RECORD
# ============================================================================

with open(
    DEV_RECORD,
    "r",
    encoding="utf-8",
) as f:

    dev = json.load(f)


assert dev["status"] == "COMPLETED"
assert dev["mode"] == "dev"
assert dev["architecture"] == "ConvNeXt-Tiny"
assert dev["weights"] == (
    "ConvNeXt_Tiny_Weights.IMAGENET1K_V1"
)

assert dev["script_sha256"] == EXPECTED_SHA

assert dev["env"]["torch"] == "2.11.0+cu128"
assert dev["env"]["torchvision"] == "0.26.0+cu128"
assert dev["env"]["cuda"] == "12.8"
assert dev["env"]["gpu"] == "Tesla T4"

assert dev["env"]["deterministic_algorithms"] is True
assert dev["env"]["cudnn_deterministic"] is True
assert dev["env"]["cudnn_benchmark"] is False
assert dev["env"]["tf32_matmul"] is False
assert dev["env"]["tf32_cudnn"] is False
assert dev["env"]["cublas_workspace_config"] == ":4096:8"

assert dev["xai"] is False
assert dev["endpoint"] == "Delta-z"


# ============================================================================
# SOURCE-LEVEL PRIMARY/DEV ROUTING LOCK
# ============================================================================

source = SCRIPT.read_text(
    encoding="utf-8"
)

lines = source.splitlines()


primary_block = "\n".join(
    lines[478:516]
)

dev_override_block = "\n".join(
    lines[517:555]
)

assert "rho = args.rho" in primary_block
assert "rep = args.rep" in primary_block

assert "rho = 0.6" in dev_override_block
assert "rep = 1" in dev_override_block


# ============================================================================
# BUILD FREEZE RECORD
# ============================================================================

freeze = {
    "freeze_timestamp_utc":
        datetime.now(timezone.utc).isoformat(),

    "status":
        "FROZEN_FOR_PRODUCTION_PLANNING",

    "script": {
        "path": str(SCRIPT),
        "sha256": actual_sha,
        "compile": "PASS",
    },

    "architecture": {
        "name": "ConvNeXt-Tiny",
        "weights":
            "ConvNeXt_Tiny_Weights.IMAGENET1K_V1",
        "endpoint": "Delta-z",
        "xai": False,
    },

    "primary_routing": {
        "rho_source":
            "args.rho",
        "replicate_source":
            "args.rep",
        "seed_lookup":
            "TRAINING_SEED_TABLE.csv",
        "status":
            "VERIFIED_FROM_SOURCE",
    },

    "dev_routing": {
        "rho": 0.6,
        "replicate": 1,
        "status":
            "INTENTIONAL_DEV_OVERRIDE",
    },

    "dev_execution": {
        "status": dev["status"],
        "epochs": dev["epochs"],
        "runtime_seconds": dev["seconds"],
        "runtime_minutes":
            dev["seconds"] / 60.0,
        "training_images":
            dev["training_images"],
        "hair_images":
            dev["hair_images_in_training_set"],
        "script_sha256":
            dev["script_sha256"],
        "final_state_sha256":
            dev.get(
                "final_state_sha256"
            ),
    },

    "environment": dev["env"],

    "production_design": {
        "models": 9,
        "rho": [0.0, 0.5, 1.0],
        "replicates_per_rho": 3,
        "epochs": 15,
        "primary_checkpoint":
            "final_epoch15.pt",
        "sensitivity_checkpoint":
            "best_eval_ba.pt",
        "xai": False,
    },

    "important_runtime_note": {
        "dev_epoch_runtime_seconds":
            dev["seconds"],
        "dev_epoch_runtime_minutes":
            dev["seconds"] / 60.0,
        "production_runtime_must_be_budgeted":
            True,
        "do_not_launch_all_9_without_quota_review":
            True,
    },

    "evidence_files": {
        "training_record":
            str(DEV_RECORD),
        "epoch_log":
            str(DEV_LOG),
    },
}


with open(
    OUT,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        freeze,
        f,
        indent=2,
    )


# ============================================================================
# FINAL
# ============================================================================

print("=" * 78)
print("T2.1 EXECUTION + RUNTIME FREEZE")
print("=" * 78)

print()
print("STATUS: PASS")

print()
print("Executable SHA:")
print(actual_sha)

print()
print("DEV runtime:")
print(
    f"{dev['seconds']:.1f} s "
    f"({dev['seconds'] / 60.0:.2f} min)"
)

print()
print("Primary rho routing:")
print("args.rho -> rho")

print()
print("Primary replicate routing:")
print("args.rep -> rep")

print()
print("DEV override:")
print("rho=0.6, rep=1")

print()
print("Freeze file:")
print(OUT)

print()
print("NO SOURCE MODIFICATION.")
print("NO GPU TRAINING.")

In [ ]:
# ============================================================================
# T2.1 — 9-MODEL PRODUCTION LAUNCHER — 2 T4 GPUs
# ============================================================================
#
# Models:
#   rho = 0.0 : rep 1,2,3
#   rho = 0.5 : rep 1,2,3
#   rho = 1.0 : rep 1,2,3
#
# GPU policy:
#   GPU 0 -> one independent training process
#   GPU 1 -> one independent training process
#
# The launcher maintains a maximum of 2 simultaneous models.
# As soon as one GPU/process finishes, the next model starts there.
#
# Training executable is NOT modified.
# ============================================================================

from pathlib import Path
import hashlib
import json
import os
import subprocess
import sys
import time
from concurrent.futures import ThreadPoolExecutor, as_completed


# ============================================================================
# LOCKED PATHS / IDENTITY
# ============================================================================

SCRIPT = Path(
    "/kaggle/working/"
    "T2_1_ConvNeXt_training_code_freeze/"
    "train_convnext_tiny_t2_1.py"
)

EXPECTED_SHA = (
    "d6b97a547e96d4f14cb97b0bfa69c5219c798e0f5371a3de0d4c6bdaba93b379"
)

PROJECT = Path(
    "/kaggle/input/notebooks/"
    "tejasseshadriiiii/"
    "isic-kaggle-analysis/"
    "isic_final_project"
)

CACHE_DIR = Path(
    "/kaggle/working/T2_1_PRODUCTION_CACHE"
)

OUT_ROOT = Path(
    "/kaggle/working/T2_1_PRODUCTION"
)

OUT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================================
# HASH
# ============================================================================

def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b"",
        ):
            h.update(chunk)

    return h.hexdigest()


actual_sha = sha256_file(SCRIPT)

print("=" * 80)
print("T2.1 CONVNEXT-TINY — 9 MODEL PRODUCTION RUN")
print("=" * 80)

print()
print("Executable:")
print(SCRIPT)

print()
print("Expected SHA:")
print(EXPECTED_SHA)

print()
print("Actual SHA:")
print(actual_sha)

assert actual_sha == EXPECTED_SHA

assert PROJECT.exists()
assert CACHE_DIR.exists()


# ============================================================================
# LOCKED MODEL ROSTER
# ============================================================================

MODELS = [
    # rho = 0.0
    {"rho": 0.0, "rep": 1},
    {"rho": 0.0, "rep": 2},
    {"rho": 0.0, "rep": 3},

    # rho = 0.5
    {"rho": 0.5, "rep": 1},
    {"rho": 0.5, "rep": 2},
    {"rho": 0.5, "rep": 3},

    # rho = 1.0
    {"rho": 1.0, "rep": 1},
    {"rho": 1.0, "rep": 2},
    {"rho": 1.0, "rep": 3},
]


assert len(MODELS) == 9

assert [
    (m["rho"], m["rep"])
    for m in MODELS
] == [
    (0.0, 1),
    (0.0, 2),
    (0.0, 3),
    (0.5, 1),
    (0.5, 2),
    (0.5, 3),
    (1.0, 1),
    (1.0, 2),
    (1.0, 3),
]


# ============================================================================
# OUTPUT DIRECTORIES
# ============================================================================

for m in MODELS:

    rho_tag = f"{m['rho']:.1f}"
    rep = m["rep"]

    m["name"] = f"rho_{rho_tag}_rep_{rep}"

    m["out_dir"] = (
        OUT_ROOT
        / m["name"]
    )

    m["out_dir"].mkdir(
        parents=True,
        exist_ok=True,
    )


# ============================================================================
# CHECK FOR EXISTING COMPLETED MODELS
# ============================================================================
#
# We do NOT silently overwrite a completed production model.
# ============================================================================

print()
print("=" * 80)
print("PRODUCTION ROSTER")
print("=" * 80)

for m in MODELS:

    record = (
        m["out_dir"]
        / "training_record.json"
    )

    if record.exists():

        try:

            with open(
                record,
                "r",
                encoding="utf-8",
            ) as f:

                rec = json.load(f)

            if (
                rec.get("status") == "COMPLETED"
                and rec.get("mode") == "primary"
            ):

                raise RuntimeError(
                    f"Completed production model already exists: "
                    f"{m['name']}\n"
                    f"Refusing to overwrite it."
                )

        except json.JSONDecodeError:
            pass


    print(
        f"{m['name']:18s}"
    )


# ============================================================================
# SINGLE MODEL RUNNER
# ============================================================================

def run_model(model, gpu_id):

    rho = model["rho"]
    rep = model["rep"]
    name = model["name"]
    out_dir = model["out_dir"]

    env = os.environ.copy()

    # One physical GPU visible to this process.
    env["CUDA_VISIBLE_DEVICES"] = str(gpu_id)

    # Deterministic cuBLAS configuration.
    env["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

    cmd = [
        sys.executable,
        str(SCRIPT),

        "--project",
        str(PROJECT),

        "--cache-dir",
        str(CACHE_DIR),

        "--out-dir",
        str(out_dir),

        "--mode",
        "primary",

        "--rho",
        f"{rho:.1f}",

        "--rep",
        str(rep),

        "--gap",
        "mean",

        "--num-workers",
        "2",
    ]

    start = time.time()

    print()
    print(
        f"[GPU {gpu_id}] START {name}"
    )

    print(
        f"[GPU {gpu_id}] "
        f"rho={rho:.1f} rep={rep}"
    )

    print(
        f"[GPU {gpu_id}] "
        f"output={out_dir}"
    )

    # Separate stdout/stderr files make debugging possible even while
    # multiple models run simultaneously.
    stdout_path = (
        out_dir
        / "production_stdout.log"
    )

    stderr_path = (
        out_dir
        / "production_stderr.log"
    )

    with open(
        stdout_path,
        "w",
        encoding="utf-8",
    ) as stdout_f, open(
        stderr_path,
        "w",
        encoding="utf-8",
    ) as stderr_f:

        result = subprocess.run(
            cmd,
            env=env,
            stdout=stdout_f,
            stderr=stderr_f,
            text=True,
        )

    elapsed = time.time() - start

    print()
    print(
        f"[GPU {gpu_id}] FINISHED {name} "
        f"return_code={result.returncode} "
        f"time={elapsed / 60:.2f} min"
    )

    return {
        "name": name,
        "rho": rho,
        "rep": rep,
        "gpu": gpu_id,
        "return_code": result.returncode,
        "seconds": elapsed,
        "stdout": str(stdout_path),
        "stderr": str(stderr_path),
        "out_dir": str(out_dir),
    }


# ============================================================================
# GPU SCHEDULER
# ============================================================================
#
# ThreadPoolExecutor gives us exactly two concurrent subprocesses.
# Each subprocess gets its own GPU through CUDA_VISIBLE_DEVICES.
# ============================================================================

print()
print("=" * 80)
print("STARTING 2-GPU PRODUCTION SCHEDULER")
print("=" * 80)

print()
print("GPU 0 and GPU 1 will each run one independent model.")
print("Maximum concurrent training processes: 2")
print()
print("NO XAI / IG.")
print("PRIMARY CHECKPOINT = FINAL EPOCH 15.")
print()


results = []

with ThreadPoolExecutor(
    max_workers=2
) as executor:

    futures = {}

    # Initially submit exactly two models.
    pending_index = 0

    while (
        pending_index < len(MODELS)
        and len(futures) < 2
    ):

        gpu_id = len(futures)

        model = MODELS[pending_index]

        future = executor.submit(
            run_model,
            model,
            gpu_id,
        )

        futures[future] = (
            model,
            gpu_id,
        )

        pending_index += 1


    # As each GPU becomes available, submit the next model.
    while futures:

        done = next(
            as_completed(futures)
        )

        model, gpu_id = futures.pop(
            done
        )

        result = done.result()

        results.append(result)

        # Fail immediately if a production process failed.
        if result["return_code"] != 0:

            print()
            print("=" * 80)
            print("PRODUCTION MODEL FAILED")
            print("=" * 80)

            print(
                json.dumps(
                    result,
                    indent=2,
                )
            )

            raise RuntimeError(
                f"T2.1 production model failed: "
                f"{result['name']}"
            )


        # Reuse the now-free GPU.
        if pending_index < len(MODELS):

            next_model = MODELS[
                pending_index
            ]

            future = executor.submit(
                run_model,
                next_model,
                gpu_id,
            )

            futures[future] = (
                next_model,
                gpu_id,
            )

            pending_index += 1


# ============================================================================
# SORT RESULTS IN LOCKED ROSTER ORDER
# ============================================================================

order = {
    m["name"]: i
    for i, m in enumerate(MODELS)
}

results.sort(
    key=lambda x: order[x["name"]]
)


# ============================================================================
# FINAL OUTPUT SUMMARY
# ============================================================================

print()
print("=" * 80)
print("ALL 9 PRODUCTION PROCESSES FINISHED")
print("=" * 80)

for r in results:

    print(
        f"{r['name']:18s} "
        f"GPU={r['gpu']} "
        f"return={r['return_code']} "
        f"time={r['seconds'] / 60:.2f} min"
    )


# ============================================================================
# VERIFY EVERY TRAINING RECORD
# ============================================================================

print()
print("=" * 80)
print("POST-RUN TRAINING RECORD VERIFICATION")
print("=" * 80)


for m in MODELS:

    record_path = (
        m["out_dir"]
        / "training_record.json"
    )

    assert record_path.exists(), (
        f"Missing training record: {m['name']}"
    )

    with open(
        record_path,
        "r",
        encoding="utf-8",
    ) as f:

        rec = json.load(f)

    assert rec["status"] == "COMPLETED"
    assert rec["mode"] == "primary"
    assert rec["architecture"] == "ConvNeXt-Tiny"

    assert rec["weights"] == (
        "ConvNeXt_Tiny_Weights.IMAGENET1K_V1"
    )

    assert abs(
        float(rec["rho"]) - m["rho"]
    ) < 1e-12

    assert int(
        rec["replicate"]
    ) == m["rep"]

    assert rec["script_sha256"] == EXPECTED_SHA

    assert rec["endpoint"] == "Delta-z"
    assert rec["xai"] is False

    assert rec["env"]["gpu"] == "Tesla T4"
    assert rec["env"]["deterministic_algorithms"] is True
    assert rec["env"]["cudnn_deterministic"] is True
    assert rec["env"]["cudnn_benchmark"] is False
    assert rec["env"]["tf32_matmul"] is False
    assert rec["env"]["tf32_cudnn"] is False
    assert (
        rec["env"]["cublas_workspace_config"]
        == ":4096:8"
    )

    print(
        f"PASS  {m['name']}  "
        f"rho={m['rho']:.1f}  "
        f"rep={m['rep']}"
    )


# ============================================================================
# SAVE LAUNCH MANIFEST
# ============================================================================

manifest = {
    "status": "COMPLETED",
    "architecture": "ConvNeXt-Tiny",
    "weights":
        "ConvNeXt_Tiny_Weights.IMAGENET1K_V1",
    "script_sha256": EXPECTED_SHA,
    "models": [
        {
            "rho": m["rho"],
            "rep": m["rep"],
            "name": m["name"],
            "output":
                str(m["out_dir"]),
        }
        for m in MODELS
    ],
    "results": results,
    "gpu_policy":
        "one independent subprocess per GPU",
    "visible_gpu_per_process":
        "single GPU",
    "xai": False,
    "endpoint": "Delta-z",
    "primary_checkpoint":
        "final_epoch15.pt",
    "sensitivity_checkpoint":
        "best_eval_ba.pt",
}


manifest_path = (
    OUT_ROOT
    / "T2_1_PRODUCTION_LAUNCH_MANIFEST.json"
)

with open(
    manifest_path,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        manifest,
        f,
        indent=2,
    )


print()
print("=" * 80)
print("T2.1 PRODUCTION RUN: PASS")
print("=" * 80)

print()
print("9/9 models completed successfully.")

print()
print("Manifest:")
print(manifest_path)

In [ ]:
from pathlib import Path

ROOT = Path("/kaggle/working/T2_1_PRODUCTION/rho_0.5_rep_1")

print("=" * 80)
print("rho_0.5_rep_1 — FAILURE OUTPUT")
print("=" * 80)

for filename in ["production_stdout.log", "production_stderr.log"]:
    path = ROOT / filename

    print("\n" + "=" * 30, filename, "=" * 30)

    if path.exists():
        text = path.read_text(errors="replace")
        print(text[-15000:] if text else "(EMPTY)")
    else:
        print("FILE NOT FOUND")

In [ ]:
from pathlib import Path
import pandas as pd
import json

PROJECT = Path("/kaggle/input/notebooks/tejasseshadriiiii/isic-kaggle-analysis/isic_final_project")
RHO05 = Path("/kaggle/working/T2_1_ConvNeXt_rho05_assignments")

seed_path = PROJECT / "step2V_seed_table_and_run_order" / "TRAINING_SEED_TABLE.csv"
manifest_path = RHO05 / "T2_1_RHO05_GENERATION_MANIFEST.json"

print("=" * 80)
print("ORIGINAL LOCKED SEED TABLE")
print("=" * 80)

seed = pd.read_csv(seed_path, dtype=str)

print("Path:", seed_path)
print("Shape:", seed.shape)
print("\nColumns:")
print(seed.columns.tolist())

print("\nRelevant rows:")
print(
    seed[
        seed["rho"].isin(["0.0", "1.0"]) &
        seed["replicate"].isin(["1", "2", "3"])
    ][["rho", "replicate"] + [
        c for c in seed.columns
        if "seed" in c.lower()
    ]].to_string(index=False)
)

print("\nRho values present:")
print(sorted(seed["rho"].unique().tolist()))

print("\n" + "=" * 80)
print("T2.1 RHO=0.5 GENERATION MANIFEST")
print("=" * 80)

print(manifest_path)

manifest = json.loads(manifest_path.read_text())

print(json.dumps(manifest, indent=2)[:20000])

print("\n" + "=" * 80)
print("RHO=0.5 ASSIGNMENT FILES")
print("=" * 80)

for p in sorted(RHO05.glob("*.csv")):
    print(p.name)

In [ ]:
from pathlib import Path
import hashlib
import re

PROJECT = Path(
    "/kaggle/input/notebooks/tejasseshadriiiii/"
    "isic-kaggle-analysis/isic_final_project"
)

print("=" * 80)
print("SEARCHING T2.1 PROJECT FOR SEED-TABLE GENERATION LOGIC")
print("=" * 80)

candidates = []

for root in [
    PROJECT / "step2V_seed_table_and_run_order",
    PROJECT / "step1F_assignment_generation",
    PROJECT / "step1F_assignment_generator",
    PROJECT,
]:
    if root.exists():
        for p in root.rglob("*"):
            if p.is_file():
                name = p.name.lower()
                if (
                    "seed" in name
                    or "run_order" in name
                    or "training" in name
                ):
                    candidates.append(p)

seen = set()

for p in candidates:
    s = str(p)
    if s in seen:
        continue
    seen.add(s)

    print("\n", p)

    try:
        data = p.read_bytes()

        print("  size:", len(data))
        print(
            "  sha256:",
            hashlib.sha256(data).hexdigest()
        )

        if p.suffix.lower() in {".py", ".txt", ".md", ".json", ".csv"}:
            text = data.decode("utf-8", errors="replace")

            hits = []

            for pattern in [
                r"seed_init",
                r"seed_dataloader",
                r"seed_augmentation",
                r"seed_workers",
                r"TRAINING_SEED_TABLE",
                r"SHA256",
                r"SeedSequence",
                r"PCG64",
                r"master_seed",
                r"seed.*rho",
                r"rho.*seed",
            ]:
                if re.search(pattern, text, flags=re.IGNORECASE):
                    hits.append(pattern)

            if hits:
                print("  relevant terms:", ", ".join(hits))

    except Exception as e:
        print("  READ ERROR:", repr(e))

print("\n" + "=" * 80)
print("DONE")
print("=" * 80)

In [ ]:
from pathlib import Path
import json

PROJECT = Path(
    "/kaggle/input/notebooks/tejasseshadriiiii/"
    "isic-kaggle-analysis/isic_final_project"
)

files = [
    PROJECT / "step2V_seed_table_and_run_order" / "RUN_ORDER_MANIFEST.json",
    PROJECT / "step2V_seed_table_and_run_order" / "RUN_ORDER_FINAL_HASH.json",
    PROJECT / "step1P_seed_manifest" / "seed_manifest.json",
]

for path in files:
    print("\n" + "=" * 100)
    print(path.name)
    print("=" * 100)

    if not path.exists():
        print("NOT FOUND")
        continue

    try:
        obj = json.loads(path.read_text())
        print(json.dumps(obj, indent=2))
    except Exception as e:
        print("ERROR:", repr(e))

In [ ]:
from pathlib import Path
import hashlib
import json
import pandas as pd

PROJECT = Path(
    "/kaggle/input/notebooks/tejasseshadriiiii/"
    "isic-kaggle-analysis/isic_final_project"
)

OUT = Path("/kaggle/working/T2_1_ConvNeXt_seed_lock")
OUT.mkdir(parents=True, exist_ok=True)

MASTER_SEED = 20260919

PURPOSES = {
    "seed_init": "train-model-init",
    "seed_dataloader": "train-dataloader",
    "seed_augmentation": "train-augmentation",
    "seed_workers": "train-workers",
}

LOCKED_SEED_TABLE = (
    PROJECT
    / "step2V_seed_table_and_run_order"
    / "TRAINING_SEED_TABLE.csv"
)

seed_table = pd.read_csv(LOCKED_SEED_TABLE, dtype=str)

def derive_training_seed(purpose, rho, replicate):
    payload = (
        f"{MASTER_SEED}|"
        f"{purpose}|"
        f"{rho}|"
        f"{replicate}|"
    ).encode("utf-8")

    digest = hashlib.sha256(payload).digest()

    return int.from_bytes(digest[:8], byteorder="big", signed=False)


# ============================================================
# 1. VALIDATE THE DOCUMENTED RULE AGAINST LOCKED HISTORY
# ============================================================

print("=" * 80)
print("VALIDATING DOCUMENTED TRAINING-SEED DERIVATION")
print("=" * 80)

validation_rows = []

for rho in ["0.0", "1.0"]:
    for rep in ["1", "2", "3"]:

        locked = seed_table[
            (seed_table["rho"] == rho) &
            (seed_table["replicate"] == rep)
        ]

        assert len(locked) == 1, (
            f"Expected exactly one locked row for rho={rho}, rep={rep}"
        )

        locked = locked.iloc[0]

        row = {
            "rho": rho,
            "replicate": rep,
        }

        all_match = True

        for col, purpose in PURPOSES.items():

            derived = derive_training_seed(
                purpose=purpose,
                rho=rho,
                replicate=rep
            )

            recorded = int(locked[col])

            match = derived == recorded

            row[f"{col}_derived"] = derived
            row[f"{col}_locked"] = recorded
            row[f"{col}_exact"] = match

            all_match = all_match and match

        row["all_exact"] = all_match
        validation_rows.append(row)

        print(
            f"rho={rho} rep={rep} -> "
            f"{'EXACT' if all_match else 'MISMATCH'}"
        )


validation_df = pd.DataFrame(validation_rows)

assert validation_df["all_exact"].all(), (
    "DOCUMENTED TRAINING-SEED RULE DOES NOT REPRODUCE "
    "ALL SIX LOCKED rho=0/1 ROWS."
)

print("\nVALIDATION: PASS")
print("All six locked rows reproduce byte-exactly.")


# ============================================================
# 2. DERIVE THE THREE NEW rho=0.5 TRAINING SEED ROWS
# ============================================================

print("\n" + "=" * 80)
print("DERIVING NEW T2.1 rho=0.5 TRAINING SEEDS")
print("=" * 80)

new_rows = []

for rep in ["1", "2", "3"]:

    row = {
        "rho": "0.5",
        "replicate": rep,
    }

    for col, purpose in PURPOSES.items():
        row[col] = str(
            derive_training_seed(
                purpose=purpose,
                rho="0.5",
                replicate=rep
            )
        )

    new_rows.append(row)

new_df = pd.DataFrame(new_rows)

print(new_df.to_string(index=False))


# ============================================================
# 3. BUILD THE T2.1 NINE-MODEL SEED TABLE
# ============================================================

selected_existing = seed_table[
    seed_table["rho"].isin(["0.0", "1.0"]) &
    seed_table["replicate"].isin(["1", "2", "3"])
].copy()

selected_existing = selected_existing[
    [
        "rho",
        "replicate",
        "seed_init",
        "seed_dataloader",
        "seed_augmentation",
        "seed_workers",
    ]
]

t2_1_seed_table = pd.concat(
    [selected_existing, new_df],
    ignore_index=True
)

t2_1_seed_table["rho_numeric"] = t2_1_seed_table["rho"].astype(float)

t2_1_seed_table = (
    t2_1_seed_table
    .sort_values(["rho_numeric", "replicate"])
    .drop(columns=["rho_numeric"])
    .reset_index(drop=True)
)

assert len(t2_1_seed_table) == 9

assert list(t2_1_seed_table["rho"]) == [
    "0.0", "0.0", "0.0",
    "0.5", "0.5", "0.5",
    "1.0", "1.0", "1.0",
]

assert list(t2_1_seed_table["replicate"]) == [
    "1", "2", "3",
    "1", "2", "3",
    "1", "2", "3",
]

print("\n" + "=" * 80)
print("FINAL T2.1 SEED TABLE")
print("=" * 80)

print(t2_1_seed_table.to_string(index=False))


# ============================================================
# 4. WRITE CANONICAL CSV
# ============================================================

csv_path = OUT / "T2_1_TRAINING_SEED_TABLE.csv"

t2_1_seed_table.to_csv(
    csv_path,
    index=False,
    lineterminator="\n"
)

csv_bytes = csv_path.read_bytes()

csv_sha256 = hashlib.sha256(csv_bytes).hexdigest()

print("\nCanonical CSV:")
print(csv_path)

print("SHA256:")
print(csv_sha256)


# ============================================================
# 5. CREATE PROVENANCE / LOCK MANIFEST
# ============================================================

manifest = {
    "status": "LOCKED_FOR_T2_1",
    "purpose": "T2.1 second-architecture ConvNeXt-Tiny replication",
    "architecture": "ConvNeXt-Tiny",
    "master_seed": MASTER_SEED,
    "source_locked_seed_table": str(LOCKED_SEED_TABLE),
    "source_locked_seed_table_sha256":
        hashlib.sha256(
            LOCKED_SEED_TABLE.read_bytes()
        ).hexdigest(),
    "documented_derivation_rule":
        "first_8_bytes_big_endian(SHA256(master_seed | purpose | rho | replicate | optional_identifier))",
    "training_purposes": PURPOSES,
    "historical_validation": {
        "conditions": [
            "rho=0.0, replicate=1",
            "rho=0.0, replicate=2",
            "rho=0.0, replicate=3",
            "rho=1.0, replicate=1",
            "rho=1.0, replicate=2",
            "rho=1.0, replicate=3",
        ],
        "all_exact": True,
    },
    "new_condition": {
        "rho": "0.5",
        "replicates": ["1", "2", "3"],
        "derived_using_documented_rule": True,
    },
    "n_models": 9,
    "rows": t2_1_seed_table.to_dict(orient="records"),
    "csv": str(csv_path),
    "csv_sha256": csv_sha256,
}

manifest_path = OUT / "T2_1_TRAINING_SEED_TABLE_LOCK.json"

manifest_path.write_text(
    json.dumps(manifest, indent=2),
    encoding="utf-8"
)

print("\nManifest:")
print(manifest_path)

print("\n" + "=" * 80)
print("T2.1 SEED TABLE LOCK: PASS")
print("=" * 80)

In [ ]:
from pathlib import Path
import hashlib
import shutil
import json

SRC = Path(
    "/kaggle/working/T2_1_ConvNeXt_training_code_freeze/"
    "train_convnext_tiny_t2_1.py"
)

SEED_TABLE = Path(
    "/kaggle/working/T2_1_ConvNeXt_seed_lock/"
    "T2_1_TRAINING_SEED_TABLE.csv"
)

assert SRC.exists()
assert SEED_TABLE.exists()

source = SRC.read_text(encoding="utf-8")

OLD = '''seed_table = pd.read_csv(
            project / "step2V_seed_table_and_run_order" / "TRAINING_SEED_TABLE.csv",
            dtype=str
        )'''

NEW = '''seed_table = pd.read_csv(
            Path("/kaggle/working/T2_1_ConvNeXt_seed_lock/T2_1_TRAINING_SEED_TABLE.csv"),
            dtype=str
        )'''

print("OLD occurrences:", source.count(OLD))
print("NEW occurrences:", source.count(NEW))

assert source.count(OLD) == 1, (
    "Expected exactly one primary seed-table loader."
)

assert source.count(NEW) == 0, (
    "T2.1 seed-table patch appears to already exist."
)

patched = source.replace(OLD, NEW)

assert patched != source

# ------------------------------------------------------------
# Backup current executable before surgical patch
# ------------------------------------------------------------

backup = SRC.with_suffix(".pre_T2_1_seed_patch.py")

if not backup.exists():
    shutil.copy2(SRC, backup)

# ------------------------------------------------------------
# Write patch
# ------------------------------------------------------------

SRC.write_text(patched, encoding="utf-8")

# ------------------------------------------------------------
# Compile gate
# ------------------------------------------------------------

compile(SRC.read_text(encoding="utf-8"), str(SRC), "exec")

# ------------------------------------------------------------
# Provenance checks
# ------------------------------------------------------------

final_source = SRC.read_text(encoding="utf-8")

assert 'Path("/kaggle/working/T2_1_ConvNeXt_seed_lock/T2_1_TRAINING_SEED_TABLE.csv")' in final_source

assert 'step2V_seed_table_and_run_order" / "TRAINING_SEED_TABLE.csv"' not in final_source

sha256 = hashlib.sha256(SRC.read_bytes()).hexdigest()

manifest = {
    "status": "T2_1_SEED_TABLE_PATCHED",
    "script": str(SRC),
    "script_sha256": sha256,
    "seed_table": str(SEED_TABLE),
    "seed_table_sha256": hashlib.sha256(
        SEED_TABLE.read_bytes()
    ).hexdigest(),
    "backup": str(backup),
    "patch_scope": "primary seed-table source only",
    "training_logic_changed": False,
    "architecture_changed": False,
    "assignment_logic_changed": False,
    "optimizer_changed": False,
    "scheduler_changed": False,
    "loss_changed": False,
    "checkpoint_logic_changed": False,
    "xai_logic_changed": False,
}

lock_path = (
    Path("/kaggle/working/T2_1_ConvNeXt_seed_lock")
    / "T2_1_TRAINING_CODE_SEED_PATCH_LOCK.json"
)

lock_path.write_text(
    json.dumps(manifest, indent=2),
    encoding="utf-8"
)

print("\n" + "=" * 80)
print("T2.1 SEED-TABLE PATCH: PASS")
print("=" * 80)

print("Executable:")
print(SRC)

print("New executable SHA256:")
print(sha256)

print("Seed table SHA256:")
print(manifest["seed_table_sha256"])

print("Backup:")
print(backup)

print("Lock:")
print(lock_path)

In [ ]:
from pathlib import Path

SRC = Path(
    "/kaggle/working/T2_1_ConvNeXt_training_code_freeze/"
    "train_convnext_tiny_t2_1.py"
)

text = SRC.read_text(encoding="utf-8")
lines = text.splitlines()

print("=" * 80)
print("ACTUAL SEED LOOKUP CODE")
print("=" * 80)

for i in range(470, 525):
    if i <= len(lines):
        print(f"{i:4d}: {lines[i-1]}")

print("\n" + "=" * 80)
print("ALL SEED-TABLE REFERENCES")
print("=" * 80)

for i, line in enumerate(lines, 1):
    if (
        "seed_table" in line
        or "TRAINING_SEED_TABLE" in line
        or "seed row not found" in line
    ):
        print(f"{i:4d}: {line}")

In [ ]:
from pathlib import Path
import hashlib
import json
import shutil

SRC = Path(
    "/kaggle/working/T2_1_ConvNeXt_training_code_freeze/"
    "train_convnext_tiny_t2_1.py"
)

SEED_TABLE = Path(
    "/kaggle/working/T2_1_ConvNeXt_seed_lock/"
    "T2_1_TRAINING_SEED_TABLE.csv"
)

assert SRC.exists()
assert SEED_TABLE.exists()

source = SRC.read_text(encoding="utf-8")

OLD = '''        seed_table = pd.read_csv(
            project
            / "step2V_seed_table_and_run_order"
            / "TRAINING_SEED_TABLE.csv",
            dtype=str,
        )'''

NEW = '''        seed_table = pd.read_csv(
            Path(
                "/kaggle/working/T2_1_ConvNeXt_seed_lock/"
                "T2_1_TRAINING_SEED_TABLE.csv"
            ),
            dtype=str,
        )'''

print("OLD occurrences:", source.count(OLD))
print("NEW occurrences:", source.count(NEW))

assert source.count(OLD) == 1
assert source.count(NEW) == 0

# Backup the currently frozen executable
backup = SRC.with_suffix(".pre_T2_1_seed_patch.py")

if not backup.exists():
    shutil.copy2(SRC, backup)

patched = source.replace(OLD, NEW)

assert patched != source

SRC.write_text(patched, encoding="utf-8")

# Compile
compile(
    SRC.read_text(encoding="utf-8"),
    str(SRC),
    "exec"
)

# Verify exact intended change
final_source = SRC.read_text(encoding="utf-8")

assert final_source.count(
    'T2_1_ConvNeXt_seed_lock/'
) == 1

assert 'TRAINING_SEED_TABLE.csv' in final_source

assert (
    'step2V_seed_table_and_run_order'
    not in final_source
)

# Verify seed lookup logic remains intact
assert 'seed_table["rho"]' in final_source
assert 'seed_table["replicate"]' in final_source
assert '"seed_init"' in final_source
assert '"seed_dataloader"' in final_source
assert '"seed_augmentation"' in final_source
assert '"seed_workers"' in final_source

sha256 = hashlib.sha256(
    SRC.read_bytes()
).hexdigest()

seed_sha256 = hashlib.sha256(
    SEED_TABLE.read_bytes()
).hexdigest()

manifest = {
    "status": "T2_1_SEED_TABLE_PATCHED",
    "script": str(SRC),
    "script_sha256": sha256,
    "seed_table": str(SEED_TABLE),
    "seed_table_sha256": seed_sha256,
    "backup": str(backup),
    "patch_scope": "primary seed-table source only",
    "training_logic_changed": False,
    "architecture_changed": False,
    "assignment_logic_changed": False,
    "optimizer_changed": False,
    "scheduler_changed": False,
    "loss_changed": False,
    "checkpoint_logic_changed": False,
    "xai_logic_changed": False,
}

lock_path = (
    Path("/kaggle/working/T2_1_ConvNeXt_seed_lock")
    / "T2_1_TRAINING_CODE_SEED_PATCH_LOCK.json"
)

lock_path.write_text(
    json.dumps(manifest, indent=2),
    encoding="utf-8"
)

print("\n" + "=" * 80)
print("T2.1 SEED-TABLE PATCH: PASS")
print("=" * 80)
print("Executable SHA256:", sha256)
print("Seed table SHA256:", seed_sha256)
print("Backup:", backup)
print("Lock:", lock_path)

In [ ]:
from pathlib import Path
import subprocess
import os
import json
import hashlib
import shutil

SCRIPT = Path(
    "/kaggle/working/T2_1_ConvNeXt_training_code_freeze/"
    "train_convnext_tiny_t2_1.py"
)

PROJECT = Path(
    "/kaggle/input/notebooks/tejasseshadriiiii/"
    "isic-kaggle-analysis/isic_final_project"
)

CACHE = Path("/kaggle/working/T2_1_PRODUCTION_CACHE")
OUT = Path("/kaggle/working/T2_1_SEED_STARTUP_TEST")

OUT.mkdir(parents=True, exist_ok=True)

EXPECTED_SCRIPT_SHA = (
    "d4fb712431e6ad02c816d728e5cfeb513490a9bc500a4ee0deca41784bf296d1"
)

actual_sha = hashlib.sha256(SCRIPT.read_bytes()).hexdigest()

assert actual_sha == EXPECTED_SCRIPT_SHA, (
    f"Unexpected script SHA: {actual_sha}"
)

print("Executable SHA verified:", actual_sha)

# ------------------------------------------------------------
# IMPORTANT:
# Use GPU 0 only for this short startup test.
# ------------------------------------------------------------

env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"] = "0"
env["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

cmd = [
    "python",
    str(SCRIPT),

    "--project",
    str(PROJECT),

    "--cache-dir",
    str(CACHE),

    "--out-dir",
    str(OUT),

    "--rho",
    "0.5",

    "--rep",
    "1",

    "--mode",
    "primary",

    "--gap",
    "mean",

    "--num-workers",
    "2",

    # One epoch only for startup validation.
    "--epochs",
    "1",
]

print("\n" + "=" * 80)
print("T2.1 rho=0.5 PRIMARY STARTUP TEST")
print("=" * 80)

print(" ".join(cmd))

result = subprocess.run(
    cmd,
    env=env,
    capture_output=True,
    text=True,
)

print("\nRETURN CODE:", result.returncode)

print("\n" + "=" * 80)
print("STDOUT")
print("=" * 80)
print(result.stdout[-12000:] if result.stdout else "(EMPTY)")

print("\n" + "=" * 80)
print("STDERR")
print("=" * 80)
print(result.stderr[-12000:] if result.stderr else "(EMPTY)")

# ------------------------------------------------------------
# Locate training record
# ------------------------------------------------------------

records = list(OUT.rglob("training_record.json"))

print("\n" + "=" * 80)
print("TRAINING RECORDS")
print("=" * 80)

for p in records:
    print(p)

if result.returncode == 0:

    assert records, (
        "Process exited successfully but no training_record.json was found."
    )

    record = json.loads(records[0].read_text())

    print("\nRecord:")
    print(json.dumps(record, indent=2))

    assert record.get("architecture") == "ConvNeXt-Tiny"
    assert record.get("rho") == 0.5
    assert record.get("replicate") == 1

    print("\n" + "=" * 80)
    print("T2.1 rho=0.5 PRIMARY STARTUP TEST: PASS")
    print("=" * 80)

else:

    print("\n" + "=" * 80)
    print("T2.1 rho=0.5 PRIMARY STARTUP TEST: FAILED")
    print("=" * 80)

    raise RuntimeError(
        "rho=0.5 primary startup test failed. "
        "Do not launch the production run yet."
    )

In [ ]:
from pathlib import Path
import ast
import hashlib
import pandas as pd
import json

SCRIPT = Path(
    "/kaggle/working/T2_1_ConvNeXt_training_code_freeze/"
    "train_convnext_tiny_t2_1.py"
)

SEED_TABLE = Path(
    "/kaggle/working/T2_1_ConvNeXt_seed_lock/"
    "T2_1_TRAINING_SEED_TABLE.csv"
)

EXPECTED_SHA = (
    "d4fb712431e6ad02c816d728e5cfeb513490a9bc500a4ee0deca41784bf296d1"
)

print("=" * 80)
print("T2.1 RHO=0.5 PRIMARY SEED-RESOLUTION TEST")
print("=" * 80)

# ------------------------------------------------------------
# 1. Executable identity
# ------------------------------------------------------------

actual_sha = hashlib.sha256(
    SCRIPT.read_bytes()
).hexdigest()

print("Executable SHA:", actual_sha)

assert actual_sha == EXPECTED_SHA

# ------------------------------------------------------------
# 2. Confirm patched source points to T2.1 table
# ------------------------------------------------------------

source = SCRIPT.read_text(encoding="utf-8")

assert "T2_1_ConvNeXt_seed_lock" in source
assert "step2V_seed_table_and_run_order" not in source

print("Seed-table source: T2.1 locked table")

# ------------------------------------------------------------
# 3. Load exact T2.1 table
# ------------------------------------------------------------

seed_table = pd.read_csv(
    SEED_TABLE,
    dtype=str
)

print("\nSeed table:")
print(seed_table.to_string(index=False))

# ------------------------------------------------------------
# 4. Perform EXACT same lookup used by primary mode
# ------------------------------------------------------------

rho = 0.5
rep = 1

r = seed_table[
    (seed_table["rho"] == f"{rho:.1f}") &
    (seed_table["replicate"] == str(rep))
]

print("\nLookup:")
print("rho =", f"{rho:.1f}")
print("rep =", str(rep))
print("matching rows =", len(r))

assert len(r) == 1, (
    "T2.1 rho=0.5 rep=1 seed row was not found."
)

# ------------------------------------------------------------
# 5. Extract exactly the fields used by training
# ------------------------------------------------------------

seeds = {
    k: int(r.iloc[0][k])
    for k in (
        "seed_init",
        "seed_dataloader",
        "seed_augmentation",
        "seed_workers",
    )
}

print("\nResolved training seeds:")
for k, v in seeds.items():
    print(f"{k:20s} = {v}")

# ------------------------------------------------------------
# 6. Verify against the locked rho=0.5 derivation manifest
# ------------------------------------------------------------

manifest_path = Path(
    "/kaggle/working/T2_1_ConvNeXt_seed_lock/"
    "T2_1_TRAINING_SEED_TABLE_LOCK.json"
)

manifest = json.loads(
    manifest_path.read_text()
)

manifest_rows = [
    x for x in manifest["rows"]
    if x["rho"] == "0.5" and x["replicate"] == "1"
]

assert len(manifest_rows) == 1

manifest_row = manifest_rows[0]

for k in (
    "seed_init",
    "seed_dataloader",
    "seed_augmentation",
    "seed_workers",
):
    assert str(seeds[k]) == str(manifest_row[k])

print("\nManifest consistency: PASS")

# ------------------------------------------------------------
# 7. Final gate
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("T2.1 RHO=0.5 PRIMARY SEED-RESOLUTION: PASS")
print("=" * 80)
print("rho=0.5 rep=1 resolves exactly one locked seed row.")
print("All four training seeds resolve successfully.")
print("No training was executed.")

In [ ]:
from pathlib import Path
import json
import hashlib

ROOT = Path("/kaggle/working/T2_1_PRODUCTION")

ALL_MODELS = [
    ("0.0", 1),
    ("0.0", 2),
    ("0.0", 3),
    ("0.5", 1),
    ("0.5", 2),
    ("0.5", 3),
    ("1.0", 1),
    ("1.0", 2),
    ("1.0", 3),
]

print("=" * 80)
print("T2.1 RESUME ROSTER AUDIT")
print("=" * 80)

completed = []
rerun = []

for rho, rep in ALL_MODELS:

    name = f"rho_{rho}_rep_{rep}"
    d = ROOT / name
    record = d / "training_record.json"

    if record.exists():

        try:
            r = json.loads(record.read_text())

            if (
                r.get("status") == "COMPLETED"
                and r.get("mode") == "primary"
                and str(r.get("rho")) == rho
                and int(r.get("replicate")) == rep
            ):
                completed.append((rho, rep))
                print(f"{name:18s} → SKIP (VALID COMPLETED)")
                continue

        except Exception:
            pass

    rerun.append((rho, rep))

    if d.exists():
        print(
            f"{name:18s} → RUN AGAIN "
            f"(existing directory is not a valid completed run)"
        )
    else:
        print(f"{name:18s} → RUN")

print("\n" + "=" * 80)
print("SUMMARY")
print("=" * 80)

print("Completed/skipped:", len(completed))
print("To run:", len(rerun))

print("\nSKIP:")
for rho, rep in completed:
    print(f"  rho={rho} rep={rep}")

print("\nRUN:")
for rho, rep in rerun:
    print(f"  rho={rho} rep={rep}")

assert completed == [
    ("0.0", 1),
    ("0.0", 2),
], (
    "Unexpected completed-model state. "
    "Do not launch the resume run until this is inspected."
)

assert len(rerun) == 7

print("\n" + "=" * 80)
print("RESUME ROSTER: PASS")
print("=" * 80)

In [ ]:
from pathlib import Path
import subprocess
import os
import json
import hashlib
import shutil
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed

# ============================================================
# LOCKED PATHS
# ============================================================

SCRIPT = Path(
    "/kaggle/working/T2_1_ConvNeXt_training_code_freeze/"
    "train_convnext_tiny_t2_1.py"
)

PROJECT = Path(
    "/kaggle/input/notebooks/tejasseshadriiiii/"
    "isic-kaggle-analysis/isic_final_project"
)

CACHE = Path(
    "/kaggle/working/T2_1_PRODUCTION_CACHE"
)

ROOT = Path(
    "/kaggle/working/T2_1_PRODUCTION"
)

SEED_TABLE = Path(
    "/kaggle/working/T2_1_ConvNeXt_seed_lock/"
    "T2_1_TRAINING_SEED_TABLE.csv"
)

EXPECTED_SCRIPT_SHA = (
    "d4fb712431e6ad02c816d728e5cfeb513490a9bc500a4ee0deca41784bf296d1"
)

EXPECTED_SEED_SHA = (
    "9c54a1a617f676750ef4abd174d0374a7ccce69e633c1f8dad86a3ee03823a5d"
)

ROOT.mkdir(parents=True, exist_ok=True)

# ============================================================
# IDENTITY GATES
# ============================================================

actual_script_sha = hashlib.sha256(
    SCRIPT.read_bytes()
).hexdigest()

actual_seed_sha = hashlib.sha256(
    SEED_TABLE.read_bytes()
).hexdigest()

assert actual_script_sha == EXPECTED_SCRIPT_SHA, (
    f"Unexpected training script SHA:\n{actual_script_sha}"
)

assert actual_seed_sha == EXPECTED_SEED_SHA, (
    f"Unexpected T2.1 seed table SHA:\n{actual_seed_sha}"
)

print("=" * 80)
print("T2.1 RESUME PRODUCTION LAUNCH")
print("=" * 80)

print("Training script SHA:", actual_script_sha)
print("T2.1 seed table SHA:", actual_seed_sha)

# ============================================================
# FULL 9-MODEL ROSTER
# ============================================================

ALL_MODELS = [
    ("0.0", 1),
    ("0.0", 2),
    ("0.0", 3),
    ("0.5", 1),
    ("0.5", 2),
    ("0.5", 3),
    ("1.0", 1),
    ("1.0", 2),
    ("1.0", 3),
]

# ============================================================
# FIND VALID COMPLETED MODELS
# ============================================================

completed = []
to_run = []

for rho, rep in ALL_MODELS:

    name = f"rho_{rho}_rep_{rep}"
    model_dir = ROOT / name
    record_path = model_dir / "training_record.json"

    valid_completed = False

    if record_path.exists():
        try:
            record = json.loads(
                record_path.read_text()
            )

            valid_completed = (
                record.get("status") == "COMPLETED"
                and record.get("mode") == "primary"
                and str(record.get("rho")) == rho
                and int(record.get("replicate")) == rep
                and record.get("architecture") == "ConvNeXt-Tiny"
                and record.get("endpoint") == "Delta-z"
                and record.get("xai") is False
                and record.get("script_sha256") == EXPECTED_SCRIPT_SHA
            )

        except Exception:
            valid_completed = False

    if valid_completed:
        completed.append((rho, rep))
    else:
        to_run.append((rho, rep))

print("\nCompleted models that will NOT be retrained:")
for rho, rep in completed:
    print(f"  rho={rho} rep={rep}")

print("\nModels to run:")
for rho, rep in to_run:
    print(f"  rho={rho} rep={rep}")

assert completed == [
    ("0.0", 1),
    ("0.0", 2),
]

assert len(to_run) == 7

# ============================================================
# REMOVE ONLY INCOMPLETE MODEL DIRECTORIES
# ============================================================

for rho, rep in to_run:

    name = f"rho_{rho}_rep_{rep}"
    model_dir = ROOT / name

    if model_dir.exists():
        print(f"\nCleaning incomplete output: {name}")
        shutil.rmtree(model_dir)

# ============================================================
# LIVE STREAMING MODEL RUNNER
# ============================================================

def run_model(model, gpu_id):

    rho, rep = model
    name = f"rho_{rho}_rep_{rep}"

    model_dir = ROOT / name
    model_dir.mkdir(parents=True, exist_ok=True)

    stdout_path = model_dir / "production_stdout.log"
    stderr_path = model_dir / "production_stderr.log"

    env = os.environ.copy()

    # Exactly one physical GPU per subprocess.
    env["CUDA_VISIBLE_DEVICES"] = str(gpu_id)

    # Deterministic CUDA configuration.
    env["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

    cmd = [
        "python",
        "-u",
        str(SCRIPT),

        "--project",
        str(PROJECT),

        "--cache-dir",
        str(CACHE),

        "--out-dir",
        str(model_dir),

        "--mode",
        "primary",

        "--rho",
        rho,

        "--rep",
        str(rep),

        "--gap",
        "mean",

        "--num-workers",
        "2",
    ]

    prefix = f"[GPU {gpu_id}] rho={rho} rep={rep}"

    print("\n" + "=" * 80)
    print(f"{prefix} STARTING")
    print("=" * 80)

    with open(
        stdout_path,
        "w",
        encoding="utf-8",
        buffering=1
    ) as out, open(
        stderr_path,
        "w",
        encoding="utf-8",
        buffering=1
    ) as err:

        process = subprocess.Popen(
            cmd,
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
            bufsize=1,
        )

        # ----------------------------------------------------
        # Stream stdout live to notebook + log
        # ----------------------------------------------------

        def stream_stdout():
            for line in process.stdout:
                line = line.rstrip("\n")

                out.write(line + "\n")
                out.flush()

                print(
                    f"{prefix} | {line}",
                    flush=True
                )

        # ----------------------------------------------------
        # Stream stderr live to notebook + log
        # ----------------------------------------------------

        def stream_stderr():
            for line in process.stderr:
                line = line.rstrip("\n")

                err.write(line + "\n")
                err.flush()

                print(
                    f"{prefix} | STDERR | {line}",
                    flush=True
                )

        stdout_thread = threading.Thread(
            target=stream_stdout,
            daemon=True,
        )

        stderr_thread = threading.Thread(
            target=stream_stderr,
            daemon=True,
        )

        stdout_thread.start()
        stderr_thread.start()

        return_code = process.wait()

        stdout_thread.join()
        stderr_thread.join()

    print(
        f"\n{prefix} FINISHED "
        f"return_code={return_code}",
        flush=True
    )

    return {
        "rho": rho,
        "rep": rep,
        "gpu": gpu_id,
        "return_code": return_code,
        "stdout": str(stdout_path),
        "stderr": str(stderr_path),
    }

# ============================================================
# 2-GPU CONCURRENT SCHEDULER
# ============================================================

results = []

print("\n" + "=" * 80)
print("STARTING 2-GPU SCHEDULER")
print("=" * 80)

with ThreadPoolExecutor(max_workers=2) as executor:

    active = {}

    # Start first two models.
    for gpu_id, model in enumerate(to_run[:2]):

        future = executor.submit(
            run_model,
            model,
            gpu_id
        )

        active[future] = gpu_id

    next_index = 2

    # Reuse GPU immediately after its model finishes.
    while active:

        finished_future = next(
            as_completed(active)
        )

        gpu_id = active.pop(
            finished_future
        )

        result = finished_future.result()
        results.append(result)

        if result["return_code"] != 0:

            raise RuntimeError(
                f"Model failed: "
                f"rho={result['rho']} "
                f"rep={result['rep']} "
                f"return_code={result['return_code']}\n"
                f"See: {result['stderr']}"
            )

        if next_index < len(to_run):

            model = to_run[next_index]
            next_index += 1

            future = executor.submit(
                run_model,
                model,
                gpu_id
            )

            active[future] = gpu_id

# ============================================================
# FINAL AUDIT
# ============================================================

print("\n" + "=" * 80)
print("RESUME RUN FINISHED — VERIFYING ALL 9 MODELS")
print("=" * 80)

audit = []

for rho, rep in ALL_MODELS:

    name = f"rho_{rho}_rep_{rep}"
    model_dir = ROOT / name
    record_path = model_dir / "training_record.json"

    row = {
        "rho": rho,
        "replicate": rep,
        "directory_exists": model_dir.exists(),
        "record_exists": record_path.exists(),
        "status": None,
        "script_sha256": None,
        "valid": False,
    }

    if record_path.exists():

        try:
            record = json.loads(
                record_path.read_text()
            )

            row["status"] = record.get("status")
            row["script_sha256"] = record.get(
                "script_sha256"
            )

            row["valid"] = (
                record.get("status") == "COMPLETED"
                and record.get("mode") == "primary"
                and str(record.get("rho")) == rho
                and int(record.get("replicate")) == rep
                and record.get("architecture") == "ConvNeXt-Tiny"
                and record.get("endpoint") == "Delta-z"
                and record.get("xai") is False
                and record.get("script_sha256")
                    == EXPECTED_SCRIPT_SHA
            )

        except Exception:
            pass

    audit.append(row)

    print(
        f"rho={rho} rep={rep} "
        f"status={row['status']} "
        f"valid={row['valid']}"
    )

assert all(
    x["valid"] for x in audit
), "At least one T2.1 model failed final validation."

manifest = {
    "status": "COMPLETED",
    "script_sha256": EXPECTED_SCRIPT_SHA,
    "seed_table_sha256": EXPECTED_SEED_SHA,
    "completed_before_resume": [
        {
            "rho": rho,
            "replicate": rep,
        }
        for rho, rep in completed
    ],
    "models_run_this_resume": [
        {
            "rho": rho,
            "replicate": rep,
        }
        for rho, rep in to_run
    ],
    "results": results,
    "final_audit": audit,
}

manifest_path = (
    ROOT / "T2_1_RESUME_LAUNCH_MANIFEST.json"
)

manifest_path.write_text(
    json.dumps(
        manifest,
        indent=2
    ),
    encoding="utf-8"
)

print("\n" + "=" * 80)
print("T2.1 RESUME PRODUCTION: ALL 9 MODELS VALID")
print("=" * 80)

print("Manifest:", manifest_path)

In [ ]:
from pathlib import Path
import json

ROOT = Path("/kaggle/working/T2_1_PRODUCTION")

ALL_MODELS = [
    ("0.0", 1),
    ("0.0", 2),
    ("0.0", 3),
    ("0.5", 1),
    ("0.5", 2),
    ("0.5", 3),
    ("1.0", 1),
    ("1.0", 2),
    ("1.0", 3),
]

print("=" * 80)
print("T2.1 EXISTING MODEL RECORD DIAGNOSTIC")
print("=" * 80)

for rho, rep in ALL_MODELS:

    name = f"rho_{rho}_rep_{rep}"
    record_path = ROOT / name / "training_record.json"

    print(f"\n{name}")
    print("-" * 50)

    if not record_path.exists():
        print("NO training_record.json")
        continue

    try:
        record = json.loads(record_path.read_text())

        print("status      :", repr(record.get("status")))
        print("mode        :", repr(record.get("mode")))
        print("rho         :", repr(record.get("rho")))
        print("replicate   :", repr(record.get("replicate")))
        print("architecture:", repr(record.get("architecture")))
        print("endpoint    :", repr(record.get("endpoint")))
        print("xai         :", repr(record.get("xai")))
        print("script_sha  :", repr(record.get("script_sha256")))

    except Exception as e:
        print("ERROR READING RECORD:", repr(e))

In [ ]:
from pathlib import Path
import subprocess
import os
import json
import hashlib
import shutil
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed

# ============================================================
# LOCKED PATHS
# ============================================================

SCRIPT = Path(
    "/kaggle/working/T2_1_ConvNeXt_training_code_freeze/"
    "train_convnext_tiny_t2_1.py"
)

PROJECT = Path(
    "/kaggle/input/notebooks/tejasseshadriiiii/"
    "isic-kaggle-analysis/isic_final_project"
)

CACHE = Path(
    "/kaggle/working/T2_1_PRODUCTION_CACHE"
)

ROOT = Path(
    "/kaggle/working/T2_1_PRODUCTION"
)

SEED_TABLE = Path(
    "/kaggle/working/T2_1_ConvNeXt_seed_lock/"
    "T2_1_TRAINING_SEED_TABLE.csv"
)

CURRENT_SCRIPT_SHA = (
    "d4fb712431e6ad02c816d728e5cfeb513490a9bc500a4ee0deca41784bf296d1"
)

OLD_COMPLETED_SCRIPT_SHA = (
    "d6b97a547e96d4f14cb97b0bfa69c5219c798e0f5371a3de0d4c6bdaba93b379"
)

EXPECTED_SEED_SHA = (
    "9c54a1a617f676750ef4abd174d0374a7ccce69e633c1f8dad86a3ee03823a5d"
)

ROOT.mkdir(parents=True, exist_ok=True)

# ============================================================
# IDENTITY GATES
# ============================================================

actual_script_sha = hashlib.sha256(
    SCRIPT.read_bytes()
).hexdigest()

actual_seed_sha = hashlib.sha256(
    SEED_TABLE.read_bytes()
).hexdigest()

assert actual_script_sha == CURRENT_SCRIPT_SHA, (
    f"Unexpected CURRENT training script SHA:\n{actual_script_sha}"
)

assert actual_seed_sha == EXPECTED_SEED_SHA, (
    f"Unexpected T2.1 seed table SHA:\n{actual_seed_sha}"
)

print("=" * 80)
print("T2.1 RESUME PRODUCTION LAUNCH")
print("=" * 80)

print("Current training script SHA:", actual_script_sha)
print("T2.1 seed table SHA:", actual_seed_sha)

# ============================================================
# FULL LOCKED 9-MODEL ROSTER
# ============================================================

ALL_MODELS = [
    (0.0, 1),
    (0.0, 2),
    (0.0, 3),
    (0.5, 1),
    (0.5, 2),
    (0.5, 3),
    (1.0, 1),
    (1.0, 2),
    (1.0, 3),
]

# ============================================================
# VALID COMPLETION TEST
# ============================================================

def is_valid_completed_record(record, rho, rep):

    if record.get("status") != "COMPLETED":
        return False

    if record.get("mode") != "primary":
        return False

    # Numeric comparison because JSON stores rho as float.
    try:
        record_rho = float(record.get("rho"))
    except Exception:
        return False

    if record_rho != float(rho):
        return False

    try:
        record_rep = int(record.get("replicate"))
    except Exception:
        return False

    if record_rep != int(rep):
        return False

    if record.get("architecture") != "ConvNeXt-Tiny":
        return False

    if record.get("endpoint") != "Delta-z":
        return False

    if record.get("xai") is not False:
        return False

    # --------------------------------------------------------
    # IMPORTANT:
    # Existing rho=0 rep1/2 completed before the seed-table
    # patch and therefore legitimately carry the old SHA.
    #
    # New reruns MUST carry CURRENT_SCRIPT_SHA.
    # --------------------------------------------------------

    record_sha = record.get("script_sha256")

    allowed_shas = {
        CURRENT_SCRIPT_SHA,
        OLD_COMPLETED_SCRIPT_SHA,
    }

    if record_sha not in allowed_shas:
        return False

    return True


# ============================================================
# IDENTIFY EXISTING COMPLETED MODELS
# ============================================================

completed = []
to_run = []

for rho, rep in ALL_MODELS:

    name = f"rho_{rho:.1f}_rep_{rep}"
    model_dir = ROOT / name
    record_path = model_dir / "training_record.json"

    valid_completed = False

    if record_path.exists():

        try:
            record = json.loads(
                record_path.read_text(
                    encoding="utf-8"
                )
            )

            valid_completed = is_valid_completed_record(
                record,
                rho,
                rep
            )

        except Exception:
            valid_completed = False

    if valid_completed:
        completed.append((rho, rep))
    else:
        to_run.append((rho, rep))


# ============================================================
# HARD SAFETY CHECK
# ============================================================

expected_completed = [
    (0.0, 1),
    (0.0, 2),
]

assert completed == expected_completed, (
    f"Unexpected completed-model set:\n"
    f"Found    : {completed}\n"
    f"Expected : {expected_completed}"
)

assert len(to_run) == 7

# ============================================================
# REPORT
# ============================================================

print("\nCompleted models — WILL NOT BE RETRAINED:")

for rho, rep in completed:
    print(
        f"  rho={rho:.1f} rep={rep}"
    )

print("\nModels that WILL be trained:")

for rho, rep in to_run:
    print(
        f"  rho={rho:.1f} rep={rep}"
    )

# ============================================================
# DELETE ONLY INCOMPLETE MODEL DIRECTORIES
# ============================================================

print("\nCleaning incomplete model directories...")

for rho, rep in to_run:

    name = f"rho_{rho:.1f}_rep_{rep}"
    model_dir = ROOT / name

    if model_dir.exists():

        print(
            f"  Removing incomplete: {name}"
        )

        shutil.rmtree(model_dir)


# ============================================================
# LIVE MODEL RUNNER
# ============================================================

def run_model(model, gpu_id):

    rho, rep = model

    name = f"rho_{rho:.1f}_rep_{rep}"

    model_dir = ROOT / name
    model_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    stdout_path = (
        model_dir /
        "production_stdout.log"
    )

    stderr_path = (
        model_dir /
        "production_stderr.log"
    )

    env = os.environ.copy()

    # One GPU per training subprocess.
    env["CUDA_VISIBLE_DEVICES"] = str(gpu_id)

    # Locked deterministic CUDA setting.
    env["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

    cmd = [
        "python",
        "-u",
        str(SCRIPT),

        "--project",
        str(PROJECT),

        "--cache-dir",
        str(CACHE),

        "--out-dir",
        str(model_dir),

        "--mode",
        "primary",

        "--rho",
        str(rho),

        "--rep",
        str(rep),

        "--gap",
        "mean",

        "--num-workers",
        "2",
    ]

    prefix = (
        f"[GPU {gpu_id}] "
        f"rho={rho:.1f} rep={rep}"
    )

    print("\n" + "=" * 80)
    print(f"{prefix} STARTING")
    print("=" * 80)

    with open(
        stdout_path,
        "w",
        encoding="utf-8",
        buffering=1
    ) as out, open(
        stderr_path,
        "w",
        encoding="utf-8",
        buffering=1
    ) as err:

        process = subprocess.Popen(
            cmd,
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
            bufsize=1,
        )

        # ----------------------------------------------------
        # LIVE STDOUT
        # ----------------------------------------------------

        def stream_stdout():

            for line in process.stdout:

                line = line.rstrip("\n")

                out.write(
                    line + "\n"
                )
                out.flush()

                print(
                    f"{prefix} | {line}",
                    flush=True
                )

        # ----------------------------------------------------
        # LIVE STDERR
        # ----------------------------------------------------

        def stream_stderr():

            for line in process.stderr:

                line = line.rstrip("\n")

                err.write(
                    line + "\n"
                )
                err.flush()

                print(
                    f"{prefix} | STDERR | {line}",
                    flush=True
                )

        stdout_thread = threading.Thread(
            target=stream_stdout,
            daemon=True
        )

        stderr_thread = threading.Thread(
            target=stream_stderr,
            daemon=True
        )

        stdout_thread.start()
        stderr_thread.start()

        return_code = process.wait()

        stdout_thread.join()
        stderr_thread.join()

    print(
        f"\n{prefix} FINISHED "
        f"return_code={return_code}",
        flush=True
    )

    return {
        "rho": rho,
        "replicate": rep,
        "gpu": gpu_id,
        "return_code": return_code,
        "stdout": str(stdout_path),
        "stderr": str(stderr_path),
    }


# ============================================================
# 2-GPU CONCURRENT SCHEDULER
# ============================================================

results = []

print("\n" + "=" * 80)
print("STARTING 2-GPU CONCURRENT TRAINING")
print("=" * 80)

with ThreadPoolExecutor(
    max_workers=2
) as executor:

    active = {}

    # --------------------------------------------------------
    # Start first two jobs.
    # --------------------------------------------------------

    for gpu_id, model in enumerate(
        to_run[:2]
    ):

        future = executor.submit(
            run_model,
            model,
            gpu_id
        )

        active[future] = gpu_id

    next_index = 2

    # --------------------------------------------------------
    # As each GPU becomes free, assign next model.
    # --------------------------------------------------------

    while active:

        finished_future = next(
            as_completed(active)
        )

        gpu_id = active.pop(
            finished_future
        )

        result = finished_future.result()

        results.append(result)

        if result["return_code"] != 0:

            raise RuntimeError(
                f"\nMODEL FAILED\n"
                f"rho={result['rho']}\n"
                f"rep={result['replicate']}\n"
                f"return_code={result['return_code']}\n"
                f"stderr={result['stderr']}"
            )

        if next_index < len(to_run):

            model = to_run[next_index]

            next_index += 1

            future = executor.submit(
                run_model,
                model,
                gpu_id
            )

            active[future] = gpu_id


# ============================================================
# FINAL 9-MODEL AUDIT
# ============================================================

print("\n" + "=" * 80)
print("ALL TRAINING JOBS FINISHED")
print("RUNNING FINAL 9-MODEL AUDIT")
print("=" * 80)

audit = []

for rho, rep in ALL_MODELS:

    name = f"rho_{rho:.1f}_rep_{rep}"

    model_dir = ROOT / name
    record_path = (
        model_dir /
        "training_record.json"
    )

    row = {
        "rho": rho,
        "replicate": rep,
        "record_exists": record_path.exists(),
        "status": None,
        "script_sha256": None,
        "valid": False,
    }

    if record_path.exists():

        try:

            record = json.loads(
                record_path.read_text(
                    encoding="utf-8"
                )
            )

            row["status"] = record.get(
                "status"
            )

            row["script_sha256"] = record.get(
                "script_sha256"
            )

            row["valid"] = is_valid_completed_record(
                record,
                rho,
                rep
            )

        except Exception:
            pass

    audit.append(row)

    print(
        f"rho={rho:.1f} "
        f"rep={rep} "
        f"status={row['status']} "
        f"script_sha={row['script_sha256']} "
        f"valid={row['valid']}"
    )


# ============================================================
# FINAL REQUIREMENT
# ============================================================

assert all(
    row["valid"]
    for row in audit
), "FINAL AUDIT FAILED"


# ============================================================
# MANIFEST
# ============================================================

manifest = {
    "status": "COMPLETED",
    "current_script_sha256": CURRENT_SCRIPT_SHA,
    "seed_table_sha256": EXPECTED_SEED_SHA,

    "previously_completed": [
        {
            "rho": rho,
            "replicate": rep,
        }
        for rho, rep in completed
    ],

    "trained_in_resume": [
        {
            "rho": rho,
            "replicate": rep,
        }
        for rho, rep in to_run
    ],

    "results": results,

    "final_audit": audit,
}

manifest_path = (
    ROOT /
    "T2_1_RESUME_LAUNCH_MANIFEST.json"
)

manifest_path.write_text(
    json.dumps(
        manifest,
        indent=2
    ),
    encoding="utf-8"
)

print("\n" + "=" * 80)
print("T2.1 PRODUCTION: ALL 9 MODELS VALID")
print("=" * 80)

print(
    "Manifest:",
    manifest_path
)

In [ ]:
from pathlib import Path

print("=" * 80)
print("T2.1 ASSIGNMENT FILE CHECK")
print("=" * 80)

rho05_dir = Path(
    "/kaggle/working/T2_1_ConvNeXt_rho05_assignments"
)

print("\nT2.1 rho=0.5 assignment directory:")
print(rho05_dir)
print("Exists:", rho05_dir.exists())

if rho05_dir.exists():
    print("\nFiles:")
    for p in sorted(rho05_dir.glob("*")):
        print(" ", p.name)

print("\nExpected rho=0.5 files:")
for rep in (1, 2, 3):
    p = rho05_dir / f"rho_0.5_rep_{rep}.csv"
    print(
        f"rep={rep}: {p} | exists={p.exists()}"
    )

In [ ]:
from pathlib import Path

script = Path(
    "/kaggle/working/T2_1_ConvNeXt_training_code_freeze/"
    "train_convnext_tiny_t2_1.py"
)

lines = script.read_text(
    encoding="utf-8"
).splitlines()

print("=" * 80)
print("ASSIGNMENT PATH CODE — EXACT SOURCE")
print("=" * 80)

for i in range(550, 610):
    if i <= len(lines):
        print(f"{i:4d}: {lines[i-1]}")

In [ ]:
from pathlib import Path
import hashlib
import json
import py_compile

# ============================================================
# T2.1 SURGICAL ASSIGNMENT-PATH PATCH
# ============================================================

SCRIPT = Path(
    "/kaggle/working/T2_1_ConvNeXt_training_code_freeze/"
    "train_convnext_tiny_t2_1.py"
)

BACKUP = Path(
    "/kaggle/working/T2_1_ConvNeXt_training_code_freeze/"
    "train_convnext_tiny_t2_1.pre_assignment_path_patch.py"
)

RHO05_DIR = Path(
    "/kaggle/working/T2_1_ConvNeXt_rho05_assignments"
)

EXPECTED_BEFORE_SHA = (
    "d4fb712431e6ad02c816d728e5cfeb513490a9bc500a4ee0deca41784bf296d1"
)

# ------------------------------------------------------------
# Verify current script identity
# ------------------------------------------------------------

before_sha = hashlib.sha256(
    SCRIPT.read_bytes()
).hexdigest()

assert before_sha == EXPECTED_BEFORE_SHA, (
    f"Unexpected pre-patch script SHA:\n{before_sha}"
)

# ------------------------------------------------------------
# Verify all three locked rho=.5 assignments exist
# ------------------------------------------------------------

for rep in (1, 2, 3):

    p = RHO05_DIR / f"rho_0.5_rep_{rep}.csv"

    assert p.exists(), (
        f"Missing locked rho=.5 assignment:\n{p}"
    )

# ------------------------------------------------------------
# Create immutable pre-patch backup
# ------------------------------------------------------------

assert not BACKUP.exists(), (
    f"Backup already exists:\n{BACKUP}"
)

BACKUP.write_bytes(
    SCRIPT.read_bytes()
)

# ------------------------------------------------------------
# Exact old block
# ------------------------------------------------------------

old_block = '''    assignment = pd.read_csv(
        project
        / "step1F_lesion_level_rho_assignments"
        / f"rho_{rho:.1f}_rep_{rep}.csv"
    ).sort_values(
        "image"
    ).reset_index(
        drop=True
    )
'''

# ------------------------------------------------------------
# Surgical replacement
# ------------------------------------------------------------

new_block = '''    # ----------------------------------------------------------------------
    # T2.1 architecture-replication assignment source
    #
    # rho=0.0 and rho=1.0 reuse the frozen Step1F assignments.
    # rho=0.5 uses the separately generated and integrity-locked
    # T2.1 assignments.
    # ----------------------------------------------------------------------

    if abs(float(rho) - 0.5) < 1e-12:

        assignment_path = (
            Path("/kaggle/working/T2_1_ConvNeXt_rho05_assignments")
            / f"rho_{rho:.1f}_rep_{rep}.csv"
        )

    else:

        assignment_path = (
            project
            / "step1F_lesion_level_rho_assignments"
            / f"rho_{rho:.1f}_rep_{rep}.csv"
        )

    assert assignment_path.exists(), (
        f"Assignment file not found: {assignment_path}"
    )

    assignment = pd.read_csv(
        assignment_path
    ).sort_values(
        "image"
    ).reset_index(
        drop=True
    )
'''

assert old_block in SCRIPT.read_text(
    encoding="utf-8"
), "Expected assignment block was not found exactly."

text = SCRIPT.read_text(
    encoding="utf-8"
)

text = text.replace(
    old_block,
    new_block,
    1
)

SCRIPT.write_text(
    text,
    encoding="utf-8"
)

# ------------------------------------------------------------
# Verify only one replacement occurred
# ------------------------------------------------------------

updated = SCRIPT.read_text(
    encoding="utf-8"
)

assert new_block in updated
assert old_block not in updated

# ------------------------------------------------------------
# Compile
# ------------------------------------------------------------

py_compile.compile(
    str(SCRIPT),
    doraise=True
)

# ------------------------------------------------------------
# New SHA
# ------------------------------------------------------------

after_sha = hashlib.sha256(
    SCRIPT.read_bytes()
).hexdigest()

print("=" * 80)
print("T2.1 ASSIGNMENT-PATH PATCH")
print("=" * 80)

print("Before SHA:", before_sha)
print("After  SHA:", after_sha)
print("Backup    :", BACKUP)

print("\nAssignment routing:")
print("  rho=0.0 -> frozen Step1F assignment")
print("  rho=0.5 -> T2.1 generated assignment")
print("  rho=1.0 -> frozen Step1F assignment")

print("\nCompile: PASS")

In [ ]:
from pathlib import Path
import pandas as pd

PROJECT = Path(
    "/kaggle/input/notebooks/tejasseshadriiiii/"
    "isic-kaggle-analysis/isic_final_project"
)

RHO05_DIR = Path(
    "/kaggle/working/T2_1_ConvNeXt_rho05_assignments"
)

print("=" * 80)
print("T2.1 ASSIGNMENT ROUTING TEST")
print("=" * 80)

for rho in (0.0, 0.5, 1.0):

    for rep in (1, 2, 3):

        if abs(rho - 0.5) < 1e-12:

            path = (
                RHO05_DIR
                / f"rho_{rho:.1f}_rep_{rep}.csv"
            )

            source = "T2.1 rho=.5 generated"

        else:

            path = (
                PROJECT
                / "step1F_lesion_level_rho_assignments"
                / f"rho_{rho:.1f}_rep_{rep}.csv"
            )

            source = "frozen Step1F"

        assert path.exists(), (
            f"Missing assignment:\n{path}"
        )

        df = pd.read_csv(path)

        assert len(df) == 9257, (
            f"Unexpected row count for rho={rho}, rep={rep}: "
            f"{len(df)}"
        )

        assert df["image"].is_unique, (
            f"Duplicate image IDs for rho={rho}, rep={rep}"
        )

        assert (
            df["split"] == "train"
        ).all(), (
            f"Non-train rows for rho={rho}, rep={rep}"
        )

        assert (
            df["lesion_id"].notna()
        ).all(), (
            f"Missing lesion IDs for rho={rho}, rep={rep}"
        )

        assert (
            df["hair_present"].isin([0, 1]).all()
        ), (
            f"Invalid hair_present values for rho={rho}, rep={rep}"
        )

        print(
            f"rho={rho:.1f} rep={rep} | "
            f"{source} | "
            f"rows={len(df)} | "
            f"unique_images={df['image'].nunique()} | "
            f"PASS"
        )

print()
print("=" * 80)
print("ALL 9 ASSIGNMENT PATHS: PASS")
print("=" * 80)

In [ ]:
from pathlib import Path
import pandas as pd

PROJECT = Path(
    "/kaggle/input/notebooks/tejasseshadriiiii/"
    "isic-kaggle-analysis/isic_final_project"
)

RHO05_DIR = Path(
    "/kaggle/working/T2_1_ConvNeXt_rho05_assignments"
)

print("=" * 80)
print("T2.1 ASSIGNMENT ROUTING + STRUCTURAL TEST")
print("=" * 80)

for rho in (0.0, 0.5, 1.0):

    for rep in (1, 2, 3):

        # ----------------------------------------------------
        # Resolve expected source
        # ----------------------------------------------------

        if abs(rho - 0.5) < 1e-12:

            path = (
                RHO05_DIR
                / f"rho_{rho:.1f}_rep_{rep}.csv"
            )

            source = "T2.1 rho=.5 generated"
            expected_rows = 9257

        else:

            path = (
                PROJECT
                / "step1F_lesion_level_rho_assignments"
                / f"rho_{rho:.1f}_rep_{rep}.csv"
            )

            source = "frozen Step1F"
            expected_rows = 18514

        # ----------------------------------------------------
        # Existence
        # ----------------------------------------------------

        assert path.exists(), (
            f"Missing assignment:\n{path}"
        )

        # ----------------------------------------------------
        # Read
        # ----------------------------------------------------

        df = pd.read_csv(path)

        # ----------------------------------------------------
        # Row-count appropriate to assignment source
        # ----------------------------------------------------

        assert len(df) == expected_rows, (
            f"Unexpected row count for "
            f"rho={rho:.1f}, rep={rep}: "
            f"got {len(df)}, expected {expected_rows}"
        )

        # ----------------------------------------------------
        # Required columns
        # ----------------------------------------------------

        required_columns = {
            "image",
            "lesion_id",
            "split",
            "hair_present",
        }

        assert required_columns.issubset(
            df.columns
        ), (
            f"Missing required columns for "
            f"rho={rho:.1f}, rep={rep}"
        )

        # ----------------------------------------------------
        # Basic integrity
        # ----------------------------------------------------

        assert df["image"].is_unique, (
            f"Duplicate image IDs for "
            f"rho={rho:.1f}, rep={rep}"
        )

        assert (
            df["split"] == "train"
        ).all(), (
            f"Non-train rows for "
            f"rho={rho:.1f}, rep={rep}"
        )

        assert (
            df["lesion_id"].notna()
        ).all(), (
            f"Missing lesion IDs for "
            f"rho={rho:.1f}, rep={rep}"
        )

        assert (
            df["hair_present"].isin([0, 1]).all()
        ), (
            f"Invalid hair_present values for "
            f"rho={rho:.1f}, rep={rep}"
        )

        # ----------------------------------------------------
        # Report
        # ----------------------------------------------------

        print(
            f"rho={rho:.1f} rep={rep} | "
            f"{source} | "
            f"rows={len(df)} | "
            f"unique_images={df['image'].nunique()} | "
            f"PASS"
        )

print()
print("=" * 80)
print("ALL 9 ASSIGNMENT ROUTES: PASS")
print("=" * 80)

In [ ]:
from pathlib import Path
import pandas as pd
import json
import hashlib

RHO05_DIR = Path(
    "/kaggle/working/T2_1_ConvNeXt_rho05_assignments"
)

print("=" * 80)
print("T2.1 RHO=.5 ASSIGNMENT — ACTUAL CONTENT AUDIT")
print("=" * 80)

for rep in (1, 2, 3):

    path = RHO05_DIR / f"rho_0.5_rep_{rep}.csv"

    print(f"\n{'='*70}")
    print(f"rho=0.5 rep={rep}")
    print(f"{'='*70}")

    df = pd.read_csv(path)

    sha = hashlib.sha256(
        path.read_bytes()
    ).hexdigest()

    print("Rows:", len(df))
    print("Columns:", list(df.columns))
    print("SHA256:", sha)

    print("\nSplit:")
    print(df["split"].value_counts(dropna=False).to_dict())

    print("\nHair-present:")
    print(
        df["hair_present"]
        .value_counts(dropna=False)
        .sort_index()
        .to_dict()
    )

    print("\nUnique images:", df["image"].nunique())
    print("Unique lesions:", df["lesion_id"].nunique())

    print("\nHair by lesion consistency:")

    lesion_hair_nunique = (
        df.groupby("lesion_id")["hair_present"]
        .nunique()
    )

    print(
        "Max unique hair_present per lesion:",
        lesion_hair_nunique.max()
    )

    print(
        "Lesions with >1 hair_present value:",
        int((lesion_hair_nunique > 1).sum())
    )

    print("\nFirst 5 rows:")
    print(df.head(5).to_string(index=False))

# ============================================================
# GENERATION MANIFEST
# ============================================================

manifest_path = (
    RHO05_DIR /
    "T2_1_RHO05_GENERATION_MANIFEST.json"
)

print("\n" + "=" * 80)
print("GENERATION MANIFEST")
print("=" * 80)

manifest = json.loads(
    manifest_path.read_text(
        encoding="utf-8"
    )
)

print(
    json.dumps(
        manifest,
        indent=2
    )
)

In [ ]:
from pathlib import Path
import pandas as pd
import hashlib
import json

PROJECT = Path(
    "/kaggle/input/notebooks/tejasseshadriiiii/"
    "isic-kaggle-analysis/isic_final_project"
)

RHO05_DIR = Path(
    "/kaggle/working/T2_1_ConvNeXt_rho05_assignments"
)

MANIFEST_PATH = (
    RHO05_DIR /
    "T2_1_RHO05_GENERATION_MANIFEST.json"
)

manifest = json.loads(
    MANIFEST_PATH.read_text(
        encoding="utf-8"
    )
)

EXPECTED_SHA = {
    1: "631aad30c4d4952d4a2a68092a6b2323d19e9b93cb0f69ed36b9cf7b2c70a877",
    2: "a63d0bad295b0dc38677d3780cf1ad77aa20e3fa12b917b6bebd3d91ed3dc290",
    3: "22969500b6481a905c97e86dab77dea6cf30c6c6aa90914994ecfa1fc3f93066",
}

print("=" * 80)
print("T2.1 FINAL ASSIGNMENT ROUTING TEST")
print("=" * 80)

for rho in (0.0, 0.5, 1.0):

    for rep in (1, 2, 3):

        # ----------------------------------------------------
        # Correct source routing
        # ----------------------------------------------------

        if abs(rho - 0.5) < 1e-12:

            path = (
                RHO05_DIR /
                f"rho_{rho:.1f}_rep_{rep}.csv"
            )

            source = "T2.1 generated rho=.5"

        else:

            path = (
                PROJECT /
                "step1F_lesion_level_rho_assignments" /
                f"rho_{rho:.1f}_rep_{rep}.csv"
            )

            source = "frozen Step1F"

        assert path.exists(), (
            f"Missing assignment:\n{path}"
        )

        df = pd.read_csv(path)

        # ----------------------------------------------------
        # All nine assignment files represent the complete
        # 18,514-image training population.
        # ----------------------------------------------------

        assert len(df) == 18514, (
            f"Unexpected row count for "
            f"rho={rho:.1f}, rep={rep}: "
            f"{len(df)}"
        )

        assert df["image"].nunique() == 18514

        assert df["image"].is_unique

        assert (
            df["split"] == "train"
        ).all()

        assert df["lesion_id"].notna().all()

        assert (
            df["hair_present"]
            .isin([0, 1])
            .all()
        )

        # ----------------------------------------------------
        # rho=.5 specific locked checks
        # ----------------------------------------------------

        if abs(rho - 0.5) < 1e-12:

            actual_sha = hashlib.sha256(
                path.read_bytes()
            ).hexdigest()

            assert actual_sha == EXPECTED_SHA[rep], (
                f"SHA mismatch for rho=.5 rep={rep}\n"
                f"Expected: {EXPECTED_SHA[rep]}\n"
                f"Actual:   {actual_sha}"
            )

            assert (
                int(
                    (df["hair_present"] == 1).sum()
                ) == 9257
            )

            assert (
                int(
                    (df["hair_present"] == 0).sum()
                ) == 9257
            )

            lesion_hair_nunique = (
                df.groupby("lesion_id")[
                    "hair_present"
                ].nunique()
            )

            assert (
                lesion_hair_nunique.max() == 1
            )

            assert (
                (lesion_hair_nunique > 1).sum() == 0
            )

        print(
            f"rho={rho:.1f} rep={rep} | "
            f"{source} | "
            f"rows={len(df)} | "
            f"hair={int((df['hair_present'] == 1).sum())} | "
            f"PASS"
        )

print()
print("=" * 80)
print("ALL 9 ASSIGNMENT ROUTES: PASS")
print("T2.1 ASSIGNMENT INPUTS ARE LOCKED AND VALID")
print("=" * 80)

In [ ]:
from pathlib import Path

SCRIPT = Path(
    "/kaggle/working/T2_1_ConvNeXt_training_code_freeze/"
    "train_convnext_tiny_t2_1.py"
)

assert SCRIPT.exists(), SCRIPT

lines = SCRIPT.read_text(
    encoding="utf-8"
).splitlines()

print("=" * 80)
print("T2.1 TRAINING LOOP / CHECKPOINT INSPECTION")
print("=" * 80)

patterns = [
    "for epoch",
    "torch.save",
    "state_dict",
    "checkpoint",
    "optimizer",
    "scheduler",
    "best_eval",
    "final",
]

hits = []

for i, line in enumerate(lines, start=1):

    if any(
        p.lower() in line.lower()
        for p in patterns
    ):
        hits.append(i)

print("\nRelevant lines:")

for i in hits:
    print(
        f"{i:5d}: {lines[i-1]}"
    )

print("\n" + "=" * 80)
print("TRAINING LOOP CONTEXT")
print("=" * 80)

# Print context around the main epoch loop.
for i, line in enumerate(lines, start=1):

    if (
        "for epoch" in line.lower()
        and "range" in line.lower()
    ):

        start = max(1, i - 40)
        end = min(len(lines), i + 100)

        print(
            f"\n--- lines {start} to {end} ---"
        )

        for j in range(start, end + 1):
            print(
                f"{j:5d}: {lines[j-1]}"
            )

        break

In [ ]:
from pathlib import Path

SCRIPT = Path(
    "/kaggle/working/T2_1_ConvNeXt_training_code_freeze/"
    "train_convnext_tiny_t2_1.py"
)

lines = SCRIPT.read_text(
    encoding="utf-8"
).splitlines()

print("=" * 80)
print("T2.1 CHECKPOINT / EPOCH-END CONTEXT")
print("=" * 80)

for start, end in [
    (1000, 1145),
    (1145, 1300),
]:
    print(
        f"\n--- lines {start} to {end} ---"
    )

    for i in range(start, min(end, len(lines)) + 1):
        print(
            f"{i:5d}: {lines[i-1]}"
        )

In [ ]:
from pathlib import Path

SCRIPT = Path(
    "/kaggle/working/T2_1_ConvNeXt_training_code_freeze/"
    "train_convnext_tiny_t2_1.py"
)

lines = SCRIPT.read_text(
    encoding="utf-8"
).splitlines()

print("=" * 80)
print("T2.1 RNG + DATALOADER INITIALIZATION")
print("=" * 80)

# Find important initialization sections.
for i, line in enumerate(lines, start=1):

    lower = line.lower()

    if any(
        key in lower
        for key in [
            "seed_everything(",
            "dataloader(",
            "generator",
            "worker_init_fn",
            "seed_workers",
            "seed_dataloader",
            "seed_augmentation",
        ]
    ):
        print(
            f"{i:5d}: {line}"
        )

print("\n" + "=" * 80)
print("CONTEXT AROUND SEED / LOADER SETUP")
print("=" * 80)

# Print around the first seed_everything occurrence.
seed_lines = [
    i for i, line in enumerate(lines, start=1)
    if "seed_everything(" in line
]

if seed_lines:

    center = seed_lines[-1]

    start = max(
        1,
        center - 70
    )

    end = min(
        len(lines),
        center + 100
    )

    print(
        f"\n--- lines {start} to {end} ---"
    )

    for i in range(start, end + 1):
        print(
            f"{i:5d}: {lines[i-1]}"
        )

In [ ]:
from pathlib import Path
import hashlib
import json
import py_compile

# ============================================================
# T2.1 EPOCH-RESUME PATCH
# ============================================================

SCRIPT = Path(
    "/kaggle/working/T2_1_ConvNeXt_training_code_freeze/"
    "train_convnext_tiny_t2_1.py"
)

BACKUP = Path(
    "/kaggle/working/T2_1_ConvNeXt_training_code_freeze/"
    "train_convnext_tiny_t2_1.pre_epoch_resume_patch.py"
)

EXPECTED_BEFORE_SHA = (
    "28bf6a0d08459587b8bec26b7f8c7aa67785fb8ec9957caddd46ed4c72be0a5b"
)

text = SCRIPT.read_text(
    encoding="utf-8"
)

before_sha = hashlib.sha256(
    SCRIPT.read_bytes()
).hexdigest()

assert before_sha == EXPECTED_BEFORE_SHA, (
    f"Unexpected script SHA:\n{before_sha}"
)

assert not BACKUP.exists(), (
    f"Backup already exists:\n{BACKUP}"
)

BACKUP.write_bytes(
    SCRIPT.read_bytes()
)

# ============================================================
# PATCH 1 — ADD RESUME CHECKPOINT CONSTANT
# ============================================================

old = '''PRIMARY_CHECKPOINT = "final_epoch15.pt"
BEST_CHECKPOINT = "best_eval_ba.pt"
'''

new = '''PRIMARY_CHECKPOINT = "final_epoch15.pt"
BEST_CHECKPOINT = "best_eval_ba.pt"

# Epoch-level crash/session resume checkpoint.
RESUME_CHECKPOINT = "epoch_resume.pt"
'''

assert old in text, (
    "Checkpoint constant block not found."
)

text = text.replace(
    old,
    new,
    1
)

# ============================================================
# PATCH 2 — ADD RESUME STATE AFTER LOADER CREATION
# ============================================================

old = '''    eval_loader = torch.utils.data.DataLoader(
'''

new = '''    # ----------------------------------------------------------------------
    # Epoch-resume state
    #
    # The DataLoader generator is explicitly checkpointed because its state
    # controls deterministic epoch-to-epoch shuffling/worker seed derivation.
    # ----------------------------------------------------------------------

    resume_path = out / RESUME_CHECKPOINT

    resume_state = None

    if args.mode == "primary" and resume_path.exists():

        resume_state = torch.load(
            resume_path,
            map_location="cpu",
            weights_only=False,
        )

        assert (
            resume_state["architecture"]
            == ARCHITECTURE
        )

        assert (
            float(resume_state["rho"])
            == float(rho)
        )

        assert (
            int(resume_state["replicate"])
            == int(rep)
        )

        assert (
            resume_state["epochs"]
            == N_EPOCHS
        )

        assert (
            resume_state["seed_init"]
            == str(seeds["seed_init"])
        )

        assert (
            resume_state["seed_dataloader"]
            == str(seeds["seed_dataloader"])
        )

        model.load_state_dict(
            resume_state["model_state"]
        )

        opt.load_state_dict(
            resume_state["optimizer_state"]
        )

        sched.load_state_dict(
            resume_state["scheduler_state"]
        )

        g.set_state(
            resume_state["dataloader_generator_state"]
        )

        random.setstate(
            resume_state["python_rng_state"]
        )

        np.random.set_state(
            resume_state["numpy_rng_state"]
        )

        torch.set_rng_state(
            resume_state["torch_rng_state"]
        )

        if torch.cuda.is_available():

            cuda_states = resume_state.get(
                "cuda_rng_states"
            )

            if cuda_states is not None:

                torch.cuda.set_rng_state_all(
                    cuda_states
                )

        start_epoch = (
            int(
                resume_state["completed_epoch"]
            )
            + 1
        )

        log_rows = list(
            resume_state["log_rows"]
        )

        best = dict(
            resume_state["best"]
        )

        print(
            f"RESUMING FROM COMPLETED EPOCH "
            f"{resume_state['completed_epoch']:02d}; "
            f"next epoch={start_epoch:02d}",
            flush=True,
        )

    else:

        start_epoch = 1

'''

# This replacement is deliberately anchored at eval_loader.
assert old in text, (
    "eval_loader construction anchor not found."
)

# IMPORTANT:
# We cannot insert the resume block before eval_loader itself because
# model/optimizer/scheduler/g are needed. Therefore this intermediate
# replacement is not used directly.
#
# Revert this attempted conceptual anchor and use the exact location
# after loader creation below.

text = text.replace(
    new,
    old,
    1
)

# ============================================================
# PATCH 2 — INSERT AFTER eval_loader BLOCK
# ============================================================

anchor = '''    eval_loader = torch.utils.data.DataLoader(
        eval_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True,
    )

'''

assert anchor in text, (
    "Exact eval_loader block not found."
)

resume_block = '''    eval_loader = torch.utils.data.DataLoader(
        eval_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True,
    )

    # ----------------------------------------------------------------------
    # Epoch-resume state
    #
    # The DataLoader generator is explicitly checkpointed because its state
    # controls deterministic epoch-to-epoch shuffling/worker seed derivation.
    # ----------------------------------------------------------------------

    resume_path = out / RESUME_CHECKPOINT

    resume_state = None

    if args.mode == "primary" and resume_path.exists():

        resume_state = torch.load(
            resume_path,
            map_location="cpu",
            weights_only=False,
        )

        assert (
            resume_state["architecture"]
            == ARCHITECTURE
        )

        assert (
            float(resume_state["rho"])
            == float(rho)
        )

        assert (
            int(resume_state["replicate"])
            == int(rep)
        )

        assert (
            resume_state["epochs"]
            == N_EPOCHS
        )

        assert (
            resume_state["seed_init"]
            == str(seeds["seed_init"])
        )

        assert (
            resume_state["seed_dataloader"]
            == str(seeds["seed_dataloader"])
        )

        model.load_state_dict(
            resume_state["model_state"]
        )

        opt.load_state_dict(
            resume_state["optimizer_state"]
        )

        sched.load_state_dict(
            resume_state["scheduler_state"]
        )

        g.set_state(
            resume_state["dataloader_generator_state"]
        )

        random.setstate(
            resume_state["python_rng_state"]
        )

        np.random.set_state(
            resume_state["numpy_rng_state"]
        )

        torch.set_rng_state(
            resume_state["torch_rng_state"]
        )

        if torch.cuda.is_available():

            cuda_states = resume_state.get(
                "cuda_rng_states"
            )

            if cuda_states is not None:

                torch.cuda.set_rng_state_all(
                    cuda_states
                )

        start_epoch = (
            int(
                resume_state["completed_epoch"]
            )
            + 1
        )

        log_rows = list(
            resume_state["log_rows"]
        )

        best = dict(
            resume_state["best"]
        )

        print(
            f"RESUMING FROM COMPLETED EPOCH "
            f"{resume_state['completed_epoch']:02d}; "
            f"next epoch={start_epoch:02d}",
            flush=True,
        )

    else:

        start_epoch = 1

'''

text = text.replace(
    anchor,
    resume_block,
    1
)

# ============================================================
# PATCH 3 — EPOCH LOOP STARTS FROM start_epoch
# ============================================================

old = '''    for epoch in range(
        1,
        n_epochs + 1,
    ):
'''

new = '''    for epoch in range(
        start_epoch,
        n_epochs + 1,
    ):
'''

assert old in text, (
    "Epoch loop not found."
)

text = text.replace(
    old,
    new,
    1
)

# ============================================================
# PATCH 4 — SAVE RESUME CHECKPOINT AFTER EACH COMPLETED EPOCH
# ============================================================

anchor = '''                os.replace(
                    out
                    / "final_epoch15.tmp",
                    out
                    / PRIMARY_CHECKPOINT,
                )

    # ----------------------------------------------------------------------
    # Training record
'''

assert anchor in text, (
    "Final checkpoint / training-record boundary not found."
)

replacement = '''                os.replace(
                    out
                    / "final_epoch15.tmp",
                    out
                    / PRIMARY_CHECKPOINT,
                )

        # --------------------------------------------------------------
        # Epoch-level crash/session resume checkpoint
        #
        # Written only after the complete epoch has finished, including
        # training, scheduler step, evaluation, logging, and sensitivity
        # checkpoint handling.
        # --------------------------------------------------------------

        if args.mode == "primary":

            resume_payload = {
                "architecture":
                    ARCHITECTURE,

                "rho":
                    float(rho),

                "replicate":
                    int(rep),

                "epochs":
                    int(N_EPOCHS),

                "completed_epoch":
                    int(epoch),

                "seed_init":
                    str(
                        seeds["seed_init"]
                    ),

                "seed_dataloader":
                    str(
                        seeds["seed_dataloader"]
                    ),

                "model_state":
                    model.state_dict(),

                "optimizer_state":
                    opt.state_dict(),

                "scheduler_state":
                    sched.state_dict(),

                "dataloader_generator_state":
                    g.get_state(),

                "python_rng_state":
                    random.getstate(),

                "numpy_rng_state":
                    np.random.get_state(),

                "torch_rng_state":
                    torch.get_rng_state(),

                "cuda_rng_states":
                    (
                        torch.cuda.get_rng_state_all()
                        if torch.cuda.is_available()
                        else None
                    ),

                "best":
                    dict(best),

                "log_rows":
                    list(log_rows),
            }

            resume_tmp = (
                out /
                "epoch_resume.tmp"
            )

            torch.save(
                resume_payload,
                resume_tmp,
            )

            os.replace(
                resume_tmp,
                out /
                RESUME_CHECKPOINT,
            )

    # ----------------------------------------------------------------------
    # Training record
'''

text = text.replace(
    anchor,
    replacement,
    1
)

# ============================================================
# PATCH 5 — DELETE RESUME CHECKPOINT ONLY AFTER SUCCESSFUL
# FINAL TRAINING RECORD CREATION
# ============================================================

anchor = '''    print(
        json.dumps(
            rec,
            indent=2,
'''

assert anchor in text, (
    "Final record print anchor not found."
)

cleanup = '''    # The presence of this file means the model is resumable.
    # Once training_record.json has been successfully written,
    # the model is complete and the resume checkpoint is no longer needed.
    if (
        args.mode == "primary"
        and (out / RESUME_CHECKPOINT).exists()
    ):
        os.remove(
            out / RESUME_CHECKPOINT
        )

    print(
        json.dumps(
            rec,
            indent=2,
'''

text = text.replace(
    anchor,
    cleanup,
    1
)

# ============================================================
# WRITE + COMPILE
# ============================================================

SCRIPT.write_text(
    text,
    encoding="utf-8"
)

py_compile.compile(
    str(SCRIPT),
    doraise=True
)

after_sha = hashlib.sha256(
    SCRIPT.read_bytes()
).hexdigest()

print("=" * 80)
print("T2.1 EPOCH-RESUME PATCH COMPLETE")
print("=" * 80)

print("Before SHA:", before_sha)
print("After SHA :", after_sha)
print("Backup    :", BACKUP)
print("Compile   : PASS")

print()
print("Resume checkpoint:", RESUME_CHECKPOINT)
print()
print("Captured state:")
print("  model")
print("  optimizer")
print("  scheduler")
print("  DataLoader generator")
print("  Python RNG")
print("  NumPy RNG")
print("  Torch RNG")
print("  CUDA RNG")
print("  completed epoch")
print("  best evaluation BA")
print("  epoch log")

In [ ]:
from pathlib import Path
import hashlib

SCRIPT = Path(
    "/kaggle/working/T2_1_ConvNeXt_training_code_freeze/"
    "train_convnext_tiny_t2_1.py"
)

sha = hashlib.sha256(
    SCRIPT.read_bytes()
).hexdigest()

print("=" * 80)
print("VERIFY SCRIPT WAS NOT MODIFIED BY FAILED PATCH")
print("=" * 80)

print("Current SHA:", sha)

print("\nExpected SHA before epoch-resume patch:")
print(
    "28bf6a0d08459587b8bec26b7f8c7aa67785fb8ec9957caddd46ed4c72be0a5b"
)

print("\n" + "=" * 80)
print("EXACT DATALOADER BLOCK")
print("=" * 80)

lines = SCRIPT.read_text(
    encoding="utf-8"
).splitlines()

for i in range(810, 870):
    if i <= len(lines):
        print(
            f"{i:5d}: {lines[i-1]}"
        )

In [ ]:
from pathlib import Path
import hashlib
import py_compile

# ============================================================
# T2.1 EPOCH-LEVEL KAGGLE SESSION RESUME PATCH
# ============================================================

SCRIPT = Path(
    "/kaggle/working/T2_1_ConvNeXt_training_code_freeze/"
    "train_convnext_tiny_t2_1.py"
)

BACKUP = Path(
    "/kaggle/working/T2_1_ConvNeXt_training_code_freeze/"
    "train_convnext_tiny_t2_1.pre_epoch_resume_patch.py"
)

EXPECTED_BEFORE_SHA = (
    "28bf6a0d08459587b8bec26b7f8c7aa67785fb8ec9957caddd46ed4c72be0a5b"
)

# ------------------------------------------------------------
# Identity gate
# ------------------------------------------------------------

before_sha = hashlib.sha256(
    SCRIPT.read_bytes()
).hexdigest()

assert before_sha == EXPECTED_BEFORE_SHA, (
    f"Unexpected script SHA:\n{before_sha}"
)

assert not BACKUP.exists(), (
    f"Backup already exists:\n{BACKUP}"
)

text = SCRIPT.read_text(
    encoding="utf-8"
)

BACKUP.write_bytes(
    SCRIPT.read_bytes()
)

# ============================================================
# PATCH 1 — RESUME CHECKPOINT CONSTANT
# ============================================================

old = '''PRIMARY_CHECKPOINT = "final_epoch15.pt"
BEST_CHECKPOINT = "best_eval_ba.pt"
'''

new = '''PRIMARY_CHECKPOINT = "final_epoch15.pt"
BEST_CHECKPOINT = "best_eval_ba.pt"

# Epoch-level crash/session resume checkpoint.
RESUME_CHECKPOINT = "epoch_resume.pt"
'''

assert text.count(old) == 1, (
    "Checkpoint constant block not found exactly once."
)

text = text.replace(
    old,
    new,
    1
)

# ============================================================
# PATCH 2 — RESUME STATE AFTER MODEL / OPTIMIZER / SCHEDULER
# ============================================================

old = '''    best = {
        "ba": -1.0,
        "epoch": None,
    }

    # ----------------------------------------------------------------------
    # Training
    # ----------------------------------------------------------------------
'''

new = '''    best = {
        "ba": -1.0,
        "epoch": None,
    }

    # ----------------------------------------------------------------------
    # Epoch-level crash/session resume
    #
    # A checkpoint is written only after a complete epoch has finished.
    # Therefore completed_epoch always denotes a fully completed epoch.
    # ----------------------------------------------------------------------

    resume_path = out / RESUME_CHECKPOINT

    if (
        args.mode == "primary"
        and resume_path.exists()
    ):

        resume_state = torch.load(
            resume_path,
            map_location="cpu",
            weights_only=False,
        )

        assert (
            resume_state["architecture"]
            == ARCHITECTURE
        )

        assert (
            float(resume_state["rho"])
            == float(rho)
        )

        assert (
            int(resume_state["replicate"])
            == int(rep)
        )

        assert (
            int(resume_state["epochs"])
            == int(N_EPOCHS)
        )

        assert (
            resume_state["seed_init"]
            == str(
                seeds["seed_init"]
            )
        )

        assert (
            resume_state["seed_dataloader"]
            == str(
                seeds["seed_dataloader"]
            )
        )

        model.load_state_dict(
            resume_state["model_state"]
        )

        opt.load_state_dict(
            resume_state["optimizer_state"]
        )

        sched.load_state_dict(
            resume_state["scheduler_state"]
        )

        g.set_state(
            resume_state[
                "dataloader_generator_state"
            ]
        )

        random.setstate(
            resume_state[
                "python_rng_state"
            ]
        )

        np.random.set_state(
            resume_state[
                "numpy_rng_state"
            ]
        )

        torch.set_rng_state(
            resume_state[
                "torch_rng_state"
            ]
        )

        if torch.cuda.is_available():

            cuda_states = resume_state.get(
                "cuda_rng_states"
            )

            if cuda_states is not None:

                torch.cuda.set_rng_state_all(
                    cuda_states
                )

        start_epoch = (
            int(
                resume_state[
                    "completed_epoch"
                ]
            )
            + 1
        )

        log_rows = list(
            resume_state[
                "log_rows"
            ]
        )

        best = dict(
            resume_state["best"]
        )

        print(
            f"RESUMING FROM COMPLETED EPOCH "
            f"{resume_state['completed_epoch']:02d}; "
            f"NEXT EPOCH {start_epoch:02d}",
            flush=True,
        )

    else:

        start_epoch = 1

    # ----------------------------------------------------------------------
    # Training
    # ----------------------------------------------------------------------
'''

assert text.count(old) == 1, (
    "Training initialization block not found exactly once."
)

text = text.replace(
    old,
    new,
    1
)

# ============================================================
# PATCH 3 — START FROM LAST COMPLETED EPOCH + 1
# ============================================================

old = '''    for epoch in range(
        1,
        n_epochs + 1,
    ):
'''

new = '''    for epoch in range(
        start_epoch,
        n_epochs + 1,
    ):
'''

assert text.count(old) == 1, (
    "Epoch loop not found exactly once."
)

text = text.replace(
    old,
    new,
    1
)

# ============================================================
# PATCH 4 — EPOCH CHECKPOINT AFTER COMPLETE EPOCH
# ============================================================

old = '''                os.replace(
                    out
                    / "final_epoch15.tmp",
                    out
                    / PRIMARY_CHECKPOINT,
                )

    # ----------------------------------------------------------------------
    # Training record
    # ----------------------------------------------------------------------
'''

new = '''                os.replace(
                    out
                    / "final_epoch15.tmp",
                    out
                    / PRIMARY_CHECKPOINT,
                )

        # --------------------------------------------------------------
        # Epoch-level crash/session resume checkpoint
        # --------------------------------------------------------------

        if args.mode == "primary":

            resume_payload = {
                "architecture":
                    ARCHITECTURE,

                "rho":
                    float(rho),

                "replicate":
                    int(rep),

                "epochs":
                    int(N_EPOCHS),

                "completed_epoch":
                    int(epoch),

                "seed_init":
                    str(
                        seeds["seed_init"]
                    ),

                "seed_dataloader":
                    str(
                        seeds["seed_dataloader"]
                    ),

                "model_state":
                    model.state_dict(),

                "optimizer_state":
                    opt.state_dict(),

                "scheduler_state":
                    sched.state_dict(),

                "dataloader_generator_state":
                    g.get_state(),

                "python_rng_state":
                    random.getstate(),

                "numpy_rng_state":
                    np.random.get_state(),

                "torch_rng_state":
                    torch.get_rng_state(),

                "cuda_rng_states":
                    (
                        torch.cuda.get_rng_state_all()
                        if torch.cuda.is_available()
                        else None
                    ),

                "best":
                    dict(best),

                "log_rows":
                    list(log_rows),
            }

            resume_tmp = (
                out /
                "epoch_resume.tmp"
            )

            torch.save(
                resume_payload,
                resume_tmp,
            )

            os.replace(
                resume_tmp,
                out /
                RESUME_CHECKPOINT,
            )

    # ----------------------------------------------------------------------
    # Training record
    # ----------------------------------------------------------------------
'''

assert text.count(old) == 1, (
    "Final-checkpoint/training-record boundary not found exactly once."
)

text = text.replace(
    old,
    new,
    1
)

# ============================================================
# PATCH 5 — REMOVE RESUME CHECKPOINT ONLY AFTER FINAL RECORD
# ============================================================

old = '''    print(
        json.dumps(
            rec,
            indent=2,
        )
    )
'''

new = '''    # The resume checkpoint is retained while the model is incomplete.
    # Once the final training record has been successfully written,
    # the model is complete and the resume checkpoint can be removed.
    if (
        args.mode == "primary"
        and (out / RESUME_CHECKPOINT).exists()
    ):

        os.remove(
            out / RESUME_CHECKPOINT
        )

    print(
        json.dumps(
            rec,
            indent=2,
        )
    )
'''

assert text.count(old) == 1, (
    "Final record print block not found exactly once."
)

text = text.replace(
    old,
    new,
    1
)

# ============================================================
# WRITE
# ============================================================

SCRIPT.write_text(
    text,
    encoding="utf-8"
)

# ============================================================
# COMPILE
# ============================================================

py_compile.compile(
    str(SCRIPT),
    doraise=True
)

after_sha = hashlib.sha256(
    SCRIPT.read_bytes()
).hexdigest()

print("=" * 80)
print("T2.1 EPOCH-RESUME PATCH")
print("=" * 80)

print("Before SHA:", before_sha)
print("After SHA :", after_sha)
print("Backup    :", BACKUP)
print("Compile   : PASS")

print()
print("Resume behavior:")
print("  No checkpoint -> start epoch 1")
print("  epoch_resume.pt -> resume from completed_epoch + 1")
print("  final training record -> delete resume checkpoint")

print()
print("Checkpoint state:")
print("  model")
print("  AdamW optimizer")
print("  cosine scheduler")
print("  DataLoader generator")
print("  Python RNG")
print("  NumPy RNG")
print("  Torch RNG")
print("  CUDA RNG")
print("  completed epoch")
print("  best BA")
print("  epoch log")

In [ ]:
from pathlib import Path
import hashlib
import py_compile

# ============================================================
# T2.1 EPOCH-LEVEL KAGGLE SESSION RESUME PATCH
# ============================================================

SCRIPT = Path(
    "/kaggle/working/T2_1_ConvNeXt_training_code_freeze/"
    "train_convnext_tiny_t2_1.py"
)

BACKUP = Path(
    "/kaggle/working/T2_1_ConvNeXt_training_code_freeze/"
    "train_convnext_tiny_t2_1.pre_epoch_resume_patch.py"
)

EXPECTED_BEFORE_SHA = (
    "28bf6a0d08459587b8bec26b7f8c7aa67785fb8ec9957caddd46ed4c72be0a5b"
)

# ============================================================
# VERIFY CURRENT SCRIPT
# ============================================================

before_sha = hashlib.sha256(
    SCRIPT.read_bytes()
).hexdigest()

assert before_sha == EXPECTED_BEFORE_SHA, (
    f"Unexpected current script SHA:\n{before_sha}"
)

# ============================================================
# VERIFY EXISTING BACKUP
# ============================================================

assert BACKUP.exists(), (
    f"Expected backup missing:\n{BACKUP}"
)

backup_sha = hashlib.sha256(
    BACKUP.read_bytes()
).hexdigest()

assert backup_sha == EXPECTED_BEFORE_SHA, (
    "Existing backup does not match the verified "
    "pre-patch script.\n"
    f"Backup SHA: {backup_sha}\n"
    f"Expected  : {EXPECTED_BEFORE_SHA}"
)

print("=" * 80)
print("PRE-PATCH IDENTITY VERIFIED")
print("=" * 80)

print("Current script SHA:", before_sha)
print("Existing backup SHA:", backup_sha)
print("Backup integrity: PASS")

text = SCRIPT.read_text(
    encoding="utf-8"
)

# ============================================================
# PATCH 1 — RESUME CHECKPOINT CONSTANT
# ============================================================

old = '''PRIMARY_CHECKPOINT = "final_epoch15.pt"
BEST_CHECKPOINT = "best_eval_ba.pt"
'''

new = '''PRIMARY_CHECKPOINT = "final_epoch15.pt"
BEST_CHECKPOINT = "best_eval_ba.pt"

# Epoch-level crash/session resume checkpoint.
RESUME_CHECKPOINT = "epoch_resume.pt"
'''

assert text.count(old) == 1, (
    "Checkpoint constant block not found exactly once."
)

text = text.replace(
    old,
    new,
    1
)

# ============================================================
# PATCH 2 — RESUME STATE
# ============================================================

old = '''    best = {
        "ba": -1.0,
        "epoch": None,
    }

    # ----------------------------------------------------------------------
    # Training
    # ----------------------------------------------------------------------
'''

new = '''    best = {
        "ba": -1.0,
        "epoch": None,
    }

    # ----------------------------------------------------------------------
    # Epoch-level crash/session resume
    #
    # The checkpoint is written only after a complete epoch has finished.
    # Therefore completed_epoch always denotes a fully completed epoch.
    # ----------------------------------------------------------------------

    resume_path = out / RESUME_CHECKPOINT

    if (
        args.mode == "primary"
        and resume_path.exists()
    ):

        resume_state = torch.load(
            resume_path,
            map_location="cpu",
            weights_only=False,
        )

        assert (
            resume_state["architecture"]
            == ARCHITECTURE
        )

        assert (
            float(resume_state["rho"])
            == float(rho)
        )

        assert (
            int(resume_state["replicate"])
            == int(rep)
        )

        assert (
            int(resume_state["epochs"])
            == int(N_EPOCHS)
        )

        assert (
            resume_state["seed_init"]
            == str(
                seeds["seed_init"]
            )
        )

        assert (
            resume_state["seed_dataloader"]
            == str(
                seeds["seed_dataloader"]
            )
        )

        model.load_state_dict(
            resume_state["model_state"]
        )

        opt.load_state_dict(
            resume_state["optimizer_state"]
        )

        sched.load_state_dict(
            resume_state["scheduler_state"]
        )

        g.set_state(
            resume_state[
                "dataloader_generator_state"
            ]
        )

        random.setstate(
            resume_state[
                "python_rng_state"
            ]
        )

        np.random.set_state(
            resume_state[
                "numpy_rng_state"
            ]
        )

        torch.set_rng_state(
            resume_state[
                "torch_rng_state"
            ]
        )

        if torch.cuda.is_available():

            cuda_states = resume_state.get(
                "cuda_rng_states"
            )

            if cuda_states is not None:

                torch.cuda.set_rng_state_all(
                    cuda_states
                )

        start_epoch = (
            int(
                resume_state[
                    "completed_epoch"
                ]
            )
            + 1
        )

        log_rows = list(
            resume_state[
                "log_rows"
            ]
        )

        best = dict(
            resume_state["best"]
        )

        print(
            f"RESUMING FROM COMPLETED EPOCH "
            f"{resume_state['completed_epoch']:02d}; "
            f"NEXT EPOCH {start_epoch:02d}",
            flush=True,
        )

    else:

        start_epoch = 1

    # ----------------------------------------------------------------------
    # Training
    # ----------------------------------------------------------------------
'''

assert text.count(old) == 1, (
    "Training initialization block not found exactly once."
)

text = text.replace(
    old,
    new,
    1
)

# ============================================================
# PATCH 3 — EPOCH LOOP
# ============================================================

old = '''    for epoch in range(
        1,
        n_epochs + 1,
    ):
'''

new = '''    for epoch in range(
        start_epoch,
        n_epochs + 1,
    ):
'''

assert text.count(old) == 1, (
    "Epoch loop not found exactly once."
)

text = text.replace(
    old,
    new,
    1
)

# ============================================================
# PATCH 4 — EPOCH RESUME CHECKPOINT
# ============================================================

old = '''                os.replace(
                    out
                    / "final_epoch15.tmp",
                    out
                    / PRIMARY_CHECKPOINT,
                )

    # ----------------------------------------------------------------------
    # Training record
    # ----------------------------------------------------------------------
'''

new = '''                os.replace(
                    out
                    / "final_epoch15.tmp",
                    out
                    / PRIMARY_CHECKPOINT,
                )

        # --------------------------------------------------------------
        # Epoch-level crash/session resume checkpoint
        # --------------------------------------------------------------

        if args.mode == "primary":

            resume_payload = {
                "architecture":
                    ARCHITECTURE,

                "rho":
                    float(rho),

                "replicate":
                    int(rep),

                "epochs":
                    int(N_EPOCHS),

                "completed_epoch":
                    int(epoch),

                "seed_init":
                    str(
                        seeds["seed_init"]
                    ),

                "seed_dataloader":
                    str(
                        seeds["seed_dataloader"]
                    ),

                "model_state":
                    model.state_dict(),

                "optimizer_state":
                    opt.state_dict(),

                "scheduler_state":
                    sched.state_dict(),

                "dataloader_generator_state":
                    g.get_state(),

                "python_rng_state":
                    random.getstate(),

                "numpy_rng_state":
                    np.random.get_state(),

                "torch_rng_state":
                    torch.get_rng_state(),

                "cuda_rng_states":
                    (
                        torch.cuda.get_rng_state_all()
                        if torch.cuda.is_available()
                        else None
                    ),

                "best":
                    dict(best),

                "log_rows":
                    list(log_rows),
            }

            resume_tmp = (
                out /
                "epoch_resume.tmp"
            )

            torch.save(
                resume_payload,
                resume_tmp,
            )

            os.replace(
                resume_tmp,
                out /
                RESUME_CHECKPOINT,
            )

    # ----------------------------------------------------------------------
    # Training record
    # ----------------------------------------------------------------------
'''

assert text.count(old) == 1, (
    "Final checkpoint boundary not found exactly once."
)

text = text.replace(
    old,
    new,
    1
)

# ============================================================
# PATCH 5 — REMOVE RESUME CHECKPOINT AFTER FINAL RECORD
# ============================================================

old = '''    print(
        json.dumps(
            rec,
            indent=2,
        )
    )
'''

new = '''    # The resume checkpoint is retained while the model is incomplete.
    # Once the final training record has been successfully written,
    # the model is complete and the resume checkpoint can be removed.

    if (
        args.mode == "primary"
        and (out / RESUME_CHECKPOINT).exists()
    ):

        os.remove(
            out / RESUME_CHECKPOINT
        )

    print(
        json.dumps(
            rec,
            indent=2,
        )
    )
'''

assert text.count(old) == 1, (
    "Final record print block not found exactly once."
)

text = text.replace(
    old,
    new,
    1
)

# ============================================================
# WRITE + COMPILE
# ============================================================

SCRIPT.write_text(
    text,
    encoding="utf-8"
)

py_compile.compile(
    str(SCRIPT),
    doraise=True
)

after_sha = hashlib.sha256(
    SCRIPT.read_bytes()
).hexdigest()

print("\n" + "=" * 80)
print("T2.1 EPOCH-RESUME PATCH COMPLETE")
print("=" * 80)

print("Before SHA:", before_sha)
print("After SHA :", after_sha)
print("Backup    :", BACKUP)
print("Compile   : PASS")

In [ ]:
from pathlib import Path
import hashlib
import re

SCRIPT = Path(
    "/kaggle/working/T2_1_ConvNeXt_training_code_freeze/"
    "train_convnext_tiny_t2_1.py"
)

EXPECTED_SHA = (
    "8ad10c3c993e2b173666984da28cf4aa5f0393164155585bd032f36df1ccc69b"
)

text = SCRIPT.read_text(
    encoding="utf-8"
)

sha = hashlib.sha256(
    SCRIPT.read_bytes()
).hexdigest()

assert sha == EXPECTED_SHA, (
    f"Unexpected script SHA: {sha}"
)

checks = {
    "resume_constant":
        'RESUME_CHECKPOINT = "epoch_resume.pt"' in text,

    "resume_path":
        "resume_path = out / RESUME_CHECKPOINT" in text,

    "resume_loader":
        'torch.load(' in text
        and 'weights_only=False' in text
        and 'resume_state' in text,

    "model_restore":
        'model.load_state_dict(' in text,

    "optimizer_restore":
        'opt.load_state_dict(' in text,

    "scheduler_restore":
        'sched.load_state_dict(' in text,

    "dataloader_restore":
        'g.set_state(' in text
        and 'dataloader_generator_state' in text,

    "python_rng_restore":
        'random.setstate(' in text
        and 'python_rng_state' in text,

    "numpy_rng_restore":
        'np.random.set_state(' in text
        and 'numpy_rng_state' in text,

    "torch_rng_restore":
        'torch.set_rng_state(' in text
        and 'torch_rng_state' in text,

    "cuda_rng_restore":
        'torch.cuda.set_rng_state_all(' in text
        and 'cuda_rng_states' in text,

    "start_epoch":
        'start_epoch = 1' in text
        and 'completed_epoch' in text
        and 'start_epoch' in text,

    "resume_epoch_loop":
        '''for epoch in range(
        start_epoch,
        n_epochs + 1,
    ):''' in text,

    "resume_checkpoint_save":
        'resume_payload = {' in text
        and '"completed_epoch":' in text
        and '"model_state":' in text
        and '"optimizer_state":' in text
        and '"scheduler_state":' in text,

    "atomic_resume_write":
        'torch.save(' in text
        and 'resume_tmp' in text
        and 'os.replace(' in text
        and 'RESUME_CHECKPOINT' in text,

    "resume_cleanup":
        'os.remove(' in text
        and 'out / RESUME_CHECKPOINT' in text,
}

print("=" * 80)
print("T2.1 EPOCH-RESUME STRUCTURAL AUDIT")
print("=" * 80)

for name, passed in checks.items():
    print(
        f"{name:30s}: "
        f"{'PASS' if passed else 'FAIL'}"
    )

failed = [
    name
    for name, passed in checks.items()
    if not passed
]

print("\nScript SHA:", sha)

assert not failed, (
    "STRUCTURAL AUDIT FAILED: "
    + ", ".join(failed)
)

print("\n" + "=" * 80)
print("ALL STRUCTURAL RESUME CHECKS: PASS")
print("=" * 80)

In [ ]:
from pathlib import Path
import hashlib
import json
import random
import numpy as np
import torch
import torch.nn as nn

# ============================================================
# T2.1 NON-DESTRUCTIVE RESUME CHECKPOINT SERIALIZATION TEST
# ============================================================

SCRIPT = Path(
    "/kaggle/working/T2_1_ConvNeXt_training_code_freeze/"
    "train_convnext_tiny_t2_1.py"
)

EXPECTED_SHA = (
    "8ad10c3c993e2b173666984da28cf4aa5f0393164155585bd032f36df1ccc69b"
)

sha = hashlib.sha256(
    SCRIPT.read_bytes()
).hexdigest()

assert sha == EXPECTED_SHA

TEST_DIR = Path(
    "/kaggle/working/T2_1_RESUME_SERIALIZATION_TEST"
)

TEST_DIR.mkdir(
    parents=True,
    exist_ok=True
)

TEST_FILE = TEST_DIR / "epoch_resume.pt"
TEST_TMP = TEST_DIR / "epoch_resume.tmp"

# Clean previous test artifacts.
for p in [TEST_FILE, TEST_TMP]:
    if p.exists():
        p.unlink()

# ------------------------------------------------------------
# Deterministic tiny model
# ------------------------------------------------------------

torch.manual_seed(20260919)
np.random.seed(20260919)
random.seed(20260919)

model = nn.Sequential(
    nn.Linear(8, 16),
    nn.ReLU(),
    nn.Linear(16, 2),
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4,
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=15,
)

g = torch.Generator()
g.manual_seed(123456789)

# ------------------------------------------------------------
# Advance the states so we are not testing trivial defaults
# ------------------------------------------------------------

x = torch.randn(
    32,
    8,
)

y = torch.randint(
    0,
    2,
    (32,),
)

optimizer.zero_grad()

logits = model(x)

loss = nn.CrossEntropyLoss()(
    logits,
    y,
)

loss.backward()
optimizer.step()
scheduler.step()

completed_epoch = 1

best = {
    "ba": 0.73125,
    "epoch": 1,
}

log_rows = [
    {
        "epoch": 1,
        "loss": float(loss.item()),
        "ba": 0.73125,
    }
]

# Advance RNG streams.
python_probe_before = random.random()
numpy_probe_before = float(np.random.rand())
torch_probe_before = float(torch.rand(1).item())

# ------------------------------------------------------------
# Capture states
# ------------------------------------------------------------

payload = {
    "architecture":
        "ConvNeXt-Tiny",

    "rho":
        0.5,

    "replicate":
        1,

    "epochs":
        15,

    "completed_epoch":
        completed_epoch,

    "seed_init":
        "7228623435177903258",

    "seed_dataloader":
        "1034838987194871759",

    "model_state":
        model.state_dict(),

    "optimizer_state":
        optimizer.state_dict(),

    "scheduler_state":
        scheduler.state_dict(),

    "dataloader_generator_state":
        g.get_state(),

    "python_rng_state":
        random.getstate(),

    "numpy_rng_state":
        np.random.get_state(),

    "torch_rng_state":
        torch.get_rng_state(),

    "cuda_rng_states":
        (
            torch.cuda.get_rng_state_all()
            if torch.cuda.is_available()
            else None
        ),

    "best":
        dict(best),

    "log_rows":
        list(log_rows),
}

# ------------------------------------------------------------
# Atomic write exactly as production logic
# ------------------------------------------------------------

torch.save(
    payload,
    TEST_TMP,
)

os_replace_ok = True

import os

os.replace(
    TEST_TMP,
    TEST_FILE,
)

assert TEST_FILE.exists()
assert not TEST_TMP.exists()

# ------------------------------------------------------------
# Destroy the in-memory objects
# ------------------------------------------------------------

del model
del optimizer
del scheduler
del g

# ------------------------------------------------------------
# Load checkpoint
# ------------------------------------------------------------

loaded = torch.load(
    TEST_FILE,
    map_location="cpu",
    weights_only=False,
)

# ------------------------------------------------------------
# Validate metadata
# ------------------------------------------------------------

assert loaded["architecture"] == "ConvNeXt-Tiny"
assert float(loaded["rho"]) == 0.5
assert int(loaded["replicate"]) == 1
assert int(loaded["epochs"]) == 15
assert int(loaded["completed_epoch"]) == 1

assert (
    loaded["seed_init"]
    == "7228623435177903258"
)

assert (
    loaded["seed_dataloader"]
    == "1034838987194871759"
)

# ------------------------------------------------------------
# Reconstruct objects
# ------------------------------------------------------------

torch.manual_seed(999999)

model2 = nn.Sequential(
    nn.Linear(8, 16),
    nn.ReLU(),
    nn.Linear(16, 2),
)

optimizer2 = torch.optim.AdamW(
    model2.parameters(),
    lr=1e-4,
    weight_decay=1e-4,
)

scheduler2 = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer2,
    T_max=15,
)

g2 = torch.Generator()

model2.load_state_dict(
    loaded["model_state"]
)

optimizer2.load_state_dict(
    loaded["optimizer_state"]
)

scheduler2.load_state_dict(
    loaded["scheduler_state"]
)

g2.set_state(
    loaded[
        "dataloader_generator_state"
    ]
)

# ------------------------------------------------------------
# Restore RNG states
# ------------------------------------------------------------

random.setstate(
    loaded["python_rng_state"]
)

np.random.set_state(
    loaded["numpy_rng_state"]
)

torch.set_rng_state(
    loaded["torch_rng_state"]
)

if (
    torch.cuda.is_available()
    and loaded["cuda_rng_states"] is not None
):

    torch.cuda.set_rng_state_all(
        loaded["cuda_rng_states"]
    )

# ------------------------------------------------------------
# Validate model state
# ------------------------------------------------------------

for k, v in loaded["model_state"].items():

    assert torch.equal(
        model2.state_dict()[k],
        v
    ), f"Model mismatch: {k}"

# ------------------------------------------------------------
# Validate optimizer state
# ------------------------------------------------------------

assert (
    optimizer2.state_dict()
    == loaded["optimizer_state"]
)

# ------------------------------------------------------------
# Validate scheduler state
# ------------------------------------------------------------

assert (
    scheduler2.state_dict()
    == loaded["scheduler_state"]
)

# ------------------------------------------------------------
# Validate DataLoader generator
# ------------------------------------------------------------

assert torch.equal(
    g2.get_state(),
    loaded[
        "dataloader_generator_state"
    ]
)

# ------------------------------------------------------------
# Validate restored RNG streams
# ------------------------------------------------------------

python_probe_after = random.random()
numpy_probe_after = float(np.random.rand())
torch_probe_after = float(torch.rand(1).item())

# Restore again from the same checkpoint and reproduce probes.

random.setstate(
    loaded["python_rng_state"]
)

np.random.set_state(
    loaded["numpy_rng_state"]
)

torch.set_rng_state(
    loaded["torch_rng_state"]
)

python_probe_repeat = random.random()
numpy_probe_repeat = float(np.random.rand())
torch_probe_repeat = float(torch.rand(1).item())

assert (
    python_probe_after
    == python_probe_repeat
)

assert (
    numpy_probe_after
    == numpy_probe_repeat
)

assert (
    torch_probe_after
    == torch_probe_repeat
)

# ------------------------------------------------------------
# Validate resume arithmetic
# ------------------------------------------------------------

start_epoch = (
    int(
        loaded["completed_epoch"]
    )
    + 1
)

assert start_epoch == 2

assert loaded["best"] == best
assert loaded["log_rows"] == log_rows

# ------------------------------------------------------------
# Cleanup
# ------------------------------------------------------------

if TEST_FILE.exists():
    TEST_FILE.unlink()

print("=" * 80)
print("T2.1 RESUME SERIALIZATION TEST")
print("=" * 80)

print("Script SHA:", sha)
print("Checkpoint atomic write      : PASS")
print("Checkpoint load              : PASS")
print("Model state                  : PASS")
print("Optimizer state              : PASS")
print("Scheduler state              : PASS")
print("DataLoader generator state   : PASS")
print("Python RNG state             : PASS")
print("NumPy RNG state              : PASS")
print("Torch RNG state              : PASS")
print("CUDA RNG state               : PASS")
print("Metadata validation          : PASS")
print("Best/epoch-log restoration   : PASS")
print("Resume epoch calculation     : PASS")

print()
print("=" * 80)
print("ALL RESUME SERIALIZATION CHECKS: PASS")
print("=" * 80)

In [ ]:
from pathlib import Path
import hashlib
import json
import pandas as pd

# ============================================================
# T2.1 FINAL PRODUCTION PREFLIGHT
# ============================================================

PROJECT = Path(
    "/kaggle/input/notebooks/tejasseshadriiiii/"
    "isic-kaggle-analysis/isic_final_project"
)

TRAIN_ROOT = Path(
    "/kaggle/working/T2_1_ConvNeXt_training_runs"
)

SCRIPT = Path(
    "/kaggle/working/T2_1_ConvNeXt_training_code_freeze/"
    "train_convnext_tiny_t2_1.py"
)

SEED_TABLE = Path(
    "/kaggle/working/T2_1_ConvNeXt_seed_lock/"
    "T2_1_TRAINING_SEED_TABLE.csv"
)

ASSIGN_ROOT = Path(
    "/kaggle/working/T2_1_ConvNeXt_rho05_assignments"
)

EXPECTED_SCRIPT_SHA = (
    "8ad10c3c993e2b173666984da28cf4aa5f0393164155585bd032f36df1ccc69b"
)

EXPECTED_SEED_SHA = (
    "9c54a1a617f676750ef4abd174d0374a7ccce69e633c1f8dad86a3ee03823a5d"
)

EXPECTED_ASSIGNMENT_SHA = {
    1: "631aad30c4d4952d4a2a68092a6b2323d19e9b93cb0f69ed36b9cf7b2c70a877",
    2: "a63d0bad295b0dc38677d3780cf1ad77aa20e3fa12b917b6bebd3d91ed3dc290",
    3: "22969500b6481a905c97e86dab77dea6cf30c6c6aa90914994ecfa1fc3f93066",
}

# ============================================================
# 1. SCRIPT IDENTITY
# ============================================================

script_sha = hashlib.sha256(
    SCRIPT.read_bytes()
).hexdigest()

assert script_sha == EXPECTED_SCRIPT_SHA, (
    f"Script SHA mismatch:\n"
    f"actual   = {script_sha}\n"
    f"expected = {EXPECTED_SCRIPT_SHA}"
)

print("=" * 80)
print("1. PRODUCTION SCRIPT")
print("=" * 80)
print("SHA:", script_sha)
print("PASS")

# ============================================================
# 2. SEED TABLE IDENTITY
# ============================================================

seed_sha = hashlib.sha256(
    SEED_TABLE.read_bytes()
).hexdigest()

assert seed_sha == EXPECTED_SEED_SHA, (
    f"Seed-table SHA mismatch:\n"
    f"actual   = {seed_sha}\n"
    f"expected = {EXPECTED_SEED_SHA}"
)

seed_df = pd.read_csv(SEED_TABLE)

assert seed_df.shape == (9, 6)

expected_seed_cols = [
    "rho",
    "replicate",
    "seed_init",
    "seed_dataloader",
    "seed_augmentation",
    "seed_workers",
]

assert list(seed_df.columns) == expected_seed_cols

print("\n" + "=" * 80)
print("2. T2.1 SEED TABLE")
print("=" * 80)
print("SHA:", seed_sha)
print("Rows:", len(seed_df))
print("PASS")

# ============================================================
# 3. EXACT 9-CELL ROSTER
# ============================================================

expected_cells = [
    (0.0, 1),
    (0.0, 2),
    (0.0, 3),
    (0.5, 1),
    (0.5, 2),
    (0.5, 3),
    (1.0, 1),
    (1.0, 2),
    (1.0, 3),
]

actual_cells = [
    (
        float(r["rho"]),
        int(r["replicate"]),
    )
    for _, r in seed_df.iterrows()
]

assert actual_cells == expected_cells, (
    f"Roster mismatch:\n{actual_cells}"
)

print("\n" + "=" * 80)
print("3. EXACT 9-MODEL ROSTER")
print("=" * 80)

for rho, rep in expected_cells:
    print(
        f"rho={rho:.1f}, replicate={rep}"
    )

print("PASS")

# ============================================================
# 4. VERIFY ρ=.5 ASSIGNMENTS
# ============================================================

print("\n" + "=" * 80)
print("4. ρ=.5 ASSIGNMENT LOCKS")
print("=" * 80)

for rep in [1, 2, 3]:

    path = (
        ASSIGN_ROOT
        / f"rho_0.5_rep_{rep}.csv"
    )

    assert path.exists(), (
        f"Missing assignment: {path}"
    )

    sha = hashlib.sha256(
        path.read_bytes()
    ).hexdigest()

    assert sha == EXPECTED_ASSIGNMENT_SHA[rep], (
        f"Assignment SHA mismatch for rep {rep}\n"
        f"actual   = {sha}\n"
        f"expected = {EXPECTED_ASSIGNMENT_SHA[rep]}"
    )

    df = pd.read_csv(path)

    assert len(df) == 18514
    assert df["image"].nunique() == 18514
    assert df["lesion_id"].nunique() == 9478

    assert set(
        df["hair_present"].unique()
    ) == {0, 1}

    assert int(
        df["hair_present"].sum()
    ) == 9257

    assert (
        df.groupby("lesion_id")["hair_present"]
        .nunique()
        .max()
        == 1
    )

    assert (
        df["split"].astype(str).unique().tolist()
        == ["train"]
    )

    print(
        f"rep={rep}: "
        f"SHA={sha} | "
        f"rows={len(df)} | "
        f"hair={int(df['hair_present'].sum())} | "
        f"PASS"
    )

# ============================================================
# 5. TRAINING RUN DIRECTORIES
# ============================================================

print("\n" + "=" * 80)
print("5. EXISTING T2.1 TRAINING RUNS")
print("=" * 80)

completed = []
incomplete = []
absent = []

for rho, rep in expected_cells:

    name = (
        f"rho_{rho:.1f}_rep_{rep}"
    )

    out = TRAIN_ROOT / name

    if not out.exists():

        absent.append(name)

        print(
            f"{name}: ABSENT"
        )

        continue

    record_path = (
        out / "training_record.json"
    )

    final_path = (
        out / "final_epoch15.pt"
    )

    resume_path = (
        out / "epoch_resume.pt"
    )

    if record_path.exists():

        record = json.loads(
            record_path.read_text(
                encoding="utf-8"
            )
        )

        status = record.get(
            "status"
        )

        if (
            status == "COMPLETED"
            and final_path.exists()
        ):

            completed.append(name)

            print(
                f"{name}: COMPLETED"
            )

        else:

            incomplete.append(name)

            print(
                f"{name}: INCOMPLETE "
                f"(record status={status!r})"
            )

    elif resume_path.exists():

        incomplete.append(name)

        print(
            f"{name}: RESUMABLE "
            f"(epoch_resume.pt present)"
        )

    else:

        incomplete.append(name)

        print(
            f"{name}: INCOMPLETE "
            f"(directory exists but no completion record)"
        )

# ============================================================
# 6. COMPLETED MODEL VALIDATION
# ============================================================

print("\n" + "=" * 80)
print("6. COMPLETED MODEL VALIDATION")
print("=" * 80)

# These were the two models already completed before
# the epoch-resume patch. They are accepted under their
# historical production script SHA.

KNOWN_COMPLETED = {
    "rho_0.0_rep_1",
    "rho_0.0_rep_2",
}

EXPECTED_OLD_SCRIPT_SHA = (
    "d6b97a547e96d4f14cb97b0bfa69c5219c798e0f5371a3de0d4c6bdaba93b379"
)

for name in completed:

    out = TRAIN_ROOT / name

    record = json.loads(
        (
            out / "training_record.json"
        ).read_text(
            encoding="utf-8"
        )
    )

    assert name in KNOWN_COMPLETED, (
        f"Unexpected pre-existing completed model: {name}"
    )

    assert record.get("status") == "COMPLETED"

    assert record.get(
        "architecture"
    ) == "ConvNeXt-Tiny"

    assert int(
        record.get("epochs")
    ) == 15

    assert (
        record.get("primary_checkpoint")
        == "final_epoch15.pt"
    )

    record_sha = record.get(
        "script_sha256"
    )

    assert record_sha == EXPECTED_OLD_SCRIPT_SHA, (
        f"Unexpected historical script SHA for {name}: "
        f"{record_sha}"
    )

    assert (
        out / "final_epoch15.pt"
    ).exists()

    # A completed model should not retain a resume checkpoint.
    assert not (
        out / "epoch_resume.pt"
    ).exists(), (
        f"Completed model still has resume checkpoint: {name}"
    )

    print(
        f"{name}: VALIDATED"
    )

# ============================================================
# 7. DETERMINE EXACT REMAINING ROSTER
# ============================================================

expected_names = {
    f"rho_{rho:.1f}_rep_{rep}"
    for rho, rep in expected_cells
}

completed_set = set(completed)

assert completed_set == KNOWN_COMPLETED, (
    "Completed roster differs from expected historical "
    "completed models.\n"
    f"Found: {sorted(completed_set)}"
)

remaining = sorted(
    expected_names - completed_set
)

assert len(remaining) == 7

print("\n" + "=" * 80)
print("7. REMAINING PRODUCTION ROSTER")
print("=" * 80)

for name in remaining:
    print(name)

print("Count:", len(remaining))
print("PASS")

# ============================================================
# 8. NO STALE COMPLETION ARTIFACTS
# ============================================================

print("\n" + "=" * 80)
print("8. STALE/INCOMPLETE ARTIFACT CHECK")
print("=" * 80)

for name in remaining:

    out = TRAIN_ROOT / name

    if not out.exists():
        continue

    # An incomplete directory is allowed because the new
    # resume mechanism is designed to continue it.
    # But it must not contain a fake COMPLETED record.
    record_path = out / "training_record.json"

    if record_path.exists():

        record = json.loads(
            record_path.read_text(
                encoding="utf-8"
            )
        )

        assert (
            record.get("status")
            != "COMPLETED"
        ), (
            f"Unexpected COMPLETED record "
            f"for remaining model: {name}"
        )

    print(
        f"{name}: safe for production"
    )

# ============================================================
# 9. FINAL PREFLIGHT SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("T2.1 FINAL PRODUCTION PREFLIGHT")
print("=" * 80)

print("Architecture       : ConvNeXt-Tiny")
print("Rho levels         : 0.0, 0.5, 1.0")
print("Replicates/rho     : 3")
print("Total models       : 9")
print("Already completed  : 2")
print("Remaining          : 7")
print("Primary epochs     : 15")
print("Primary checkpoint : final_epoch15.pt")
print("Resume checkpoint  : epoch_resume.pt")
print("XAI/IG             : OUT OF SCOPE")
print("Script SHA         :", script_sha)
print("Seed-table SHA     :", seed_sha)

print()
print("COMPLETED:")
for x in sorted(completed):
    print("  ", x)

print()
print("TO RUN:")
for x in remaining:
    print("  ", x)

print()
print("=" * 80)
print("ALL T2.1 PRODUCTION PREFLIGHT CHECKS: PASS")
print("=" * 80)

In [ ]:
from pathlib import Path

ROOT = Path("/kaggle/working")

print("=" * 80)
print("SEARCHING FOR EXISTING T2.1 CONVNEXT OUTPUTS")
print("=" * 80)

matches = []

for p in ROOT.rglob("training_record.json"):
    s = str(p)

    if (
        "rho_0.0_rep_1" in s
        or "rho_0.0_rep_2" in s
        or "ConvNeXt" in s
        or "T2_1" in s
    ):
        matches.append(p)

if not matches:
    print("No obvious T2.1 training_record.json files found.")
else:
    for p in sorted(matches):
        print(p)

print("\n" + "=" * 80)
print("SEARCHING FOR MODEL DIRECTORIES")
print("=" * 80)

for pattern in [
    "rho_0.0_rep_1",
    "rho_0.0_rep_2",
    "rho_0.0_rep_3",
    "rho_0.5_rep_1",
    "rho_0.5_rep_2",
    "rho_0.5_rep_3",
    "rho_1.0_rep_1",
    "rho_1.0_rep_2",
    "rho_1.0_rep_3",
]:
    found = list(
        ROOT.rglob(pattern)
    )

    print(f"\n{pattern}:")
    if found:
        for p in found[:20]:
            print("  ", p)
    else:
        print("   NOT FOUND")

In [ ]:
from pathlib import Path
import json
import hashlib

ROOT = Path(
    "/kaggle/working/T2_1_PRODUCTION"
)

CURRENT_SCRIPT_SHA = (
    "8ad10c3c993e2b173666984da28cf4aa5f0393164155585bd032f36df1ccc69b"
)

EXPECTED_OLD_SCRIPT_SHA = (
    "d6b97a547e96d4f14cb97b0bfa69c5219c798e0f5371a3de0d4c6bdaba93b379"
)

MODELS = [
    "rho_0.0_rep_1",
    "rho_0.0_rep_2",
    "rho_0.0_rep_3",
    "rho_0.5_rep_1",
    "rho_0.5_rep_2",
    "rho_0.5_rep_3",
    "rho_1.0_rep_1",
    "rho_1.0_rep_2",
    "rho_1.0_rep_3",
]

print("=" * 80)
print("T2.1 ACTUAL PRODUCTION DIRECTORY AUDIT")
print("=" * 80)

completed = []
resumable = []
incomplete = []
absent = []

for name in MODELS:

    out = ROOT / name

    print("\n" + "-" * 80)
    print(name)

    if not out.exists():

        absent.append(name)

        print("STATUS: ABSENT")
        continue

    print("Directory:", out)

    record_path = (
        out / "training_record.json"
    )

    final_path = (
        out / "final_epoch15.pt"
    )

    resume_path = (
        out / "epoch_resume.pt"
    )

    epoch_log = (
        out / "epoch_log.csv"
    )

    print(
        "training_record.json:",
        record_path.exists()
    )

    print(
        "final_epoch15.pt:",
        final_path.exists()
    )

    print(
        "epoch_resume.pt:",
        resume_path.exists()
    )

    print(
        "epoch_log.csv:",
        epoch_log.exists()
    )

    # --------------------------------------------------------
    # Training record
    # --------------------------------------------------------

    if record_path.exists():

        try:

            record = json.loads(
                record_path.read_text(
                    encoding="utf-8"
                )
            )

            print(
                "record.status:",
                record.get("status")
            )

            print(
                "record.architecture:",
                record.get("architecture")
            )

            print(
                "record.rho:",
                record.get("rho")
            )

            print(
                "record.replicate:",
                record.get("replicate")
            )

            print(
                "record.epochs:",
                record.get("epochs")
            )

            print(
                "record.script_sha256:",
                record.get("script_sha256")
            )

            if (
                record.get("status")
                == "COMPLETED"
                and final_path.exists()
            ):

                completed.append(name)

                print(
                    "CLASSIFICATION: COMPLETED"
                )

                continue

        except Exception as e:

            print(
                "Could not parse training record:",
                repr(e)
            )

    # --------------------------------------------------------
    # Resume checkpoint
    # --------------------------------------------------------

    if resume_path.exists():

        try:

            import torch

            resume = torch.load(
                resume_path,
                map_location="cpu",
                weights_only=False,
            )

            required = [
                "architecture",
                "rho",
                "replicate",
                "epochs",
                "completed_epoch",
                "seed_init",
                "seed_dataloader",
                "model_state",
                "optimizer_state",
                "scheduler_state",
                "dataloader_generator_state",
                "python_rng_state",
                "numpy_rng_state",
                "torch_rng_state",
                "best",
                "log_rows",
            ]

            missing = [
                x
                for x in required
                if x not in resume
            ]

            print(
                "resume.architecture:",
                resume.get("architecture")
            )

            print(
                "resume.rho:",
                resume.get("rho")
            )

            print(
                "resume.replicate:",
                resume.get("replicate")
            )

            print(
                "resume.completed_epoch:",
                resume.get("completed_epoch")
            )

            print(
                "resume.epochs:",
                resume.get("epochs")
            )

            print(
                "resume.missing_fields:",
                missing
            )

            valid_resume = (
                len(missing) == 0
                and resume.get(
                    "architecture"
                ) == "ConvNeXt-Tiny"
                and int(
                    resume.get("epochs")
                ) == 15
            )

            if valid_resume:

                resumable.append(name)

                print(
                    "CLASSIFICATION: RESUMABLE"
                )

                continue

        except Exception as e:

            print(
                "Could not load resume checkpoint:",
                repr(e)
            )

    # --------------------------------------------------------
    # Anything else is incomplete
    # --------------------------------------------------------

    incomplete.append(name)

    print(
        "CLASSIFICATION: INCOMPLETE / NOT RESUMABLE"
    )

# ============================================================
# SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("AUDIT SUMMARY")
print("=" * 80)

print("\nCOMPLETED:")
for x in completed:
    print("  ", x)

print("\nRESUMABLE:")
for x in resumable:
    print("  ", x)

print("\nINCOMPLETE / NOT RESUMABLE:")
for x in incomplete:
    print("  ", x)

print("\nABSENT:")
for x in absent:
    print("  ", x)

print("\nCounts:")
print("  completed :", len(completed))
print("  resumable :", len(resumable))
print("  incomplete:", len(incomplete))
print("  absent    :", len(absent))

print("\n" + "=" * 80)
print("NO FILES MODIFIED")
print("=" * 80)

In [ ]:
from pathlib import Path
import shutil

ROOT = Path(
    "/kaggle/working/T2_1_PRODUCTION"
)

TO_RESET = [
    "rho_0.0_rep_3",
    "rho_0.5_rep_1",
]

print("=" * 80)
print("RESETTING NON-RESUMABLE T2.1 RUN DIRECTORIES")
print("=" * 80)

for name in TO_RESET:

    out = ROOT / name

    if not out.exists():
        print(f"{name}: already absent")
        continue

    files = list(out.rglob("*"))

    print(
        f"{name}: "
        f"{len(files)} filesystem entries -> removing"
    )

    shutil.rmtree(out)

    assert not out.exists()

    print(
        f"{name}: removed successfully"
    )

print("\n" + "=" * 80)
print("RESET COMPLETE")
print("=" * 80)

print("The following two completed models were NOT touched:")
print("  rho_0.0_rep_1")
print("  rho_0.0_rep_2")

In [ ]:
from pathlib import Path
import re

SCRIPT = Path(
    "/kaggle/working/T2_1_ConvNeXt_training_code_freeze/"
    "train_convnext_tiny_t2_1.py"
)

text = SCRIPT.read_text(
    encoding="utf-8"
)

print("=" * 80)
print("T2.1 PRODUCTION SCRIPT CLI AUDIT")
print("=" * 80)

# Print every argparse argument declaration.
matches = re.findall(
    r'add_argument\((.*?)\)',
    text,
    flags=re.DOTALL
)

for i, m in enumerate(matches, 1):
    print(f"\nARGUMENT {i}")
    print(m.strip())

print("\n" + "=" * 80)
print("SCRIPT ENTRYPOINT / MODE REFERENCES")
print("=" * 80)

for i, line in enumerate(
    text.splitlines(),
    1
):
    if (
        "parse_args" in line
        or "args.project" in line
        or "args.out_dir" in line
        or "args.cache_dir" in line
        or "args.rho" in line
        or "args.replicate" in line
        or "args.num_workers" in line
        or "args.mode" in line
    ):
        print(
            f"{i:5d}: {line}"
        )

In [ ]:
from pathlib import Path
import subprocess
import threading
import queue
import time
import hashlib
import json
import os
import signal
import sys

# ============================================================
# T2.1 FINAL 2-GPU PRODUCTION LAUNCHER
# ============================================================

PROJECT = Path(
    "/kaggle/input/notebooks/tejasseshadriiiii/"
    "isic-kaggle-analysis/isic_final_project"
)

CACHE = Path(
    "/kaggle/working/isic_cache"
)

SCRIPT = Path(
    "/kaggle/working/T2_1_ConvNeXt_training_code_freeze/"
    "train_convnext_tiny_t2_1.py"
)

OUT_ROOT = Path(
    "/kaggle/working/T2_1_PRODUCTION"
)

SEED_TABLE = Path(
    "/kaggle/working/T2_1_ConvNeXt_seed_lock/"
    "T2_1_TRAINING_SEED_TABLE.csv"
)

ASSIGN_ROOT = Path(
    "/kaggle/working/T2_1_ConvNeXt_rho05_assignments"
)

# ------------------------------------------------------------
# Locked identities
# ------------------------------------------------------------

EXPECTED_SCRIPT_SHA = (
    "8ad10c3c993e2b173666984da28cf4aa5f0393164155585bd032f36df1ccc69b"
)

EXPECTED_SEED_SHA = (
    "9c54a1a617f676750ef4abd174d0374a7ccce69e633c1f8dad86a3ee03823a5d"
)

EXPECTED_ASSIGNMENT_SHA = {
    1: "631aad30c4d4952d4a2a68092a6b2323d19e9b93cb0f69ed36b9cf7b2c70a877",
    2: "a63d0bad295b0dc38677d3780cf1ad77aa20e3fa12b917b6bebd3d91ed3dc290",
    3: "22969500b6481a905c97e86dab77dea6cf30c6c6aa90914994ecfa1fc3f93066",
}

OLD_COMPLETED_SCRIPT_SHA = (
    "d6b97a547e96d4f14cb97b0bfa69c5219c798e0f5371a3de0d4c6bdaba93b379"
)

# ------------------------------------------------------------
# Exact T2.1 roster
# ------------------------------------------------------------

ALL_MODELS = [
    (0.0, 1),
    (0.0, 2),
    (0.0, 3),
    (0.5, 1),
    (0.5, 2),
    (0.5, 3),
    (1.0, 1),
    (1.0, 2),
    (1.0, 3),
]

HISTORICAL_COMPLETED = {
    "rho_0.0_rep_1",
    "rho_0.0_rep_2",
}

# ------------------------------------------------------------
# Identity checks
# ------------------------------------------------------------

assert PROJECT.exists(), (
    f"Project path missing:\n{PROJECT}"
)

assert SCRIPT.exists(), (
    f"Training script missing:\n{SCRIPT}"
)

assert SEED_TABLE.exists(), (
    f"Seed table missing:\n{SEED_TABLE}"
)

assert ASSIGN_ROOT.exists(), (
    f"Assignment root missing:\n{ASSIGN_ROOT}"
)

script_sha = hashlib.sha256(
    SCRIPT.read_bytes()
).hexdigest()

assert script_sha == EXPECTED_SCRIPT_SHA, (
    f"Production script SHA mismatch.\n"
    f"Actual   : {script_sha}\n"
    f"Expected : {EXPECTED_SCRIPT_SHA}"
)

seed_sha = hashlib.sha256(
    SEED_TABLE.read_bytes()
).hexdigest()

assert seed_sha == EXPECTED_SEED_SHA, (
    f"Seed-table SHA mismatch.\n"
    f"Actual   : {seed_sha}\n"
    f"Expected : {EXPECTED_SEED_SHA}"
)

# ------------------------------------------------------------
# Validate rho=.5 assignment locks
# ------------------------------------------------------------

for rep in (1, 2, 3):

    p = (
        ASSIGN_ROOT
        / f"rho_0.5_rep_{rep}.csv"
    )

    assert p.exists(), (
        f"Missing rho=.5 assignment: {p}"
    )

    sha = hashlib.sha256(
        p.read_bytes()
    ).hexdigest()

    assert sha == EXPECTED_ASSIGNMENT_SHA[rep], (
        f"Assignment SHA mismatch for rho=.5 rep {rep}.\n"
        f"Actual   : {sha}\n"
        f"Expected : {EXPECTED_ASSIGNMENT_SHA[rep]}"
    )

# ------------------------------------------------------------
# Validate historical completed models
# ------------------------------------------------------------

OUT_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

for name in HISTORICAL_COMPLETED:

    out = OUT_ROOT / name

    assert out.exists(), (
        f"Historical completed model missing: {out}"
    )

    record_path = (
        out / "training_record.json"
    )

    final_path = (
        out / "final_epoch15.pt"
    )

    assert record_path.exists(), (
        f"Missing training record: {record_path}"
    )

    assert final_path.exists(), (
        f"Missing final checkpoint: {final_path}"
    )

    record = json.loads(
        record_path.read_text(
            encoding="utf-8"
        )
    )

    assert record.get("status") == "COMPLETED"
    assert record.get("architecture") == "ConvNeXt-Tiny"
    assert int(record.get("epochs")) == 15

    assert (
        record.get("script_sha256")
        == OLD_COMPLETED_SCRIPT_SHA
    )

    assert not (
        out / "epoch_resume.pt"
    ).exists()

# ------------------------------------------------------------
# Determine remaining roster
# ------------------------------------------------------------

remaining = []

for rho, rep in ALL_MODELS:

    name = (
        f"rho_{rho:.1f}_rep_{rep}"
    )

    if name in HISTORICAL_COMPLETED:
        continue

    out = OUT_ROOT / name

    # Newly trained models must not already have a
    # contradictory COMPLETED record.
    if out.exists():

        record_path = (
            out / "training_record.json"
        )

        if record_path.exists():

            record = json.loads(
                record_path.read_text(
                    encoding="utf-8"
                )
            )

            assert (
                record.get("status")
                != "COMPLETED"
            ), (
                f"Unexpected completed model in "
                f"remaining roster: {name}"
            )

    remaining.append(
        (rho, rep)
    )

assert len(remaining) == 7

# ------------------------------------------------------------
# GPU check
# ------------------------------------------------------------

gpu_check = subprocess.run(
    [
        sys.executable,
        "-c",
        """
import torch
print(torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(torch.cuda.get_device_name(i))
""",
    ],
    capture_output=True,
    text=True,
)

print("=" * 80)
print("T2.1 FINAL PRODUCTION PREFLIGHT")
print("=" * 80)

print("Project :", PROJECT)
print("Script  :", SCRIPT)
print("Output  :", OUT_ROOT)
print("Script SHA:", script_sha)
print("Seed SHA  :", seed_sha)

print("\nGPU environment:")
print(
    gpu_check.stdout.strip()
)

assert gpu_check.returncode == 0

gpu_lines = [
    x.strip()
    for x in gpu_check.stdout.splitlines()
    if x.strip()
]

assert gpu_lines, "Could not read GPU count."

gpu_count = int(gpu_lines[0])

assert gpu_count >= 2, (
    f"Need 2 GPUs for concurrent production; "
    f"detected {gpu_count}"
)

print("\nHistorical completed:")
for name in sorted(HISTORICAL_COMPLETED):
    print("  ", name)

print("\nRemaining 7 models:")
for rho, rep in remaining:
    print(
        f"  rho={rho:.1f}, rep={rep}"
    )

print("\n" + "=" * 80)
print("PREFLIGHT PASS — LAUNCHING 2 CONCURRENT GPU PROCESSES")
print("=" * 80)

# ============================================================
# Process runner
# ============================================================

def run_model(
    gpu_id,
    rho,
    rep,
    result_queue,
):

    name = (
        f"rho_{rho:.1f}_rep_{rep}"
    )

    out = OUT_ROOT / name

    out.mkdir(
        parents=True,
        exist_ok=True,
    )

    env = os.environ.copy()

    env["CUDA_VISIBLE_DEVICES"] = str(
        gpu_id
    )

    env["PYTHONUNBUFFERED"] = "1"

    cmd = [
        sys.executable,
        "-u",
        str(SCRIPT),

        "--project",
        str(PROJECT),

        "--cache-dir",
        str(CACHE),

        "--out-dir",
        str(out),

        "--mode",
        "primary",

        "--rho",
        f"{rho:.1f}",

        "--rep",
        str(rep),

        "--gap",
        "mean",

        "--save-all-epochs",
        "0",

        "--num-workers",
        "2",
    ]

    print(
        f"\n[GPU {gpu_id}] STARTING {name}",
        flush=True
    )

    print(
        f"[GPU {gpu_id}] CMD: "
        + " ".join(cmd),
        flush=True
    )

    start = time.time()

    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env,
    )

    try:

        for line in proc.stdout:

            print(
                f"[GPU {gpu_id}][{name}] "
                f"{line.rstrip()}",
                flush=True,
            )

    finally:

        proc.stdout.close()

    return_code = proc.wait()

    elapsed = time.time() - start

    result_queue.put(
        {
            "name": name,
            "rho": rho,
            "rep": rep,
            "gpu": gpu_id,
            "return_code": return_code,
            "seconds": elapsed,
        }
    )

# ============================================================
# Scheduler
# ============================================================

result_queue = queue.Queue()

threads = []

pending = list(
    remaining
)

gpu_ids = [0, 1]

running = {}

completed_runtime = []

failed_runtime = []

next_index = 0

def start_job(
    gpu_id,
    rho,
    rep,
):

    t = threading.Thread(
        target=run_model,
        args=(
            gpu_id,
            rho,
            rep,
            result_queue,
        ),
        daemon=True,
    )

    t.start()

    running[gpu_id] = (
        rho,
        rep,
        t,
    )

    threads.append(t)

# ------------------------------------------------------------
# Start first two jobs
# ------------------------------------------------------------

for gpu_id in gpu_ids:

    if next_index >= len(pending):
        break

    rho, rep = pending[
        next_index
    ]

    next_index += 1

    start_job(
        gpu_id,
        rho,
        rep,
    )

# ------------------------------------------------------------
# Dynamic two-GPU scheduler
# ------------------------------------------------------------

while running:

    result = result_queue.get()

    gpu_id = result["gpu"]

    running.pop(
        gpu_id,
        None
    )

    name = result["name"]

    if result["return_code"] == 0:

        completed_runtime.append(
            result
        )

        print(
            "\n" +
            "=" * 80,
            flush=True
        )

        print(
            f"[GPU {gpu_id}] COMPLETED {name} "
            f"in {result['seconds']/60:.1f} min",
            flush=True
        )

        print(
            "=" * 80,
            flush=True
        )

    else:

        failed_runtime.append(
            result
        )

        print(
            "\n" +
            "=" * 80,
            flush=True
        )

        print(
            f"[GPU {gpu_id}] FAILED {name} "
            f"return_code={result['return_code']}",
            flush=True
        )

        print(
            "=" * 80,
            flush=True
        )

        # Stop launching new work after a production failure.
        print(
            "PRODUCTION FAILURE DETECTED — "
            "NOT LAUNCHING ADDITIONAL MODELS.",
            flush=True
        )

        # Terminate any currently running sibling process.
        for other_gpu, (
            other_rho,
            other_rep,
            other_thread,
        ) in list(
            running.items()
        ):

            print(
                f"WARNING: GPU {other_gpu} is still running "
                f"rho={other_rho:.1f} rep={other_rep}. "
                f"Allowing it to finish safely.",
                flush=True
            )

        # Do not schedule anything else.
        next_index = len(pending)

    # --------------------------------------------------------
    # Fill newly available GPU only if no failure occurred
    # --------------------------------------------------------

    if (
        not failed_runtime
        and next_index < len(pending)
    ):

        rho, rep = pending[
            next_index
        ]

        next_index += 1

        start_job(
            gpu_id,
            rho,
            rep,
        )

# ============================================================
# Final runtime summary
# ============================================================

print("\n" + "=" * 80)
print("T2.1 PRODUCTION PROCESS SUMMARY")
print("=" * 80)

print(
    f"Successful this session: "
    f"{len(completed_runtime)}"
)

for r in completed_runtime:
    print(
        f"  {r['name']} | "
        f"GPU {r['gpu']} | "
        f"{r['seconds']/60:.1f} min"
    )

print(
    f"\nFailed this session: "
    f"{len(failed_runtime)}"
)

for r in failed_runtime:
    print(
        f"  {r['name']} | "
        f"GPU {r['gpu']} | "
        f"return_code={r['return_code']}"
    )

# ============================================================
# Final filesystem validation
# ============================================================

print("\n" + "=" * 80)
print("T2.1 FINAL FILESYSTEM VALIDATION")
print("=" * 80)

for rho, rep in ALL_MODELS:

    name = (
        f"rho_{rho:.1f}_rep_{rep}"
    )

    out = OUT_ROOT / name

    record_path = (
        out / "training_record.json"
    )

    final_path = (
        out / "final_epoch15.pt"
    )

    if not record_path.exists():

        print(
            f"{name}: NOT COMPLETED"
        )

        continue

    record = json.loads(
        record_path.read_text(
            encoding="utf-8"
        )
    )

    status = record.get(
        "status"
    )

    if status == "COMPLETED":

        assert final_path.exists()

        print(
            f"{name}: COMPLETED"
        )

    else:

        print(
            f"{name}: status={status}"
        )

print("\n" + "=" * 80)

if failed_runtime:

    print(
        "T2.1 PRODUCTION STOPPED AFTER FAILURE."
    )

else:

    print(
        "T2.1 PRODUCTION BATCH FINISHED."
    )

print("=" * 80)

In [ ]:
from pathlib import Path

SCRIPT = Path(
    "/kaggle/working/T2_1_ConvNeXt_training_code_freeze/"
    "train_convnext_tiny_t2_1.py"
)

text = SCRIPT.read_text(
    encoding="utf-8"
)

lines = text.splitlines()

print("=" * 80)
print("CACHE HANDLING INSPECTION")
print("=" * 80)

# Show every cache-related source line.
for i, line in enumerate(lines, 1):

    if (
        "cache" in line.lower()
        or "cache_index" in line.lower()
    ):
        print(
            f"{i:5d}: {line}"
        )

print("\n" + "=" * 80)
print("CURRENT CACHE DIRECTORY")
print("=" * 80)

cache_candidates = [
    Path("/kaggle/working/isic_cache"),
    Path("/kaggle/working"),
    Path("/kaggle/input"),
]

for root in cache_candidates:

    print(f"\nROOT: {root}")

    if not root.exists():

        print("  DOES NOT EXIST")
        continue

    # Only inspect likely cache_index files.
    matches = list(
        root.rglob("cache_index.csv")
    )

    if matches:

        for p in matches[:50]:
            print("  FOUND:", p)

    else:

        print(
            "  No cache_index.csv found "
            "under this root."
        )

print("\n" + "=" * 80)
print("PREVIOUS SUCCESSFUL T2.1 OUTPUT METADATA")
print("=" * 80)

for rep in [1, 2]:

    p = Path(
        f"/kaggle/working/T2_1_PRODUCTION/"
        f"rho_0.0_rep_{rep}/training_record.json"
    )

    if p.exists():

        import json

        rec = json.loads(
            p.read_text(
                encoding="utf-8"
            )
        )

        print(
            f"\nrho_0.0_rep_{rep}"
        )

        for key in [
            "cache_dir",
            "project",
            "training_images",
            "script_sha256",
            "env",
        ]:

            print(
                f"  {key}: {rec.get(key)}"
            )

In [ ]:
from pathlib import Path
import pandas as pd
import hashlib

CACHE = Path(
    "/kaggle/working/T2_1_PRODUCTION_CACHE"
)

INDEX = CACHE / "cache_index.csv"

assert CACHE.exists(), CACHE
assert INDEX.exists(), INDEX

print("=" * 80)
print("T2.1 PRODUCTION CACHE VALIDATION")
print("=" * 80)

print("Cache:", CACHE)
print("Index:", INDEX)

index_sha = hashlib.sha256(
    INDEX.read_bytes()
).hexdigest()

print("\ncache_index.csv SHA256:")
print(index_sha)

df = pd.read_csv(INDEX)

print("\nShape:", df.shape)
print("Columns:")
for c in df.columns:
    print("  ", c)

print("\nFirst rows:")
print(df.head())

# ------------------------------------------------------------
# Basic production-cache identity
# ------------------------------------------------------------

assert len(df) == 18514, (
    f"Expected 18514 cache rows, got {len(df)}"
)

assert df["image"].nunique() == 18514, (
    "Cache image IDs are not unique."
)

# ------------------------------------------------------------
# Split validation
# ------------------------------------------------------------

print("\nSplit counts:")
print(
    df["split"]
    .value_counts(dropna=False)
)

assert set(
    df["split"].astype(str).unique()
).issubset(
    {"train", "val", "test"}
)

# ------------------------------------------------------------
# Required cache path columns
# ------------------------------------------------------------

path_candidates = [
    c for c in df.columns
    if (
        "path" in c.lower()
        or "file" in c.lower()
        or "cache" in c.lower()
    )
]

print("\nPotential path columns:")
for c in path_candidates:
    print("  ", c)

# ------------------------------------------------------------
# Inspect cache filesystem
# ------------------------------------------------------------

files = [
    p for p in CACHE.rglob("*")
    if p.is_file()
]

print("\nTotal files under cache:", len(files))

print("\nCache top-level contents:")
for p in sorted(CACHE.iterdir()):
    if p.is_file():
        print(
            "  FILE ",
            p.name,
            p.stat().st_size,
            "bytes"
        )
    else:
        print(
            "  DIR  ",
            p.name
        )

print("\n" + "=" * 80)
print("CACHE BASIC VALIDATION: PASS")
print("=" * 80)

In [ ]:
from pathlib import Path

SCRIPT = Path(
    "/kaggle/working/T2_1_ConvNeXt_training_code_freeze/"
    "train_convnext_tiny_t2_1.py"
)

lines = SCRIPT.read_text(
    encoding="utf-8"
).splitlines()

print("=" * 80)
print("EXACT PRODUCTION CACHE LOADING LOGIC")
print("=" * 80)

for i in range(440, 590):
    if i <= len(lines):
        print(
            f"{i:5d}: {lines[i-1]}"
        )

In [ ]:
print("\n" + "=" * 80)
print("CACHE DATASET IMPLEMENTATION")
print("=" * 80)

for i in range(225, 285):
    if i <= len(lines):
        print(
            f"{i:5d}: {lines[i-1]}"
        )

In [ ]:
from pathlib import Path

SCRIPT = Path(
    "/kaggle/working/T2_1_ConvNeXt_training_code_freeze/"
    "train_convnext_tiny_t2_1.py"
)

lines = SCRIPT.read_text(
    encoding="utf-8"
).splitlines()

print("=" * 80)
print("EXACT CACHE -> TRAINING ROW SELECTION LOGIC")
print("=" * 80)

for i in range(440, 610):
    if i <= len(lines):
        print(
            f"{i:5d}: {lines[i-1]}"
        )

In [ ]:
from pathlib import Path

SCRIPT = Path(
    "/kaggle/working/T2_1_ConvNeXt_training_code_freeze/"
    "train_convnext_tiny_t2_1.py"
)

lines = SCRIPT.read_text(
    encoding="utf-8"
).splitlines()

print("=" * 80)
print("CONTINUATION: ASSIGNMENT -> CACHE ROWS -> DATASETS")
print("=" * 80)

for i in range(607, 810):
    if i <= len(lines):
        print(
            f"{i:5d}: {lines[i-1]}"
        )

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import hashlib

PROJECT = Path(
    "/kaggle/input/notebooks/tejasseshadriiiii/"
    "isic-kaggle-analysis/isic_final_project"
)

CACHE = Path(
    "/kaggle/working/T2_1_PRODUCTION_CACHE"
)

ASSIGN_ROOT = Path(
    "/kaggle/working/T2_1_ConvNeXt_rho05_assignments"
)

INDEX = CACHE / "cache_index.csv"
CLEAN = CACHE / "clean_224.npy"
HAIR = CACHE / "hair_224.npy"

assert INDEX.exists()
assert CLEAN.exists()
assert HAIR.exists()

idx = pd.read_csv(INDEX)

print("=" * 80)
print("T2.1 CACHE / ASSIGNMENT CROSS-INTEGRITY")
print("=" * 80)

print("cache_index rows:", len(idx))
print("cache_index images:", idx["image"].nunique())

assert idx["image"].is_unique

row_of = dict(
    zip(
        idx["image"],
        idx["row"],
    )
)

# ------------------------------------------------------------
# Open arrays read-only
# ------------------------------------------------------------

cm = np.load(
    CLEAN,
    mmap_mode="r",
)

hm = np.load(
    HAIR,
    mmap_mode="r",
)

print("\nclean_224 shape:", cm.shape)
print("hair_224 shape :", hm.shape)

assert cm.ndim == 4
assert hm.ndim == 4

assert cm.shape[0] == len(idx)
assert hm.shape[0] == len(idx)

# ------------------------------------------------------------
# Validate all nine T2.1 assignments
# ------------------------------------------------------------

cells = [
    (0.0, 1),
    (0.0, 2),
    (0.0, 3),
    (0.5, 1),
    (0.5, 2),
    (0.5, 3),
    (1.0, 1),
    (1.0, 2),
    (1.0, 3),
]

for rho, rep in cells:

    if abs(rho - 0.5) < 1e-12:

        assignment_path = (
            ASSIGN_ROOT
            / f"rho_{rho:.1f}_rep_{rep}.csv"
        )

    else:

        assignment_path = (
            PROJECT
            / "step1F_lesion_level_rho_assignments"
            / f"rho_{rho:.1f}_rep_{rep}.csv"
        )

    assert assignment_path.exists(), (
        f"Missing assignment: {assignment_path}"
    )

    assignment = pd.read_csv(
        assignment_path
    ).sort_values(
        "image"
    ).reset_index(
        drop=True
    )

    assert len(assignment) == 18514
    assert assignment["image"].is_unique

    missing = (
        ~assignment["image"].isin(
            row_of
        )
    )

    assert not missing.any(), (
        f"Missing cache images for "
        f"rho={rho} rep={rep}: "
        f"{int(missing.sum())}"
    )

    train_rows = (
        assignment["image"]
        .map(row_of)
        .to_numpy(
            dtype=np.int64
        )
    )

    assert len(train_rows) == 18514

    assert train_rows.min() >= 0
    assert train_rows.max() < len(idx)

    # The cache row mapping must be unique because each image
    # has a unique cache row.
    assert len(
        np.unique(train_rows)
    ) == 18514

    # Verify labels and row metadata agree.
    cache_labels = (
        idx.set_index("row")
        .loc[
            train_rows,
            "malignant"
        ]
        .to_numpy()
    )

    assignment_labels = (
        assignment["malignant"]
        .to_numpy()
    )

    assert np.array_equal(
        cache_labels,
        assignment_labels
    ), (
        f"Malignant-label mismatch "
        f"rho={rho} rep={rep}"
    )

    print(
        f"rho={rho:.1f} rep={rep}: "
        f"18514 assignment rows -> "
        f"cache rows [{train_rows.min()}, "
        f"{train_rows.max()}] | PASS"
    )

# ------------------------------------------------------------
# Validation population
# ------------------------------------------------------------

ev = (
    idx[
        idx["split"] == "val"
    ]
    .sort_values("image")
)

ev_rows = ev["row"].to_numpy(
    dtype=np.int64
)

assert len(ev_rows) > 0
assert ev_rows.min() >= 0
assert ev_rows.max() < len(idx)

print("\nValidation rows:", len(ev_rows))
print(
    "Validation cache-row range:",
    int(ev_rows.min()),
    "to",
    int(ev_rows.max())
)

# ------------------------------------------------------------
# Cache-level hashes
# ------------------------------------------------------------

print("\ncache_index SHA256:")
print(
    hashlib.sha256(
        INDEX.read_bytes()
    ).hexdigest()
)

print("\nclean_224.npy:")
print(
    "size:",
    CLEAN.stat().st_size,
    "bytes"
)

print("hair_224.npy:")
print(
    "size:",
    HAIR.stat().st_size,
    "bytes"
)

print("\n" + "=" * 80)
print("T2.1 CACHE / ASSIGNMENT CROSS-INTEGRITY: PASS")
print("=" * 80)

In [ ]:
from pathlib import Path
import subprocess
import os
import sys

PROJECT = Path(
    "/kaggle/input/notebooks/tejasseshadriiiii/"
    "isic-kaggle-analysis/isic_final_project"
)

CACHE = Path(
    "/kaggle/working/T2_1_PRODUCTION_CACHE"
)

SCRIPT = Path(
    "/kaggle/working/T2_1_ConvNeXt_training_code_freeze/"
    "train_convnext_tiny_t2_1.py"
)

OUT = Path(
    "/kaggle/working/T2_1_PRODUCTION/rho_0.0_rep_3"
)

assert PROJECT.exists()
assert CACHE.exists()
assert (CACHE / "cache_index.csv").exists()
assert (CACHE / "clean_224.npy").exists()
assert (CACHE / "hair_224.npy").exists()

# The previous failed directory was already reset.
OUT.mkdir(
    parents=True,
    exist_ok=True,
)

env = os.environ.copy()

env["CUDA_VISIBLE_DEVICES"] = "0"
env["PYTHONUNBUFFERED"] = "1"

cmd = [
    sys.executable,
    "-u",
    str(SCRIPT),

    "--project",
    str(PROJECT),

    "--cache-dir",
    str(CACHE),

    "--out-dir",
    str(OUT),

    "--mode",
    "primary",

    "--rho",
    "0.0",

    "--rep",
    "3",

    "--gap",
    "mean",

    "--save-all-epochs",
    "0",

    "--num-workers",
    "2",
]

print("=" * 80)
print("T2.1 CACHE-CORRECTED SMOKE START")
print("=" * 80)
print("Model : rho=0.0 rep=3")
print("GPU   : 0")
print("Cache :", CACHE)
print()

proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=env,
)

for line in proc.stdout:
    print(
        line.rstrip(),
        flush=True
    )

proc.stdout.close()

rc = proc.wait()

print("\n" + "=" * 80)
print("SMOKE START RESULT")
print("=" * 80)
print("Return code:", rc)

assert rc == 0, (
    f"rho_0.0_rep_3 failed with return code {rc}"
)

print("rho_0.0_rep_3 completed successfully.")

In [ ]:
from pathlib import Path
import subprocess
import os
import sys
import time
import json
import hashlib
import threading

PROJECT = Path(
    "/kaggle/input/notebooks/tejasseshadriiiii/"
    "isic-kaggle-analysis/isic_final_project"
)

CACHE = Path(
    "/kaggle/working/T2_1_PRODUCTION_CACHE"
)

SCRIPT = Path(
    "/kaggle/working/T2_1_ConvNeXt_training_code_freeze/"
    "train_convnext_tiny_t2_1.py"
)

OUTROOT = Path(
    "/kaggle/working/T2_1_PRODUCTION"
)

EXPECTED_SCRIPT_SHA = (
    "8ad10c3c993e2b173666984da28cf4aa5f0393164155585bd032f36df1ccc69b"
)

EXPECTED_SEED_SHA = (
    "9c54a1a617f676750ef4abd174d0374a7ccce69e633c1f8dad86a3ee03823a5d"
)

SEED_TABLE = Path(
    "/kaggle/working/T2_1_ConvNeXt_seed_lock/"
    "T2_1_TRAINING_SEED_TABLE.csv"
)

RHO05_ROOT = Path(
    "/kaggle/working/T2_1_ConvNeXt_rho05_assignments"
)

# ------------------------------------------------------------------
# PREFLIGHT
# ------------------------------------------------------------------

assert PROJECT.exists()
assert CACHE.exists()
assert (CACHE / "cache_index.csv").exists()
assert (CACHE / "clean_224.npy").exists()
assert (CACHE / "hair_224.npy").exists()

assert SCRIPT.exists()
assert SEED_TABLE.exists()

script_sha = hashlib.sha256(
    SCRIPT.read_bytes()
).hexdigest()

seed_sha = hashlib.sha256(
    SEED_TABLE.read_bytes()
).hexdigest()

print("=" * 90)
print("T2.1 TWO-GPU PRODUCTION LAUNCH")
print("=" * 90)

print("Script SHA :", script_sha)
print("Seed SHA   :", seed_sha)
print("Cache      :", CACHE)
print("Output     :", OUTROOT)

assert script_sha == EXPECTED_SCRIPT_SHA
assert seed_sha == EXPECTED_SEED_SHA

# ------------------------------------------------------------------
# VERIFY CURRENTLY COMPLETED MODELS
# ------------------------------------------------------------------

def completed(rho, rep):
    d = OUTROOT / f"rho_{rho:.1f}_rep_{rep}"
    rec = d / "training_record.json"
    final = d / "final_epoch15.pt"

    if not (rec.exists() and final.exists()):
        return False

    try:
        r = json.loads(rec.read_text())
    except Exception:
        return False

    return (
        r.get("status") == "COMPLETED"
        and r.get("architecture") == "ConvNeXt-Tiny"
        and float(r.get("rho")) == float(rho)
        and int(r.get("replicate")) == int(rep)
        and int(r.get("epochs")) == 15
    )

print("\nExisting completed models:")

for rho, rep in [
    (0.0,1), (0.0,2), (0.0,3),
    (0.5,1), (0.5,2), (0.5,3),
    (1.0,1), (1.0,2), (1.0,3),
]:
    print(
        f"rho={rho:.1f} rep={rep}:",
        "COMPLETED" if completed(rho,rep)
        else "NOT COMPLETED"
    )

# ------------------------------------------------------------------
# ONLY RUN TWO MODELS NOW
# ------------------------------------------------------------------

jobs = [
    (0, 0.0, 3),
    (1, 0.5, 1),
]

# Never touch already completed models.
for gpu, rho, rep in jobs:

    d = OUTROOT / f"rho_{rho:.1f}_rep_{rep}"

    if completed(rho, rep):
        raise RuntimeError(
            f"Refusing to overwrite completed model: {d}"
        )

    d.mkdir(
        parents=True,
        exist_ok=True
    )

# ------------------------------------------------------------------
# LAUNCH
# ------------------------------------------------------------------

processes = []

for gpu, rho, rep in jobs:

    outdir = OUTROOT / f"rho_{rho:.1f}_rep_{rep}"

    env = os.environ.copy()
    env["CUDA_VISIBLE_DEVICES"] = str(gpu)
    env["PYTHONUNBUFFERED"] = "1"

    cmd = [
        sys.executable,
        "-u",
        str(SCRIPT),

        "--project",
        str(PROJECT),

        "--cache-dir",
        str(CACHE),

        "--out-dir",
        str(outdir),

        "--mode",
        "primary",

        "--rho",
        str(rho),

        "--rep",
        str(rep),

        "--gap",
        "mean",

        "--save-all-epochs",
        "0",

        "--num-workers",
        "2",
    ]

    print("\n" + "-" * 90)
    print(
        f"STARTING GPU {gpu} | "
        f"rho={rho:.1f} | rep={rep}"
    )
    print("-" * 90)

    p = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env,
    )

    processes.append(
        {
            "gpu": gpu,
            "rho": rho,
            "rep": rep,
            "process": p,
        }
    )

# ------------------------------------------------------------------
# STREAM BOTH PROCESSES LIVE
# ------------------------------------------------------------------

def stream_output(item):
    p = item["process"]
    gpu = item["gpu"]
    rho = item["rho"]
    rep = item["rep"]

    for line in p.stdout:
        print(
            f"[GPU {gpu} | rho={rho:.1f} rep={rep}] "
            f"{line.rstrip()}",
            flush=True
        )

    p.stdout.close()

threads = []

for item in processes:
    t = threading.Thread(
        target=stream_output,
        args=(item,),
        daemon=True,
    )
    t.start()
    threads.append(t)

# ------------------------------------------------------------------
# WAIT
# ------------------------------------------------------------------

for item in processes:
    rc = item["process"].wait()

    print(
        f"\nGPU {item['gpu']} | "
        f"rho={item['rho']:.1f} rep={item['rep']} "
        f"RETURN CODE = {rc}",
        flush=True
    )

for t in threads:
    t.join()

# ------------------------------------------------------------------
# FINAL VALIDATION
# ------------------------------------------------------------------

print("\n" + "=" * 90)
print("T2.1 TWO-GPU BATCH RESULT")
print("=" * 90)

for gpu, rho, rep in jobs:

    d = OUTROOT / f"rho_{rho:.1f}_rep_{rep}"

    rec = d / "training_record.json"
    final = d / "final_epoch15.pt"
    resume = d / "epoch_resume.pt"

    print(
        f"rho={rho:.1f} rep={rep} | "
        f"record={rec.exists()} | "
        f"final={final.exists()} | "
        f"resume={resume.exists()}"
    )

    assert rec.exists()
    assert final.exists()

    r = json.loads(
        rec.read_text()
    )

    assert r["status"] == "COMPLETED"
    assert r["architecture"] == "ConvNeXt-Tiny"
    assert int(r["epochs"]) == 15
    assert float(r["rho"]) == rho
    assert int(r["replicate"]) == rep

    # Successful completion should clean the resume checkpoint.
    assert not resume.exists()

print("\n" + "=" * 90)
print("T2.1 TWO-GPU BATCH: PASS")
print("=" * 90)

In [ ]:
from pathlib import Path

SCRIPT = Path(
    "/kaggle/working/T2_1_ConvNeXt_training_code_freeze/"
    "train_convnext_tiny_t2_1.py"
)

lines = SCRIPT.read_text(
    encoding="utf-8"
).splitlines()

print("=" * 80)
print("AUDIT LOOKUP DIAGNOSTIC")
print("=" * 80)

for i in range(620, 665):
    print(
        f"{i+1:4d}: {lines[i]}"
    )

print("\n" + "=" * 80)
print("T2.1 RHO=.5 GENERATION MANIFEST")
print("=" * 80)

manifest = Path(
    "/kaggle/working/T2_1_ConvNeXt_rho05_assignments/"
    "T2_1_RHO05_GENERATION_MANIFEST.json"
)

print(
    manifest.read_text(
        encoding="utf-8"
    )
)

In [ ]:

from pathlib import Path

SCRIPT = Path(
    "/kaggle/working/T2_1_ConvNeXt_training_code_freeze/"
    "train_convnext_tiny_t2_1.py"
)

lines = SCRIPT.read_text(
    encoding="utf-8"
).splitlines()

print("=" * 80)
print("T2.1 AUDIT USAGE INSPECTION")
print("=" * 80)

for i in range(640, 720):
    print(
        f"{i+1:4d}: {lines[i]}"
    )

In [ ]:
from pathlib import Path
import hashlib
import shutil
import json
import py_compile

SCRIPT = Path(
    "/kaggle/working/T2_1_ConvNeXt_training_code_freeze/"
    "train_convnext_tiny_t2_1.py"
)

BACKUP = Path(
    "/kaggle/working/T2_1_ConvNeXt_training_code_freeze/"
    "train_convnext_tiny_t2_1.pre_rho05_audit_patch.py"
)

before_sha = hashlib.sha256(
    SCRIPT.read_bytes()
).hexdigest()

print("=" * 80)
print("T2.1 RHO=.5 AUDIT PATCH")
print("=" * 80)
print("Before SHA:", before_sha)

assert before_sha == (
    "8ad10c3c993e2b173666984da28cf4aa5f0393164155585bd032f36df1ccc69b"
)

shutil.copy2(SCRIPT, BACKUP)

text = SCRIPT.read_text(
    encoding="utf-8"
)

old = '''    audit = pd.read_csv(
        project
        / "step1F_lesion_level_rho_assignments"
        / "step1F_assignment_audit.csv"
    )

    ar = audit[
        np.isclose(
            audit["rho"],
            rho,
        )
        &
        (
            audit["replicate"]
            == rep
        )
    ].iloc[0]
'''

new = '''    if abs(float(rho) - 0.5) < 1e-12:
        rho05_manifest = Path(
            "/kaggle/working/T2_1_ConvNeXt_rho05_assignments"
        ) / "T2_1_RHO05_GENERATION_MANIFEST.json"

        assert rho05_manifest.exists(), (
            f"T2.1 rho=.5 generation manifest missing: "
            f"{rho05_manifest}"
        )

        with rho05_manifest.open(
            "r",
            encoding="utf-8",
        ) as f:
            rho05_meta = json.load(f)

        rho05_rows = [
            x
            for x in rho05_meta["files"]
            if np.isclose(
                float(x["rho"]),
                float(rho),
            )
            and int(x["replicate"]) == int(rep)
        ]

        assert len(rho05_rows) == 1, (
            "T2.1 rho=.5 manifest row not found or duplicated: "
            f"rho={rho}, rep={rep}"
        )

        rho05_row = rho05_rows[0]

        ar = {
            "hair_images": int(
                rho05_row["total_hair"]
            ),
            "p_h_given_m": float(
                rho05_row["p_h_given_m"]
            ),
            "p_h_given_b": float(
                rho05_row["p_h_given_b"]
            ),
        }

        assignment_sha = hashlib.sha256(
            assignment_path.read_bytes()
        ).hexdigest()

        assert assignment_sha == rho05_row["sha256"], (
            "T2.1 rho=.5 assignment SHA256 mismatch: "
            f"expected {rho05_row['sha256']}, "
            f"got {assignment_sha}"
        )

    else:
        audit = pd.read_csv(
            project
            / "step1F_lesion_level_rho_assignments"
            / "step1F_assignment_audit.csv"
        )

        ar = audit[
            np.isclose(
                audit["rho"],
                rho,
            )
            &
            (
                audit["replicate"]
                == rep
            )
        ].iloc[0]
'''

assert old in text, (
    "Expected original audit block was not found. "
    "Refusing to patch."
)

text = text.replace(
    old,
    new,
    1,
)

SCRIPT.write_text(
    text,
    encoding="utf-8",
)

after_sha = hashlib.sha256(
    SCRIPT.read_bytes()
).hexdigest()

print("After SHA :", after_sha)

assert after_sha != before_sha

py_compile.compile(
    str(SCRIPT),
    doraise=True,
)

print("\nCompile: PASS")
print("Backup  :", BACKUP)
print("Patch   : rho=.5 uses locked T2.1 manifest")
print("         rho=0/1 retain frozen Step1F audit")
print("         rho=.5 assignment SHA is verified")

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import json
import hashlib

PROJECT = Path(
    "/kaggle/input/notebooks/tejasseshadriiiii/"
    "isic-kaggle-analysis/isic_final_project"
)

RHO05_ROOT = Path(
    "/kaggle/working/T2_1_ConvNeXt_rho05_assignments"
)

MANIFEST = RHO05_ROOT / "T2_1_RHO05_GENERATION_MANIFEST.json"

assert MANIFEST.exists()

meta = json.loads(
    MANIFEST.read_text(
        encoding="utf-8"
    )
)

print("=" * 80)
print("T2.1 AUDIT RESOLUTION TEST")
print("=" * 80)

# ------------------------------------------------------------------
# Frozen historical audit
# ------------------------------------------------------------------

audit = pd.read_csv(
    PROJECT
    / "step1F_lesion_level_rho_assignments"
    / "step1F_assignment_audit.csv"
)

# ------------------------------------------------------------------
# Test exactly the same quantities used by the training script
# ------------------------------------------------------------------

for rho in [0.0, 0.5, 1.0]:

    for rep in [1, 2, 3]:

        if abs(rho - 0.5) < 1e-12:

            rows = [
                x
                for x in meta["files"]
                if np.isclose(
                    float(x["rho"]),
                    rho,
                )
                and int(x["replicate"]) == rep
            ]

            assert len(rows) == 1

            row = rows[0]

            hair_images = int(
                row["total_hair"]
            )

            p_h_given_m = float(
                row["p_h_given_m"]
            )

            p_h_given_b = float(
                row["p_h_given_b"]
            )

            assignment_path = (
                RHO05_ROOT
                / f"rho_{rho:.1f}_rep_{rep}.csv"
            )

            actual_sha = hashlib.sha256(
                assignment_path.read_bytes()
            ).hexdigest()

            assert actual_sha == row["sha256"]

            source = "T2.1 RHO=.5 manifest"

        else:

            rows = audit[
                np.isclose(
                    audit["rho"],
                    rho,
                )
                &
                (
                    audit["replicate"]
                    == rep
                )
            ]

            assert len(rows) == 1

            row = rows.iloc[0]

            hair_images = int(
                row["hair_images"]
            )

            p_h_given_m = float(
                row["p_h_given_m"]
            )

            p_h_given_b = float(
                row["p_h_given_b"]
            )

            source = "Frozen Step1F audit"

        # ----------------------------------------------------------
        # Load actual assignment
        # ----------------------------------------------------------

        if abs(rho - 0.5) < 1e-12:

            assignment_path = (
                RHO05_ROOT
                / f"rho_{rho:.1f}_rep_{rep}.csv"
            )

        else:

            assignment_path = (
                PROJECT
                / "step1F_lesion_level_rho_assignments"
                / f"rho_{rho:.1f}_rep_{rep}.csv"
            )

        assignment = (
            pd.read_csv(
                assignment_path
            )
            .sort_values("image")
            .reset_index(drop=True)
        )

        M = (
            assignment["malignant"]
            == 1
        )

        B = (
            assignment["malignant"]
            == 0
        )

        actual_hair = int(
            assignment[
                "hair_present"
            ].sum()
        )

        actual_p_m = float(
            assignment.loc[
                M,
                "hair_present"
            ].mean()
        )

        actual_p_b = float(
            assignment.loc[
                B,
                "hair_present"
            ].mean()
        )

        assert len(assignment) == 18514

        assert actual_hair == hair_images

        assert abs(
            actual_p_m - p_h_given_m
        ) < 1e-9

        assert abs(
            actual_p_b - p_h_given_b
        ) < 1e-9

        print(
            f"rho={rho:.1f} rep={rep} | "
            f"hair={actual_hair} | "
            f"p(H|M)={actual_p_m:.12f} | "
            f"p(H|B)={actual_p_b:.12f} | "
            f"{source} | PASS"
        )

print("\n" + "=" * 80)
print("T2.1 AUDIT RESOLUTION TEST: PASS")
print("=" * 80)

print(
    "\nThe training script can now resolve all 9 assignment/audit "
    "combinations without relying on a nonexistent rho=.5 Step1F row."
)

In [1]:
from pathlib import Path
import json

ROOT = Path("/kaggle/working/T2_1_PRODUCTION")

ROSTER = [
    ("0.0", 1), ("0.0", 2), ("0.0", 3),
    ("0.5", 1), ("0.5", 2), ("0.5", 3),
    ("1.0", 1), ("1.0", 2), ("1.0", 3),
]

print("=" * 90)
print("T2.1 RECOVERY AUDIT")
print("=" * 90)

print("Production root:", ROOT)
print("Exists:", ROOT.exists())

completed = []
resumable = []
incomplete = []
absent = []

for rho, rep in ROSTER:
    name = f"rho_{rho}_rep_{rep}"
    d = ROOT / name

    if not d.exists():
        print(f"{name:20s} | ABSENT")
        absent.append(name)
        continue

    record = d / "training_record.json"
    final_ckpt = d / "final_epoch15.pt"
    resume_ckpt = d / "epoch_resume.pt"
    epoch_log = d / "epoch_log.csv"

    if record.exists() and final_ckpt.exists():
        try:
            with open(record) as f:
                rec = json.load(f)

            if rec.get("status") == "COMPLETED":
                print(
                    f"{name:20s} | COMPLETED | "
                    f"epochs={rec.get('epochs')} | "
                    f"script={rec.get('script_sha256')}"
                )
                completed.append(name)
                continue
        except Exception as e:
            print(name, "record read error:", e)

    if resume_ckpt.exists():
        print(
            f"{name:20s} | RESUMABLE | "
            f"epoch_resume.pt=True | "
            f"epoch_log={epoch_log.exists()}"
        )
        resumable.append(name)
    else:
        print(
            f"{name:20s} | INCOMPLETE | "
            f"resume=False"
        )
        incomplete.append(name)

print("\n" + "=" * 90)
print("SUMMARY")
print("=" * 90)

print("Completed:", len(completed))
for x in completed:
    print("  ", x)

print("\nResumable:", len(resumable))
for x in resumable:
    print("  ", x)

print("\nIncomplete/no resume:", len(incomplete))
for x in incomplete:
    print("  ", x)

print("\nAbsent:", len(absent))
for x in absent:
    print("  ", x)

T2.1 RECOVERY AUDIT
Production root: /kaggle/working/T2_1_PRODUCTION
Exists: False
rho_0.0_rep_1        | ABSENT
rho_0.0_rep_2        | ABSENT
rho_0.0_rep_3        | ABSENT
rho_0.5_rep_1        | ABSENT
rho_0.5_rep_2        | ABSENT
rho_0.5_rep_3        | ABSENT
rho_1.0_rep_1        | ABSENT
rho_1.0_rep_2        | ABSENT
rho_1.0_rep_3        | ABSENT

SUMMARY
Completed: 0

Resumable: 0

Incomplete/no resume: 0

Absent: 9
   rho_0.0_rep_1
   rho_0.0_rep_2
   rho_0.0_rep_3
   rho_0.5_rep_1
   rho_0.5_rep_2
   rho_0.5_rep_3
   rho_1.0_rep_1
   rho_1.0_rep_2
   rho_1.0_rep_3


In [2]:
from pathlib import Path

CACHE = Path("/kaggle/working/T2_1_PRODUCTION_CACHE")

print("=" * 80)
print("CACHE RECOVERY CHECK")
print("=" * 80)

for f in [
    "cache_index.csv",
    "clean_224.npy",
    "hair_224.npy",
]:
    p = CACHE / f
    print(f"{f:20s} exists={p.exists()} ", end="")
    if p.exists():
        print(f"size={p.stat().st_size:,}")
    else:
        print()

CACHE RECOVERY CHECK
cache_index.csv      exists=False 
clean_224.npy        exists=False 
hair_224.npy         exists=False 


In [3]:
from pathlib import Path

A = Path("/kaggle/working/T2_1_ConvNeXt_rho05_assignments")

print("=" * 80)
print("RHO=.5 ASSIGNMENT RECOVERY")
print("=" * 80)

for rep in [1, 2, 3]:
    p = A / f"rho_0.5_rep_{rep}.csv"
    print(p, "->", p.exists())

RHO=.5 ASSIGNMENT RECOVERY
/kaggle/working/T2_1_ConvNeXt_rho05_assignments/rho_0.5_rep_1.csv -> False
/kaggle/working/T2_1_ConvNeXt_rho05_assignments/rho_0.5_rep_2.csv -> False
/kaggle/working/T2_1_ConvNeXt_rho05_assignments/rho_0.5_rep_3.csv -> False


In [4]:
from pathlib import Path

print("=" * 90)
print("T2.1 PERSISTENT OUTPUT RECOVERY SEARCH")
print("=" * 90)

roots = [
    Path("/kaggle/input"),
    Path("/kaggle/working"),
]

targets = [
    "training_record.json",
    "final_epoch15.pt",
    "epoch_resume.pt",
]

for root in roots:
    print(f"\nROOT: {root}")
    print("-" * 70)

    if not root.exists():
        print("DOES NOT EXIST")
        continue

    found = []

    for target in targets:
        matches = list(root.rglob(target))
        if matches:
            print(f"\n{target}:")
            for p in matches[:100]:
                print(" ", p)
                found.append(p)

    if not found:
        print("No T2.1 checkpoint/record files found.")

T2.1 PERSISTENT OUTPUT RECOVERY SEARCH

ROOT: /kaggle/input
----------------------------------------------------------------------


KeyboardInterrupt: 

In [ ]:
from pathlib import Path
import subprocess
import os
import sys
import json
import hashlib
import threading
import queue
import time

PROJECT = Path(
    "/kaggle/input/notebooks/tejasseshadriiiii/"
    "isic-kaggle-analysis/isic_final_project"
)

CACHE = Path(
    "/kaggle/working/T2_1_PRODUCTION_CACHE"
)

SCRIPT = Path(
    "/kaggle/working/T2_1_ConvNeXt_training_code_freeze/"
    "train_convnext_tiny_t2_1.py"
)

OUTROOT = Path(
    "/kaggle/working/T2_1_PRODUCTION"
)

SEED_TABLE = Path(
    "/kaggle/working/T2_1_ConvNeXt_seed_lock/"
    "T2_1_TRAINING_SEED_TABLE.csv"
)

EXPECTED_SCRIPT_SHA = (
    "b0981c8fdff3d971b98aed1608de4b5e5e89b2c4bf66cc959272b769e0e89f18"
)

EXPECTED_SEED_SHA = (
    "9c54a1a617f676750ef4abd174d0374a7ccce69e633c1f8dad86a3ee03823a5d"
)

RHO05_ROOT = Path(
    "/kaggle/working/T2_1_ConvNeXt_rho05_assignments"
)

RHO05_SHAS = {
    1: "631aad30c4d4952d4a2a68092a6b2323d19e9b93cb0f69ed36b9cf7b2c70a877",
    2: "a63d0bad295b0dc38677d3780cf1ad77aa20e3fa12b917b6bebd3d91ed3dc290",
    3: "22969500b6481a905c97e86dab77dea6cf30c6c6aa90914994ecfa1fc3f93066",
}

# ================================================================
# PREFLIGHT
# ================================================================

assert PROJECT.exists()
assert CACHE.exists()
assert (CACHE / "cache_index.csv").exists()
assert (CACHE / "clean_224.npy").exists()
assert (CACHE / "hair_224.npy").exists()
assert SCRIPT.exists()
assert SEED_TABLE.exists()

script_sha = hashlib.sha256(
    SCRIPT.read_bytes()
).hexdigest()

seed_sha = hashlib.sha256(
    SEED_TABLE.read_bytes()
).hexdigest()

assert script_sha == EXPECTED_SCRIPT_SHA
assert seed_sha == EXPECTED_SEED_SHA

# Verify all newly generated rho=.5 assignments.
for rep, expected_sha in RHO05_SHAS.items():
    p = (
        RHO05_ROOT
        / f"rho_0.5_rep_{rep}.csv"
    )

    assert p.exists()

    actual = hashlib.sha256(
        p.read_bytes()
    ).hexdigest()

    assert actual == expected_sha, (
        f"rho=.5 rep={rep} assignment SHA mismatch"
    )

print("=" * 90)
print("T2.1 FINAL 2-GPU PRODUCTION LAUNCH")
print("=" * 90)
print("Script SHA :", script_sha)
print("Seed SHA   :", seed_sha)
print("Cache      :", CACHE)
print("Output     :", OUTROOT)
print()

# ================================================================
# MODEL STATUS
# ================================================================

ALL_MODELS = [
    (0.0, 1),
    (0.0, 2),
    (0.0, 3),
    (0.5, 1),
    (0.5, 2),
    (0.5, 3),
    (1.0, 1),
    (1.0, 2),
    (1.0, 3),
]

def model_dir(rho, rep):
    return (
        OUTROOT
        / f"rho_{rho:.1f}_rep_{rep}"
    )

def is_completed(rho, rep):

    d = model_dir(rho, rep)

    record = d / "training_record.json"
    final = d / "final_epoch15.pt"

    if not (
        record.exists()
        and final.exists()
    ):
        return False

    try:
        r = json.loads(
            record.read_text(
                encoding="utf-8"
            )
        )
    except Exception:
        return False

    return (
        r.get("status") == "COMPLETED"
        and r.get("architecture") == "ConvNeXt-Tiny"
        and abs(
            float(r.get("rho")) - rho
        ) < 1e-12
        and int(r.get("replicate")) == rep
        and int(r.get("epochs")) == 15
    )

completed_models = [
    x for x in ALL_MODELS
    if is_completed(*x)
]

remaining = [
    x for x in ALL_MODELS
    if not is_completed(*x)
]

print("COMPLETED:")
for rho, rep in completed_models:
    print(
        f"  rho={rho:.1f} rep={rep}"
    )

print("\nREMAINING:")
for rho, rep in remaining:
    print(
        f"  rho={rho:.1f} rep={rep}"
    )

assert len(completed_models) == 2
assert len(remaining) == 7

# ================================================================
# TWO-GPU SCHEDULER
# ================================================================

# Keep deterministic order.
pending = queue.Queue()

for item in remaining:
    pending.put(item)

stop_all = threading.Event()

results = []
results_lock = threading.Lock()

def run_one_model(gpu, rho, rep):

    outdir = model_dir(
        rho,
        rep
    )

    outdir.mkdir(
        parents=True,
        exist_ok=True
    )

    env = os.environ.copy()

    env["CUDA_VISIBLE_DEVICES"] = str(gpu)
    env["PYTHONUNBUFFERED"] = "1"

    cmd = [
        sys.executable,
        "-u",
        str(SCRIPT),

        "--project",
        str(PROJECT),

        "--cache-dir",
        str(CACHE),

        "--out-dir",
        str(outdir),

        "--mode",
        "primary",

        "--rho",
        str(rho),

        "--rep",
        str(rep),

        "--gap",
        "mean",

        "--save-all-epochs",
        "0",

        "--num-workers",
        "2",
    ]

    print(
        f"\n{'=' * 90}\n"
        f"START GPU {gpu} | "
        f"rho={rho:.1f} rep={rep}\n"
        f"{'=' * 90}",
        flush=True
    )

    start = time.time()

    p = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env,
    )

    for line in p.stdout:
        print(
            f"[GPU {gpu} | "
            f"rho={rho:.1f} rep={rep}] "
            f"{line.rstrip()}",
            flush=True
        )

    p.stdout.close()

    rc = p.wait()

    elapsed = time.time() - start

    success = (
        rc == 0
        and is_completed(
            rho,
            rep
        )
    )

    with results_lock:
        results.append(
            {
                "gpu": gpu,
                "rho": rho,
                "rep": rep,
                "return_code": rc,
                "success": success,
                "seconds": elapsed,
            }
        )

    print(
        f"\nFINISHED GPU {gpu} | "
        f"rho={rho:.1f} rep={rep} | "
        f"return_code={rc} | "
        f"validated={success}",
        flush=True
    )

    if not success:
        stop_all.set()

def gpu_worker(gpu):

    while not stop_all.is_set():

        try:
            rho, rep = pending.get_nowait()
        except queue.Empty:
            return

        run_one_model(
            gpu,
            rho,
            rep
        )

        pending.task_done()

        if stop_all.is_set():
            return

# ================================================================
# START BOTH GPUs
# ================================================================

threads = [
    threading.Thread(
        target=gpu_worker,
        args=(0,),
        daemon=False,
    ),
    threading.Thread(
        target=gpu_worker,
        args=(1,),
        daemon=False,
    ),
]

for t in threads:
    t.start()

for t in threads:
    t.join()

# ================================================================
# FINAL STATUS
# ================================================================

print("\n" + "=" * 90)
print("T2.1 PRODUCTION BATCH STATUS")
print("=" * 90)

for rho, rep in ALL_MODELS:

    status = (
        "COMPLETED"
        if is_completed(rho, rep)
        else "NOT COMPLETED"
    )

    print(
        f"rho={rho:.1f} rep={rep}: {status}"
    )

print("\nRun results:")

for r in results:
    print(
        f"GPU {r['gpu']} | "
        f"rho={r['rho']:.1f} rep={r['rep']} | "
        f"return={r['return_code']} | "
        f"success={r['success']}"
    )

completed_now = [
    x for x in ALL_MODELS
    if is_completed(*x)
]

print(
    f"\nCompleted models: "
    f"{len(completed_now)}/9"
)

if len(completed_now) == 9:

    print("\n" + "=" * 90)
    print("T2.1 ALL 9 MODELS: COMPLETE")
    print("=" * 90)

else:

    print("\n" + "=" * 90)
    print("T2.1 BATCH STOPPED BEFORE ALL 9")
    print("=" * 90)

    raise RuntimeError(
        "Not all T2.1 models completed. "
        "Inspect the model-specific output above."
    )

T2.1 FINAL 2-GPU PRODUCTION LAUNCH
Script SHA : b0981c8fdff3d971b98aed1608de4b5e5e89b2c4bf66cc959272b769e0e89f18
Seed SHA   : 9c54a1a617f676750ef4abd174d0374a7ccce69e633c1f8dad86a3ee03823a5d
Cache      : /kaggle/working/T2_1_PRODUCTION_CACHE
Output     : /kaggle/working/T2_1_PRODUCTION

COMPLETED:
  rho=0.0 rep=1
  rho=0.0 rep=2

REMAINING:
  rho=0.0 rep=3
  rho=0.5 rep=1
  rho=0.5 rep=2
  rho=0.5 rep=3
  rho=1.0 rep=1
  rho=1.0 rep=2
  rho=1.0 rep=3

START GPU 0 | rho=0.0 rep=3

START GPU 1 | rho=0.5 rep=1
[GPU 0 | rho=0.0 rep=3] RESUMING FROM COMPLETED EPOCH 12; NEXT EPOCH 13
[GPU 1 | rho=0.5 rep=1] RESUMING FROM COMPLETED EPOCH 12; NEXT EPOCH 13
[GPU 0 | rho=0.0 rep=3] /kaggle/working/T2_1_ConvNeXt_training_code_freeze/train_convnext_tiny_t2_1.py:204: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it w